In [ ]:
# -*- coding: utf-8 -*-
# ===== БЛОК 1: МИРЭА =====
# Образцы — МТУСИ, Политех, МИИТ, Губкин, МЭИ, СТАНКИН. Страница списков — React
# SPA с пустым HTML; найден открытый JSON API в бандле main.*.js (Selenium НЕ
# нужен, старый селениумный блок МИРЭА устарел):
#   1. Каталог: GET https://priem.mirea.ru/competitions_api?edu_level_id=2&
#      edu_form_id=1&org_unit_id=1484028700495285107 (бак/спец, очная, Москва —
#      параметры из URL ТЗ). Ответ: 97 программ, у каждой competitions[] c
#      compTypeId: '4' = БЮДЖЕТНЫЙ общий конкурс (наша строка по ТЗ; plan уже за
#      вычетом квот: у 01.03.02 план 48, общий 35 = 48−5−3−5), '1' = бюджетный БВИ
#      (отдельный список, план общий с типом 4), '2' особая, '3' целевая,
#      '7' отдельная квоты, '6'/'5' — платные общий/БВИ.
#   2. Списки: GET https://priem.mirea.ru/competitions_api/entrants?
#      competitions[]=<id>&... (можно несколько id за раз) -> data[] конкурсов с
#      updatedAt и entrants[]. Поля абитуриента: superCode=Код, priority,
#      finalMark/entranceMark/achievementMark (×1000), marks=["100 76 100"]
#      (порядок = examSetVariant конкурса), place, origIn (ОРИГИНАЛ документа —
#      в UI колонка согласия у типа 4 рендерится из origIn, но на 18.07.2026
#      origIn=0 у ВСЕХ: оригиналы бюджета МИРЭА ещё не отражает), accepted
#      (согласие на зачисление — в UI им рендерится согласие остальных типов
#      конкурса; проверено кросс-сверкой с согласиями других вузов: 1
#      противоречие на 46 пересечений). «Согласие» = accepted ИЛИ origIn.
#      iHP = сайтовый «Основной высший приоритет» (галочка «да»),
#      iHPO = «Высший проходной» (с согласием), isActive (0 = выбыл, статус в s),
#      isBVI, benefitCategoryTitle.
#
# «Основной ВП» = сайтовый iHP (по ТЗ: «в мирэа ВП рассчитывается правильно,
# просто берем данные как есть») — сайт считает по всем конкурсам. Как и для
# СТАНКИНа, наш deferred acceptance остаётся КОНТРОЛЬНЫМ (колонка «ВП расч.»,
# в таблицы не идёт) со сверкой в логе; для контрольного DA места = план типа 4
# минус активные БВИ из списка типа 1 (БВИ занимают места общего конкурса).
# Сайт ставит iHP и людям ниже порога — «да» может быть меньше мест (норма).
# Проходной ВП — наш расчёт: свободные места = план − активные БВИ − сайтовые
# «да» по ПОЛНОМУ списку (до порога), кандидаты — не имеющие сайтового «да» нигде.
#
# Неактивные строки (isActive=0 — «Отозвано» и пр.) исключаем из таблиц.
# Порог (как у всех): 277, если в examSetVariant есть физика, иначе 269
# (у целевых ИТ-программ МИРЭА — Математика/Информатика/Русский -> 269).
#
# Кэш: сигнатура по каталогу (id, план, число заявлений общего и БВИ конкурсов) —
# applicationsCount растёт при каждом обновлении списков; дата updatedAt
# сохраняется в данных для отображения.

import time
import os
import re
import pickle
import collections
import requests
import urllib3
import pandas as pd
from IPython.display import display, Markdown

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

start_time_mirea = time.time()

CATALOG_URL_MIREA = ('https://priem.mirea.ru/competitions_api'
                     '?edu_level_id=2&edu_form_id=1&org_unit_id=1484028700495285107')
ENTRANTS_URL_MIREA = 'https://priem.mirea.ru/competitions_api/entrants'
HTTP_HEADERS_MIREA = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'),
    'Accept-Language': 'ru-RU,ru;q=0.9',
    'Referer': 'https://priem.mirea.ru/accepted-entrants-list/',
}
CACHE_MIREA = 'mirea_cache.pkl'
# Версия схемы парсинга. Инкрементируй при изменении логики parse_program_mirea —
# иначе при неизменившейся сигнатуре каталога из кэша подтянутся старые данные.
# v3: «Проходной ВП» = сайтовое поле iHPO (ТЗ: берём данные как есть). Свой
#     расчёт снят — он давал «нет» всегда (см. комментарий у присвоения колонки).
# v2: Согласие = accepted|origIn (origIn у бюджета пока всегда 0 — сайт не
#     отражает оригиналы; accepted = согласие на зачисление, см. шапку).
# v1: каталог (тип 4 бюджетный общий + тип 1 БВИ), entrants одним запросом,
#     Основной ВП = сайтовый iHP, контрольный DA (план − активные БВИ) со сверкой,
#     isActive=0 исключаем, порог 277/269 по examSetVariant, суммы ×1000.
# v4 (04.08.2026): три изменения после зачисления по квотам и БВИ.
#   1. «№_orig» — это НОМЕР СТРОКИ НА САЙТЕ, а не сырое поле place. МИРЭА удалил
#      зачисленных из выдачи, но place у оставшихся НЕ пересчитал: на 10.05.04 у
#      кода 2033142 place=151, а строк выше него всего 137, то есть на сайте он
#      138-й (max place 1299 при 1155 строках — в нумерации дыры). Теперь №_orig
#      считается сквозным счётчиком по активным строкам в порядке place, а сырое
#      значение сохранено в колонке «place API» для диагностики.
#   2. Конкурсов типа 1 (бюджетный БВИ) и типов 2/3/7 (квоты) в каталоге больше
#      нет — остались 4 (бюджетный общий), 5 и 6 (платные). План типа 4 стал
#      ОСТАТКОМ мест общего конкурса: после розыгрыша квот он вырос у 72 программ
#      из 80 (10.05.04: было 30, стало 42). Берём как есть, ничего не вычитаем.
#   3. Каталог отдаёт готовые проходные баллы: minScore = минимальный балл среди
#      получивших iHPO (реальная картина, с учётом поданных согласий),
#      minScoreByAll = минимальный балл среди получивших iHP (модель «согласились
#      все»). Проверено на 84 программах: совпадение 84/84 по обоим, и число iHP
#      = число iHPO = план. Эти два балла НЕ обрезаны нашим порогом отбора, то
#      есть свободны от главной грабли проекта, и добавлены в сигнатуру кэша —
#      applicationsCount после закрытия приёма замер, а проходной живой.
PARSE_VERSION_MIREA = 4

# --- Целевые программы из ТЗ: «<профиль> <код направление>». ---
# Матчим по normalize(title + ' ' + programSubjectTitle) каталога.
# Список обновлён 04.08.2026 (новое ТЗ бро): 13 программ вместо прежних 36 —
# оставлены только те, куда реально поданы документы. Все 13 сверены с живым
# каталогом: найдены, бюджет > 0. Прежний список сохранён в CHANGELOG.
TARGET_MIREA = [
    'Аппаратное программирование встраиваемых систем 09.03.01 Информатика и вычислительная техника',
    'Интеллектуальные системы управления и обработки информации 09.03.01 Информатика и вычислительная техника',
    'Инфраструктура информационных технологий 09.03.01 Информатика и вычислительная техника',
    'Квантовые коммуникации и вычисления 09.03.01 Информатика и вычислительная техника',
    'Киберфизические системы 09.03.01 Информатика и вычислительная техника',
    'Технологии и системы искусственного интеллекта в здравоохранении 09.03.01 Информатика и вычислительная техника',
    'Умные сервисы высокотехнологичных производств 09.03.01 Информатика и вычислительная техника',
    'Цифровые комплексы, системы и сети 09.03.01 Информатика и вычислительная техника',
    'Анализ безопасности компьютерных систем 10.05.01 Компьютерная безопасность',
    'Разработка защищенного программного обеспечения 10.05.01 Компьютерная безопасность',
    'Разработка защищенных телекоммуникационных систем 10.05.02 Информационная безопасность телекоммуникационных систем',
    'Технологии информационно-аналитического мониторинга 10.05.04 Информационно-аналитические системы безопасности',
    'Инженерия автоматизированных систем 27.03.03 Системный анализ и управление',
]

session_mirea = requests.Session()
session_mirea.headers.update(HTTP_HEADERS_MIREA)


RETRY_DELAYS_MIREA = (3, 6, 12, 24, 40, 60)  # 23.07.2026: сервер МИРЭА отдавал 502


def api_mirea(url, params=None, retries=len(RETRY_DELAYS_MIREA) + 1):
    """GET JSON API МИРЭА с повторами (экспоненциальный бэкофф до ~2.5 мин).

    Шлюз МИРЭА периодически отдаёт 502/503/504 — это временный перегрев
    сервера, а не ошибка запроса, поэтому ждём дольше, чем при обычном сбое.
    """
    last_error = None
    for attempt in range(retries):
        try:
            response = session_mirea.get(url, params=params, timeout=180, verify=False)
            response.raise_for_status()
            return response.json()
        except Exception as error:
            last_error = error
            if attempt == retries - 1:
                break
            pause = RETRY_DELAYS_MIREA[min(attempt, len(RETRY_DELAYS_MIREA) - 1)]
            print(f'  МИРЭА не ответил ({type(error).__name__}: {error}). '
                  f'Повтор {attempt + 2}/{retries} через {pause} с...')
            time.sleep(pause)
    raise RuntimeError(f'Не удалось вызвать {url}: {last_error}')


def _norm_mirea(text):
    return re.sub(r'\s+', ' ', str(text).replace('ё', 'е').replace('Ё', 'Е')
                  .replace('–', '-').replace('—', '-').replace('«', '"')
                  .replace('»', '"')).strip().lower()


def _mark1000_mirea(value):
    try:
        return int(value) // 1000
    except (TypeError, ValueError):
        return 0


def subject_label_mirea(exam_name):
    """Подпись предмета по элементу examSetVariant (физика приоритетнее)."""
    joined = str(exam_name).lower()
    if 'физик' in joined:
        return 'Физ.'
    if 'информатик' in joined:
        return 'Инф.'
    if 'математик' in joined:
        return 'Мат.'
    if 'русск' in joined:
        return 'Рус.'
    if 'хими' in joined:
        return 'Хим.'
    if 'общество' in joined:
        return 'Обществ.'
    word = str(exam_name).split()[0] if str(exam_name).split() else '—'
    return word[:8]


def parse_program_mirea(prog, is_target):
    """Качает entrants бюджетного общего конкурса (тип 4) + БВИ (тип 1) одной
    программы, собирает DataFrame (только активные, только выше порога)."""
    comp_ids = list(prog['comp_ids']) + list(prog['bvi_ids'])
    params = [('competitions[]', cid) for cid in comp_ids] + [('edu_level', '2')]
    payload = api_mirea(ENTRANTS_URL_MIREA, params=params)
    competitions = payload.get('data') or []

    main_entrants, bvi_active, update_date, exam_set = [], 0, None, []
    for comp in competitions:
        entrants = comp.get('entrants') or []
        if str(comp.get('compType', '')).strip() == 'общий конкурс':
            main_entrants.extend(entrants)
            update_date = comp.get('updatedAt') or update_date
            exam_set = comp.get('examSetVariant') or exam_set
        elif 'без вступительных испытаний' in str(comp.get('compType', '')):
            bvi_active += sum(1 for e in entrants if e.get('isActive'))

    if not main_entrants:
        return None

    slot_labels = [subject_label_mirea(x) for x in exam_set]
    has_physics = 'физик' in _norm_mirea(' '.join(map(str, exam_set)))
    threshold = 277 if has_physics else 269

    # Сайтовые «да» и коды считаем по ПОЛНОМУ списку активных (до порога) —
    # для точных свободных мест Проходного ВП и глобального enrolled.
    min_score = prog.get('min_score')            # проходной среди подавших согласие
    min_score_all = prog.get('min_score_all')    # проходной, «если бы согласились все»

    site_yes_full = 0
    site_enrolled_codes = set()
    inactive_count = 0
    consent_full = 0          # согласий по ПОЛНОМУ списку (до обрезки порогом)
    site_row = 0              # номер строки на сайте: сквозной по активным
    rows = []
    for entrant in sorted(main_entrants, key=lambda e: e.get('place') or 10 ** 9):
        if not entrant.get('isActive'):
            inactive_count += 1
            continue
        # Нумерация сайта. Поле place МИРЭА после удаления зачисленных НЕ
        # пересчитал (дыры: max place 1299 при 1155 строках), а на странице строки
        # пронумерованы подряд. Считаем сами — тогда №_orig совпадает с тем, что
        # бро видит на сайте, и сверка глазами перестаёт расходиться.
        site_row += 1
        code = str(entrant.get('superCode') or '').strip()
        if not code:
            continue
        if entrant.get('iHP'):
            site_yes_full += 1
            site_enrolled_codes.add(code)
        if entrant.get('accepted') or entrant.get('origIn'):
            consent_full += 1
        total = _mark1000_mirea(entrant.get('finalMark'))
        if total < threshold:
            continue

        mark_parts = str((entrant.get('marks') or [''])[0]).split()
        marks = ', '.join(
            f'{slot_labels[i] if i < len(slot_labels) else f"ВИ{i + 1}"}: {m}'
            for i, m in enumerate(mark_parts) if m.isdigit()
        ) or '—'
        note_parts = []
        if entrant.get('isBVI'):
            note_parts.append('БВИ')
        if entrant.get('benefitCategoryTitle'):
            note_parts.append(str(entrant['benefitCategoryTitle']))
        rows.append({
            '№_orig': site_row,
            'place API': int(entrant.get('place') or 0),
            'Код': code,
            'Сумма': total,
            'Сумма без ИД': _mark1000_mirea(entrant.get('entranceMark')),
            'Оценки': marks,
            'Согласие': 'да' if (entrant.get('accepted') or entrant.get('origIn')) else '',
            'Приоритет': int(entrant.get('priority') or 99),
            'Осн.ВП сайт': 'да' if entrant.get('iHP') else 'нет',
            # iHPO — сайтовый «Высший проходной приоритет»: реальная картина с
            # учётом того, кто подал согласие (iHP — модель «как будто согласие
            # подали все»). Берём как есть по ТЗ; свой расчёт этой колонки давал
            # ноль всегда, потому что список обрезан порогом.
            'Прох.ВП сайт': 'да' if entrant.get('iHPO') else 'нет',
            'Примечание': '; '.join(note_parts),
        })

    if not rows:
        return None

    df = pd.DataFrame(rows)
    # Дедуп по Коду на всякий случай; сортировка по Сумме (с ИД) по убыванию,
    # тай-брейк по №_orig (place сайта) — детерминизм.
    df = (df.sort_values(['Сумма', '№_orig'], ascending=[False, True], kind='stable')
            .drop_duplicates('Код', keep='first')
            .reset_index(drop=True))
    df.insert(0, '№', range(1, len(df) + 1))

    # «Хватает балла» — сравнение с проходным баллом самого вуза (minScore).
    # Этот балл считается по ПОЛНОМУ списку и не обрезан нашим порогом отбора,
    # поэтому ему можно верить там, где нельзя верить выводам из усечённых таблиц
    # (главная грабля проекта). Смысл: «да» — при подаче согласия в МИРЭА строка
    # вытеснила бы слабейшего из проходящих; «на грани» — ровно на границе,
    # решает тай-брейк вуза.
    if min_score is not None:
        df['Хватает балла'] = [
            'да' if s > min_score else ('на грани' if s == min_score else 'нет')
            for s in df['Сумма']
        ]
    else:
        df['Хватает балла'] = '—'

    return {
        'df': df,
        'budget': prog['plan'],                     # места общего конкурса (сайт)
        'bvi_active': bvi_active,                   # активные БВИ (занимают эти места)
        'budget_da': max(0, prog['plan'] - bvi_active),   # для контрольного DA
        'site_yes_full': site_yes_full,             # сайтовые «да» по полному списку
        'site_enrolled_codes': site_enrolled_codes,
        'inactive': inactive_count,
        'min_score': min_score,                     # проходной среди подавших согласие
        'min_score_all': min_score_all,             # проходной «если согласились все»
        'consent_full': consent_full,               # согласий по полному списку
        'active_total': site_row,                   # строк на сайте (активных)
        'threshold': threshold,
        'form': 'Очная',
        'university': 'МИРЭА',
        'competition': prog['display'],
        'is_target': is_target,
        'update_date': update_date,
    }


# ===== Сбор данных: каталог -> бюджетные общие конкурсы =====
print('Загрузка каталога конкурсов МИРЭА (бак/спец, очная, Москва)...')
catalog_mirea = api_mirea(CATALOG_URL_MIREA)
print(f'Программ в каталоге: {len(catalog_mirea)}.')

all_programs_mirea = []
for prog in catalog_mirea:
    comp_main = next((c for c in prog.get('competitions') or []
                      if c.get('compTypeId') == '4'), None)
    if comp_main is None or int(comp_main.get('plan') or 0) <= 0:
        continue
    # Конкурс типа 1 (бюджетный БВИ) после зачисления 30.07–02.08 из каталога
    # пропал — остались только 4 (бюджетный общий), 5 и 6 (платные). Отсутствие
    # штатно: bvi_ids пуст, bvi_active = 0, план типа 4 уже за вычетом зачисленных.
    comp_bvi = next((c for c in prog.get('competitions') or []
                     if c.get('compTypeId') == '1'), None)
    all_programs_mirea.append({
        'id': str(prog['id']),
        'title': str(prog.get('title', '')).strip(),
        'subject': str(prog.get('programSubjectTitle', '')).strip(),
        'display': f"{str(prog.get('title', '')).strip()} "
                   f"({str(prog.get('programSubjectTitle', '')).strip()})",
        'plan': int(comp_main['plan']),
        'min_score': comp_main.get('minScore'),
        'min_score_all': comp_main.get('minScoreByAll'),
        'is_final': bool(comp_main.get('isFinal')),
        'comp_ids': [str(x) for x in comp_main.get('compIds') or []],
        'app_count': int(comp_main.get('applicationsCount') or 0),
        'bvi_ids': [str(x) for x in (comp_bvi or {}).get('compIds') or []],
        'bvi_app_count': int((comp_bvi or {}).get('applicationsCount') or 0),
    })
print(f'Программ с бюджетным общим конкурсом (план > 0): {len(all_programs_mirea)}.')

target_ids_mirea = []
for tz_line in TARGET_MIREA:
    prog_hit = next(
        (p for p in all_programs_mirea
         if _norm_mirea(f"{p['title']} {p['subject']}") == _norm_mirea(tz_line)), None)
    if prog_hit is None:
        print(f'  ВНИМАНИЕ: целевая «{tz_line[:60]}» не найдена в каталоге с бюджетом.')
        continue
    if prog_hit['id'] not in target_ids_mirea:
        target_ids_mirea.append(prog_hit['id'])
target_names_mirea = {p['id']: p['display'] for p in all_programs_mirea
                      if p['id'] in target_ids_mirea}
print(f'Целевых программ: {len(target_ids_mirea)} (из {len(TARGET_MIREA)} строк ТЗ).')

# Сигнатура кэша: applicationsCount растёт при обновлении списков, а после
# закрытия приёма замирает — поэтому с 04.08.2026 в сигнатуру добавлены проходные
# баллы каталога (minScore/minScoreByAll). Они двигаются при каждом изменении
# состава проходящих, то есть ловят ровно то, что счётчик заявлений уже не ловит.
programs_signature_mirea = tuple(sorted(
    (p['id'], p['plan'], p['app_count'], p['bvi_app_count'],
     p['min_score'], p['min_score_all']) for p in all_programs_mirea))

cached_mirea = None
if os.path.exists(CACHE_MIREA):
    with open(CACHE_MIREA, 'rb') as f:
        cached_mirea = pickle.load(f)

# Страховка по возрасту кэша: в сигнатуре МИРЭА нет ни даты, ни согласий —
# только план и число заявлений (applicationsCount). После закрытия приёма
# (25.07) счётчик замирает, а согласия и флаги iHP/iHPO продолжают меняться до
# зачисления, и кэш повис бы навсегда. Старше TTL — перекачиваем всегда.
# 04.08.2026 — день решения, списки правятся вузом в течение дня, поэтому TTL
# снижен с 3 ч до 30 мин: перекачка 84 программ занимает ~2 мин, это дешевле, чем
# принять решение по получасовой давности данным.
CACHE_TTL_HOURS_MIREA = 0.5
cache_fresh_mirea = (
    os.path.exists(CACHE_MIREA)
    and (time.time() - os.path.getmtime(CACHE_MIREA)) < CACHE_TTL_HOURS_MIREA * 3600
)

cache_valid_mirea = (
    cached_mirea is not None
    and cache_fresh_mirea
    and cached_mirea.get('version') == PARSE_VERSION_MIREA
    and cached_mirea.get('signature') == programs_signature_mirea
    and cached_mirea.get('data')
)
if cached_mirea is not None and not cache_fresh_mirea:
    print(f'Кэш старше {CACHE_TTL_HOURS_MIREA} ч — перекачиваем списки '
          f'(согласия меняются без изменения числа заявлений).')

all_mirea_data = {}
if cache_valid_mirea:
    all_mirea_data = cached_mirea['data']
    print('Каталог не изменился — используем кэш.')
else:
    failed_mirea = []
    for idx, prog in enumerate(all_programs_mirea, 1):
        is_target = prog['id'] in target_ids_mirea
        tag = 'ЦЕЛЕВАЯ' if is_target else 'общая'
        try:
            data = parse_program_mirea(prog, is_target)
        except Exception as error:
            print(f'  [{idx}/{len(all_programs_mirea)}] Ошибка {prog["title"][:40]}: {error}')
            failed_mirea.append(prog['title'][:40])
            continue
        if data is None:
            print(f'  [{idx}/{len(all_programs_mirea)}] {tag}: {prog["title"][:45]} — '
                  f'нет строк выше порога (творческие/слабобалльные), пропуск.')
            continue
        all_mirea_data[prog['id']] = data
        bvi_note = f' − БВИ {data["bvi_active"]}' if data['bvi_active'] else ''
        print(f'  [{idx}/{len(all_programs_mirea)}] {tag}: {prog["title"][:45]} '
              f'[{data["form"]}] мест общ.конкурса={data["budget"]}{bvi_note}, '
              f'порог={data["threshold"]}, строк={len(data["df"])} '
              f'(активных в списке {len(data["df"]) and data["site_yes_full"] or 0} «да» сайта), '
              f'обновлено {data["update_date"]}')

    # Кэш пишем ТОЛЬКО при полной загрузке (см. пояснение в блоке МТУСИ).
    if all_mirea_data and not failed_mirea:
        with open(CACHE_MIREA, 'wb') as f:
            pickle.dump({
                'version': PARSE_VERSION_MIREA,
                'signature': programs_signature_mirea,
                'data': all_mirea_data,
            }, f)
        print('Данные сохранены в кэш.')
    elif failed_mirea:
        print(f'  ВНИМАНИЕ: не загрузились программы ({len(failed_mirea)}): '
              f'{"; ".join(failed_mirea[:3])} — кэш НЕ сохранён, '
              f'работаем на неполных данных.')

if not all_mirea_data:
    raise SystemExit('Нет данных ни по одной программе МИРЭА.')

loaded_targets_mirea = [pid for pid in target_ids_mirea if pid in all_mirea_data]
print(f'Загружено программ: {len(all_mirea_data)}; '
      f'целевых с данными: {len(loaded_targets_mirea)}/{len(target_ids_mirea)}.')

# ===== Контрольный расчёт Основного ВП (отложенная приёмка по ВСЕМ программам) =====
# Кандидат-предлагающая deferred acceptance, как в предыдущих вузах. Для МИРЭА это
# КОНТРОЛЬНЫЙ расчёт (сверка с сайтом) — в таблицы идёт сайтовый iHP (ТЗ).
# Места для DA = план общего конкурса минус активные БВИ.
prog_rank_mirea = {}
prog_budget_mirea = {}
cand_prefs_mirea = {}
for key, data in all_mirea_data.items():
    df = data['df']
    prog_budget_mirea[key] = data['budget_da']
    prog_rank_mirea[key] = {code: i for i, code in enumerate(df['Код'])}
    for code, priority in zip(df['Код'], df['Приоритет']):
        cand_prefs_mirea.setdefault(code, []).append((int(priority), key))
for code, pairs in cand_prefs_mirea.items():
    cand_prefs_mirea[code] = [key for _, key in sorted(pairs, key=lambda x: x[0])]

admitted_mirea = {key: {} for key in all_mirea_data}
next_choice_mirea = {code: 0 for code in cand_prefs_mirea}
pending_mirea = collections.deque(cand_prefs_mirea)

while pending_mirea:
    code = pending_mirea.popleft()
    prefs = cand_prefs_mirea[code]
    while next_choice_mirea[code] < len(prefs):
        key = prefs[next_choice_mirea[code]]
        budget = prog_budget_mirea[key]
        if budget <= 0:
            next_choice_mirea[code] += 1
            continue
        rank = prog_rank_mirea[key][code]
        held = admitted_mirea[key]
        if len(held) < budget:
            held[code] = rank
            break
        worst_code = max(held, key=held.get)   # держим слабейшего (наибольший ранг)
        if rank < held[worst_code]:
            del held[worst_code]
            held[code] = rank
            next_choice_mirea[worst_code] += 1
            pending_mirea.append(worst_code)   # вытесненный ищет следующий приоритет
            break
        next_choice_mirea[code] += 1           # здесь не прошёл — следующий приоритет

# «Основной ВП» в таблицах = сайтовый iHP (ТЗ); наш DA — контрольная колонка.
enrolled_mirea = {}
for key, data in all_mirea_data.items():
    df = data['df']
    admitted_codes = set(admitted_mirea[key])
    df['ВП расч.'] = df['Код'].isin(admitted_codes).map({True: 'да', False: 'нет'})
    df['Основной ВП'] = df['Осн.ВП сайт']
    enrolled_mirea.update((code, True) for code in data['site_enrolled_codes'])

# ===== Сверка контрольного расчёта с сайтовым iHP =====
match_mirea = mismatch_mirea = 0
mismatch_examples_mirea = []
for key, data in all_mirea_data.items():
    df = data['df']
    for ours, site, code in zip(df['ВП расч.'], df['Осн.ВП сайт'], df['Код']):
        if ours == site:
            match_mirea += 1
        else:
            mismatch_mirea += 1
            if len(mismatch_examples_mirea) < 8:
                mismatch_examples_mirea.append(
                    f'{code}@{data["competition"][:25]} расч={ours} сайт={site}')
total_checked_mirea = match_mirea + mismatch_mirea
print(f'\nСверка контрольного DA с сайтовым iHP («Основной высший приоритет»): '
      f'совпало {match_mirea}/{total_checked_mirea} '
      f'({100.0 * match_mirea / max(1, total_checked_mirea):.1f}%), '
      f'расхождений {mismatch_mirea}.')
if mismatch_examples_mirea:
    print('  Примеры расхождений (сайт считает по всем конкурсам, включая квоты и '
          f'платное, мы — по бюджетным общим): {"; ".join(mismatch_examples_mirea)}')

# ===== Проходной ВП — сайтовый iHPO (ТЗ: «берём данные как есть») =====
# Свой расчёт этой колонки был снят: он давал «нет» ВСЕГДА. Причина не в ранге,
# а в самом определении — свободные места считались как план − сайтовые «да» по
# полному списку, а сайт раздаёт места ровно по плану, поэтому free всегда 0.
# Даже при free > 0 после отложенной приёмки не остаётся незачисленных
# кандидатов: тот, кого нигде не взяли, дошёл бы до свободного места и занял его.
#
# Что такое iHP и iHPO — проверено живьём 04.08.2026 на 30 программах:
#   iHPO=1 стоит у 851 строки, и у ВСЕХ 851 есть согласие (accepted=1), без
#   единого исключения. iHP=1 стоит у тех же 851, но согласие есть лишь у 13.5 %.
#   Значит iHP — модель «как будто согласие подали все» (сравнение по баллам),
#   а iHPO — реальная картина ТОЛЬКО среди подавших согласие в МИРЭА.
# Практическое следствие, важное для решения: «Проходной ВП = нет» у строки без
# согласия НЕ означает «балла не хватает». Хватает ли балла, показывает отдельная
# колонка «Хватает балла» (сравнение с minScore, проходным баллом самого вуза).
for key, data in all_mirea_data.items():
    df = data['df']
    df['Проходной ВП'] = df['Прох.ВП сайт']

# ===== Отдельные таблицы по программам (только целевые, порядок ТЗ) =====
display(Markdown('# БЛОК 1: МИРЭА'))
for key in target_ids_mirea:
    data = all_mirea_data.get(key)
    if data is None:
        display(Markdown(f'### {target_names_mirea.get(key, key)} — нет данных'))
        print('-' * 80)
        continue
    df = data['df']
    count_main = (df['Основной ВП'] == 'да').sum()
    count_pass = (df['Проходной ВП'] == 'да').sum()

    display(Markdown(f"### {data['competition']} ({data['form']})"))
    bvi_note = f' − активных БВИ {data["bvi_active"]}' if data['bvi_active'] else ''
    display(Markdown(
        f"**Мест общего конкурса (сайт):** {data['budget']}{bvi_note}  |  "
        f"**Порог отбора:** {data['threshold']}  |  **Всего строк:** {len(df)}  |  "
        f"**Обновлено:** {data['update_date']}"
    ))
    display(Markdown(
        f'**Основной ВП (да):** {count_main}  |  **Проходной ВП (да):** {count_pass}  |  '
        f'**Сумма (да):** {count_main + count_pass}'
    ))
    display(Markdown(
        f"**Проходной балл вуза сейчас (среди подавших согласие):** "
        f"{data.get('min_score')}  |  **Проходной, если бы согласились все:** "
        f"{data.get('min_score_all')}  |  **Согласий в полном списке:** "
        f"{data.get('consent_full')} из {data.get('active_total')} строк"
    ))
    display_cols = ['№', '№_orig', 'Код', 'Сумма', 'Сумма без ИД', 'Оценки', 'Согласие',
                    'Приоритет', 'Основной ВП', 'Проходной ВП', 'Хватает балла',
                    'Примечание']
    display(df[[c for c in display_cols if c in df.columns]].style.hide(axis='index'))
    print('-' * 80)

print(f'\nВремя работы БЛОКА 1 (МИРЭА): {time.time() - start_time_mirea:.1f} сек.')
print('Данные для сводной готовы: all_mirea_data, target_ids_mirea — '
      'см. «БЛОК 2: СВОДНАЯ ТАБЛИЦА МИРЭА».')

In [ ]:
# ===== БЛОК 1: МТУСИ =====
# Парсинг, проверка обновления, расчёт ВП, таблицы по программам, время работы.
# Сводная таблица вынесена в отдельный блок «БЛОК 2: СВОДНАЯ ТАБЛИЦА МТУСИ».
#
# Selenium не нужен: сайт отдаёт готовую таблицу обычным GET-запросом.
# Критично: пробелы в параметрах кодируются как %20 (quote), а не как '+' (quote_plus) —
# иначе Bitrix не распознаёт параметр group и отдаёт страницу без таблицы.
#
# Нюанс расчёта ВП: чтобы корректно посчитать «Основной ВП», нужно моделировать
# зачисление не только по нашим целевым программам, а по ВСЕМ программам вуза,
# удовлетворяющим условиям (город Москва, бюджетных мест > 0, приём на общих основаниях),
# причём по ВСЕМ формам обучения — очной, очно-заочной и заочной. Иначе абитуриент,
# у которого выше приоритетом стоит нецелевая программа (в т.ч. заочная/очно-заочная),
# ошибочно занимает место на нашей целевой. Поэтому:
#   - all_programs_mtuci (19 шт.) — участвуют в симуляции зачисления;
#   - target_programs_mtuci (11 шт.) — только для них строим таблицы.
# Очно-заочные и заочные программы нужны ИСКЛЮЧИТЕЛЬНО для расчёта ВП: в таблицы,
# в mtuci_data и в сводную они не попадают (там только target_programs_mtuci).
#
# Порог отсечения (277/269): строки с суммой ниже порога в DataFrame не сохраняются —
# они не влияют на результат анализа. Применяется и к целевым, и к добавочным программам.
#
# Два режима выдачи (параметр originalView) показывают РАЗНОЕ и датированы по-разному:
#   'all'    — все поданные заявления, «Свободно мест» равно «Количеству мест» на всех
#              19 программах из 19 и потому бесполезно; страница отстаёт по времени;
#   'rating' — только подавшие согласие, то есть те, кто реально может занять место.
#              Здесь вуз печатает настоящий остаток и обрывает очередь строкой
#              «Достижение предела зачисления на конкурс».
# Блок качает оба: таблицы и расчёт ВП строятся по 'all', числа мест и вердикт вуза
# берутся из 'rating'.

import time
import os
import re
import pickle
import collections
import requests
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

start_time_mtuci = time.time()

BASE_URL_MTUCI = 'https://abitur.mtuci.ru/ranked_lists/spisok.php'
HTTP_HEADERS_MTUCI = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'),
    'Accept-Language': 'ru-RU,ru;q=0.9',
}
# Кэш переехал с mtuci_data.pkl на mtuci_cache.pkl: старый кэш хранил только 11
# программ, а теперь для расчёта ВП нужны 14 — переиспользовать его нельзя.
CACHE_MTUCI = 'mtuci_cache.pkl'
# Версия схемы парсинга. Инкрементируй при изменении логики parse_program_mtuci /
# short_subject_name — иначе при неизменившейся дате из кэша подтянутся старые данные.
# v3: форма обучения берётся из названия программы (build_params_mtuci), в расчёт ВП
#     добавлены очно-заочная и заочная формы (other_form_programs_mtuci).
# v4: порог 269 там, где предмет по выбору включает информатику (ОТКАЧЕНО в v5).
# v5: порог снова ПО ПРОГРАММЕ (ТЗ): есть физика среди ВИ, пусть даже как
#     альтернатива информатике -> 277; 269 только там, где ВИ исключительно
#     информатика. Понижение порога на программах с физикой давало лишний шум.
# v6 (04.08.2026): после зачисления по квотам и БВИ 30.07-02.08 режим 'all' врёт по
#     местам и отстаёт по времени. Пять изменений.
#   1. Реальный остаток мест — из режима 'rating': «Свободно мест» там равно
#      «Количеству мест» минус уже зачисленные БВИ (формула сошлась на всех 19
#      программах из 19). Разрыв доходил до 40 мест: 09.03.01 план 51, реально
#      свободно 11; 11.03.01 план 40, свободно 6. План остаётся в data['budget']
#      для сверки, реальный остаток уходит в data['budget_da'] — его читают
#      финал_статистика.py и модель_шансов.py. Зачисленные БВИ вуз из остатка уже
#      вычел, поэтому в отложенную приёмку они не идут (data['no_general_codes'] +
#      непустая колонка «Квота»), иначе одно место занималось бы дважды.
#   2. Даты актуальности у режимов РАЗНЫЕ: 03.08.2026 12:14:57 в 'all' против
#      17:28:48 в 'rating', то есть очередь свежее списка на пять часов. В сигнатуру
#      кэша пошли обе даты — иначе обновлённая очередь молча подтягивалась бы из
#      кэша, собранного по старой.
#   3. Сайтовые колонки «Основной высший приоритет» и «Высший проходной приоритет»
#      сохраняются как «ОВП сайта»/«ВПП сайта», а вердикт rating-режима — как
#      «Проходит (сайт)». Число строк с вердиктом «проходит» в rating-режиме точно
#      равно «Свободно мест» (09.03.02 15/15, 10.05.02 9/9, 15.03.04 38/38), так что
#      это независимая сверка нашей отложенной приёмки. Минимальный балл среди
#      проводимых уходит в data['min_score']: он посчитан вузом по ПОЛНОМУ списку и
#      потому свободен от главной грабли проекта — наши таблицы обрезаны порогом.
#   4. Тихий ноль закрыт с двух сторон: пустой tbody при разобранной шапке — теперь
#      ошибка с диагнозом (программа уходит в failed, кэш не пишется), а законно
#      пустая программа (все ниже порога) попадает в отдельный список empty и входит
#      в сигнатуру кэша. Раньше такая программа не попадала ни в all_mtuci_data, ни в
#      failed_mtuci, а кэш всё равно писался как полный.
#   5. Строка-разделитель «Достижение предела зачисления на конкурс» идёт одной
#      ячейкой с colspan на всю ширину и на прежнем детекте БВИ «есть любой colspan»
#      давала ложное срабатывание. Опорный признак БВИ теперь — заполненное
#      «Основание приема без вступительных испытаний».
# v7: 04.08.2026, из отложенной приёмки выведены НЕзачисленные БВИ. Строки БВИ блок
#     ставит наверх списка и порогом не режет, но вуз держит место только за уже
#     зачисленными: незачисленный БВИ согласия не подавал, места за ним нет, а в
#     нашей приёмке он забирал место сверху с суммой 8-10 (одни баллы за ИД, без
#     ЕГЭ). Живьём таких строк 9 на 6 программах, 5 внутри целевых; на 02.03.01 так
#     уходило ЕДИНСТВЕННОЕ свободное место, и живой абитуриент с 307 баллами
#     оказывался «не проходит». Теперь они помечены «Квота: БВИ без зачисления» и
#     в приёмку не идут; если такой человек донесёт согласие сегодня, вуз зачислит
#     его по БВИ и мест станет меньше — об этом печатается предупреждение.
PARSE_VERSION_MTUCI = 7

# Наш код ЕПГУ. Конкурсный балл на МТУСИ = 277 и это РОВНО порог отсечения: запас
# нулевой, и при малейшем пересчёте баллов строка исчезнет из таблиц молча. Поэтому
# блок явно проверяет её наличие и положение относительно порога (см. конец блока).
OUR_CODE_MTUCI = '2033142'

# Разделитель rating-режима: ниже него абитуриенты на места уже не претендуют.
LIMIT_ROW_MTUCI = 'Достижение предела зачисления'

# Порог: у МТУСИ первый предметный столбец — экзамен по выбору
# «Информатика и ИКТ / Основы прикладной физики / Информатика / Физика»,
# то есть физика в перечне присутствует => 277. Чтобы принудительно задать порог,
# поставь FORCE_THRESHOLD_MTUCI = 269 (или 277).
FORCE_THRESHOLD_MTUCI = None

# --- Целевые программы (11 шт.): только для них строим таблицы и итоговый DataFrame. ---
target_programs_mtuci = [
    '01.03.02 Прикладная математика и программирование (Бюджет, Очная) (2026)',
    '02.03.01 Инженерия данных (Бюджет, Очная) (2026)',
    '09.03.01 ТОП-ИИ: Инженерия систем искусственного интеллекта (Бюджет, Очная) (2026)',
    '09.03.02 Инженерия DevSecOps (Бюджет, Очная) (2026)',
    '09.03.03 Прикладные информационные системы (Бюджет, Очная) (2026)',
    '09.03.04 Разработка и сопровождение программного обеспечения (Бюджет, Очная) (2026)',
    '10.03.01 Безопасность автоматизированных систем (по отрасли или в сфере профессиональной деятельности) (Бюджет, Очная) (2026)',
    '10.03.01 Безопасность компьютерных систем (по отрасли или в сфере профессиональной деятельности) (Бюджет, Очная) (2026)',
    '10.05.02 Управление безопасностью телекоммуникационных систем и сетей (Бюджет, Очная) (2026)',
    '15.03.04 Промышленный интернет вещей и робототехника (Бюджет, Очная) (2026)',
    '27.03.04 Разработка и программирование автономных робототехнических систем (Бюджет, Очная) (2026)',
]

# --- Добавочные программы (3 шт.): Москва, очная, бюджет > 0, но НЕ целевые. ---
# Нужны исключительно для корректной симуляции зачисления (Основной/Проходной ВП),
# в таблицы и в mtuci_data не попадают.
extra_programs_mtuci = [
    '11.03.01 Радиотехника и аудиовизуальные технологии (Бюджет, Очная) (2026)',
    '11.03.02 Беспроводная связь (Бюджет, Очная) (2026)',
    '11.03.02 Сетевая инженерия телекоммуникаций (Бюджет, Очная) (2026)',
]

# --- Программы очно-заочной и заочной форм (5 шт.): Москва, бюджет > 0, общие основания. ---
# Только для симуляции ВП: если у абитуриента такая программа стоит приоритетнее нашей
# целевой очной, зачисление на неё снимает его с конкурса на целевую. В таблицы, mtuci_data
# и сводную НЕ попадают. Форма определяется по названию, отдельный параметр не нужен.
other_form_programs_mtuci = [
    '09.03.01 Организация и технологии защиты информации (Бюджет, Очно-заочная) (2026)',
    '09.03.02 Инженерия DevSecOps (Бюджет, Заочная) (2026)',
    '11.03.02 Сетевая инженерия телекоммуникаций (Бюджет, Заочная) (2026)',
    '15.03.04 Промышленный интернет вещей и робототехника (Бюджет, Заочная) (2026)',
    '27.03.04 Разработка и программирование автономных робототехнических систем (Бюджет, Заочная) (2026)',
]

# Все программы вуза, участвующие в расчёте ВП (очная + очно-заочная + заочная).
all_programs_mtuci = target_programs_mtuci + extra_programs_mtuci + other_form_programs_mtuci

session_mtuci = requests.Session()
session_mtuci.headers.update(HTTP_HEADERS_MTUCI)


def form_from_name_mtuci(program_name):
    """Форма обучения из названия конкурса: «... (Бюджет, Очно-заочная) (2026)» -> «Очно-заочная».

    Параметр form должен совпадать с формой в названии программы, иначе Bitrix отдаёт
    страницу без таблицы. По умолчанию — «Очная».
    """
    match = re.search(r'\(Бюджет,\s*([^)]+)\)', program_name)
    return match.group(1).strip() if match else 'Очная'


def build_params_mtuci(program_name, view='all'):
    """Фиксированные фильтры: бюджет, общие основания, бюджетная основа.

    Форма обучения (очная/очно-заочная/заочная) берётся из названия программы.
    view='all' — все поданные заявления (списки и расчёт ВП),
    view='rating' — только подавшие согласие (реальный остаток мест и вердикт вуза).
    """
    return {
        'valueSearch': '',
        'priznakViev': 'budg',
        'levelTarget': 'bak_main',
        'group': program_name,
        'form': form_from_name_mtuci(program_name),
        'category': 'На общих основаниях',
        'footings': 'Бюджетная основа',
        'originalFilter': '',
        'search_type': 'uniqueID',
        'originalView': view,
    }


# 23.07.2026: сайты вузов в горячую фазу выпадают на минуту-две (502, таймаут
# соединения). Ждём до ~2.5 мин вместо прежних 12 с; connect отдельно от read.
RETRY_DELAYS_MTUCI = (3, 6, 12, 24, 40, 60)
TIMEOUT_MTUCI = (15, 60)  # (соединение, чтение)


def fetch_page_mtuci(program_name, view='all', retries=len(RETRY_DELAYS_MTUCI) + 1):
    """GET-запрос страницы конкурсного списка. requests сам кодирует пробелы как %20."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session_mtuci.get(
                BASE_URL_MTUCI, params=build_params_mtuci(program_name, view),
                timeout=TIMEOUT_MTUCI
            )
            response.raise_for_status()
            return response.text
        except Exception as error:
            last_error = error
            if attempt == retries - 1:
                break
            pause = RETRY_DELAYS_MTUCI[min(attempt, len(RETRY_DELAYS_MTUCI) - 1)]
            print(f'  МТУСИ не ответил по «{program_name[:40]}» [{view}] '
                  f'({type(error).__name__}: {error}). '
                  f'Повтор {attempt + 2}/{retries} через {pause} с...')
            time.sleep(pause)
    raise RuntimeError(f'Не удалось загрузить страницу: {last_error}')


def get_update_date_mtuci(html):
    """Дата актуальности списков из div.list-specialty."""
    soup = BeautifulSoup(html, 'html.parser')
    spec = soup.find('div', class_='list-specialty')
    if not spec:
        return None
    match = re.search(r'актуален на\s*([\d.]+\s+[\d:]+)', spec.get_text(' ', strip=True))
    return match.group(1) if match else None


def expand_cells(tr):
    """Разворачивает colspan: у строк БВИ одна ячейка занимает три предметных столбца."""
    cells = []
    for td in tr.find_all('td'):
        span = int(td.get('colspan') or 1)
        cells.extend([td.get_text(' ', strip=True)] * span)
    return cells


def is_limit_row_mtuci(cells):
    """Строка-разделитель «Достижение предела зачисления на конкурс».

    Вуз рисует её одной ячейкой с colspan на всю ширину таблицы, поэтому после
    expand_cells она выглядит как абитуриент, у которого во всех столбцах один и тот
    же текст. Проверять её надо ДО всего остального разбора: на прежнем детекте БВИ
    «есть хоть один colspan» она давала ложное срабатывание, а её текст уезжал в
    поле «Уникальный код».
    """
    return bool(cells) and LIMIT_ROW_MTUCI in cells[0]


def short_subject_name(header):
    """Короткая подпись предмета для колонки «Оценки».

    Если в перечне есть физика (в т.ч. как альтернатива информатике) — «Физ.»;
    если только информатика без физики — «Инф.».
    """
    has_inf = 'Информатика' in header
    has_phys = 'Физика' in header or 'физик' in header
    if has_phys:
        return 'Физ.'
    if has_inf:
        return 'Инф.'
    if 'Математика' in header:
        return 'Мат.'
    if 'Русский' in header:
        return 'Рус.'
    return header.split()[0][:8] if header else '—'


def parse_program_mtuci(program_name):
    """Парсит страницу программы: бюджет, предметы, порог, таблицу абитуриентов.

    Строки ниже порога не сохраняются в DataFrame — на результат анализа они не влияют.
    """
    html = fetch_page_mtuci(program_name)
    soup = BeautifulSoup(html, 'html.parser')

    # --- Бюджетные места и фактическое название конкурса ---
    budget_places = 0
    competition_name = program_name
    head_list = soup.find('ul', class_='list-head')
    if head_list:
        head_text = head_list.get_text(' | ', strip=True)
        match_places = re.search(r'Количество мест:\s*\|\s*(\d+)', head_text)
        if match_places:
            budget_places = int(match_places.group(1))
        match_name = re.search(r'Конкурс:\s*\|\s*([^|]+)', head_text)
        if match_name:
            competition_name = match_name.group(1).strip()

    # --- Таблица ---
    block = soup.find('div', class_='block_table')
    table = block.find('table') if block else soup.find('table')
    if not table:
        raise ValueError('Таблица не найдена')

    headers = [th.get_text(' ', strip=True) for th in table.find('thead').find_all('th')]
    idx_code = headers.index('Уникальный код')
    idx_sum = headers.index('Сумма баллов')
    idx_id = headers.index('Сумма баллов за ИД')
    idx_consent = headers.index('Согласие')
    idx_priority = headers.index('Приоритет')
    # Собственные колонки вуза: номер приоритета, на который он проводит абитуриента.
    # Совпал с «Приоритетом» строки — вуз сажает человека сюда. Блок раньше их
    # выбрасывал, хотя это независимая сверка нашего расчёта ВП.
    idx_main_prio = headers.index('Основной высший приоритет')
    idx_pass_prio = headers.index('Высший проходной приоритет')
    idx_bvi_reason = headers.index('Основание приема без вступительных испытаний')
    idx_subjects = list(range(idx_sum + 1, idx_id))
    if len(idx_subjects) != 3:
        raise ValueError(f'Ожидались 3 предметных столбца, найдено {len(idx_subjects)}')

    subject_headers = [headers[i] for i in idx_subjects]
    subject_names = [short_subject_name(h) for h in subject_headers]

    # Порог по ТЗ — ПО ПРОГРАММЕ, а не по строке: если среди ВИ есть физика
    # (в т.ч. как альтернатива информатике: «Информатика и ИКТ / Основы
    # прикладной физики / Информатика / Физика») — 277. Порог 269 берём только
    # там, где ВИ — исключительно информатика, без физики в вариантах.
    headers_joined = ' '.join(subject_headers).lower()
    has_physics = 'физик' in headers_joined
    threshold = FORCE_THRESHOLD_MTUCI or (277 if has_physics else 269)

    rows = []
    rows_seen = 0        # строк-абитуриентов в таблице ДО отсечения порогом
    for tr in table.find('tbody').find_all('tr'):
        cells = expand_cells(tr)
        if is_limit_row_mtuci(cells):
            continue
        if len(cells) <= idx_bvi_reason:
            continue
        code = cells[idx_code]
        if not code:
            continue
        rows_seen += 1

        # БВИ ловим по заполненному «Основанию приема без вступительных испытаний»:
        # прежний признак «есть любой colspan» срабатывал ещё и на строке-разделителе
        # rating-режима. colspan оставлен запасным признаком — у строки БВИ он ровно
        # по числу предметных столбцов (три), у разделителя — на всю ширину.
        bvi_reason = cells[idx_bvi_reason].strip()
        cell_spans = [int(td.get('colspan') or 1) for td in tr.find_all('td')]
        is_bvi = bool(bvi_reason) or any(s == len(idx_subjects) for s in cell_spans)
        score_texts = [cells[i] for i in idx_subjects]

        try:
            total = int(cells[idx_sum]) if cells[idx_sum].isdigit() else 0
        except ValueError:
            total = 0
        try:
            bonus = int(cells[idx_id]) if cells[idx_id].isdigit() else 0
        except ValueError:
            bonus = 0
        try:
            priority = int(cells[idx_priority]) if cells[idx_priority].isdigit() else 99
        except ValueError:
            priority = 99

        # БВИ проходят без порога: у них «Сумма баллов» = только баллы за ИД.
        # Остальные ниже порога отсекаем — в DataFrame не попадают.
        if not is_bvi and total < threshold:
            continue

        if is_bvi:
            marks = 'БВИ'
        else:
            marks = ', '.join(f'{n}: {s}' for n, s in zip(subject_names, score_texts))

        rows.append({
            '№_orig': int(cells[0]) if cells[0].isdigit() else 0,
            'Код': code,
            'Сумма': total,
            'Сумма без ИД': total - bonus,
            'Оценки': marks,
            'Согласие': cells[idx_consent].strip().lower(),
            'Приоритет': priority,
            'БВИ': 'да' if is_bvi else 'нет',
            # Числа самого вуза, не наш расчёт. Пустое значение = вуз не проводит
            # человека никуда (ВПП пуст ещё и у всех, кто не подал согласие).
            'ОВП сайта': cells[idx_main_prio].strip() or '—',
            'ВПП сайта': cells[idx_pass_prio].strip() or '—',
            'Примечание': cells[-1] if cells else '',
        })

    # Тихий ноль: шапку разобрали, а строк абитуриентов нет вообще. Это поломка
    # разметки, а не пустой конкурс, — падаем с диагнозом, чтобы программа ушла в
    # failed и кэш не записался как полный.
    if rows_seen == 0:
        raise ValueError(
            f'тихий ноль: шапка разобрана ({len(headers)} столбцов), но в tbody '
            f'нет ни одной строки абитуриента — разметка вуза изменилась'
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        # БВИ всегда наверху, остальные — по убыванию суммы.
        df = df.sort_values(
            ['БВИ', 'Сумма'], ascending=[True, False],
            key=lambda c: c.map({'да': 0, 'нет': 1}) if c.name == 'БВИ' else c
        ).reset_index(drop=True)
        df.insert(0, '№', range(1, len(df) + 1))

    return {
        'df': df,
        'budget': budget_places,          # «Количество мест» = ПЛАН, для сверки
        'rows_seen': rows_seen,
        'subject_names': subject_names,
        'threshold': threshold,
        'university': 'МТУСИ',
        'competition': competition_name,
        'is_target': program_name in target_programs_mtuci,
        'update_date': get_update_date_mtuci(html),
    }


def parse_rating_mtuci(program_name):
    """Реальный остаток мест и вердикт вуза из режима originalView='rating'.

    В этом режиме вуз показывает только подавших согласие — тех, кто реально может
    занять место, — сам печатает остаток за вычетом уже зачисленных БВИ и обрывает
    очередь строкой «Достижение предела зачисления на конкурс». Всё это посчитано по
    ПОЛНОМУ списку, а не по нашему, обрезанному порогом 277/269.
    """
    html = fetch_page_mtuci(program_name, view='rating')
    soup = BeautifulSoup(html, 'html.parser')

    places_rating = free_places = None
    head_list = soup.find('ul', class_='list-head')
    if head_list:
        head_text = head_list.get_text(' | ', strip=True)
        match_places = re.search(r'Количество мест:\s*\|\s*(-?\d+)', head_text)
        match_free = re.search(r'Свободно мест:\s*\|\s*(-?\d+)', head_text)
        if match_places:
            places_rating = int(match_places.group(1))
        if match_free:
            free_places = int(match_free.group(1))
    if free_places is None:
        raise ValueError('в rating-режиме нет «Свободно мест» — разметка изменилась')

    block = soup.find('div', class_='block_table')
    table = block.find('table') if block else soup.find('table')
    if not table:
        raise ValueError('таблица rating-режима не найдена')

    headers = [th.get_text(' ', strip=True) for th in table.find('thead').find_all('th')]
    idx_code = headers.index('Уникальный код')
    idx_sum = headers.index('Сумма баллов')
    idx_priority = headers.index('Приоритет')
    idx_main_prio = headers.index('Основной высший приоритет')
    idx_enrolled = headers.index('Зачислен')
    idx_bvi_reason = headers.index('Основание приема без вступительных испытаний')

    listed_codes, enrolled_codes, pass_codes, pass_scores = [], [], [], []
    limit_reached = False
    rows_seen = 0
    for tr in table.find('tbody').find_all('tr'):
        cells = expand_cells(tr)
        if is_limit_row_mtuci(cells):
            limit_reached = True
            continue
        if len(cells) <= idx_bvi_reason:
            continue
        code = cells[idx_code]
        if not code:
            continue
        rows_seen += 1
        listed_codes.append(code)
        if cells[idx_enrolled].strip().lower() == 'да':
            enrolled_codes.append(code)
        if limit_reached:
            continue
        # Вердикт вуза: «Основной высший приоритет» совпал с приоритетом строки —
        # значит вуз проводит абитуриента именно сюда. Число таких строк до
        # разделителя равно «Свободно мест» (проверено на 18 программах из 19;
        # исключение — программа, где мест не осталось вовсе и разделителя нет).
        main_prio = cells[idx_main_prio].strip()
        if not main_prio or main_prio != cells[idx_priority].strip():
            continue
        pass_codes.append(code)
        # У БВИ в «Сумме баллов» стоят только баллы за ИД, минимальным проходным
        # такое число называть нельзя.
        if not cells[idx_bvi_reason].strip() and cells[idx_sum].isdigit():
            pass_scores.append(int(cells[idx_sum]))

    if rows_seen == 0:
        raise ValueError('тихий ноль в rating-режиме: ни одной строки абитуриента')

    return {
        'places': places_rating,
        'free': free_places,
        'listed_codes': listed_codes,
        'enrolled_codes': enrolled_codes,
        'pass_codes': pass_codes,
        'cutoff': min(pass_scores) if pass_scores else None,
        'limit_reached': limit_reached,
        'rows': rows_seen,
        'update_date': get_update_date_mtuci(html),
    }


def merge_rating_mtuci(data, rating):
    """Кладёт рядом с планом реальный остаток мест и вердикт вуза.

    data['budget'] остаётся ПЛАНОМ — он нужен для сверки. Реальный остаток уходит в
    data['budget_da'], который читают финал_статистика.py и модель_шансов.py. Уже
    зачисленных БВИ вуз из остатка вычел, поэтому в отложенную приёмку они не идут:
    иначе одно место занималось бы дважды. Строки таких абитуриентов из таблиц не
    убираем (в списках вуза они есть, и их надо видеть) — помечаем колонкой «Квота»,
    по непустому значению которой модель_шансов.py исключает их из конкурентов.
    """
    enrolled = set(rating['enrolled_codes'])
    passing = set(rating['pass_codes'])
    listed = set(rating['listed_codes'])

    data['budget_free_rating'] = rating['free']
    data['budget_da'] = rating['free']
    data['bvi_enrolled'] = len(enrolled)
    data['no_general_codes'] = sorted(enrolled)
    data['site_pass_codes'] = sorted(passing)
    data['rating_date'] = rating['update_date']
    # Минимальный балл среди тех, кого вуз проводит прямо сейчас. Считан по полному
    # списку вуза, поэтому свободен от главной грабли проекта: наши таблицы обрезаны
    # порогом, и минимум по ним всегда врёт вверх. Читает модель_шансов.py.
    data['min_score'] = rating['cutoff']

    df = data['df']
    bvi_pending = []
    if not df.empty:
        # Строки БВИ блок ставит НАВЕРХ списка (льгота сильнее любого балла) и не
        # режет порогом. Но вуз держит место только за теми БВИ, кого уже зачислил;
        # незачисленный БВИ согласия не подавал, места за ним нет, а в нашей
        # приёмке он всё равно забирал место сверху — с суммой 8-10, то есть это
        # были одни баллы за ИД без ЕГЭ. На 02.03.01 так уходило ЕДИНСТВЕННОЕ
        # свободное место (код 2403679, сумма 10), и живой абитуриент с 307 баллами
        # оказывался «не проходит». Поэтому незачисленных БВИ из приёмки тоже
        # выводим — строку оставляем видимой с пометкой.
        is_bvi = df['БВИ'].astype(str).str.strip() == 'да' if 'БВИ' in df.columns \
            else pd.Series(False, index=df.index)
        bvi_pending = sorted({code for code, flag in zip(df['Код'], is_bvi)
                              if flag and code not in enrolled})
        df['Квота'] = [
            'зачислен БВИ' if code in enrolled
            else ('БВИ без зачисления' if code in set(bvi_pending) else '')
            for code in df['Код']
        ]
        # «—» = кода нет в rating-режиме, то есть согласие не подано и в реальной
        # очереди человек не стоит вовсе. Это не «не проходит», а «не участвует».
        df['Проходит (сайт)'] = [
            'да' if code in passing else ('нет' if code in listed else '—')
            for code in df['Код']
        ]
    data['bvi_pending'] = bvi_pending
    data['no_general_codes'] = sorted(enrolled | set(bvi_pending))

    warnings = []
    if bvi_pending:
        # Не молчим: если такой человек донесёт согласие сегодня, вуз зачислит его
        # по БВИ и свободных мест станет меньше. Наш расчёт этого не моделирует.
        warnings.append(f'БВИ без зачисления: {len(bvi_pending)} — из приёмки '
                        f'выведены, но при подаче ими согласия свободных мест '
                        f'станет меньше')
    plan = data['budget']
    if rating['places'] is not None and rating['places'] != plan:
        warnings.append(f'«Количество мест» разошлось: all={plan}, '
                        f'rating={rating["places"]}')
    if plan - len(enrolled) != rating['free']:
        warnings.append(f'«Свободно мест» {rating["free"]} != план {plan} − '
                        f'зачисленных БВИ {len(enrolled)}')
    if rating['limit_reached'] and len(passing) != rating['free']:
        warnings.append(f'вуз проводит {len(passing)} строк при «Свободно мест» '
                        f'{rating["free"]}')
    return warnings


# ===== Сбор данных по всем программам вуза (с кэшем по дате + составу программ) =====
print('Проверка обновления списков МТУСИ...')
current_date_mtuci = get_update_date_mtuci(fetch_page_mtuci(all_programs_mtuci[0]))
# Дата у режимов своя: 03.08.2026 очередь ('rating') была на пять часов свежее
# списка ('all'). Кэш, сверенный только по 'all', подсунул бы старую очередь.
current_rating_date_mtuci = get_update_date_mtuci(
    fetch_page_mtuci(all_programs_mtuci[0], view='rating'))
print(f'Дата актуальности на сайте: списки {current_date_mtuci}, '
      f'очередь согласий {current_rating_date_mtuci}')

cached_mtuci = None
if os.path.exists(CACHE_MTUCI):
    with open(CACHE_MTUCI, 'rb') as f:
        cached_mtuci = pickle.load(f)

# Кэш валиден только если совпали ОБЕ даты актуальности И состав программ. Условие
# «данные + законно пустые == все программы» закрывает прежнюю дыру: программа с
# пустым DataFrame не попадала ни в data, ни в failed, и кэш писался как полный.
cache_valid_mtuci = (
    cached_mtuci is not None
    and current_date_mtuci is not None
    and cached_mtuci.get('version') == PARSE_VERSION_MTUCI
    and cached_mtuci.get('date') == current_date_mtuci
    and cached_mtuci.get('rating_date') == current_rating_date_mtuci
    and set(cached_mtuci.get('programs', [])) == set(all_programs_mtuci)
    and (set(cached_mtuci.get('data', {})) | set(cached_mtuci.get('empty', [])))
        == set(all_programs_mtuci)
    and cached_mtuci.get('data')
)

all_mtuci_data = {}
empty_mtuci = []
if cache_valid_mtuci:
    all_mtuci_data = cached_mtuci['data']
    empty_mtuci = list(cached_mtuci.get('empty', []))
    print('Обновления не было и состав программ не менялся — используем кэш.')
else:
    failed_mtuci = []
    for idx, program in enumerate(all_programs_mtuci, 1):
        if program in target_programs_mtuci:
            tag = 'целевая'
        elif program in other_form_programs_mtuci:
            tag = 'др.форма'
        else:
            tag = 'добавочная'
        print(f'  {idx}/{len(all_programs_mtuci)} [{tag}]: {program[:55]}...')
        try:
            data = parse_program_mtuci(program)
            rating_mtuci = parse_rating_mtuci(program)
            for warning in merge_rating_mtuci(data, rating_mtuci):
                print(f'    ВНИМАНИЕ: {warning}')
            if data['df'].empty:
                # Законно пустая программа: строки в таблице есть, но все ниже
                # порога. Раньше она молча выпадала и из data, и из failed, а кэш
                # писался как полный — теперь её видно в логе и в сигнатуре кэша.
                print(f"    Нет абитуриентов с суммой >= {data['threshold']} "
                      f"(строк в таблице {data['rows_seen']}), "
                      f"мест свободно {data['budget_da']} — программа пустая")
                empty_mtuci.append(program)
                continue
            all_mtuci_data[program] = data
            bvi_count = (data['df']['БВИ'] == 'да').sum()
            site_pass = len(data['site_pass_codes'])
            print(f"    План: {data['budget']}, свободно (rating): "
                  f"{data['budget_da']} (зачислено БВИ {data['bvi_enrolled']}), "
                  f"вуз проводит: {site_pass}, проходной вуза: "
                  f"{data['min_score'] if data['min_score'] is not None else '—'}, "
                  f"БВИ в списке: {bvi_count}, порог: {data['threshold']}, "
                  f"строк: {len(data['df'])}")
        except Exception as error:
            print(f'    Ошибка: {error}')
            failed_mtuci.append(program)
            continue

    # Кэш пишем ТОЛЬКО при полной загрузке: сигнатура хранит весь каталог, и
    # частичный набор при следующем запуске подтянулся бы как валидный, молча
    # убрав из расчёта ВП конкурентов с упавших программ. Пустые программы тоже
    # учтены: без них проверка «data + empty == каталог» не сойдётся.
    complete_mtuci = (set(all_mtuci_data) | set(empty_mtuci)) == set(all_programs_mtuci)
    if all_mtuci_data and not failed_mtuci and complete_mtuci:
        with open(CACHE_MTUCI, 'wb') as f:
            pickle.dump({
                'version': PARSE_VERSION_MTUCI,
                'date': current_date_mtuci,
                'rating_date': current_rating_date_mtuci,
                'programs': list(all_programs_mtuci),
                'data': all_mtuci_data,
                'empty': list(empty_mtuci),
            }, f)
        print('Данные сохранены.')
    elif failed_mtuci:
        print(f'  ВНИМАНИЕ: не загрузились программы ({len(failed_mtuci)}): '
              f'{"; ".join(p[:40] for p in failed_mtuci[:3])} — кэш НЕ сохранён, '
              f'работаем на неполных данных.')
    else:
        print(f'  ВНИМАНИЕ: разобрано {len(all_mtuci_data)} + пустых '
              f'{len(empty_mtuci)} из {len(all_programs_mtuci)} программ каталога — '
              f'кэш НЕ сохранён, работаем на неполных данных.')

if not all_mtuci_data:
    raise SystemExit('Нет данных ни по одной программе МТУСИ.')

loaded_targets = [p for p in target_programs_mtuci if p in all_mtuci_data]
loaded_extra = [p for p in extra_programs_mtuci if p in all_mtuci_data]
loaded_other = [p for p in other_form_programs_mtuci if p in all_mtuci_data]
print(f'Загружено программ: {len(all_mtuci_data)} '
      f'(целевых {len(loaded_targets)}/{len(target_programs_mtuci)}, '
      f'добавочных очных {len(loaded_extra)}/{len(extra_programs_mtuci)}, '
      f'очно-заочных/заочных {len(loaded_other)}/{len(other_form_programs_mtuci)}).')
if empty_mtuci:
    print(f'  Пустых программ (все абитуриенты ниже порога): {len(empty_mtuci)} — '
          f'{"; ".join(p[:45] for p in empty_mtuci)}. В расчёте ВП не участвуют.')

# ===== Расчёт Основного ВП (отложенная приёмка по ВСЕМ программам вуза) =====
# Модель Минобра «конкурс с приоритетами»: абитуриент зачисляется на программу с
# НАИВЫСШИМ приоритетом (меньший номер), где он реально проходит по баллам — с учётом
# того, что часть более сильных конкурентов уходит на свои более приоритетные программы
# и освобождает места (эффект вытеснения). Прежний «добор по приоритетам подряд» этот
# эффект игнорировал: он мог посадить абитуриента на приоритет 5, хотя после ухода
# конкурентов он проходит на приоритет 2. Отсюда — кандидат-предлагающая отложенная
# приёмка (deferred acceptance), сходящаяся к устойчивому распределению:
#   1) ранг внутри программы = позиция в отсортированном списке (БВИ уже сверху);
#   2) абитуриент «предлагается» своей самой приоритетной программе;
#   3) программа держит лучших budget по рангу, лишних отклоняет;
#   4) отклонённый/вытесненный идёт на следующий приоритет — до стабилизации.
# Мест раздаём РЕАЛЬНЫЙ остаток (budget_da), а не план: план 09.03.01 — 51 место,
# свободных после зачисления БВИ — 11. Зачисленные БВИ (no_general_codes) из остатка
# уже вычтены вузом, поэтому в приёмку их не пускаем: иначе их место считалось бы
# дважды и наши шансы поехали бы вниз на все зачисленные строки сразу.
locked_codes_mtuci = set()
for data in all_mtuci_data.values():
    locked_codes_mtuci.update(data.get('no_general_codes', ()))

prog_rank_mtuci = {}      # программа -> {Код: позиция в списке, меньше = сильнее}
prog_budget_mtuci = {}    # программа -> число свободных бюджетных мест
cand_prefs_mtuci = {}     # Код -> программы по возрастанию приоритета
for prog, data in all_mtuci_data.items():
    df = data['df']
    prog_budget_mtuci[prog] = int(data.get('budget_da', data['budget']) or 0)
    prog_rank_mtuci[prog] = {code: i for i, code in enumerate(df['Код'])}
    for code, priority in zip(df['Код'], df['Приоритет']):
        if code in locked_codes_mtuci:
            continue
        cand_prefs_mtuci.setdefault(code, []).append((int(priority), prog))
for code, pairs in cand_prefs_mtuci.items():
    cand_prefs_mtuci[code] = [prog for _, prog in sorted(pairs, key=lambda x: x[0])]

admitted_mtuci = {prog: {} for prog in all_mtuci_data}   # программа -> {Код: ранг}
next_choice_mtuci = {code: 0 for code in cand_prefs_mtuci}
pending_mtuci = collections.deque(cand_prefs_mtuci)

while pending_mtuci:
    code = pending_mtuci.popleft()
    prefs = cand_prefs_mtuci[code]
    while next_choice_mtuci[code] < len(prefs):
        prog = prefs[next_choice_mtuci[code]]
        budget = prog_budget_mtuci[prog]
        if budget <= 0:
            next_choice_mtuci[code] += 1
            continue
        rank = prog_rank_mtuci[prog][code]
        held = admitted_mtuci[prog]
        if len(held) < budget:
            held[code] = rank
            break
        worst_code = max(held, key=held.get)   # держим слабейшего (наибольший ранг)
        if rank < held[worst_code]:
            del held[worst_code]
            held[code] = rank
            next_choice_mtuci[worst_code] += 1
            pending_mtuci.append(worst_code)    # вытесненный ищет следующий приоритет
            break
        next_choice_mtuci[code] += 1            # здесь не прошёл — следующий приоритет

enrolled_mtuci = {}
for prog, data in all_mtuci_data.items():
    admitted_codes = set(admitted_mtuci[prog])
    data['df']['Основной ВП'] = (
        data['df']['Код'].isin(admitted_codes).map({True: 'да', False: 'нет'})
    )
    enrolled_mtuci.update((code, True) for code in admitted_codes)

# ===== Свободные места после Основного ВП =====
# «Проходной ВП» своим расчётом получить НЕЛЬЗЯ: после отложенной приёмки у
# программы со свободными местами не остаётся незачисленных кандидатов — тот,
# кого не взяли нигде, дошёл бы до этого места и занял его. Прежний расчёт
# давал «нет» во всех строках и создавал ложное впечатление второй проверки.
# Полезен сам остаток: места не разобраны теми, кто выше порога, значит
# проходной балл программы НИЖЕ нашего порога отбора.
free_places_mtuci = {}
for prog, data in all_mtuci_data.items():
    df = data['df']
    df['Проходной ВП'] = 'нет'
    free_places_mtuci[prog] = max(
        0, prog_budget_mtuci[prog] - int((df['Основной ВП'] == 'да').sum()))

# ===== Отдельные таблицы по программам (только целевые) =====
display(Markdown('# БЛОК 1: МТУСИ'))
for prog in target_programs_mtuci:
    data = all_mtuci_data.get(prog)
    if data is None:
        continue
    df = data['df']
    count_main = (df['Основной ВП'] == 'да').sum()
    count_pass = (df['Проходной ВП'] == 'да').sum()
    count_bvi = (df['БВИ'] == 'да').sum()

    count_site = int((df['Проходит (сайт)'] == 'да').sum())
    display(Markdown(f"### {data['competition']}"))
    display(Markdown(
        f"**Мест свободно:** {data['budget_da']} (план {data['budget']} − зачислено "
        f"БВИ {data['bvi_enrolled']})  |  **БВИ в списке:** {count_bvi}  |  "
        f"**Порог отбора:** {data['threshold']}  |  **Всего строк:** {len(df)}"
    ))
    display(Markdown(
        f'**Основной ВП (да):** {count_main}  |  '
        f'**Мест не занято абитуриентами выше порога:** '
        f'{free_places_mtuci.get(prog, 0)} (если больше нуля — проходной балл ниже нашего порога отбора)'
    ))
    # Вердикт вуза считается по подавшим согласие и по полному списку, наш Основной
    # ВП — по всем поданным заявлениям. Числа и должны расходиться: это две разные
    # картины, «если согласятся все» против «как есть сейчас».
    display(Markdown(
        f'**Вуз проводит сейчас (rating):** {len(data["site_pass_codes"])} чел. '
        f'(в нашей таблице видно {count_site}, остальные ниже порога '
        f'{data["threshold"]})  |  **Проходной балл вуза:** '
        f'{data["min_score"] if data["min_score"] is not None else "мест не осталось"}  |  '
        f'**Очередь актуальна на:** {data.get("rating_date") or "—"}'
    ))
    display_cols = ['№', '№_orig', 'Код', 'Сумма', 'Сумма без ИД', 'Оценки', 'Согласие',
                    'Приоритет', 'БВИ', 'Квота', 'ОВП сайта', 'ВПП сайта',
                    'Проходит (сайт)', 'Основной ВП', 'Проходной ВП', 'Примечание']
    display(df[[c for c in display_cols if c in df.columns]].style.hide(axis='index'))
    print('-' * 80)

# ===== Контроль нашей строки: нулевой запас по порогу =====
# Конкурсный балл 277 на МТУСИ равен порогу отсечения РОВНО. Любая пересдача ИД или
# пересчёт баллов уводит строку ниже порога, и она пропадёт из таблиц без единого
# сообщения — вместе с ней пропадёт и весь анализ по вузу. Поэтому положение строки
# относительно порога проверяем явно на каждом прогоне.
print(f'\nКонтроль кода {OUR_CODE_MTUCI}:')
our_rows_mtuci = 0
for prog in all_programs_mtuci:
    data = all_mtuci_data.get(prog)
    if data is None:
        continue
    df = data['df']
    ours = df[df['Код'] == OUR_CODE_MTUCI]
    for _, row in ours.iterrows():
        our_rows_mtuci += 1
        margin = int(row['Сумма']) - data['threshold']
        alarm = ('  <<< ЗАПАС НУЛЕВОЙ: строка ровно на пороге, потеря 1 балла '
                 'убирает её из списка молча' if margin == 0 else '')
        print(f"  {data['competition'][:52]:54s} №{row['№']}/{len(df)} "
              f"сумма {row['Сумма']} (порог {data['threshold']}, запас {margin:+d}), "
              f"приоритет {row['Приоритет']}, согласие {row['Согласие']}, "
              f"наш Основной ВП: {row['Основной ВП']}, вуз: {row['Проходит (сайт)']} "
              f"(ОВП сайта {row['ОВП сайта']}, ВПП сайта {row['ВПП сайта']}){alarm}")
if not our_rows_mtuci:
    print(f'  ВНИМАНИЕ: кода {OUR_CODE_MTUCI} нет ни в одном списке МТУСИ. Либо он '
          f'ушёл из конкурса, либо балл опустился ниже порога отбора — проверь руками.')

print(f'\nВремя работы БЛОКА 1 (МТУСИ): {time.time() - start_time_mtuci:.1f} сек.')
print('Данные для сводной готовы: all_mtuci_data, target_programs_mtuci — '
      'см. «БЛОК 2: СВОДНАЯ ТАБЛИЦА МТУСИ».')

In [ ]:
# -*- coding: utf-8 -*-
# ===== БЛОК 1: МЭИ =====
# Образцы — МТУСИ, Политех, МИИТ, Губкин. Тот же принцип: «Основной ВП» считаем
# по ВСЕМ конкурсным группам вуза, удовлетворяющим условиям, таблицы — по целевым.
#
# Источник — статические HTML-страницы (Selenium не нужен):
#   1. Индекс https://pk.mpei.ru/info/entrants_list — фильтры на странице чисто
#      клиентские (CSS-классы), в HTML сразу вся структура:
#      секция бакалавриата/специалитета class="groupFilterBasSpec" -> внутри секции
#      формы class="groupFilterFormO/OZ/Z" -> внутри формы tbody
#      class="groupFilterMoscow" (Москва; филиалы — другие tbody, отсекаются сами).
#      Строка = конкурсная группа: имя, курсивом коды программ, ссылки на списки.
#      Ссылка бюджета БЕЗ квот: class содержит listFilterBudget и НЕ содержит
#      listFilterTarget/Special/Separate/Contract. Проверено 17.07.2026: на
#      очно-заочной и заочной формах Москвы бюджетных ссылок нет вообще (только
#      платные) — как и сказано в ТЗ, для ВП их смотреть не надо. Значит ВП — по
#      всем очным бюджетным группам Москвы (27 шт.).
#   2. Страница списка entrants_listNN.html: шапка «Cписок поступающих (данные на
#      ЧЧ:ММ ДД.ММ.ГГГГ)» (первая C — ЛАТИНСКАЯ, в regex не матчить букву),
#      «Количество вакантных мест: N». Далее таблицы, каждая с текстом-заголовком
#      перед ней: «В рамках отдельной квоты (без проведения вступительных
#      испытаний)», «Без вступительных испытаний» и т.п., последняя — «По конкурсу»
#      (основной конкурс). По ТЗ людей из таблиц ДО «По конкурсу» (БВИ и льготники)
#      вычитаем из вакантных мест — но с 30.07.2026 только тех из них, кого вуз ещё
#      НЕ зачислил (см. п.4).
#   3. По ТЗ mei2.txt (18.07.2026) дополнительно вычитаем места КВОТНЫХ списков
#      группы: ссылки индекса с классом listFilterBudget + listFilterTarget
#      (целевая квота, ссылок несколько), + listFilterSpecial (особая квота),
#      + listFilterSeparate (отдельная квота) — с каждой такой страницы берём
#      «Количество вакантных мест». НЕ вычитаются только платные списки
#      (listFilterContract, в т.ч. AbroadPay).
#   4. 30.07–02.08.2026 прошло приоритетное зачисление (квоты и БВИ), и МЭИ сменил
#      СМЫСЛ числа: «Количество вакантных мест» и на бюджетной, и на квотных
#      страницах теперь ОСТАТОК (план минус зачисленные), а не план. Проверено на
#      живых данных 04.08.2026 против кэша 23.07.2026: падение вакантных = падение
#      остатка квот + число БВИ, подавших согласие, — сходится в ноль на 6 целевых
#      группах из 7. Значит зачисленных БВИ вычитать второй раз НЕЛЬЗЯ: у ФИИТ это
#      давало 6 − 7 − 0 = −1 место вместо 6, и группа вылетала из расчёта ВП. Из
#      предтаблицы вычитаем только строки, где согласие ещё НЕ «да»: за таким БВИ
#      место продолжает висеть, он может подать согласие сегодня. Итого: места
#      общего конкурса = вакантные − БВИ без согласия − остаток квот группы.
#      Седьмая целевая (ИВТ) даёт +4: вакантные упали на 31 (26 зачислено по квотам
#      + 5 БВИ), а остаток квот — на 30, то есть 4 недобранных целевых места вуз
#      вернул в общий конкурс. Формула «вакантные − остаток квот» ловит этот
#      возврат сама, отдельной правки не нужно.
#   5. С 30.07.2026 в таблице «По конкурсу» есть строки-призраки, которые в конкурсе
#      уже не участвуют: <tr class="Excluded"> (примечание «Забрал документы»,
#      7749 строк по вузу — класс и текст совпадают один в один) и примечание
#      «Зачислен в другой КГ» (2271 строка: человек зачислен по квоте/БВИ в другую
#      конкурсную группу МЭИ). Проверено: пометка «Зачислен в другой КГ» стоит во
#      ВСЕХ группах этого кода, «чистой» строки не остаётся ни у одного из 448
#      таких кодов — второй раз он нигде не сядет. Класс строки виден, только если
#      ловить атрибуты тега, поэтому regex строк теперь <tr([^>]*)>. Забравших
#      документы ловим по классу ИЛИ по тексту примечания и печатаем расхождение:
#      сегодня признаки совпадают один в один, и если вуз переименует класс или
#      сменит формулировку, выбывший не вернётся в конкурс молча. Призраки не
#      претендуют на места: в Основной ВП и в сквозной номер №_orig не идут, в
#      таблице остаются с пометкой в колонке «Выбыл».
#
# Колонки «По конкурсу»: Уникальный код, Сумма, Сумма без ИД, Баллы ВИ (colspan =
# число слотов, обычно 3: слот 1 Мат.|Инж.мат., слот 2 Физ.|ИТ Проф.|Инф., слот 3
# Рус.), ИД, Преимущ. право (colspan=2: ч.9/ч.10 ст.71), Согласие, Приоритет,
# Основной высший, Высший проходной, Общежитие, Примечание. Строка данных =
# 12 + n_slots ячеек, хвост разбираем отрицательными индексами. Сайтовые колонки
# «Основной высший»/«Высший проходной» НЕ используем — ВП считаем сами.
#
# Порог (как у всех): 277, если среди предметов есть физика, иначе 269.
# Строки ниже порога (по сумме С ИД) не сохраняем; хвост списка без баллов
# отсекается порогом автоматически.
#
# Кэш: на каждой странице списка есть «данные на ЧЧ:ММ ДД.ММ.ГГГГ» — страницы
# генерируются пакетно, дата у всех одна. Валидность кэша: дата первой бюджетной
# страницы + состав бюджетных и квотных ссылок индекса.

import time
import os
import re
import pickle
import collections
import requests
import urllib3
import pandas as pd
from IPython.display import display, Markdown

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

start_time_mei = time.time()

INDEX_URL_MEI = 'https://pk.mpei.ru/info/entrants_list'
BASE_URL_MEI = 'https://pk.mpei.ru/info/'
HTTP_HEADERS_MEI = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'),
    'Accept-Language': 'ru-RU,ru;q=0.9',
    'Referer': INDEX_URL_MEI,
}
CACHE_MEI = 'mei_cache.pkl'
# Версия схемы парсинга. Инкрементируй при изменении логики parse_list_page_mei —
# иначе при неизменившейся дате из кэша подтянутся старые данные.
# v1: индекс (бак/спец, очная, Москва, бюджет без квот) + страницы списков; места
#     общ.конкурса = вакантные - люди таблиц до «По конкурсу» (БВИ/льготники, ТЗ);
#     слоты предметов по colspan «Баллы ВИ», порог 277/269 по сумме с ИД,
#     Основной ВП = deferred acceptance.
# v2: из мест общего конкурса дополнительно вычитаются места квотных списков
#     группы («Количество вакантных мест» страниц целевых/особой/отдельной квот,
#     ТЗ mei2.txt); платные списки не вычитаются.
# v3: 04.08.2026, после приоритетного зачисления. «Вакантных» — остаток, поэтому из
#     предтаблицы БВИ вычитаются только строки БЕЗ согласия (было: все подряд):
#     по 7 целевым мест 57/−1/133/36/22/21/8 -> 64/6/138/37/22/24/8. Строки
#     <tr class="Excluded"> («Забрал документы») и «Зачислен в другой КГ» больше не
#     считаются конкурентами: выброшены из Основного ВП и из №_orig, в таблице
#     помечены колонкой «Выбыл» (по 7 целевым 2323 + 434 = 2757 строк). Нехватка
#     мест (остаток квот больше вакантных) и список без строк выше порога больше не
#     глушатся молча — печатается диагноз и число в data['places_short'].
# v4: 04.08.2026, дедуп по коду больше не выбрасывает ЖИВУЮ строку в пользу
#     выбывшей. Одна конкурсная группа покрывает несколько программ, человек может
#     стоять в ней дважды — по одной заявке живым, по другой с пометкой «Забрал
#     документы»; призраки счётчик №_orig не двигают, поэтому у призрака номер на
#     единицу меньше, и при равной сумме он выигрывал сортировку. Живая заявка
#     молча получала «Выбыл» и вылетала из конкурса — направление ошибки опасное,
#     конкурент исчезает и наши шансы завышаются. Живьём 04.08 таких кодов 12 выше
#     порога, в 4 терялась живая строка (на состав принятых не повлияло — удача).
PARSE_VERSION_MEI = 4

# --- Целевые конкурсные группы из ТЗ (mei.txt) — матчим по имени группы в индексе. ---
TARGET_MEI = [
    'Прикладная математика, информатика, математическое моделирование',
    'Фундаментальная информатика и информационные технологии',
    'Информатика и вычислительная техника',
    'ИТНО. Информационные системы и технологии',
    'Прикладная информатика',
    'Информационная безопасность',
    'Бизнес-информатика',
]

session_mei = requests.Session()
session_mei.headers.update(HTTP_HEADERS_MEI)


# 23.07.2026: сайты вузов в горячую фазу выпадают на минуту-две (502, таймаут
# соединения). Ждём до ~2.5 мин вместо прежних 12 с; connect отдельно от read.
RETRY_DELAYS_MEI = (3, 6, 12, 24, 40, 60)
TIMEOUT_MEI = (15, 90)  # (соединение, чтение)


def fetch_mei(url, retries=len(RETRY_DELAYS_MEI) + 1):
    """GET страницы МЭИ с повторами (экспоненциальный бэкофф)."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session_mei.get(url, timeout=TIMEOUT_MEI, verify=False)
            response.raise_for_status()
            response.encoding = 'utf-8'
            return response.text
        except Exception as error:
            last_error = error
            if attempt == retries - 1:
                break
            pause = RETRY_DELAYS_MEI[min(attempt, len(RETRY_DELAYS_MEI) - 1)]
            print(f'  МЭИ не ответил ({type(error).__name__}: {error}). '
                  f'Повтор {attempt + 2}/{retries} через {pause} с...')
            time.sleep(pause)
    raise RuntimeError(f'Не удалось загрузить {url}: {last_error}')


def _clean_mei(html_fragment):
    txt = re.sub(r'<[^>]+>', ' ', html_fragment).replace('&nbsp;', ' ')
    return re.sub(r'\s+', ' ', txt).strip()


def _norm_name_mei(name):
    return re.sub(r'\s+', ' ', name.replace('–', '-').replace('—', '-')).strip().lower()


def parse_index_mei(html):
    """Индекс -> список бюджетных конкурсных групп Москвы, очная форма бак/спец.

    Возвращает [{'name', 'codes', 'href', 'display', 'quota_hrefs'}] в порядке
    индекса; quota_hrefs — [(вид квоты, href)] по квотным спискам группы
    (целевые — несколько ссылок, особая, отдельная; платные не входят).
    """
    bas_start = html.find('class="groupFilterBasSpec"')
    bas_end = html.find('class="groupFilterMag"')
    if bas_start < 0:
        raise RuntimeError('Индекс МЭИ: не найдена секция groupFilterBasSpec.')
    section = html[bas_start:bas_end if bas_end > 0 else len(html)]

    form_start = section.find('class="groupFilterFormO"')
    form_end = section.find('class="groupFilterFormOZ"')
    if form_start < 0:
        raise RuntimeError('Индекс МЭИ: не найдена очная форма groupFilterFormO.')
    form_zone = section[form_start:form_end if form_end > 0 else len(section)]

    tbody_match = re.search(r'<tbody class="groupFilterMoscow">(.*?)</tbody>', form_zone, re.S)
    if tbody_match is None:
        raise RuntimeError('Индекс МЭИ: не найден tbody groupFilterMoscow (очная форма).')

    programs = []
    for tr in re.findall(r'<tr>(.*?)</tr>', tbody_match.group(1), re.S):
        tds = re.findall(r'<td[^>]*>(.*?)</td>', tr, re.S)
        if len(tds) < 2:
            continue
        codes_match = re.search(r'<div[^>]*>\((.*?)\)</div>', tds[0], re.S)
        codes = _clean_mei(codes_match.group(1)) if codes_match else ''
        name = _clean_mei(re.sub(r'<div.*?</div>', '', tds[0], flags=re.S))
        budget_href = None
        quota_hrefs = []
        for a_attrs in re.findall(r'<a\b([^>]*)>', tds[1]):
            cls_match = re.search(r'class="([^"]*)"', a_attrs)
            href_match = re.search(r'href="([^"]+)"', a_attrs)
            if not cls_match or not href_match:
                continue
            cls = cls_match.group(1)
            href = href_match.group(1)
            if 'listFilterContract' in cls or 'listFilterBudget' not in cls:
                continue   # платные списки по ТЗ не вычитаются вообще
            if 'listFilterTarget' in cls:
                quota_hrefs.append(('целевая', href))
            elif 'listFilterSpecial' in cls:
                quota_hrefs.append(('особая', href))
            elif 'listFilterSeparate' in cls:
                quota_hrefs.append(('отдельная', href))
            elif budget_href is None:
                budget_href = href   # бюджет без квот — основной список группы
        if budget_href is None:
            continue   # у группы нет бюджетных мест (только платно) — для ВП не нужна
        programs.append({
            'name': name,
            'codes': codes,
            'href': budget_href,
            'display': f'{name} ({codes})' if codes else name,
            'quota_hrefs': quota_hrefs,
        })
    return programs


def subject_label_mei(header_text):
    """Подпись слота по тексту ячейки шапки (физика приоритетнее)."""
    joined = header_text.lower()
    if 'физ' in joined:
        return 'Физ.'
    if 'инф' in joined or 'ит проф' in joined:
        return 'Инф.'
    if 'мат' in joined:
        return 'Мат.'
    if 'рус' in joined:
        return 'Рус.'
    if 'общество' in joined or 'общ.' in joined:
        return 'Обществ.'
    if 'ин.яз' in joined or 'иностр' in joined:
        return 'Ин.яз.'
    return header_text.split()[0][:8] if header_text.strip() else '—'


def _to_int_mei(value, default=0):
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return default


def _pre_table_label_mei(heading_lower):
    if 'отдельной квоты' in heading_lower or 'отдельная квота' in heading_lower:
        return 'отдельная квота без ВИ'
    if 'без вступительных испытаний' in heading_lower:
        return 'БВИ'
    if 'особ' in heading_lower:
        return 'особые права'
    return 'льготники'


def quota_places_mei(prog):
    """Остаток мест квотных списков группы (целевые/особая/отдельная).

    С каждой квотной страницы берётся «Количество вакантных мест» (ТЗ mei2.txt).
    С 30.07.2026 это уже не план квоты, а её ОСТАТОК: зачисленных по квоте вуз из
    числа вычел, и недобранные целевые места вуз возвращает в общий конкурс —
    поэтому «вакантные общей страницы − этот остаток» само даёт места общего
    конкурса. Возвращает (всего мест, разбивка строкой по видам квот).
    """
    total = 0
    by_kind = {}
    for kind, href in prog.get('quota_hrefs', []):
        quota_html = fetch_mei(BASE_URL_MEI + href)
        places_match = re.search(r'Количество вакантных мест[^0-9<]*(\d+)', quota_html)
        if places_match is None:
            print(f'  ВНИМАНИЕ: {prog["name"][:50]} — на квотной странице {href} '
                  f'нет «Количество вакантных мест», считаем 0.')
            continue
        places = int(places_match.group(1))
        total += places
        by_kind[kind] = by_kind.get(kind, 0) + places
    breakdown = '; '.join(f'{kind}: {places}' for kind, places in by_kind.items())
    return total, breakdown or 'нет'


def parse_list_page_mei(html, prog, is_target, quota_places, quota_breakdown):
    """Парсит страницу бюджетного списка одной конкурсной группы.

    Места общего конкурса = «Количество вакантных мест» (после 30.07.2026 это
    остаток: план минус зачисленные) минус те люди из таблиц ДО «По конкурсу»,
    кого вуз ещё НЕ зачислил (согласие не «да»), минус остаток квотных списков
    группы (quota_places, ТЗ mei2.txt). Зачисленных БВИ вуз из вакантных уже
    вычел — второй раз их вычитать нельзя.
    Строки ниже порога (по сумме с ИД) не сохраняются. Строки-призраки («Забрал
    документы» = class="Excluded", «Зачислен в другой КГ») сохраняются с пометкой
    «Выбыл», но не нумеруются в №_orig и не участвуют в конкурсе.
    """
    date_match = re.search(r'писок поступающих\s*\(данные на\s*([^)]+)\)', html)
    update_date = date_match.group(1).strip() if date_match else None
    vacant_match = re.search(r'Количество вакантных мест[^0-9<]*(\d+)', html)
    vacant = int(vacant_match.group(1)) if vacant_match else 0

    main_table = None
    pre_pending = 0      # БВИ/льготники без согласия — место за ними ещё висит
    pre_enrolled = 0     # с согласием: вуз их уже зачислил и вычел из вакантных
    pre_breakdown = []
    prev_end = 0
    for table_match in re.finditer(r'<table[^>]*>.*?</table>', html, re.S):
        heading_zone = re.sub(r'<script.*?</script>', '', html[prev_end:table_match.start()],
                              flags=re.S)
        heading = _clean_mei(heading_zone)[-250:].lower()
        prev_end = table_match.end()
        if 'по конкурсу' in heading:
            main_table = table_match.group(0)
            break
        # В предтаблице 16 ячеек (лишняя колонка «Основание»), но хвост тот же —
        # согласие берём тем же отрицательным индексом, что и в основной таблице.
        people = [
            cells for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', table_match.group(0), re.S)
            for cells in [[_clean_mei(c) for c in
                           re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', tr, re.S)]]
            if cells and re.fullmatch(r'\d{5,9}', cells[0])
        ]
        if not people:
            continue
        # Строку, у которой хвост не читается, считаем незачисленной — так место
        # останется вычтенным, то есть ошибка сыграет в минус нашим шансам.
        enrolled = sum(1 for cells in people
                       if len(cells) >= 13 and cells[-6].lower() == 'да')
        pre_enrolled += enrolled
        pre_pending += len(people) - enrolled
        pre_breakdown.append(f'{_pre_table_label_mei(heading)}: {len(people)} '
                             f'(зачислено {enrolled}, ждут согласия {len(people) - enrolled})')

    if main_table is None:
        print(f'  ВНИМАНИЕ: {prog["name"][:50]} — таблица «По конкурсу» не найдена, пропуск.')
        return None

    # Атрибуты <tr> нужны целиком: «Забрал документы» видно только по class="Excluded".
    rows_html = re.findall(r'<tr([^>]*)>(.*?)</tr>', main_table, re.S)
    if len(rows_html) < 3:
        raise RuntimeError(f'{prog["name"][:50]}: в таблице «По конкурсу» {len(rows_html)} '
                           f'строк — разметка сайта изменилась.')

    # Шапка: colspan ячейки «Баллы ВИ» = число слотов предметов; подписи слотов —
    # из первых n_slots ячеек второй строки шапки («Мат.Инж.мат.» и т.п.).
    balls_match = re.search(r'colspan="?(\d+)"?[^>]*>\s*Баллы\s*ВИ', rows_html[0][1])
    n_slots = int(balls_match.group(1)) if balls_match else 3
    header2_cells = [_clean_mei(c) for c in
                     re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', rows_html[1][1], re.S)]
    slot_labels = [subject_label_mei(c) for c in header2_cells[:n_slots]]
    has_physics = any(label == 'Физ.' for label in slot_labels)
    threshold = 277 if has_physics else 269

    rows = []
    position = 0          # сквозной номер ЖИВОЙ строки: призраки его не двигают
    people_total = 0
    n_withdrawn = 0       # «Забрал документы»
    n_other_kg = 0        # «Зачислен в другой КГ»
    n_class_only = 0      # класс есть, а текста нет (и наоборот) — расхождение
    for attrs, tr in rows_html[2:]:
        cells = [_clean_mei(c) for c in re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', tr, re.S)]
        if not cells or not re.fullmatch(r'\d{5,9}', cells[0]):
            continue
        people_total += 1
        if len(cells) < 13:
            continue
        class_match = re.search(r'class="([^"]*)"', attrs)
        row_class = class_match.group(1) if class_match else ''
        note_lower = cells[-1].lower()
        # 04.08.2026 класс Excluded и текст «Забрал документы» совпадают один в один
        # на всех 96 367 строках вуза. Берём ИЛИ, чтобы переименование класса или
        # смена формулировки не вернули выбывшего в конкурс молча, а расхождение
        # печатаем — это сигнал, что разметка поехала.
        by_class = 'Excluded' in row_class
        by_note = 'забрал документы' in note_lower
        withdrawn = by_class or by_note
        other_kg = 'зачислен в другой' in note_lower   # уже сидит в другой КГ МЭИ
        n_withdrawn += withdrawn
        n_other_kg += other_kg
        n_class_only += by_class != by_note
        if not (withdrawn or other_kg):
            position += 1
        row_slots = len(cells) - 12
        total = _to_int_mei(cells[1])
        if total < threshold:
            continue

        marks = ', '.join(
            f'{slot_labels[i] if i < len(slot_labels) else f"ВИ{i + 1}"}: {cells[3 + i]}'
            for i in range(row_slots) if cells[3 + i].isdigit()
        ) or '—'
        note_parts = []
        if cells[-1]:
            note_parts.append(cells[-1])
        # Преимущ. право из колонок ч.9/ч.10 ст.71 — только если сайт сам не написал
        # «Преимущественное право» в примечании (иначе дубль).
        if (cells[-8] == 'да' or cells[-7] == 'да') and 'преимущ' not in cells[-1].lower():
            note_parts.append('преим. право')
        rows.append({
            # У призрака №_orig = номер последней живой строки перед ним: «стоял бы
            # сразу за ней». Живые строки нумеруются подряд без дыр.
            '№_orig': position,
            'Код': cells[0],
            'Сумма': total,
            'Сумма без ИД': _to_int_mei(cells[2]),
            'Оценки': marks,
            'Согласие': 'да' if cells[-6].lower() == 'да' else '',
            'Приоритет': _to_int_mei(cells[-5], 99),
            'Выбыл': 'да' if (withdrawn or other_kg) else '',
            'Примечание': '; '.join(note_parts),
        })

    if people_total == 0:
        raise RuntimeError(f'{prog["name"][:50]}: в таблице «По конкурсу» разобрано ноль '
                           f'строк-людей при {len(rows_html)} строках HTML — сломалась '
                           f'разметка, а не пустой список.')
    if n_class_only:
        print(f'  ВНИМАНИЕ: {prog["name"][:50]} — у {n_class_only} строк class="Excluded" '
              f'и примечание «Забрал документы» разошлись; выбывших считаем по любому '
              f'из двух признаков, разметку сайта надо перепроверить.')
    if not rows:
        print(f'  ВНИМАНИЕ: {prog["name"][:50]} — все {people_total} строк ниже порога '
              f'{threshold}, группа в расчёт Основного ВП не идёт.')
        return None

    places_general = vacant - pre_pending - quota_places
    places_short = 0
    if places_general < 0:
        # Раньше молча клампилось в 0 и группа выпадала из расчёта. Теперь громко:
        # это либо возврат квотных мест, который вуз ещё не отразил, либо наша
        # ошибка в разборе квотных страниц — человек должен это увидеть.
        places_short = -places_general
        print(f'  ВНИМАНИЕ: {prog["name"][:50]} — остаток квот ({quota_places}) плюс '
              f'БВИ без согласия ({pre_pending}) больше вакантных ({vacant}) на '
              f'{places_short}; мест общего конкурса считаем 0, число под вопросом.')
        places_general = 0

    df = pd.DataFrame(rows)
    # Дедуп по Коду; сортировка по Сумме (с ИД) по убыванию, тай-брейк по №_orig
    # (порядок сайта) — детерминизм.
    #
    # ЖИВАЯ строка обязана идти РАНЬШЕ выбывшей: одна конкурсная группа покрывает
    # несколько программ, и человек может стоять в ней дважды — по одной заявке
    # живым, по другой с пометкой «Забрал документы» / «Зачислен в другой КГ».
    # Призраки счётчик №_orig не двигают, поэтому у призрака номер на единицу
    # меньше, чем у его же живой строки, и при равной сумме он выигрывал дедуп:
    # живая заявка молча получала «Выбыл» и вылетала из конкурса. Направление
    # ошибки опасное — конкурент исчезает и наши шансы завышаются. Живьём 04.08
    # таких кодов 12 выше порога, в 4 терялась живая строка (на состав принятых
    # сегодня не повлияло, но это удача, а не свойство).
    sort_cols = ['Сумма', '№_orig']
    ascending = [False, True]
    if 'Выбыл' in df.columns:
        df['_живой'] = (df['Выбыл'].astype(str).str.strip() == '').astype(int)
        sort_cols = ['Сумма', '_живой', '№_orig']
        ascending = [False, False, True]
    df = (df.sort_values(sort_cols, ascending=ascending, kind='stable')
            .drop_duplicates('Код', keep='first')
            .drop(columns=['_живой'], errors='ignore')
            .reset_index(drop=True))
    df.insert(0, '№', range(1, len(df) + 1))

    return {
        'df': df,
        'budget': places_general,
        'places_short': places_short,
        'vacant': vacant,
        'pre_people': pre_pending,
        'pre_enrolled': pre_enrolled,
        'pre_breakdown': '; '.join(pre_breakdown) if pre_breakdown else 'нет',
        'quota_places': quota_places,
        'quota_breakdown': quota_breakdown,
        'people_total': people_total,
        'n_withdrawn': n_withdrawn,
        'n_other_kg': n_other_kg,
        'threshold': threshold,
        'form': 'Очная',
        'university': 'МЭИ',
        'competition': prog['display'],
        'is_target': is_target,
        'update_date': update_date,
    }


# ===== Сбор данных: индекс -> бюджетные группы Москвы (очная, бак/спец) =====
print('Загрузка индекса списков МЭИ...')
all_programs_mei = parse_index_mei(fetch_mei(INDEX_URL_MEI))
print(f'Бюджетных конкурсных групп (Москва, очная, бак/спец): {len(all_programs_mei)}.')

target_hrefs_mei = []
for tz_name in TARGET_MEI:
    prog_hit = next((p for p in all_programs_mei
                     if _norm_name_mei(p['name']) == _norm_name_mei(tz_name)), None)
    if prog_hit is None:
        print(f'  ВНИМАНИЕ: целевая «{tz_name[:55]}» не найдена в индексе с бюджетом.')
        continue
    if prog_hit['href'] not in target_hrefs_mei:
        target_hrefs_mei.append(prog_hit['href'])
target_ids_mei = target_hrefs_mei
target_names_mei = {p['href']: p['display'] for p in all_programs_mei
                    if p['href'] in target_ids_mei}
print(f'Целевых групп: {len(target_ids_mei)} (из {len(TARGET_MEI)} строк ТЗ).')

# Дата актуальности — с первой бюджетной страницы (страницы генерируются пакетно,
# дата у всех одна). Сигнатура кэша: дата + состав бюджетных и квотных ссылок.
first_prog_mei = all_programs_mei[0]
first_html_mei = fetch_mei(BASE_URL_MEI + first_prog_mei['href'])
first_date_match = re.search(r'писок поступающих\s*\(данные на\s*([^)]+)\)', first_html_mei)
update_date_mei = first_date_match.group(1).strip() if first_date_match else None
print(f'Данные списков МЭИ на: {update_date_mei}')

programs_signature_mei = (
    update_date_mei,
    tuple(sorted(p['href'] for p in all_programs_mei)),
    tuple(sorted(href for p in all_programs_mei for _, href in p['quota_hrefs'])),
)

cached_mei = None
if os.path.exists(CACHE_MEI):
    with open(CACHE_MEI, 'rb') as f:
        cached_mei = pickle.load(f)

cache_valid_mei = (
    cached_mei is not None
    and cached_mei.get('version') == PARSE_VERSION_MEI
    and cached_mei.get('signature') == programs_signature_mei
    and cached_mei.get('data')
)

all_mei_data = {}
if cache_valid_mei:
    all_mei_data = cached_mei['data']
    print('Дата списков не изменилась — используем кэш.')
else:
    failed_mei = []
    for idx, prog in enumerate(all_programs_mei, 1):
        is_target = prog['href'] in target_ids_mei
        tag = 'ЦЕЛЕВАЯ' if is_target else 'общая'
        try:
            page_html = (first_html_mei if prog['href'] == first_prog_mei['href']
                         else fetch_mei(BASE_URL_MEI + prog['href']))
            quota_places, quota_breakdown = quota_places_mei(prog)
            data = parse_list_page_mei(page_html, prog, is_target,
                                       quota_places, quota_breakdown)
        except Exception as error:
            print(f'  [{idx}/{len(all_programs_mei)}] Ошибка {prog["name"][:45]}: {error}')
            failed_mei.append(prog['name'][:45])
            continue
        if data is None:
            continue
        all_mei_data[prog['href']] = data
        print(f'  [{idx}/{len(all_programs_mei)}] {tag}: {prog["name"][:45]} '
              f'[{data["form"]}] мест общ.конкурса={data["budget"]} '
              f'(вакантных {data["vacant"]} - БВИ без согласия {data["pre_people"]} '
              f'[{data["pre_breakdown"]}] - остаток квот {data["quota_places"]} '
              f'[{data["quota_breakdown"]}]), порог={data["threshold"]}, '
              f'строк={len(data["df"])}, выбыло по списку '
              f'{data["n_withdrawn"]} забрали документы + {data["n_other_kg"]} '
              f'зачислены в другую КГ из {data["people_total"]}')

    # Кэш пишем ТОЛЬКО при полной загрузке (см. пояснение в блоке МТУСИ).
    if all_mei_data and not failed_mei:
        with open(CACHE_MEI, 'wb') as f:
            pickle.dump({
                'version': PARSE_VERSION_MEI,
                'signature': programs_signature_mei,
                'data': all_mei_data,
            }, f)
        print('Данные сохранены в кэш.')
    elif failed_mei:
        print(f'  ВНИМАНИЕ: не загрузились группы ({len(failed_mei)}): '
              f'{"; ".join(failed_mei[:3])} — кэш НЕ сохранён, '
              f'работаем на неполных данных.')

if not all_mei_data:
    raise SystemExit('Нет данных ни по одной конкурсной группе МЭИ.')

loaded_targets_mei = [pid for pid in target_ids_mei if pid in all_mei_data]
print(f'Загружено групп: {len(all_mei_data)}; '
      f'целевых с данными: {len(loaded_targets_mei)}/{len(target_ids_mei)}.')

# ===== Расчёт Основного ВП (отложенная приёмка по ВСЕМ конкурсным группам) =====
# Кандидат-предлагающая deferred acceptance, как в мтуси_блок3.py / политех_блок4.py /
# миит_блок5.py / губкин_блок6.py: абитуриент садится на группу с наивысшим
# приоритетом, где реально проходит по баллам после ухода конкурентов (вытеснение).
prog_rank_mei = {}      # href -> {Код: позиция в списке, меньше = сильнее}
prog_budget_mei = {}    # href -> места общего конкурса (вакантные - БВИ без согл. - квоты)
cand_prefs_mei = {}     # Код -> href групп по возрастанию приоритета
for key, data in all_mei_data.items():
    # Забравшие документы и зачисленные в другую КГ на места не претендуют —
    # в конкурсе участвуют только живые строки.
    df_live = data['df'][data['df']['Выбыл'] == '']
    prog_budget_mei[key] = data['budget']
    prog_rank_mei[key] = {code: i for i, code in enumerate(df_live['Код'])}
    for code, priority in zip(df_live['Код'], df_live['Приоритет']):
        cand_prefs_mei.setdefault(code, []).append((int(priority), key))
for code, pairs in cand_prefs_mei.items():
    cand_prefs_mei[code] = [key for _, key in sorted(pairs, key=lambda x: x[0])]

admitted_mei = {key: {} for key in all_mei_data}   # href -> {Код: ранг}
next_choice_mei = {code: 0 for code in cand_prefs_mei}
pending_mei = collections.deque(cand_prefs_mei)

while pending_mei:
    code = pending_mei.popleft()
    prefs = cand_prefs_mei[code]
    while next_choice_mei[code] < len(prefs):
        key = prefs[next_choice_mei[code]]
        budget = prog_budget_mei[key]
        if budget <= 0:
            next_choice_mei[code] += 1
            continue
        rank = prog_rank_mei[key][code]
        held = admitted_mei[key]
        if len(held) < budget:
            held[code] = rank
            break
        worst_code = max(held, key=held.get)   # держим слабейшего (наибольший ранг)
        if rank < held[worst_code]:
            del held[worst_code]
            held[code] = rank
            next_choice_mei[worst_code] += 1
            pending_mei.append(worst_code)   # вытесненный ищет следующий приоритет
            break
        next_choice_mei[code] += 1           # здесь не прошёл — следующий приоритет

enrolled_mei = {}
for key, data in all_mei_data.items():
    admitted_codes = set(admitted_mei[key])
    data['df']['Основной ВП'] = (
        data['df']['Код'].isin(admitted_codes).map({True: 'да', False: 'нет'})
    )
    enrolled_mei.update((code, True) for code in admitted_codes)

# ===== Свободные места после Основного ВП =====
# «Проходной ВП» своим расчётом получить НЕЛЬЗЯ: после отложенной приёмки у
# группы со свободными местами не остаётся незачисленных кандидатов — тот, кого
# не взяли нигде, дошёл бы до этого свободного места и занял его. Прежний расчёт
# давал «нет» во всех строках всех групп и создавал ложное впечатление проверки.
# Полезный сигнал — сам остаток: если места не разобраны абитуриентами выше
# порога, значит проходной балл группы НИЖЕ нашего порога отбора.
free_places_mei = {}
for key, data in all_mei_data.items():
    df = data['df']
    df['Проходной ВП'] = 'нет'
    free_places_mei[key] = max(
        0, data['budget'] - int((df['Основной ВП'] == 'да').sum()))

# ===== Отдельные таблицы по конкурсным группам (только целевые, порядок ТЗ) =====
display(Markdown('# БЛОК 1: МЭИ'))
for key in target_ids_mei:
    data = all_mei_data.get(key)
    if data is None:
        display(Markdown(f'### {target_names_mei.get(key, key)} — нет данных'))
        print('-' * 80)
        continue
    df = data['df']
    count_main = (df['Основной ВП'] == 'да').sum()
    count_pass = (df['Проходной ВП'] == 'да').sum()

    n_left = int((df['Выбыл'] == 'да').sum())

    display(Markdown(f"### {data['competition']} ({data['form']})"))
    display(Markdown(
        f"**Мест общего конкурса:** {data['budget']} "
        f"(вакантных-остаток {data['vacant']} − БВИ без согласия {data['pre_people']} "
        f"[{data['pre_breakdown']}] − остаток квот {data['quota_places']} "
        f"[{data['quota_breakdown']}])"
        + (f'  |  **МЕСТ НЕ ХВАТИЛО НА {data["places_short"]} — число под вопросом**'
           if data.get('places_short') else '')
        + f"  |  **Порог отбора:** {data['threshold']}  |  "
        f"**Всего строк:** {len(df)} (из них выбывших {n_left})"
    ))
    display(Markdown(
        f"**Выбыли из конкурса по всему списку группы:** "
        f"{data['n_withdrawn']} забрали документы + {data['n_other_kg']} зачислены "
        f"в другую КГ из {data['people_total']} строк сайта — в местах, конкурентах "
        f"и №_orig они не считаются"
    ))
    display(Markdown(
        f'**Основной ВП (да):** {count_main}  |  '
        f'**Мест не занято абитуриентами выше порога:** {free_places_mei.get(key, 0)} '
        f'(если больше нуля — проходной балл группы ниже нашего порога отбора)'
    ))
    display_cols = ['№', '№_orig', 'Код', 'Сумма', 'Сумма без ИД', 'Оценки', 'Согласие',
                    'Приоритет', 'Выбыл', 'Основной ВП', 'Проходной ВП', 'Примечание']
    display(df[[c for c in display_cols if c in df.columns]].style.hide(axis='index'))
    print('-' * 80)

print(f'\nВремя работы БЛОКА 1 (МЭИ): {time.time() - start_time_mei:.1f} сек.')
print('Данные для сводной готовы: all_mei_data, target_ids_mei — '
      'см. «БЛОК 2: СВОДНАЯ ТАБЛИЦА МЭИ».')

In [ ]:
# -*- coding: utf-8 -*-
# ===== БЛОК 1: МОСКОВСКИЙ ПОЛИТЕХ =====
# Образец — рабочий код МТУСИ (БЛОК 1: МТУСИ, мтуси_блок3.py). Тот же принцип:
# «Основной ВП»
# считаем по ВСЕМ программам вуза, удовлетворяющим условиям, а таблицы строим
# только по целевым.
#
# Условия для программ, участвующих в расчёте ВП:
#   - площадка Москва (бакалавриат/специалитет), select1=000000066_01;
#   - ОБЩИЙ конкурс (в строках держим только Льгота=0, квоты отбрасываем);
#   - формы очная / очно-заочная / заочная;
#   - мест общего конкурса > 0.
# Перебираем ВСЕ программы площадки (их ~65) × 3 формы; оставляем те, где есть
# места общего конкурса и участники. Таблицы и итоговый DataFrame — только по 15
# целевым программам (очная).
#
# Selenium/Playwright не нужны: списки отдаёт AJAX-эндпоинт обычным POST-запросом.
#   Страница рейтинга (в ней select1=площадка, select2=программа со spec_code,
#   eduForm, eduFin):
#     https://mospolytech.ru/postupayushchim/priem-v-universitet/rating-abiturientov/
#   Эндпоинт: <base>/rating-abiturientov/fio_list_curl.php
#     POST select1=000000066_01, specCode=<из атрибута option spec_code>,
#          eduForm=<Очная|Очно-заочная|Заочная>, eduFin=Бюджетная основа, f=1.
#   Критично: и запрос, и ответ в UTF-8.
#
# Места общего конкурса = «Количество бюджетных мест*» (КЦП) − (целевая + особая +
# отдельная квоты) − БВИ. БВИ — строки с цифрой 5 или 6 в колонке «Льгота»:
# победители/призёры олимпиад, зачисляются без вступительных испытаний и занимают
# места общего конкурса. В верхней сводке ответа строка программы =
# [код, название, КЦП, целевая, особая, отдельная, проходной прошлого года].
#
# Раскладка колонок списка НЕ фиксированная: у обычных программ 23 колонки
# (3 предметных балла), у творческих (дизайн, графика) — 24 (4 экзамена), и все
# колонки после предметных сдвинуты на +1. Поэтому индексы определяем динамически
# по строке заголовков таблицы («Уникальный код», «Льгота», «Статус в Госуслугах»...).
#
# Порог отсечения (как у МТУСИ): 277, если у программы среди предметов есть физика
# (в т.ч. как альтернатива информатике), иначе 269. Строки общего конкурса ниже
# порога не сохраняем — на результат анализа они не влияют.
#
# Оценки: если в предметном столбце присутствует физика (пусть даже часть
# абитуриентов сдавала информатику как альтернативу) — подписываем «Физ.» всем,
# но балл оставляем как есть.

import time
import os
import re
import pickle
import collections
import requests
import urllib3
import pandas as pd
from urllib.parse import quote
from IPython.display import display, Markdown

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

start_time_polytech = time.time()

RATING_URL_POLYTECH = ('https://mospolytech.ru/postupayushchim/priem-v-universitet/'
                       'rating-abiturientov/')
BASE_URL_POLYTECH = RATING_URL_POLYTECH + 'fio_list_curl.php'
HTTP_HEADERS_POLYTECH = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'),
    'Accept-Language': 'ru-RU,ru;q=0.9',
    'Referer': RATING_URL_POLYTECH,
    'Content-Type': 'application/x-www-form-urlencoded',
}
CACHE_POLYTECH = 'polytech_cache.pkl'
# Версия схемы парсинга. Инкрементируй при изменении логики parse_program_polytech —
# иначе при неизменившейся дате из кэша подтянутся старые данные.
# v1: полный перебор программ площадки × 3 формы, места общего конкурса = КЦП−квоты,
#     только общий конкурс (Льгота=0), порог 277/269, Основной ВП = deferred acceptance.
# v2: без фильтра по «Статусу в Госуслугах» (учитываем всех выше порога, как МТУСИ);
#     стабильный тай-брейк по №_orig при равном конкурсном балле.
# v3: места общего конкурса = КЦП − квоты − БВИ (Льгота 5/6, олимпиадники без ВИ);
#     индексы колонок определяются по строке заголовков (у творческих программ
#     24 колонки вместо 23 — раньше их Льгота/баллы читались из чужих колонок).
# v4: порог ПОСТРОЧНЫЙ по подсказке предмета строки (ОТКАЧЕНО в v5).
# v5: порог снова ПО ПРОГРАММЕ (ТЗ): физика среди ВИ, пусть даже как альтернатива
#     информатике, -> 277; 269 только там, где ВИ — исключительно информатика.
#     Построчный порог тянул в таблицы лишний шум.
# v6 (29.07.2026): строки списка берём по НОМЕРУ строки и коду, а не по классу
#     <tr>. Раньше ловили только `<tr class="rnotes">`, а строки «реального
#     рейтинга» Политех красит зелёным и отдаёт как `<tr bgcolor="#c2ffe0">` —
#     они молча выпадали. В 10.05.03 так терялись 4 строки из 93, и все четыре
#     были с СОГЛАСИЕМ и стояли на вершине реального рейтинга (коды 1146548 —
#     307 баллов, реальный рейтинг 1; 957138; 1854305; 1415184). Плюс колонка
#     «Реальный рейтинг» теперь выводится, а квоты разыгрываются (см. ниже).
# v7 (04.08.2026): верхняя сводка мест читается ПО НАЗВАНИЯМ КОЛОНОК, а места
#     общего конкурса берутся из новой колонки сайта «Остаток бюджетных мест на
#     общий конкурс». После зачисления по квотам и БВИ (30.07–02.08) Политех
#     перекроил сводку: было 7 ячеек [код, название, КЦП, целевая, особая,
#     отдельная, проходной прошлого года], стало 9 — и числа квот теперь означают
#     не ПЛАН квоты, а ЗАЧИСЛЕННЫХ по ней. Из-за этого розыгрыш квот возвращал
#     места к полному КЦП: квотных строк в списках больше нет, значит «занято по
#     квоте» = 0, значит «свободных квотных мест» = все квотные места, и формула
#     КЦП − квоты + свободные_квотные давала ровно КЦП. На 10.05.03 выходило 91
#     место вместо 86, на 10.03.01 — 99 вместо 77 (завышение на 22 места, то есть
#     завышение наших шансов). Теперь при наличии колонки «Остаток» места берутся
#     из неё как есть, а розыгрыш квот их не трогает.
PARSE_VERSION_POLYTECH = 7

# Фиксированные фильтры конкурсного списка.
SELECT1_POLYTECH = '000000066_01'   # Москва (бакалавриат/специалитет)
EDU_FIN_POLYTECH = 'Бюджетная основа'
FORMS_POLYTECH = ['Очная', 'Очно-заочная', 'Заочная']

# Учитывать ВСЕ строки общего конкурса выше порога независимо от «Статуса в
# Госуслугах» (как в МТУСИ — там фильтра по статусу нет; порог по баллу сам отсекает
# строки без итогового конкурсного балла). Поставь True, если нужно оставить только
# «Участвует в конкурсе».
ONLY_IN_COMPETITION_POLYTECH = False

# --- Розыгрыш квот (29.07.2026, ТЗ «этап 2», п.5) ---
# Политех сам дублирует квотника в общий конкурс: у 10.05.03 из 169 квотников 139
# стоят в списке ВТОРОЙ строкой с «Льгота = 0». То есть добавлять их в таблицы не
# нужно — они там уже есть, как только починена выборка строк. Неверно другое:
# сейчас квотник конкурирует за место общего конкурса ДАЖЕ если проходит по своей
# квоте, хотя место квоты мы из КЦП уже вычли. Такой «фантом» стоит выше нас и
# занижает шансы.
# Поэтому квоты разыгрываем явно, по всем программам вуза сразу:
#   1. Каждая квота разыгрывается ОТДЕЛЬНО, по официальной легенде сайта
#      (вкладка «Перечень и коды льготных категорий», проверена 19.07.2026):
#        0 — без льготы (общий конкурс);
#        1 — целевая квота        -> мест = «Целевая квота»;
#        2 — особая квота         -> мест = «Особая квота»;
#        3 и 4 — отдельная квота (у Политеха она ведётся двумя списками)
#                                 -> мест = «Отдельная квота» на две группы вместе;
#        5, 6 — БВИ (олимпиадники), см. ниже.
#      Косвенно легенда подтверждается данными: целевые поля («Номер предложения»,
#      «Код конкурсной группы», «Имеется заявка») заполнены ровно у льготы 1, и
#      только на программах, где «Целевая квота» больше нуля.
#   2. Кто прошёл по квоте — место общего конкурса НЕ занимает (исключается из
#      отложенной приёмки, строка в таблице остаётся с пометкой в колонке «Квота»).
#   3. Незанятые квотные места переходят в общий конкурс (так требует Порядок
#      приёма) и добавляются к местам программы.
#   4. Не прошедшие по квоте остаются в общем конкурсе — они уже в списке.
# Внутри квоты ранг — по конкурсному баллу (при равенстве по номеру строки сайта).
QUOTA_ROUND_POLYTECH = True
# группа -> (коды «Льготы», индекс числа мест в кортеже quotas=(целевая, особая, отдельная))
QUOTA_GROUPS_POLYTECH = {
    'целевая': (('1',), 0),
    'особая': (('2',), 1),
    'отдельная': (('3', '4'), 2),
}
LGOTA_BVI_POLYTECH = ('5', '6')             # без вступительных испытаний

# --- Целевые программы (15 шт.): точечный код из ТЗ -> человекочитаемое имя. ---
# spec_code резолвится по точечному коду со страницы рейтинга (см. discover).
# Таблицы и итоговый DataFrame строим только по ним (очная форма).
TARGET_DOTTED_POLYTECH = [
    ('01.03.02',    'Прикладная математика и информатика (Программирование и интеллектуальные системы управления транспортом)'),
    ('10.05.03',    'Информационная безопасность автоматизированных систем'),
    ('10.03.01',    'Информационная безопасность'),
    ('09.03.03.02', 'Прикладная информатика (Большие и открытые данные)'),
    ('09.03.01.07', 'Информатика и вычислительная техника (Искусственный интеллект и машинное обучение)'),
    ('09.03.01.05', 'Информатика и вычислительная техника (Системная и программная инженерия)'),
    ('09.03.01.02', 'Информатика и вычислительная техника (Разработка инженерного программного обеспечения)'),
    ('09.03.01.06', 'Информатика и вычислительная техника (Программирование электронных устройств и систем; Информационные системы умных пространств)'),
    ('09.03.03.03', 'Прикладная информатика (Информационные технологии управления бизнесом)'),
    ('09.03.03.04', 'Прикладная информатика (Корпоративные решения на платформе 1С)'),
    ('09.03.03.01', 'Прикладная информатика (Разработка и интеграция бизнес-приложений)'),
    ('09.03.01.04', 'Информатика и вычислительная техника (Интеллектуальные беспилотные системы)'),
    ('09.03.02.01', 'Информационные системы и технологии (АСОИУ; ИТ в креативных индустриях; AR/VR; ПО игр)'),
    ('09.03.02.03', 'Информационные системы и технологии (Интеллектуальные информационно-измерительные системы)'),
    ('15.03.04.01', 'Автоматизация технологических процессов и производств (Роботы и робототехнические комплексы)'),
]

session_polytech = requests.Session()
session_polytech.headers.update(HTTP_HEADERS_POLYTECH)


def program_key_polytech(spec_code, form):
    """Уникальный ключ программы в all_polytech_data: spec_code + форма."""
    return f'{spec_code}|{form}'


def build_body_polytech(spec_code, form):
    """Тело POST-запроса. Пробелы/кириллица кодируются в UTF-8 (quote)."""
    return '&'.join([
        'select1=' + quote(SELECT1_POLYTECH),
        'specCode=' + quote(spec_code),
        'eduForm=' + quote(form),
        'eduFin=' + quote(EDU_FIN_POLYTECH),
        'f=1',
    ]).encode('ascii')


# 23.07.2026: сайты вузов в горячую фазу выпадают на минуту-две (502, таймаут
# соединения). Ждём до ~2.5 мин вместо прежних 12 с; connect отдельно от read.
RETRY_DELAYS_POLYTECH = (3, 6, 12, 24, 40, 60)
TIMEOUT_POLYTECH = (15, 120)  # (соединение, чтение)


def _retry_polytech(what, attempt, retries, error):
    """Пауза перед повтором; True — пробуем ещё, False — попытки кончились."""
    if attempt == retries - 1:
        return False
    pause = RETRY_DELAYS_POLYTECH[min(attempt, len(RETRY_DELAYS_POLYTECH) - 1)]
    print(f'  Политех не ответил по {what} ({type(error).__name__}: {error}). '
          f'Повтор {attempt + 2}/{retries} через {pause} с...')
    time.sleep(pause)
    return True


def fetch_list_polytech(spec_code, form, retries=len(RETRY_DELAYS_POLYTECH) + 1):
    """POST-запрос конкурсного списка. Ответ — UTF-8."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session_polytech.post(
                BASE_URL_POLYTECH, data=build_body_polytech(spec_code, form),
                timeout=TIMEOUT_POLYTECH, verify=False
            )
            response.raise_for_status()
            return response.content.decode('utf-8', errors='replace')
        except Exception as error:
            last_error = error
            if not _retry_polytech(f'{spec_code} [{form}]', attempt, retries, error):
                break
    raise RuntimeError(f'Не удалось загрузить {spec_code} [{form}]: {last_error}')


def fetch_rating_page_polytech(retries=len(RETRY_DELAYS_POLYTECH) + 1):
    """Страница рейтинга — из неё берём список всех программ площадки (spec_code)."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session_polytech.get(RATING_URL_POLYTECH,
                                            timeout=TIMEOUT_POLYTECH, verify=False)
            response.raise_for_status()
            return response.text
        except Exception as error:
            last_error = error
            if not _retry_polytech('страницу рейтинга', attempt, retries, error):
                break
    raise RuntimeError(f'Не удалось загрузить страницу рейтинга: {last_error}')


def discover_programs_polytech():
    """Все программы площадки Москва (бак/спец): список (spec_code, dotted, display).

    dotted — точечный код из видимого текста опции (09.03.01.02 и т.п.), по нему
    сопоставляем целевые программы из ТЗ.
    """
    html = fetch_rating_page_polytech()
    programs = []
    pattern = re.compile(
        r'<option[^>]*value="' + re.escape(SELECT1_POLYTECH) +
        r'"[^>]*spec_code="([^"]+)"[^>]*>(.*?)</option>', re.S
    )
    for match in pattern.finditer(html):
        spec_code = match.group(1)
        display_text = re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', '', match.group(2))).strip()
        dotted_match = re.match(r'(\d{2}\.\d{2}\.\d{2}(?:\.\d{2})?)', display_text)
        dotted = dotted_match.group(1) if dotted_match else ''
        if spec_code:
            programs.append((spec_code, dotted, display_text))
    return programs


def resolve_targets_polytech(programs):
    """Сопоставить точечные коды целевых из ТЗ со spec_code (по видимому тексту опции).

    Возвращает упорядоченный список (dotted, spec_code, name_из_ТЗ) в порядке ТЗ.
    """
    resolved = []
    for dotted, name in TARGET_DOTTED_POLYTECH:
        spec_code = None
        for sc, dot, _ in programs:
            if dot == dotted:
                spec_code = sc
                break
        if spec_code is None:
            # Запасной матч, если сайт сменил гранулярность точечного кода
            # (напр. в ТЗ 01.03.02, а на сайте 01.03.02.05 — или наоборот).
            for sc, dot, disp in programs:
                if (dot and (dot.startswith(dotted + '.') or dotted.startswith(dot + '.'))) \
                        or disp.startswith(dotted + ' '):
                    spec_code = sc
                    break
        if spec_code is None:
            print(f'    ВНИМАНИЕ: целевая {dotted} ({name[:40]}) не найдена на странице рейтинга')
            continue
        resolved.append((dotted, spec_code, name))
    return resolved


def get_update_date_polytech(html):
    """Дата актуальности: «Последнее обновление произведено: ДД.ММ.ГГГГ ЧЧ:ММ»."""
    match = re.search(
        r'Последнее обновление произведено:\s*(?:<[^>]*>\s*)*([\d.]+\s+[\d:]+)', html
    )
    return match.group(1).strip('. ') if match else None


def subject_label_polytech(titles):
    """Короткая подпись предметного столбца по множеству встреченных названий.

    Если среди названий есть физика (в т.ч. «Основы инженерной/прикладной физики»)
    — «Физ.» (физика приоритетнее информатики, как требует ТЗ). Иначе информатика,
    математика, русский и т.д.
    """
    joined = ' '.join(titles).lower()
    if 'физик' in joined:
        return 'Физ.'
    if 'информатик' in joined or 'информационные технологии' in joined:
        return 'Инф.'
    if 'математик' in joined:
        return 'Мат.'
    if 'русск' in joined:
        return 'Рус.'
    if 'иностран' in joined or 'англ' in joined:
        return 'Ин.яз.'
    if 'литератур' in joined:
        return 'Лит.'
    if 'географ' in joined:
        return 'Геогр.'
    if 'общество' in joined:
        return 'Обществ.'
    if 'истори' in joined:
        return 'Ист.'
    if 'хими' in joined:
        return 'Хим.'
    if 'биолог' in joined:
        return 'Биол.'
    for t in titles:
        if t:
            return t.split()[0][:8]
    return '—'


def _cell_text_polytech(td_html):
    """Текст ячейки без тегов, комментариев и &nbsp;."""
    txt = re.sub(r'<!--.*?-->', '', td_html, flags=re.S)
    txt = re.sub(r'<[^>]+>', '', txt)
    txt = txt.replace('&nbsp;', ' ')
    return re.sub(r'\s+', ' ', txt).strip()


def _to_int_polytech(value, default=0):
    m = re.search(r'-?\d+', value or '')
    return int(m.group(0)) if m else default


# Раскладка по умолчанию — обычная программа с 3 предметными баллами (23 колонки).
# real_rating — «Реальный рейтинг» Политеха: место среди тех, кто подал согласие и
# для кого эта программа — основной высший приоритет. Заполнен только у строк,
# подсвеченных зелёным (проверено 29.07.2026: 65 зелёных строк = 65 заполненных
# рейтингов, ни одной заполненной у обычных строк).
# site_main — «Основной высший приоритет»: НОМЕР приоритета, по которому сайт
# проводит абитуриента. Если он равен приоритету строки — сайт считает, что
# абитуриент проходит именно сюда. Держим для сверки с нашей отложенной приёмкой.
DEFAULT_COLUMNS_POLYTECH = {
    'code': 2, 'real_rating': 1, 'subjects': [4, 5, 6], 'sum_subjects': 7, 'total': 10,
    'consent': 12, 'priority': 13, 'site_main': 14, 'lgota': 16, 'status': 22,
}


def detect_columns_polytech(clean_html):
    """Индексы колонок из строки заголовков таблицы списка.

    У обычных программ 23 колонки (3 «Балл приоритет N»), у творческих — 24
    (4 экзамена), и всё после предметных сдвинуто на +1. Ищем первый <tr>,
    содержащий «Льгота», и сопоставляем заголовки по нормализованному тексту.
    Если строка заголовков не найдена — возвращаем раскладку по умолчанию.
    """
    for tr in re.findall(r'<tr[^>]*>.*?</tr>', clean_html, re.S):
        if 'Льгот' not in tr:
            continue
        headers = [_cell_text_polytech(td)
                   for td in re.findall(r'<t[dh][^>]*>.*?</t[dh]>', tr, re.S)]
        if len(headers) < 15:
            continue
        norm = [re.sub(r'\s+', '', h).lower() for h in headers]

        def exact(name, default):
            return norm.index(name) if name in norm else default

        subjects = [i for i, h in enumerate(norm) if h.startswith('баллприоритет')]
        return {
            'code': exact('уникальныйкод', DEFAULT_COLUMNS_POLYTECH['code']),
            'real_rating': exact('реальныйрейтинг',
                                 DEFAULT_COLUMNS_POLYTECH['real_rating']),
            'subjects': subjects or DEFAULT_COLUMNS_POLYTECH['subjects'],
            'sum_subjects': exact('суммабалловпопредметам',
                                  DEFAULT_COLUMNS_POLYTECH['sum_subjects']),
            'total': exact('конкурсныйбалл', DEFAULT_COLUMNS_POLYTECH['total']),
            'consent': exact('согласие', DEFAULT_COLUMNS_POLYTECH['consent']),
            'priority': exact('приоритет', DEFAULT_COLUMNS_POLYTECH['priority']),
            'site_main': exact('основнойвысшийприоритет',
                               DEFAULT_COLUMNS_POLYTECH['site_main']),
            'lgota': exact('льгота', DEFAULT_COLUMNS_POLYTECH['lgota']),
            'status': exact('статусвгосуслугах', DEFAULT_COLUMNS_POLYTECH['status']),
        }
    return dict(DEFAULT_COLUMNS_POLYTECH)


# Подписи колонок верхней сводки мест (нормализованные: без пробелов, в нижнем
# регистре, звёздочка у «Количество бюджетных мест*» отбрасывается сравнением по
# префиксу). Сопоставляем ПО НАЗВАНИЮ, а не по номеру ячейки — на позиционном
# разборе этот проект уже терял вуз целиком (финка 29.07.2026 вставила колонку «№»).
SUMMARY_LABELS_POLYTECH = [
    ('kcp',        'количествобюджетныхмест'),
    ('celevaya',   'зачисленныхпоцелевойквоте'),
    ('osobaya',    'зачисленныхпоособойквоте'),
    ('otdelnaya',  'зачисленныхпоотдельнойквоте'),
    ('bvi',        'зачисленныхбезвступительныхиспытаний'),
    ('remainder',  'остатокбюджетныхместнаобщийконкурс'),
]


def parse_summary_polytech(top_block):
    """Верхняя сводка мест программы.

    Возвращает dict: kcp, celevaya, osobaya, otdelnaya, bvi_enrolled,
    remainder (int или None), competition.

    Раскладку берём из строки заголовков сводки: её подписи описывают ячейки
    строки программы, начиная с третьей (первые две — код и название, последняя —
    проходной балл прошлого года). Смещение вычисляем, а не вшиваем, чтобы
    пережить добавление ещё одной колонки.

    Если строку заголовков распознать не удалось, откатываемся на прежний
    позиционный разбор [код, название, КЦП, целевая, особая, отдельная, ...] —
    он верен для формата до 03.08.2026. В этом случае remainder = None, и места
    считаются старой формулой КЦП − квоты − БВИ.
    """
    rows = []
    for tr in re.findall(r'<tr[^>]*>.*?</tr>', top_block, re.S):
        cells = [_cell_text_polytech(td)
                 for td in re.findall(r'<t[dh]\b[^>]*>.*?</t[dh]>', tr, re.S)]
        if cells:
            rows.append(cells)

    # Строка заголовков сводки — та, где есть «Количество бюджетных мест».
    labels = None
    for cells in rows:
        norm = [re.sub(r'\s+', '', c).lower() for c in cells]
        if any(n.startswith(SUMMARY_LABELS_POLYTECH[0][1]) for n in norm):
            labels = norm
            break

    # Строка программы — первая, начинающаяся с точечного кода направления.
    prog_cells = None
    for cells in rows:
        if re.match(r'\d{2}\.\d{2}\.\d{2}', cells[0]) and len(cells) >= 6:
            prog_cells = cells
            break
    if prog_cells is None:
        return None

    result = {'kcp': 0, 'celevaya': 0, 'osobaya': 0, 'otdelnaya': 0,
              'bvi_enrolled': 0, 'remainder': None,
              'competition': f'{prog_cells[0]} {prog_cells[1]}'.strip()}

    if labels:
        # Ячейки [0] код и [1] название описаны отдельной шапкой; последняя ячейка
        # строки — «Проходной балл в прошлом году». Отсюда сдвиг подписей.
        offset = len(prog_cells) - len(labels) - 1
        if offset >= 0:
            hit = {}
            for i, name in enumerate(labels):
                for key, prefix in SUMMARY_LABELS_POLYTECH:
                    if name.startswith(prefix) and i + offset < len(prog_cells):
                        hit[key] = prog_cells[i + offset]
            if 'kcp' in hit:
                result['kcp'] = _to_int_polytech(hit['kcp'])
                result['celevaya'] = _to_int_polytech(hit.get('celevaya'))
                result['osobaya'] = _to_int_polytech(hit.get('osobaya'))
                result['otdelnaya'] = _to_int_polytech(hit.get('otdelnaya'))
                result['bvi_enrolled'] = _to_int_polytech(hit.get('bvi'))
                if 'remainder' in hit:
                    result['remainder'] = _to_int_polytech(hit['remainder'], -1)
                    if result['remainder'] < 0:
                        result['remainder'] = None
                return result

    # Откат: прежняя позиционная раскладка.
    result['kcp'] = _to_int_polytech(prog_cells[2])
    result['celevaya'] = _to_int_polytech(prog_cells[3])
    result['osobaya'] = _to_int_polytech(prog_cells[4])
    result['otdelnaya'] = _to_int_polytech(prog_cells[5])
    return result


def iter_rows_polytech(clean, cols, min_cells):
    """Строки списка абитуриентов: (ячейки, исходные <td>).

    Отбираем НЕ по классу <tr>, а по содержимому: первая ячейка — номер строки,
    ячейка кода — 4-10 цифр. Причина (29.07.2026): Политех отдаёт строки двумя
    разными тегами — обычные как `<tr class="rnotes" id='pkm_'>`, а строки
    «реального рейтинга» как `<tr bgcolor="#c2ffe0" id='pkm_'>`. Прежняя выборка
    по `<tr\\s+class="rnotes"` вторые молча теряла, а это как раз абитуриенты с
    согласием — то есть самые важные для анализа. Отбор по содержимому переживёт
    и следующую смену оформления, а строки верхней сводки не захватит: там в
    первой ячейке код направления «10.05.03», а не номер.
    """
    code_idx = cols['code']
    for tr in re.findall(r'<tr[^>]*>.*?</tr>', clean, re.S):
        tds = re.findall(r'<td\b[^>]*>.*?</td>', tr, re.S)
        if len(tds) < min_cells:
            continue
        cells = [_cell_text_polytech(td) for td in tds]
        if not re.fullmatch(r'\d+', cells[0]):
            continue
        if not re.fullmatch(r'\d{4,10}', cells[code_idx]):
            continue
        yield cells, tds


def parse_program_polytech(spec_code, form, is_target, pool=None):
    """Парсит конкурсный список программы (одна форма).

    Места общего конкурса = КЦП − (целевая + особая + отдельная) − БВИ, где БВИ —
    уникальные коды со значением 5 или 6 в колонке «Льгота» (олимпиадники, зачисление
    без ВИ). Возвращает dict с df/бюджетом(общий конкурс)/названием или None, если у
    программы нет мест общего конкурса или нет участников общего конкурса выше порога.
    Строки общего конкурса ниже порога не сохраняются.

    В pool (если передан) кладём данные для розыгрыша квот по ВСЕМ программам —
    даже по тем, что вернут None: квотник может осесть на программе, которая в наши
    таблицы не попадает, и тогда на нашей он место общего конкурса не занимает.
    Квотные строки берём БЕЗ порога: квотник с любым баллом занимает место квоты.
    """
    html = fetch_list_polytech(spec_code, form)
    clean = re.sub(r'<!--.*?-->', '', html, flags=re.S)
    update_date = get_update_date_polytech(html)

    rows_start = clean.find("id='pkm_'")
    if rows_start < 0:
        rows_start = clean.find('class="rnotes"')
    top_block = clean[:rows_start] if rows_start > 0 else clean

    # --- Верхняя сводка: КЦП, квоты, БВИ и «Остаток мест на общий конкурс» ---
    summary = parse_summary_polytech(top_block)
    if summary is None:
        summary = {'kcp': 0, 'celevaya': 0, 'osobaya': 0, 'otdelnaya': 0,
                   'bvi_enrolled': 0, 'remainder': None, 'competition': spec_code}
    kcp = summary['kcp']
    celevaya = summary['celevaya']
    osobaya = summary['osobaya']
    otdelnaya = summary['otdelnaya']
    remainder = summary['remainder']
    competition = summary['competition'] or spec_code

    cols = detect_columns_polytech(clean)
    min_cells = max(cols['status'], cols['lgota'], cols['total'], cols['code']) + 1

    # --- Строки абитуриентов: общий конкурс (Льгота 0/пусто), квоты (1-4), БВИ (5/6) ---
    raw_rows = []
    bvi_rows = []
    quota_rows = []
    for cells, tds in iter_rows_polytech(clean, cols, min_cells):
        code = cells[cols['code']]
        lgota = cells[cols['lgota']]
        row_no = _to_int_polytech(cells[0])
        total = _to_int_polytech(cells[cols['total']])
        priority = _to_int_polytech(cells[cols['priority']], 99)

        # Льгота 5/6 — олимпиадники БВИ: зачисляются без ВИ и занимают бюджетное
        # место. Место снимаем не здесь: БВИ подаёт на несколько программ, а осядет
        # ровно на одной (см. розыгрыш ниже) — раньше его вычитали на каждой.
        if lgota in LGOTA_BVI_POLYTECH:
            bvi_rows.append({'Код': code, 'Приоритет': priority, '№_orig': row_no})
            continue
        # Квоты: в общий конкурс эти строки не идут, но нужны для розыгрыша квот.
        if lgota not in ('', '0'):
            quota_rows.append({'Код': code, 'Приоритет': priority, 'Сумма': total,
                               '№_orig': row_no, 'Льгота': lgota})
            continue

        status = cells[cols['status']]
        if ONLY_IN_COMPETITION_POLYTECH and not status.lower().startswith('участв'):
            continue

        titles = []
        for idx in cols['subjects']:
            tm = re.search(r'title="([^"]*)"', tds[idx])
            titles.append(tm.group(1) if tm else '')

        raw_rows.append({
            '№_orig': row_no,
            'Код': code,
            'Сумма': total,                                  # Конкурсный балл (с ИД)
            'Сумма без ИД': _to_int_polytech(cells[cols['sum_subjects']]),
            'Приоритет': priority,
            'Согласие': 'да' if cells[cols['consent']].strip().lower() == 'да' else '',
            'Реальный рейтинг': cells[cols['real_rating']],
            'ОВП сайта': ('да' if (cells[cols['site_main']]
                                   and cells[cols['site_main']]
                                   == cells[cols['priority']]) else 'нет'),
            'Статус': status,
            '_scores': [cells[idx] for idx in cols['subjects']],
            '_titles': titles,
        })

    # Пул для розыгрыша квот: наполняем ВСЕГДА, даже если программа дальше отсеется.
    if pool is not None:
        pool[program_key_polytech(spec_code, form)] = {
            'quota_rows': quota_rows,
            'bvi_rows': bvi_rows,
            'kcp': kcp,
            'quotas': (celevaya, osobaya, otdelnaya),
            'remainder': remainder,
            'bvi_enrolled': summary['bvi_enrolled'],
        }

    bvi_codes = {row['Код'] for row in bvi_rows}
    if remainder is not None:
        # Сайт сам публикует остаток: КЦП минус ЗАЧИСЛЕННЫЕ по квотам и БВИ.
        # Незанятые квотные места в него уже включены, повторно ничего не вычитаем
        # и не прибавляем — иначе получаем обратно полный КЦП (см. v7 в шапке).
        budget_general = remainder
    else:
        if kcp - celevaya - osobaya - otdelnaya <= 0:
            return None
        budget_general = kcp - celevaya - osobaya - otdelnaya - len(bvi_codes)
    if budget_general <= 0 or not raw_rows:
        return None

    # Подписи предметных столбцов + наличие физики (агрегируем по всем строкам).
    n_subjects = len(cols['subjects'])
    slot_titles = [set() for _ in range(n_subjects)]
    for row in raw_rows:
        for j, title in enumerate(row['_titles']):
            if title:
                slot_titles[j].add(title)
    slot_labels = [subject_label_polytech(s) for s in slot_titles]
    all_titles = ' '.join(t for slot in slot_titles for t in slot).lower()
    # Порог ПО ПРОГРАММЕ (ТЗ): если среди ВИ есть физика — пусть даже как
    # альтернатива информатике в одном столбце — порог 277. Порог 269 только у
    # программ, где ВИ — исключительно информатика. Подпись в «Оценках» здесь
    # заодно общая по столбцу («Физ.» всем, кто сдавал альтернативу).
    threshold = 277 if 'физик' in all_titles else 269

    rows = []
    for row in raw_rows:
        if row['Сумма'] < threshold:
            continue
        marks = ', '.join(
            f'{slot_labels[j]}: {row["_scores"][j]}'
            for j in range(n_subjects) if row['_scores'][j]
        )
        rows.append({
            '№_orig': row['№_orig'],
            'Код': row['Код'],
            'Сумма': row['Сумма'],
            'Сумма без ИД': row['Сумма без ИД'],
            'Оценки': marks if marks else '—',
            'Согласие': row['Согласие'],
            'Приоритет': row['Приоритет'],
            'Реальный рейтинг': row['Реальный рейтинг'],
            'ОВП сайта': row['ОВП сайта'],
            'Статус': row['Статус'],
        })

    if not rows:
        return None

    df = pd.DataFrame(rows)
    # Один абитуриент — одна строка общего конкурса на программу; дедуп на всякий
    # случай (оставляем строку с наибольшей суммой). При равном конкурсном балле
    # тай-брейк по №_orig (порядок сайта) — детерминизм и совпадение с рейтингом.
    df = (df.sort_values(['Сумма', '№_orig'], ascending=[False, True], kind='stable')
            .drop_duplicates('Код', keep='first')
            .reset_index(drop=True))
    df.insert(0, '№', range(1, len(df) + 1))

    return {
        'df': df,
        'budget': budget_general,
        'kcp': kcp,
        'quotas': (celevaya, osobaya, otdelnaya),
        'bvi': len(bvi_codes),
        'bvi_enrolled': summary['bvi_enrolled'],
        'remainder': remainder,
        'threshold': threshold,
        'form': form,
        'university': 'Московский Политех',
        'competition': competition,
        'is_target': is_target,
        'update_date': update_date,
    }


# ===== Сбор данных по всем программам площадки (с кэшем по дате + составу) =====
print('Загрузка списка программ Московского Политеха...')
all_programs_polytech = discover_programs_polytech()
targets_resolved_polytech = resolve_targets_polytech(all_programs_polytech)
target_speccodes_polytech = {sc for _, sc, _ in targets_resolved_polytech}
# Порядок целевых для вывода (как в ТЗ), ключи all_polytech_data — очная форма.
# dict.fromkeys — на случай, если два точечных кода резолвятся в один spec_code.
target_keys_polytech = list(dict.fromkeys(
    program_key_polytech(sc, 'Очная') for _, sc, _ in targets_resolved_polytech))
target_names_polytech = {program_key_polytech(sc, 'Очная'): name
                         for _, sc, name in targets_resolved_polytech}
print(f'Программ площадки Москва (бак/спец): {len(all_programs_polytech)}; '
      f'целевых сопоставлено: {len(targets_resolved_polytech)}/{len(TARGET_DOTTED_POLYTECH)}.')

# Дата актуальности — с первой целевой программы (очная).
current_date_polytech = None
if targets_resolved_polytech:
    first_sc = targets_resolved_polytech[0][1]
    current_date_polytech = get_update_date_polytech(
        fetch_list_polytech(first_sc, 'Очная'))
print(f'Дата актуальности на сайте: {current_date_polytech}')

cached_polytech = None
if os.path.exists(CACHE_POLYTECH):
    with open(CACHE_POLYTECH, 'rb') as f:
        cached_polytech = pickle.load(f)

programs_signature_polytech = sorted(sc for sc, _, _ in all_programs_polytech)
cache_valid_polytech = (
    cached_polytech is not None
    and current_date_polytech is not None
    and cached_polytech.get('version') == PARSE_VERSION_POLYTECH
    and cached_polytech.get('date') == current_date_polytech
    and cached_polytech.get('programs') == programs_signature_polytech
    and cached_polytech.get('data')
)

all_polytech_data = {}
quota_pool_polytech = {}
if cache_valid_polytech and cached_polytech.get('quota_pool') is not None:
    all_polytech_data = cached_polytech['data']
    quota_pool_polytech = cached_polytech['quota_pool']
    print('Обновления не было и состав программ не менялся — используем кэш.')
else:
    if cache_valid_polytech:
        print('Кэш без данных по квотам (старая версия) — перезагружаем.')
    total_combos = len(all_programs_polytech) * len(FORMS_POLYTECH)
    done = 0
    failed_polytech = []
    for spec_code, dotted, display_text in all_programs_polytech:
        for form in FORMS_POLYTECH:
            done += 1
            is_target = (form == 'Очная' and spec_code in target_speccodes_polytech)
            try:
                data = parse_program_polytech(spec_code, form, is_target,
                                              pool=quota_pool_polytech)
            except Exception as error:
                print(f'  [{done}/{total_combos}] Ошибка {dotted} [{form}]: {error}')
                failed_polytech.append(f'{dotted} [{form}]')
                continue
            if data is None:
                continue
            key = program_key_polytech(spec_code, form)
            all_polytech_data[key] = data
            tag = 'ЦЕЛЕВАЯ' if is_target else 'общая'
            print(f'  [{done}/{total_combos}] {tag}: {dotted} [{form}] '
                  f'мест общ.конкурса={data["budget"]} (БВИ 5/6: {data["bvi"]}), '
                  f'порог={data["threshold"]}, строк={len(data["df"])}')

    # Кэш пишем ТОЛЬКО при полной загрузке (см. пояснение в блоке МТУСИ):
    # сигнатура хранит весь каталог, частичный набор подтянулся бы как валидный.
    if all_polytech_data and not failed_polytech:
        with open(CACHE_POLYTECH, 'wb') as f:
            pickle.dump({
                'version': PARSE_VERSION_POLYTECH,
                'date': current_date_polytech,
                'programs': programs_signature_polytech,
                'data': all_polytech_data,
                'quota_pool': quota_pool_polytech,
            }, f)
        print('Данные сохранены в кэш.')
    elif failed_polytech:
        print(f'  ВНИМАНИЕ: не загрузились программы ({len(failed_polytech)}): '
              f'{"; ".join(failed_polytech[:3])} — кэш НЕ сохранён, '
              f'работаем на неполных данных.')

if not all_polytech_data:
    raise SystemExit('Нет данных ни по одной программе Московского Политеха.')

loaded_targets_polytech = [k for k in target_keys_polytech if k in all_polytech_data]
print(f'Загружено программ (все формы): {len(all_polytech_data)}; '
      f'целевых с данными: {len(loaded_targets_polytech)}/{len(target_keys_polytech)}.')


# ===== Розыгрыш БВИ и квот (см. QUOTA_ROUND_POLYTECH) =====
def _run_quota_round_polytech(pool):
    """Кто оседает по БВИ и по квотам -> (bvi_taken, quota_taken, free_quota).

    bvi_taken  — {ключ программы: сколько БВИ реально заняли здесь место};
    quota_taken— {Код: (ключ программы, 'целевая'|'иная квота')};
    free_quota — {ключ программы: сколько квотных мест осталось незанятыми}.

    БВИ зачисляют без конкурса, поэтому каждый оседает на программе своего
    ВЫСШЕГО приоритета — и место снимается только там. Прежний код вычитал БВИ на
    каждой программе, где тот подал заявление, и занижал места сразу на нескольких.

    Квоты — та же отложенная приёмка, что и в общем конкурсе, но по местам квот и
    только среди квотников: абитуриент предлагает себя программам по возрастанию
    приоритета, программа держит сильнейших по конкурсному баллу.
    """
    # --- БВИ: высший приоритет забирает место ---
    bvi_best = {}
    for key, entry in pool.items():
        for row in entry['bvi_rows']:
            best = bvi_best.get(row['Код'])
            if best is None or row['Приоритет'] < best[0]:
                bvi_best[row['Код']] = (row['Приоритет'], key)
    bvi_taken = collections.Counter(key for _, key in bvi_best.values())

    # --- Квоты: места и ранги по группам ---
    seats = {}          # (ключ, группа) -> мест
    rank = {}           # (ключ, группа) -> {Код: ранг}
    prefs = {}          # Код -> [(приоритет, (ключ, группа))]
    for key, entry in pool.items():
        for group, (lgotas, places_index) in QUOTA_GROUPS_POLYTECH.items():
            places = entry['quotas'][places_index]
            rows = [r for r in entry['quota_rows'] if r['Льгота'] in lgotas]
            if places <= 0 or not rows:
                continue
            slot = (key, group)
            seats[slot] = places
            ordered = sorted(rows, key=lambda r: (-r['Сумма'], r['№_orig']))
            # Один код — одна строка в группе (на всякий случай, дедуп по лучшему).
            rank[slot] = {}
            for row in ordered:
                rank[slot].setdefault(row['Код'], len(rank[slot]))
                prefs.setdefault(row['Код'], []).append((row['Приоритет'], slot))

    for code, pairs in prefs.items():
        # Один и тот же слот может прийти дважды (дубли строк) — оставляем лучший.
        best = {}
        for priority, slot in pairs:
            if slot not in best or priority < best[slot]:
                best[slot] = priority
        prefs[code] = [slot for slot, _ in sorted(best.items(), key=lambda x: x[1])]

    held = {slot: {} for slot in seats}
    next_choice = {code: 0 for code in prefs}
    pending = collections.deque(prefs)
    while pending:
        code = pending.popleft()
        options = prefs[code]
        while next_choice[code] < len(options):
            slot = options[next_choice[code]]
            place = held[slot]
            position = rank[slot][code]
            if len(place) < seats[slot]:
                place[code] = position
                break
            worst = max(place, key=place.get)
            if position < place[worst]:
                del place[worst]
                place[code] = position
                next_choice[worst] += 1
                pending.append(worst)
                break
            next_choice[code] += 1

    quota_taken = {}
    free_quota = collections.Counter()
    for key, entry in pool.items():
        # Свободные места считаем ПО КАЖДОЙ квоте отдельно и складываем: недобор в
        # одной квоте не закрывается переполнением другой, в общий конкурс уходит
        # именно неразобранный остаток.
        free = 0
        for group, (_, places_index) in QUOTA_GROUPS_POLYTECH.items():
            slot = (key, group)
            taken = held.get(slot, {})
            for code in taken:
                quota_taken[code] = (key, group)
            free += max(0, entry['quotas'][places_index] - len(taken))
        free_quota[key] = free
    return bvi_taken, quota_taken, free_quota


quota_taken_polytech = {}
free_quota_polytech = collections.Counter()
budget_before_polytech = {k: d['budget'] for k, d in all_polytech_data.items()}
if QUOTA_ROUND_POLYTECH and quota_pool_polytech:
    bvi_taken_polytech, quota_taken_polytech, free_quota_polytech = \
        _run_quota_round_polytech(quota_pool_polytech)
    for key, data in all_polytech_data.items():
        entry = quota_pool_polytech.get(key)
        if entry is None:
            continue
        if entry.get('remainder') is not None:
            # Формат сайта с 03.08.2026: числа квот — это ЗАЧИСЛЕННЫЕ по квоте, а
            # «Остаток бюджетных мест на общий конкурс» уже учитывает и их, и БВИ,
            # и незанятые квотные места. Пересчитывать нельзя: квотных строк в
            # списках больше нет, поэтому «занято по квоте» вышло бы 0,
            # «свободных квотных мест» — все квотные, и КЦП − квоты + свободные
            # вернуло бы полный КЦП (91 вместо 86 на 10.05.03).
            data['bvi'] = int(entry.get('bvi_enrolled') or 0)
            data['free_quota'] = 0
            continue
        celevaya, osobaya, otdelnaya = entry['quotas']
        # Места общего конкурса = КЦП − квоты − БВИ, осевшие ИМЕННО здесь,
        # + квотные места, которые никто не занял (по Порядку приёма они
        # переходят в общий конкурс).
        data['bvi'] = int(bvi_taken_polytech.get(key, 0))
        data['free_quota'] = int(free_quota_polytech.get(key, 0))
        data['budget'] = max(0, entry['kcp'] - celevaya - osobaya - otdelnaya
                             - data['bvi'] + data['free_quota'])
    # Кто прошёл по квоте — место общего конкурса не занимает. Строку в таблице
    # оставляем (она в списке есть, и её надо видеть), но помечаем и выводим из
    # отложенной приёмки. Так же поступаем с БВИ, осевшими на этой программе.
    for key, data in all_polytech_data.items():
        df = data['df']
        marks = []
        locked = set()
        for code in df['Код']:
            slot = quota_taken_polytech.get(code)
            if slot is not None:
                locked.add(code)
                marks.append('прошёл по квоте'
                             if slot[0] == key else 'квота на др. программе')
            else:
                marks.append('')
        df['Квота'] = marks
        data['no_general_codes'] = sorted(locked)
    changed = sum(1 for k in all_polytech_data
                  if all_polytech_data[k]['budget'] != budget_before_polytech[k])
    print(f'Розыгрыш квот: программ с изменившимися местами {changed}; '
          f'кодов, проходящих по квоте (не занимают общий конкурс): '
          f'{len(quota_taken_polytech)}.')
else:
    for data in all_polytech_data.values():
        data['df']['Квота'] = ''
        data['free_quota'] = 0
        data['no_general_codes'] = []

# ===== Расчёт Основного ВП (отложенная приёмка по ВСЕМ программам) =====
# Модель Минобра «конкурс с приоритетами»: абитуриент зачисляется на программу с
# наивысшим приоритетом, где реально проходит по баллам после ухода конкурентов на
# их приоритетные программы (эффект вытеснения). Кандидат-предлагающая отложенная
# приёмка (deferred acceptance), как в мтуси_блок3.py. У Политеха в общем конкурсе
# БВИ отдельным проходом не выделяются — ранг чисто по конкурсному баллу.
prog_rank_polytech = {}      # ключ -> {Код: позиция в списке, меньше = сильнее}
prog_budget_polytech = {}    # ключ -> число мест общего конкурса
cand_prefs_polytech = {}     # Код -> ключи программ по возрастанию приоритета
# Коды, прошедшие по квоте или БВИ: место общего конкурса они не занимают, иначе
# считались бы дважды (место квоты из КЦП уже вычтено) и вытесняли бы нас вниз.
locked_codes_polytech = set()
for data in all_polytech_data.values():
    locked_codes_polytech.update(data.get('no_general_codes', ()))
for key, data in all_polytech_data.items():
    df = data['df']
    prog_budget_polytech[key] = data['budget']
    prog_rank_polytech[key] = {code: i for i, code in enumerate(df['Код'])}
    for code, priority in zip(df['Код'], df['Приоритет']):
        if code in locked_codes_polytech:
            continue
        cand_prefs_polytech.setdefault(code, []).append((int(priority), key))
for code, pairs in cand_prefs_polytech.items():
    cand_prefs_polytech[code] = [key for _, key in sorted(pairs, key=lambda x: x[0])]

admitted_polytech = {key: {} for key in all_polytech_data}   # ключ -> {Код: ранг}
next_choice_polytech = {code: 0 for code in cand_prefs_polytech}
pending_polytech = collections.deque(cand_prefs_polytech)

while pending_polytech:
    code = pending_polytech.popleft()
    prefs = cand_prefs_polytech[code]
    while next_choice_polytech[code] < len(prefs):
        key = prefs[next_choice_polytech[code]]
        budget = prog_budget_polytech[key]
        if budget <= 0:
            next_choice_polytech[code] += 1
            continue
        rank = prog_rank_polytech[key][code]
        held = admitted_polytech[key]
        if len(held) < budget:
            held[code] = rank
            break
        worst_code = max(held, key=held.get)   # держим слабейшего (наибольший ранг)
        if rank < held[worst_code]:
            del held[worst_code]
            held[code] = rank
            next_choice_polytech[worst_code] += 1
            pending_polytech.append(worst_code)   # вытесненный ищет следующий приоритет
            break
        next_choice_polytech[code] += 1           # здесь не прошёл — следующий приоритет

enrolled_polytech = {}
for key, data in all_polytech_data.items():
    admitted_codes = set(admitted_polytech[key])
    data['df']['Основной ВП'] = (
        data['df']['Код'].isin(admitted_codes).map({True: 'да', False: 'нет'})
    )
    enrolled_polytech.update((code, True) for code in admitted_codes)

# ===== Свободные места после Основного ВП =====
# «Проходной ВП» своим расчётом получить НЕЛЬЗЯ: после отложенной приёмки у
# программы со свободными местами не остаётся незачисленных кандидатов — тот,
# кого не взяли нигде, дошёл бы до этого места и занял его. Прежний расчёт
# давал «нет» во всех строках и создавал ложное впечатление второй проверки.
# Полезен сам остаток: места не разобраны теми, кто выше порога, значит
# проходной балл программы НИЖЕ нашего порога отбора.
free_places_polytech = {}
for key, data in all_polytech_data.items():
    df = data['df']
    df['Проходной ВП'] = 'нет'
    free_places_polytech[key] = max(
        0, data['budget'] - int((df['Основной ВП'] == 'да').sum()))

# ===== Отдельные таблицы по программам (только целевые, порядок ТЗ) =====
display(Markdown('# БЛОК 1: МОСКОВСКИЙ ПОЛИТЕХ'))
for key in target_keys_polytech:
    data = all_polytech_data.get(key)
    if data is None:
        display(Markdown(f'### {target_names_polytech.get(key, key)} — нет данных'))
        print('-' * 80)
        continue
    df = data['df']
    count_main = (df['Основной ВП'] == 'да').sum()
    count_pass = (df['Проходной ВП'] == 'да').sum()

    count_quota = int((df['Квота'] != '').sum()) if 'Квота' in df.columns else 0
    count_site = int((df['ОВП сайта'] == 'да').sum()) if 'ОВП сайта' in df.columns else 0

    display(Markdown(f"### {data['competition']} ({data['form']})"))
    if data.get('remainder') is not None:
        celevaya, osobaya, otdelnaya = data['quotas']
        places_note = (
            f"**Мест общего конкурса:** {data['budget']} "
            f"(колонка сайта «Остаток бюджетных мест на общий конкурс»)  |  "
            f"КЦП {data['kcp']} − зачислено по квотам "
            f"{celevaya}+{osobaya}+{otdelnaya} − БВИ {data.get('bvi_enrolled', 0)}"
        )
    else:
        places_note = (
            f"**Мест общего конкурса:** {data['budget']} (КЦП − квоты − БВИ)  |  "
            f"**БВИ (Льгота 5/6):** {data.get('bvi', 0)}  |  "
            f"**Свободных мест из квот:** {data.get('free_quota', 0)}"
        )
    display(Markdown(
        f"{places_note}  |  **Порог отбора:** {data['threshold']}  |  "
        f"**Всего строк:** {len(df)}  |  **Обновлено:** {data.get('update_date')}"
    ))
    display(Markdown(
        f'**Основной ВП (да):** {count_main}  |  '
        f'**Мест не занято абитуриентами выше порога:** '
        f'{free_places_polytech.get(key, 0)} (если больше нуля — проходной балл ниже нашего порога отбора)'
    ))
    display(Markdown(
        f'**Строк выше порога, проходящих по квоте (место общего конкурса не '
        f'занимают):** {count_quota}  |  **ОВП по расчёту сайта:** {count_site}'
    ))
    display_cols = ['№', '№_orig', 'Реальный рейтинг', 'Код', 'Сумма', 'Сумма без ИД',
                    'Оценки', 'Согласие', 'Приоритет', 'Квота', 'Основной ВП',
                    'ОВП сайта', 'Проходной ВП', 'Статус']
    display(df[[c for c in display_cols if c in df.columns]].style.hide(axis='index'))
    print('-' * 80)

print(f'\nВремя работы БЛОКА 1 (Московский Политех): {time.time() - start_time_polytech:.1f} сек.')
print('Данные для сводной готовы: all_polytech_data, target_keys_polytech — '
      'см. «БЛОК 2: СВОДНАЯ ТАБЛИЦА МОСКОВСКИЙ ПОЛИТЕХ».')

In [ ]:
# -*- coding: utf-8 -*-
# ===== БЛОК 1: МИИТ =====
# Образцы — МТУСИ (мтуси_блок3.py) и Московский Политех (политех_блок4.py).
# Тот же принцип: «Основной ВП» считаем по ВСЕМ программам вуза, удовлетворяющим
# условиям, а таблицы строим только по целевым.
#
# Условия для программ, участвующих в расчёте ВП:
#   - уровень «Базовое высшее образование» (бакалавриат + специалитет);
#   - г. Москва (у РУТ (МИИТ) в API один город);
#   - формы очная / очно-заочная / заочная;
#   - мест общего конкурса > 0.
#
# Selenium/Playwright не нужны: сайт miit.ru — React SPA, данные отдаёт открытый
# JSON API 1С (найден в бандле /static/js/main.*.js):
#   - Каталог: GET /api-1c/miit/hs/AdmissionPlan/reception-plan?year=2026 —
#     все конкурсные группы (planReceptionUrl -> id, klfName, ktName, specQualifier,
#     specName, профили, planNumbers). Поле disciplines в каталоге ПУСТОЕ.
#   - Карточка: GET .../concourse-group-page-dto?id=<id>&year=2026 —
#     «Вступительные испытания» (disciplines, напр. «Математика, Русский язык,
#     (Информатика или Физика)») и planNumbers.integers.CONCOURSE.PLAN —
#     «Общий конкурс N» — места общего конкурса, берём КАК ЕСТЬ, не пересчитываем.
#   - Список: GET .../rating?id=<id>&year=2026 — competitions[] по видам конкурса
#     («Общий конкурс», «Особая/Отдельная/Целевая квота», «Места по договорам»).
#     Берём ТОЛЬКО «Общий конкурс»; внутри подгруппы «Участвуют в конкурсе...»,
#     «Нет баллов...», «Баллы ниже минимума...» (дата актуальности — в названии
#     подгруппы «...по состоянию на ДД.ММ.ГГГГ»). Порог сам отсекает лишние.
#
# У каждого competition ДВЕ параллельные ветки строк с одним и тем же составом:
#   - concourseEntrantGroups (CEG) — сплошной список по убыванию баллов. Считаем
#     по нему: ranking честный, ratingIndex = номер по баллам.
#   - projectEntrantGroups (PEG) — тот же состав, но разложенный по ВЕРДИКТУ вуза:
#     code=1 «Зачислены», code=2 «Проходят», code=0 «Не проходят на бюджет на все
#     выбранные специальности» и code=0 «Нет согласия на зачисление». Кто в CEG
#     есть, а в PEG нет — тот проходит на ДРУГУЮ программу МИИТ (у всех таких
#     согласие представлено, проверено 33/33 группами).
#     «Зачислены»+«Проходят» = РОВНО CONCOURSE.PLAN человек (33 группы из 33) и
#     все со сданным согласием — это готовый результат отложенной приёмки самого
#     вуза по ПОЛНОМУ списку, без нашей грабли «список обрезан порогом 277/269».
#     Забираем в колонку «Вердикт вуза» и печатаем минимальный балл проходящих:
#     это реальный проходной, а не «мест не занято N из M» по обрезку.
#
# Поля абитуриента в rating: displayName=ID (наш «Код»), points=сумма С ИД,
# pointsDisc=сумма без ИД, pointsInd=ИД, pointList='99; 94; 98' (баллы по предметам
# в порядке disciplines программы), preferenceIndexDisplay='1 (Основной ВП)' ->
# приоритет = число в начале, originalDocumentStatus='Представлено' -> согласие,
# ratingIndex=позиция в СВОЕЙ ветке. Сайтовые «Основной ВП» из
# preferenceIndexDisplay НЕ используем — считаем сами. Профили абитуриента в
# таблицы не пишем (по ТЗ).
#
# ВНИМАНИЕ на нумерацию. По умолчанию сайт рисует PEG (в бандле useState("Конкурс")
# и children:r.ratingIndex), а ratingIndex в PEG сквозной ПО ПОРЯДКУ ГРУПП вердикта,
# а не по баллам. Поэтому число на экране и наш №_orig — разные шкалы: код 2033142
# на 10.05.01 это №_orig=33 (CEG, по баллам) и «№ сайта»=243 (PEG). Храним оба,
# иначе сверка глазами расходится и кажется, что парсер врёт.
#
# БВИ в «Общем конкурсе» МИИТ единичны (по одному в 5 группах из 33), обрабатываем:
# withoutExamsText/foundationBVI непустые -> строка «БВИ», без порога, наверху
# списка (как у МТУСИ). У них pointsDisc=null (вступительных не сдавали), поэтому
# «Сумму без ИД» добираем как points - pointsInd, иначе в таблице стоял голый 0.
#
# Порог отсечения (как у МТУСИ/Политеха): 277, если среди предметов программы есть
# физика (в т.ч. как альтернатива информатике), иначе 269. Строки ниже порога не
# сохраняем — на результат анализа они не влияют.
#
# Кэш: даты с точностью до минут у МИИТ нет (только «по состоянию на ДД.ММ.ГГГГ»),
# поэтому валидность кэша проверяем по сигнатуре каталога: (id, места общего
# конкурса, число заявлений) по всем группам — при обновлении списков число
# заявлений меняется и кэш пересобирается.
#
# 30.07–02.08.2026 прошло зачисление по квотам и БВИ, и МИИТ ПЕРЕСЧИТАЛ CONCOURSE.PLAN:
# теперь это остаток бюджета на общий конкурс. Проверено арифметикой по всем 33
# группам: BUDGET.PLAN == CONCOURSE.PLAN + SPECIAL_QUOTA + QUOTA_07 + TARGET_QUOTA +
# CRIMEA + QUOTA_05, расхождений 0. Значит формула «budget = CONCOURSE.PLAN как есть»
# осталась ВЕРНОЙ, чинить нечего. Но кэш от 25.07 занижал места на 304 по вузу
# (09.03.01 id=1000012: 114 -> 130; 10.05.01 id=1000025: 63 -> 70), поэтому смена
# PARSE_VERSION здесь обязательна: сигнатура каталога сама по себе кэш добьёт
# (места входят в неё), но полагаться на это нельзя — TTL 3 ч и версия надёжнее.

import time
import os
import re
import pickle
import collections
import requests
import urllib3
import pandas as pd
from IPython.display import display, Markdown

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

start_time_miit = time.time()

BASE_MIIT = 'https://www.miit.ru'
API_MIIT = BASE_MIIT + '/api-1c/miit/hs/AdmissionPlan'
YEAR_MIIT = '2026'
HTTP_HEADERS_MIIT = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'),
    'Accept-Language': 'ru-RU,ru;q=0.9',
    'Referer': BASE_MIIT + '/admissions/degrees',
}
CACHE_MIIT = 'miit_cache.pkl'
# Версия схемы парсинга. Инкрементируй при изменении логики parse_program_miit —
# иначе при неизменившейся сигнатуре каталога из кэша подтянутся старые данные.
# v1: JSON API (reception-plan + concourse-group-page-dto + rating), только
#     «Общий конкурс», места = CONCOURSE.PLAN как есть, порог 277/269,
#     Основной ВП = deferred acceptance, сорт по сумме с ИД ↓.
# v2: порог ПОСТРОЧНЫЙ по pointListDisplay строки (ОТКАЧЕНО в v3).
# v3: порог снова ПО ПРОГРАММЕ (ТЗ): физика среди ВИ, пусть даже как альтернатива
#     информатике («(Информатика или Физика)») -> 277; 269 только там, где ВИ —
#     исключительно информатика. Построчный порог тянул в таблицы лишний шум.
# v4 (04.08.2026, после квотного зачисления 30.07–02.08): забираем ВЕРДИКТ ВУЗА из
#     projectEntrantGroups — колонки «Вердикт вуза» и «№ сайта», в отчёте вместо
#     бесполезного «мест не занято 60 из 70» печатаем реальный минимальный балл
#     проходящих (на 10.05.01 это 224). Плюс: 4 инварианта с диагнозом вместо
#     тихого нуля («Зачислены»+«Проходят»==CONCOURSE.PLAN; подгруппа CEG
#     «Зачислены» не выкидывается и мест не освобождает; выпавшие из PEG — только
#     со сданным согласием; код 2033142 не исчез из 5 своих групп) и «Сумма без ИД»
#     у БВИ = points - pointsInd вместо нуля (pointsDisc у них null).
#     Мест по вузу стало 1952 против 1648 в кэше от 25.07 (+304): PLAN пересчитан.
PARSE_VERSION_MIIT = 4

# Уровни и формы, участвующие в расчёте ВП.
KLF_ALLOWED_MIIT = {'Базовое высшее образование', 'Бакалавриат', 'Специалитет'}
FORMS_ALLOWED_MIIT = {'Очная', 'Очно-заочная', 'Заочная'}

# Код ЕПГУ, ради которого считается весь конвейер. Наш конкурсный балл в МИИТ = 277
# (вуз берёт лучшее из «Информатика или Физика», у нас это физика 86) — РОВНО порог
# отсечения 277, запас нулевой. Одна лишняя строка «total < threshold» вместо «<=» —
# и наша строка молча пропадает из всех таблиц. Поэтому после сбора проверяем, что
# код на месте во всех OUR_TARGET_COUNT_MIIT своих группах (04.08.2026 их 5:
# 10.05.01 приоритет 1, 10.03.01 — 2, 09.03.01 — 3, 09.03.02 — 4, 09.03.03 — 5).
OUR_CODE_MIIT = '2033142'
OUR_TARGET_COUNT_MIIT = 5

# --- Целевые программы из ТЗ (miit.txt): (код, название/профиль). ---
# На сайте это 7 конкурсных групп: часть строк ТЗ — профили одной группы
# (напр. оба профиля 10.05.01 живут в группе «Компьютерная безопасность»).
# Матчим по коду + точному названию specName ИЛИ профиля; название программы в
# таблицах = «код specName (профиль1 | профиль2 | ...)» — профили по факту.
TARGET_MIIT = [
    ('01.03.02', 'Прикладная математика и информатика'),
    ('02.03.02', 'Фундаментальная информатика и информационные технологии'),
    ('10.05.01', 'Информационная безопасность объектов информатизации на базе компьютерных систем'),
    ('10.05.01', 'Безопасность компьютерных систем и сетей (в сфере связи, информационных и коммуникационных технологий)'),
    ('10.03.01', 'Безопасность компьютерных систем'),
    ('09.03.01', 'IT-сервисы и технологии обработки данных на транспорте'),
    ('09.03.01', 'Вычислительные системы и сети'),
    ('09.03.01', 'Системы автоматизированного проектирования'),
    ('09.03.02', 'Информационные системы и технологии на транспорте'),
    ('09.03.02', 'Технологии искусственного интеллекта в транспортных системах'),
    ('09.03.03', 'Прикладная информатика в экономике и бизнесе'),
]

session_miit = requests.Session()
session_miit.headers.update(HTTP_HEADERS_MIIT)


# 23.07.2026: сайты вузов в горячую фазу выпадают на минуту-две (502, таймаут
# соединения). Ждём до ~2.5 мин вместо прежних 12 с; connect отдельно от read.
#
# 04.08.2026, день зачисления: их 1С под нагрузкой отвечает МЕДЛЕННО, но отвечает.
# Замер живьём в момент падения: /reception-plan — 22.4 сек, /rating — 74.2 сек,
# /concourse-group-page-dto — 0.5 сек, все три HTTP 200 с валидным JSON. При connect
# timeout 15 сек соединение рвалось до ответа (ConnectTimeout), а следом их шлюз
# начинал отдавать 404 — ровно как МПГУ 31.07: у обоих 404 под нагрузкой означает
# «сейчас не могу», а не «страница переехала». Поэтому connect поднят до 40 сек,
# read до 420, а на каталоге (единственный запрос, без которого блок не стартует)
# — отдельная, более длинная лесенка ожиданий.
RETRY_DELAYS_MIIT = (3, 6, 12, 24, 40, 60)
# Лесенка для каталога: суммарно около 8 минут против прежних 2.5. Платится
# только тогда, когда вуз действительно лежит.
RETRY_DELAYS_CATALOG_MIIT = (5, 10, 20, 40, 60, 90, 120, 150)
TIMEOUT_MIIT = (40, 420)  # (соединение, чтение)


def fetch_json_miit(path, params, retries=None, delays=RETRY_DELAYS_MIIT):
    """GET JSON-эндпоинта API 1С МИИТ с повторами (экспоненциальный бэкофф).

    404 здесь — такой же временный отказ, как таймаут: их шлюз под нагрузкой
    подменяет им 5xx. Поэтому он тоже ретраится, а не считается «нет адреса».
    """
    if retries is None:
        retries = len(delays) + 1
    last_error = None
    for attempt in range(retries):
        try:
            response = session_miit.get(
                API_MIIT + path, params=params, timeout=TIMEOUT_MIIT, verify=False
            )
            response.raise_for_status()
            return response.json()
        except Exception as error:
            last_error = error
            if attempt == retries - 1:
                break
            pause = delays[min(attempt, len(delays) - 1)]
            print(f'  МИИТ не ответил на {path} ({type(error).__name__}: {error}). '
                  f'Повтор {attempt + 2}/{retries} через {pause} с...')
            time.sleep(pause)
    raise RuntimeError(
        f'Не удалось загрузить {path} {params}: {last_error}. Если это 404 или '
        f'таймаут — их 1С под нагрузкой, адрес живой (проверено 04.08.2026): '
        f'подожди и запусти блок ещё раз.')


def _norm_text_miit(text):
    """Нормализация названия для сравнения: регистр, пробелы, ё."""
    return re.sub(r'\s+', ' ', str(text or '')).strip().lower().replace('ё', 'е')


def _plan_int_miit(group, section, field='PLAN'):
    """planNumbers.integers.<section>.<field> или 0."""
    integers = ((group.get('planNumbers') or {}).get('integers') or {})
    value = (integers.get(section) or {}).get(field)
    return int(value) if isinstance(value, (int, float)) else 0


def discover_programs_miit():
    """Все конкурсные группы базового ВО с местами общего конкурса > 0.

    Возвращает список dict: id, spec, spec_name, form, profiles, conc, entrants.
    """
    # Каталог — единственный запрос, без которого блок не стартует вовсе,
    # поэтому ждём его дольше остальных (см. RETRY_DELAYS_CATALOG_MIIT).
    catalog = fetch_json_miit('/reception-plan', {'year': YEAR_MIIT},
                              delays=RETRY_DELAYS_CATALOG_MIIT)
    programs = []
    for institute in catalog.get('result', []):
        for group in institute.get('concourseGroups', []):
            if group.get('klfName') not in KLF_ALLOWED_MIIT:
                continue
            if group.get('ktName') not in FORMS_ALLOWED_MIIT:
                continue
            conc = _plan_int_miit(group, 'CONCOURSE')
            if conc <= 0:
                continue
            match = re.search(r'/degrees/(\d+)', group.get('planReceptionUrl') or '')
            if not match:
                continue
            programs.append({
                'id': match.group(1),
                'spec': group.get('specQualifier') or '',
                'spec_name': (group.get('specName') or '').strip(),
                'form': group.get('ktName') or '',
                'profiles': [str(p.get('specNote') or '').strip()
                             for p in (group.get('profiles') or [])],
                'conc': conc,
                'entrants': _plan_int_miit(group, 'ALL', 'ENTRANTS'),
            })
    return programs


def resolve_targets_miit(programs):
    """Сопоставить 11 целевых строк ТЗ с конкурсными группами.

    Строка ТЗ матчится на группу по коду + точному (нормализованному) совпадению
    с specName или с одним из профилей группы. Возвращает упорядоченный список
    id групп (без дублей, в порядке первого упоминания в ТЗ).
    """
    resolved_ids = []
    for dotted, name in TARGET_MIIT:
        norm_name = _norm_text_miit(name)
        found = None
        for prog in programs:
            if prog['spec'] != dotted:
                continue
            candidates = [_norm_text_miit(prog['spec_name'])] + \
                         [_norm_text_miit(p) for p in prog['profiles']]
            if norm_name in candidates:
                found = prog['id']
                break
        if found is None:
            # Запасной матч по вхождению подстроки (если сайт переформулировал).
            for prog in programs:
                if prog['spec'] != dotted:
                    continue
                haystack = _norm_text_miit(prog['spec_name'] + ' ' + ' '.join(prog['profiles']))
                if norm_name in haystack:
                    found = prog['id']
                    break
        if found is None:
            print(f'    ВНИМАНИЕ: целевая {dotted} «{name[:50]}» не найдена в каталоге')
            continue
        if found not in resolved_ids:
            resolved_ids.append(found)
    return resolved_ids


def program_display_miit(prog):
    """Название программы для таблиц: «код specName (профиль1 | профиль2)»."""
    profiles = ' | '.join(p for p in prog['profiles'] if p)
    tail = f' ({profiles})' if profiles else ''
    return f"{prog['spec']} {prog['spec_name']}{tail}"


def split_disciplines_miit(disciplines):
    """Слоты предметов из «Вступительных испытаний» карточки программы.

    «Математика, Русский язык, (Информатика или Физика)» ->
    ['Математика', 'Русский язык', '(Информатика или Физика)'] —
    запятые внутри скобок не разделяют.
    """
    slots, buf, depth = [], '', 0
    for ch in disciplines or '':
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth = max(0, depth - 1)
        if ch == ',' and depth == 0:
            slots.append(buf.strip())
            buf = ''
        else:
            buf += ch
    if buf.strip():
        slots.append(buf.strip())
    return slots


def subject_label_miit(slot_text):
    """Короткая подпись слота. Физика приоритетнее информатики (как в образцах)."""
    lowered = _norm_text_miit(slot_text)
    if 'физик' in lowered:
        return 'Физ.'
    if 'информатик' in lowered or 'информационные технологии' in lowered:
        return 'Инф.'
    if 'математик' in lowered:
        return 'Мат.'
    if 'русск' in lowered:
        return 'Рус.'
    if 'иностран' in lowered or 'англ' in lowered:
        return 'Ин.яз.'
    if 'общество' in lowered:
        return 'Обществ.'
    if 'истори' in lowered:
        return 'Ист.'
    if 'хими' in lowered:
        return 'Хим.'
    if 'биолог' in lowered:
        return 'Биол.'
    cleaned = re.sub(r'[()]', '', str(slot_text or '')).strip()
    return cleaned.split()[0][:8] if cleaned else '—'


def _priority_miit(entrant):
    """Приоритет из preferenceIndexDisplay: «1 (Основной ВП)» -> 1."""
    match = re.match(r'\s*(\d+)', str(entrant.get('preferenceIndexDisplay') or ''))
    return int(match.group(1)) if match else 99


def _to_int_miit(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def get_update_date_miit(rating):
    """Дата актуальности из названия подгруппы: «...по состоянию на ДД.ММ.ГГГГ»."""
    for comp in rating.get('competitions', []):
        for group in comp.get('concourseEntrantGroups', []):
            match = re.search(r'по состоянию на\s*([\d.]+)', str(group.get('name') or ''))
            if match:
                return match.group(1).strip('. ')
    return None


def _strip_state_date_miit(name):
    """«Проходят по состоянию на 04.08.2026» -> «Проходят»."""
    return re.sub(r'\s*по состоянию на.*$', '', str(name or '')).strip()


# Короткие подписи вердикта вуза для колонки «Вердикт вуза». Названия групп PEG
# длинные и с датой, в таблицу их не поместить.
VERDICT_ENROLLED_MIIT = 'Зачислен'
VERDICT_PASS_MIIT = 'Проходит'
VERDICT_FAIL_MIIT = 'Не проходит'
VERDICT_NO_CONSENT_MIIT = 'Нет согласия'
VERDICT_ELSEWHERE_MIIT = 'Проходит на другую'


def verdict_label_miit(group):
    """Вердикт вуза по одной группе projectEntrantGroups.

    Ориентируемся на code (1/2 — машинные, вуз ими красит строки), а текстовые
    названия распознаём по подстроке: даты в них меняются каждый день, а
    формулировку «Не проходят на бюджет на все выбранные специальности» вуз уже
    один раз переписывал.
    """
    code = group.get('code')
    if code == 1:
        return VERDICT_ENROLLED_MIIT
    if code == 2:
        return VERDICT_PASS_MIIT
    lowered = _norm_text_miit(group.get('name'))
    if 'нет согласия' in lowered:
        return VERDICT_NO_CONSENT_MIIT
    if 'не проход' in lowered:
        return VERDICT_FAIL_MIIT
    return _strip_state_date_miit(group.get('name')) or '—'


def parse_program_miit(prog, is_target):
    """Парсит одну конкурсную группу: карточка (предметы, места) + список.

    Места общего конкурса = CONCOURSE.PLAN из карточки, КАК ЕСТЬ (по ТЗ): после
    квотного зачисления 30.07–02.08 вуз сам пересчитал это поле в остаток бюджета.
    Берём только конкурс «Общий конкурс» и ВСЕ его подгруппы, включая «Зачислены»
    («Нет баллов» и «Баллы ниже минимума» отсекаются порогом). Строки ниже порога
    не сохраняются. Из ветки projectEntrantGroups добираем вердикт вуза и номер
    строки на экране сайта, а также реальный проходной балл по полному списку.
    """
    dto = fetch_json_miit('/concourse-group-page-dto', {'id': prog['id'], 'year': YEAR_MIIT})
    card = dto.get('concourseGroup') or {}
    budget = _plan_int_miit(card, 'CONCOURSE')
    if budget <= 0:
        budget = prog['conc']
    if budget <= 0:
        return None

    disciplines = (card.get('disciplines') or '').strip()
    slot_labels = [subject_label_miit(s) for s in split_disciplines_miit(disciplines)]
    # Порог ПО ПРОГРАММЕ (ТЗ): есть физика среди ВИ — в т.ч. как альтернатива
    # информатике, «(Информатика или Физика)» — порог 277. Порог 269 только у
    # программ, где ВИ — исключительно информатика.
    disciplines_norm = _norm_text_miit(disciplines)
    threshold = 277 if 'физик' in disciplines_norm else 269

    rating = fetch_json_miit('/rating', {'id': prog['id'], 'year': YEAR_MIIT})
    update_date = get_update_date_miit(rating)

    general = None
    for comp in rating.get('competitions', []):
        if str(comp.get('competitionName') or '').startswith('Общий'):
            general = comp
            break
    if general is None:
        raise RuntimeError(
            f'{prog["spec"]} [{prog["form"]}] id={prog["id"]}: в /rating нет конкурса '
            f'«Общий конкурс» (виды: '
            f'{[c.get("competitionName") for c in rating.get("competitions", [])]})')

    # --- Ветка PEG: вердикт вуза + номер строки на экране сайта. ---
    verdict_by_code, site_index_by_code = {}, {}
    vuz_pass_codes, vuz_enrolled_codes = set(), set()
    for group in general.get('projectEntrantGroups', []):
        label = verdict_label_miit(group)
        for entrant in group.get('entrants', []):
            code = str(entrant.get('displayName') or '').strip()
            if not code:
                continue
            verdict_by_code[code] = label
            site_index_by_code[code] = _to_int_miit(entrant.get('ratingIndex'))
            if label == VERDICT_PASS_MIIT:
                vuz_pass_codes.add(code)
            elif label == VERDICT_ENROLLED_MIIT:
                vuz_enrolled_codes.add(code)
    vuz_holders = vuz_pass_codes | vuz_enrolled_codes

    # ИНВАРИАНТ 1. «Зачислены» + «Проходят» = ровно CONCOURSE.PLAN (33 группы из 33
    # на 04.08.2026). Если вуз перестанет раздавать ветку ровно по числу мест,
    # колонка «Вердикт вуза» превратится в тыкву — узнать об этом надо сразу.
    if vuz_holders and len(vuz_holders) != budget:
        print(f'    ВНИМАНИЕ {prog["spec"]} [{prog["form"]}] id={prog["id"]}: вердикт вуза '
              f'раздан на {len(vuz_holders)} чел., а мест общего конкурса {budget} — '
              f'ветка «Проходят/Зачислены» больше не равна плану, колонке не верить.')

    rows = []
    ceg_codes, ceg_enrolled_seen, hidden_without_consent = set(), 0, 0
    for group in general.get('concourseEntrantGroups', []):
        status = _strip_state_date_miit(group.get('name'))
        # ИНВАРИАНТ 2. Подгруппу «Зачислены по состоянию на ...» берём наравне с
        # «Участвуют в конкурсе»: CONCOURSE.PLAN зачисленных НЕ вычитает, значит их
        # места внутри плана и заняты. Если когда-нибудь «оптимизировать» разбор до
        # одной подгруппы «Участвуют», занятые места молча станут свободными и
        # шансы взлетят на пустом месте.
        if group.get('code') == 1 or 'ачислен' in _norm_text_miit(status):
            ceg_enrolled_seen += len(group.get('entrants') or [])
        for entrant in group.get('entrants', []):
            code = str(entrant.get('displayName') or '').strip()
            if not code:
                continue
            ceg_codes.add(code)
            total = _to_int_miit(entrant.get('points'))
            is_bvi = bool(str(entrant.get('withoutExamsText') or '').strip()
                          or str(entrant.get('foundationBVI') or '').strip())
            consent = str(entrant.get('originalDocumentStatus') or '').strip().lower()
            has_consent = consent.startswith(('представлено', 'да'))
            # Считаем по ВСЕМУ списку, а не только выше порога: правило «нет в PEG =
            # проходит на другую программу» надо проверять на полной выборке.
            if code not in verdict_by_code and not has_consent:
                hidden_without_consent += 1
            # У БВИ pointsDisc=null (вступительных не сдавали) — добираем разностью,
            # иначе в колонке «Сумма без ИД» стоял бы ноль, который модель шансов
            # читает как признак БВИ, а человек — как «у него нет баллов».
            if entrant.get('pointsDisc') is None:
                bonus_free = total - _to_int_miit(entrant.get('pointsInd'))
            else:
                bonus_free = _to_int_miit(entrant.get('pointsDisc'))
            # Порог не применяется к БВИ (по образцу МТУСИ).
            if not is_bvi and total < threshold:
                continue

            if is_bvi:
                marks = 'БВИ'
            else:
                scores = [s.strip() for s in str(entrant.get('pointList') or '').split(';')]
                marks = ', '.join(
                    f'{slot_labels[j]}: {scores[j]}'
                    for j in range(min(len(slot_labels), len(scores))) if scores[j]
                ) or '—'

            # Кого в PEG нет вовсе — тот проходит на другую программу МИИТ (свой
            # приоритет выше), поэтому в этом списке вуз его не показывает.
            verdict = verdict_by_code.get(
                code, VERDICT_ELSEWHERE_MIIT if has_consent else '—')
            rows.append({
                '№_orig': _to_int_miit(entrant.get('ratingIndex')),
                # «№ сайта» = 0 у тех, кого в ветке вердиктов нет: они проходят на
                # другую программу, и на экране этого списка вуз их не рисует вовсе.
                '№ сайта': site_index_by_code.get(code, 0),
                'Код': code,
                'Сумма': total,
                'Сумма без ИД': bonus_free,
                'Оценки': marks,
                'Согласие': 'да' if has_consent else '',
                'Приоритет': _priority_miit(entrant),
                'БВИ': 'да' if is_bvi else 'нет',
                'Вердикт вуза': verdict,
                'Статус': status,
            })

    # ИНВАРИАНТ 2 (проверка). Зачисленных в CEG ровно столько же, сколько в ветке
    # вердиктов. Если кто-то «оптимизирует» разбор до одной подгруппы «Участвуют»,
    # здесь станет 0 против ненулевого вердикта — и мы это увидим, а не отдадим
    # молча освободившиеся места тем, кто на них не проходит.
    if ceg_enrolled_seen != len(vuz_enrolled_codes):
        print(f'    ВНИМАНИЕ {prog["spec"]} [{prog["form"]}] id={prog["id"]}: зачисленных '
              f'в списке {ceg_enrolled_seen}, а по вердикту вуза {len(vuz_enrolled_codes)} — '
              f'подгруппа «Зачислены» разобрана не полностью, их места считаются свободными.')

    # ИНВАРИАНТ 3. Выпасть из PEG может только тот, кто прошёл на другую программу,
    # а туда без согласия не берут. Если появятся выпавшие без согласия — значит
    # вуз стал прятать строки по другому правилу и «Проходит на другую» врёт.
    if hidden_without_consent:
        print(f'    ВНИМАНИЕ {prog["spec"]} [{prog["form"]}] id={prog["id"]}: '
              f'{hidden_without_consent} чел. нет в ветке вердиктов и при этом без '
              f'согласия — правило «нет в PEG = проходит на другую» больше не держится.')

    # ИНВАРИАНТ 4. Разобрали ноль строк — это ошибка с диагнозом, а не пустой список.
    if not rows:
        if not ceg_codes:
            raise RuntimeError(
                f'{prog["spec"]} [{prog["form"]}] id={prog["id"]}: «Общий конкурс» есть, '
                f'но в concourseEntrantGroups ноль абитуриентов — разметку сломали.')
        return None

    # Минимальный балл среди тех, кому вуз отдал места, — реальный проходной по
    # ПОЛНОМУ списку. БВИ считаем отдельно: у них points=0 и они утянули бы минимум
    # в ноль. Это единственное честное число про проходной балл в блоке: наш
    # «остаток мест» считается по обрезанному порогом списку и всегда завышен.
    holder_points = [_to_int_miit(e.get('points'))
                     for group in general.get('projectEntrantGroups', [])
                     if verdict_label_miit(group) in (VERDICT_PASS_MIIT, VERDICT_ENROLLED_MIIT)
                     for e in group.get('entrants', [])
                     if not (str(e.get('withoutExamsText') or '').strip()
                             or str(e.get('foundationBVI') or '').strip())]

    df = pd.DataFrame(rows)
    # Один абитуриент — одна строка общего конкурса на группу; дедуп на всякий
    # случай (оставляем строку с наибольшей суммой). БВИ всегда наверху, остальные
    # по Сумме (с ИД) по убыванию; тай-брейк по №_orig — детерминизм и совпадение
    # с порядком сайта.
    df = (df.sort_values(
            ['БВИ', 'Сумма', '№_orig'], ascending=[True, False, True], kind='stable',
            key=lambda c: c.map({'да': 0, 'нет': 1}) if c.name == 'БВИ' else c)
            .drop_duplicates('Код', keep='first')
            .reset_index(drop=True))
    df.insert(0, '№', range(1, len(df) + 1))

    return {
        'df': df,
        'budget': budget,
        'threshold': threshold,
        'slot_labels': slot_labels,
        'disciplines': disciplines,
        'form': prog['form'],
        'university': 'МИИТ',
        'competition': program_display_miit(prog),
        'is_target': is_target,
        'update_date': update_date,
        # Вердикт вуза по полному списку: сколько мест он уже раздал, скольких
        # зачислил и какой у проходящих минимальный балл.
        'vuz_holders': len(vuz_holders),
        'vuz_enrolled': len(vuz_enrolled_codes),
        'vuz_pass_min': min(holder_points) if holder_points else None,
        'ceg_enrolled': ceg_enrolled_seen,
    }


# ===== Сбор данных по всем программам вуза (кэш по сигнатуре каталога) =====
print('Загрузка каталога программ МИИТ (РУТ)...')
all_programs_miit = discover_programs_miit()
target_ids_miit = resolve_targets_miit(all_programs_miit)
programs_by_id_miit = {prog['id']: prog for prog in all_programs_miit}
target_names_miit = {pid: program_display_miit(programs_by_id_miit[pid])
                     for pid in target_ids_miit}
print(f'Групп базового ВО с местами общего конкурса: {len(all_programs_miit)}; '
      f'целевых групп сопоставлено: {len(target_ids_miit)} '
      f'(из {len(TARGET_MIIT)} строк ТЗ).')

# Сигнатура каталога: при обновлении списков меняется число заявлений.
programs_signature_miit = sorted(
    (prog['id'], prog['conc'], prog['entrants']) for prog in all_programs_miit)

cached_miit = None
if os.path.exists(CACHE_MIIT):
    with open(CACHE_MIIT, 'rb') as f:
        cached_miit = pickle.load(f)

# Страховка по возрасту кэша: в сигнатуре МИИТ нет ни даты, ни согласий — только
# места и число заявлений. После закрытия приёма (25.07) счётчик заявлений
# замирает, а согласия продолжают меняться до самого зачисления, и кэш повис бы
# навсегда на устаревших данных. Поэтому старше TTL — перекачиваем всегда.
CACHE_TTL_HOURS_MIIT = 3
cache_fresh_miit = (
    os.path.exists(CACHE_MIIT)
    and (time.time() - os.path.getmtime(CACHE_MIIT)) < CACHE_TTL_HOURS_MIIT * 3600
)

cache_valid_miit = (
    cached_miit is not None
    and cache_fresh_miit
    and cached_miit.get('version') == PARSE_VERSION_MIIT
    and cached_miit.get('signature') == programs_signature_miit
    and cached_miit.get('data')
)
if cached_miit is not None and not cache_fresh_miit:
    print(f'Кэш старше {CACHE_TTL_HOURS_MIIT} ч — перекачиваем списки '
          f'(согласия меняются без изменения числа заявлений).')

all_miit_data = {}
if cache_valid_miit:
    all_miit_data = cached_miit['data']
    print('Каталог не изменился (места и число заявлений те же) — используем кэш.')
else:
    failed_miit = []
    for idx, prog in enumerate(all_programs_miit, 1):
        is_target = prog['id'] in target_ids_miit
        tag = 'ЦЕЛЕВАЯ' if is_target else 'общая'
        try:
            data = parse_program_miit(prog, is_target)
        except Exception as error:
            print(f'  [{idx}/{len(all_programs_miit)}] Ошибка {prog["spec"]} [{prog["form"]}]: {error}')
            failed_miit.append(f'{prog["spec"]} [{prog["form"]}]')
            continue
        if data is None:
            continue
        all_miit_data[prog['id']] = data
        print(f'  [{idx}/{len(all_programs_miit)}] {tag}: {prog["spec"]} '
              f'{prog["spec_name"][:40]} [{prog["form"]}] мест общ.конкурса={data["budget"]}, '
              f'порог={data["threshold"]}, строк={len(data["df"])}')

    # Кэш пишем ТОЛЬКО при полной загрузке (см. пояснение в блоке МТУСИ).
    if all_miit_data and not failed_miit:
        with open(CACHE_MIIT, 'wb') as f:
            pickle.dump({
                'version': PARSE_VERSION_MIIT,
                'signature': programs_signature_miit,
                'data': all_miit_data,
            }, f)
        print('Данные сохранены в кэш.')
    elif failed_miit:
        print(f'  ВНИМАНИЕ: не загрузились программы ({len(failed_miit)}): '
              f'{"; ".join(failed_miit[:3])} — кэш НЕ сохранён, '
              f'работаем на неполных данных.')

if not all_miit_data:
    raise SystemExit('Нет данных ни по одной программе МИИТ.')

current_date_miit = next(
    (d['update_date'] for d in all_miit_data.values() if d.get('update_date')), None)
loaded_targets_miit = [pid for pid in target_ids_miit if pid in all_miit_data]
print(f'Дата актуальности списков: {current_date_miit}')
print(f'Загружено программ: {len(all_miit_data)}; '
      f'целевых с данными: {len(loaded_targets_miit)}/{len(target_ids_miit)}.')

# ===== Расчёт Основного ВП (отложенная приёмка по ВСЕМ программам вуза) =====
# Модель Минобра «конкурс с приоритетами»: абитуриент зачисляется на программу с
# наивысшим приоритетом, где реально проходит по баллам после ухода конкурентов на
# их приоритетные программы (эффект вытеснения). Кандидат-предлагающая отложенная
# приёмка (deferred acceptance), как в мтуси_блок3.py / политех_блок4.py.
prog_rank_miit = {}      # id -> {Код: позиция в списке, меньше = сильнее}
prog_budget_miit = {}    # id -> число мест общего конкурса
cand_prefs_miit = {}     # Код -> id программ по возрастанию приоритета
for key, data in all_miit_data.items():
    df = data['df']
    prog_budget_miit[key] = data['budget']
    prog_rank_miit[key] = {code: i for i, code in enumerate(df['Код'])}
    for code, priority in zip(df['Код'], df['Приоритет']):
        cand_prefs_miit.setdefault(code, []).append((int(priority), key))
for code, pairs in cand_prefs_miit.items():
    cand_prefs_miit[code] = [key for _, key in sorted(pairs, key=lambda x: x[0])]

admitted_miit = {key: {} for key in all_miit_data}   # id -> {Код: ранг}
next_choice_miit = {code: 0 for code in cand_prefs_miit}
pending_miit = collections.deque(cand_prefs_miit)

while pending_miit:
    code = pending_miit.popleft()
    prefs = cand_prefs_miit[code]
    while next_choice_miit[code] < len(prefs):
        key = prefs[next_choice_miit[code]]
        budget = prog_budget_miit[key]
        if budget <= 0:
            next_choice_miit[code] += 1
            continue
        rank = prog_rank_miit[key][code]
        held = admitted_miit[key]
        if len(held) < budget:
            held[code] = rank
            break
        worst_code = max(held, key=held.get)   # держим слабейшего (наибольший ранг)
        if rank < held[worst_code]:
            del held[worst_code]
            held[code] = rank
            next_choice_miit[worst_code] += 1
            pending_miit.append(worst_code)    # вытесненный ищет следующий приоритет
            break
        next_choice_miit[code] += 1            # здесь не прошёл — следующий приоритет

enrolled_miit = {}
for key, data in all_miit_data.items():
    admitted_codes = set(admitted_miit[key])
    data['df']['Основной ВП'] = (
        data['df']['Код'].isin(admitted_codes).map({True: 'да', False: 'нет'})
    )
    enrolled_miit.update((code, True) for code in admitted_codes)

# ===== Свободные места после Основного ВП =====
# «Проходной ВП» своим расчётом получить НЕЛЬЗЯ: после отложенной приёмки у
# программы со свободными местами не остаётся незачисленных кандидатов — тот,
# кого не взяли нигде, дошёл бы до этого места и занял его. Прежний расчёт
# давал «нет» во всех строках и создавал ложное впечатление второй проверки.
# Полезен сам остаток: места не разобраны теми, кто выше порога, значит
# проходной балл программы НИЖЕ нашего порога отбора.
free_places_miit = {}
for key, data in all_miit_data.items():
    df = data['df']
    df['Проходной ВП'] = 'нет'
    free_places_miit[key] = max(
        0, data['budget'] - int((df['Основной ВП'] == 'да').sum()))

# ===== Отдельные таблицы по программам (только целевые, порядок ТЗ) =====
display(Markdown('# БЛОК 1: МИИТ'))
for key in target_ids_miit:
    data = all_miit_data.get(key)
    if data is None:
        display(Markdown(f'### {target_names_miit.get(key, key)} — нет данных'))
        print('-' * 80)
        continue
    df = data['df']
    count_main = (df['Основной ВП'] == 'да').sum()
    count_bvi = (df['БВИ'] == 'да').sum()

    display(Markdown(f"### {data['competition']} ({data['form']})"))
    display(Markdown(f"**Вступительные испытания:** {data['disciplines'] or '—'}"))
    display(Markdown(
        f"**Мест общего конкурса:** {data['budget']}  |  **БВИ в списке:** {count_bvi}  |  "
        f"**Порог отбора:** {data['threshold']}  |  **Всего строк:** {len(df)}"
    ))
    # Вердикт вуза по ПОЛНОМУ списку — вместо прежнего «мест не занято N из M».
    # Тот остаток считался по обрезку выше порога и всегда завышал свободные места:
    # на 10.05.01 он писал «не занято 60 из 70», хотя вуз раздал все 70 и нижний
    # проходящий там с 224 баллами.
    pass_min = data.get('vuz_pass_min')
    display(Markdown(
        f'**Основной ВП (да):** {count_main}  |  '
        f"**Вердикт вуза:** мест роздано {data.get('vuz_holders', 0)} из {data['budget']} "
        f"(в т.ч. уже зачислено {data.get('vuz_enrolled', 0)})  |  "
        f'**Реальный проходной балл (минимум среди проходящих):** '
        f"{pass_min if pass_min is not None else '—'}  |  "
        f'**Свободно по нашему обрезку:** {free_places_miit.get(key, 0)} '
        f'(число завышено — список обрезан порогом {data["threshold"]})'
    ))
    display_cols = ['№', '№_orig', '№ сайта', 'Код', 'Сумма', 'Сумма без ИД', 'Оценки',
                    'Согласие', 'Приоритет', 'БВИ', 'Основной ВП', 'Проходной ВП',
                    'Вердикт вуза', 'Статус']
    display(df[[c for c in display_cols if c in df.columns]].style.hide(axis='index'))
    print('-' * 80)

# ===== Где стоит наш код (сторож против тихого исчезновения строки) =====
# Наш балл 277 = ровно порог отсечения, запас нулевой: любая правка сравнения или
# смена набора ВИ у вуза выкинет строку из всех таблиц, и блок отработает «успешно»
# с пустым местом на месте главного ответа. Печатаем строки явно и ругаемся, если
# код нашёлся не во всех своих целевых группах.
our_rows_miit = []
for key, data in all_miit_data.items():
    hit = data['df'][data['df']['Код'] == OUR_CODE_MIIT]
    for _, row in hit.iterrows():
        our_rows_miit.append((key, data, row))

display(Markdown(f'### Наш код {OUR_CODE_MIIT} в списках МИИТ'))
if not our_rows_miit:
    print(f'  ОШИБКА: код {OUR_CODE_MIIT} не найден НИ В ОДНОЙ программе МИИТ. '
          f'Это не «нас нет в списках», это сломанный разбор: проверь порог '
          f'(наш балл 277 = ровно порог, сравнение должно быть строгим "<") '
          f'и поле displayName.')
else:
    for key, data, row in sorted(our_rows_miit, key=lambda t: t[2]['Приоритет']):
        print(f"  приоритет {row['Приоритет']}: {data['competition']} [{data['form']}] "
              f"— №_orig {row['№_orig']} (на сайте № {row['№ сайта']}) из {len(data['df'])} "
              f"строк выше порога, балл {row['Сумма']}, мест {data['budget']}, "
              f"согласие «{row['Согласие'] or 'нет'}», вердикт вуза «{row['Вердикт вуза']}», "
              f"Основной ВП «{row['Основной ВП']}», "
              f"реальный проходной {data.get('vuz_pass_min')}")

our_target_hits_miit = sum(1 for key, _, _ in our_rows_miit if key in target_ids_miit)
if our_target_hits_miit < OUR_TARGET_COUNT_MIIT:
    print(f'  ВНИМАНИЕ: код {OUR_CODE_MIIT} найден только в {our_target_hits_miit} '
          f'целевых группах из ожидаемых {OUR_TARGET_COUNT_MIIT}. Либо вуз снял заявление, '
          f'либо разбор потерял строки — разбирайся руками, молча этому верить нельзя.')
print('-' * 80)

print(f'\nВремя работы БЛОКА 1 (МИИТ): {time.time() - start_time_miit:.1f} сек.')
print('Данные для сводной готовы: all_miit_data, target_ids_miit — '
      'см. «БЛОК 2: СВОДНАЯ ТАБЛИЦА МИИТ».')

In [ ]:
# -*- coding: utf-8 -*-
# ===== БЛОК 1: РГУ ГУБКИНА =====
# Образцы — МТУСИ, Московский Политех, МИИТ. Тот же принцип: «Основной ВП» считаем
# по ВСЕМ конкурсным группам вуза, удовлетворяющим условиям, таблицы — по целевым.
#
# У Губкина конкурс идёт по КОНКУРСНЫМ ГРУППАМ (КГ): одна КГ может объединять
# несколько программ (профилей) с общими местами. Целевые строки ТЗ (9 точечных
# кодов) ложатся в 5 КГ: КГ17 (все 09.03.01.xx), КГ18, КГ26, КГ27 (обе 10.05.03),
# КГ28.
#
# Источники (Selenium не нужен):
#   1. «Онлайн-табло» https://transfer.priem.gubkin.ru/live/ — обычная HTML-таблица
#      (г. Москва): КГ -> программы (точечные коды) и колонка «Места на общий
#      конкурс (чел.)» — берём КАК ЕСТЬ, пересчитывать не надо (ТЗ). rowspan-грабли:
#      у КГ с несколькими программами строки-продолжения содержат только
#      [код, название]. Табло — только Москва, поэтому филиалы (Оренбург «О»,
#      Ташкент «Т») и коммерческие группы отсекаются сами: их в табло нет.
#   2. Списки: Angular SPA https://transfer.priem.gubkin.ru/abiturients_list/,
#      открытый JSON API (найден в бандле main.*.js):
#      GET api/api.php?act=search&method=...
#        getForm -> формы {1: Очная, 2: Очно-заочная, 3: Заочная};
#        getFaculties(educationFormId) -> факультеты, нужен id=666
#          «Бакалавриат/специалитет»;
#        getGroups(educationFormId, facultyId) -> КГ (id, short_name, full_name,
#          without_exam, ...);
#        getEducationTypes(contestGroupId) -> типы; нужен «Основные места в рамках
#          КЦП» (в ТЗ «КЦД») — если у КГ его нет, общего конкурса у неё нет;
#        get(educationTypeId, contestGroupId) -> список абитуриентов.
#
# Поля строки списка: fio=ID (наш «Код»), position=№_orig, totalBalls=сумма С ИД,
# entranceBalls=сумма без ИД, individualAchievementsBalls=ИД, priority=приоритет,
# enrollmentAgreement=галочка «Согласие» (true -> «да»), benefit («преим. право» ->
# в Примечание), ballsBySubjects=[{name, ball, priority}] — priority здесь =
# НОМЕР СЛОТА предметов: слот 1 = Математика|Основы инж. вычислений, слот 2 =
# Физика|Информатика|Экзамен по специальности, слот 3 = Русский язык.
# Если сданы и физика, и информатика — в сумму идёт максимум (проверено:
# entranceBalls = сумма максимумов по слотам). Слот с физикой подписываем «Физ.»
# всем (как в Политехе), балл — фактический из суммы (максимум слота).
#
# Порог (как у всех): 277, если среди предметов есть физика, иначе 269.
# Строки ниже порога (по сумме С ИД) не сохраняем.
#
# Кэш: точной даты обновления у Губкина нет (на табло только день), поэтому
# валидность кэша — по сигнатуре табло (все числовые колонки всех КГ + дата):
# при обновлении списков меняется «Принято заявлений» и кэш пересобирается.

import time
import os
import re
import pickle
import collections
import requests
import urllib3
import pandas as pd
from IPython.display import display, Markdown

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

start_time_gubkin = time.time()

LIVE_URL_GUBKIN = 'https://transfer.priem.gubkin.ru/live/'
API_GUBKIN = 'https://transfer.priem.gubkin.ru/abiturients_list/api/api.php'
HTTP_HEADERS_GUBKIN = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'),
    'Accept-Language': 'ru-RU,ru;q=0.9',
    'Referer': 'https://transfer.priem.gubkin.ru/abiturients_list/',
}
CACHE_GUBKIN = 'gubkin_cache.pkl'
# Версия схемы парсинга. Инкрементируй при изменении логики parse_group_gubkin —
# иначе при неизменившейся сигнатуре табло из кэша подтянутся старые данные.
# v1: табло (места общ.конкурса как есть) + JSON API списков, тип «Основные места
#     в рамках КЦП», слоты предметов по priority, порог 277/269 по сумме с ИД,
#     Основной ВП = deferred acceptance.
# v2: порог ПОСТРОЧНЫЙ по предмету строки из ballsBySubjects (ОТКАЧЕНО в v3).
# v3: порог снова ПО КГ (ТЗ): физика среди предметов слотов, пусть даже как
#     альтернатива информатике, -> 277; 269 только там, где ВИ — исключительно
#     информатика. Построчный порог тянул в таблицы лишний шум.
PARSE_VERSION_GUBKIN = 3

FACULTY_BAKSPEC_GUBKIN = 666          # «Бакалавриат/специалитет»
FORM_IDS_GUBKIN = [1, 2, 3]           # Очная / Очно-заочная / Заочная

# --- Целевые программы из ТЗ (gubki.txt): точечный код -> название. ---
# Матчим по коду программы в табло; целевая единица — вся КГ, в которую входит код.
TARGET_GUBKIN = [
    ('09.03.01.02', 'Интегрированные автоматизированные информационные системы'),
    ('09.03.01.06', 'Искусственный интеллект в автоматизированных системах обработки информации и управления'),
    ('09.03.01.01', 'Автоматизированные системы обработки информации и управления'),
    ('09.03.01.05', 'Программная инженерия в управлении техническими системами'),
    ('01.03.04.01', 'Математическое моделирование в технике и экономике'),
    ('10.03.01.05', 'Безопасность автоматизированных систем (в топливно-энергетическом комплексе)'),
    ('10.05.03.04', 'Безопасность автоматизированных систем управления технологическими процессами'),
    ('10.05.03.03', 'Анализ безопасности информационных систем'),
    ('10.05.04.02', 'Информационная безопасность финансовых и экономических структур'),
]

session_gubkin = requests.Session()
session_gubkin.headers.update(HTTP_HEADERS_GUBKIN)


# 23.07.2026: сайты вузов в горячую фазу выпадают на минуту-две (502, таймаут
# соединения). Ждём до ~2.5 мин вместо прежних 12 с; connect отдельно от read.
RETRY_DELAYS_GUBKIN = (3, 6, 12, 24, 40, 60)
TIMEOUT_GUBKIN = (15, 120)  # (соединение, чтение)


def _retry_gubkin(what, attempt, retries, error):
    """Пауза перед повтором; True — пробуем ещё, False — попытки кончились."""
    if attempt == retries - 1:
        return False
    pause = RETRY_DELAYS_GUBKIN[min(attempt, len(RETRY_DELAYS_GUBKIN) - 1)]
    print(f'  Губкин не ответил на {what} ({type(error).__name__}: {error}). '
          f'Повтор {attempt + 2}/{retries} через {pause} с...')
    time.sleep(pause)
    return True


def fetch_live_gubkin(retries=len(RETRY_DELAYS_GUBKIN) + 1):
    """HTML онлайн-табло (г. Москва)."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session_gubkin.get(LIVE_URL_GUBKIN,
                                          timeout=TIMEOUT_GUBKIN, verify=False)
            response.raise_for_status()
            return response.text
        except Exception as error:
            last_error = error
            if not _retry_gubkin('онлайн-табло', attempt, retries, error):
                break
    raise RuntimeError(f'Не удалось загрузить онлайн-табло: {last_error}')


def api_gubkin(params, retries=len(RETRY_DELAYS_GUBKIN) + 1):
    """GET api.php?act=search&... -> поле data из JSON-ответа."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session_gubkin.get(
                API_GUBKIN, params=dict(params, act='search'),
                timeout=TIMEOUT_GUBKIN, verify=False
            )
            response.raise_for_status()
            payload = response.json()
            if not payload.get('success', True):
                raise ValueError(f'API success=false: {str(payload)[:120]}')
            return payload.get('data')
        except Exception as error:
            last_error = error
            if not _retry_gubkin(f'API {params}', attempt, retries, error):
                break
    raise RuntimeError(f'Не удалось вызвать API {params}: {last_error}')


def _cell_text_gubkin(td_html):
    txt = re.sub(r'<[^>]+>', '', td_html).replace('&nbsp;', ' ')
    return re.sub(r'\s+', ' ', txt).strip()


def parse_live_gubkin(html):
    """Табло -> (дата, {КГ: {'codes', 'names', 'nums', 'places_general'}}).

    Строка новой КГ: [«КГ N», код, название, 10 числовых колонок(rowspan)],
    строка-продолжение той же КГ: [код, название]. «Места на общий конкурс (чел.)» —
    5-я числовая колонка (nums[4]).
    """
    date_match = re.search(r'Табло[^<]*</b><br/>\s*([^<]+)', html)
    live_date = date_match.group(1).strip() if date_match else None

    kg_map = {}
    current = None
    for tr in re.findall(r'<tr[^>]*>.*?</tr>', html, re.S):
        cells = [_cell_text_gubkin(td) for td in re.findall(r'<td[^>]*>.*?</td>', tr, re.S)]
        if not cells:
            continue
        if re.match(r'^[А-ЯA-Z]{1,3}\s?\d+$', cells[0]) and len(cells) >= 8 \
                and re.match(r'\d{2}\.\d{2}\.\d{2}', cells[1]):
            kg = cells[0].replace(' ', '')
            nums = cells[3:]
            places = int(nums[4]) if len(nums) > 4 and nums[4].isdigit() else 0
            current = {'codes': [cells[1]], 'names': [cells[2]],
                       'nums': nums, 'places_general': places}
            kg_map[kg] = current
        elif len(cells) == 2 and current is not None \
                and re.match(r'\d{2}\.\d{2}\.\d{2}', cells[0]):
            current['codes'].append(cells[0])
            current['names'].append(cells[1])
    return live_date, kg_map


def subject_label_gubkin(names):
    """Подпись слота по множеству названий предметов (физика приоритетнее)."""
    joined = ' '.join(names).lower()
    if 'физик' in joined:
        return 'Физ.'
    if 'информатик' in joined or 'информационные технологии' in joined:
        return 'Инф.'
    if 'математик' in joined or 'вычислен' in joined:
        return 'Мат.'
    if 'русск' in joined:
        return 'Рус.'
    if 'хими' in joined:
        return 'Хим.'
    if 'общество' in joined:
        return 'Обществ.'
    for name in names:
        if name:
            return name.split()[0][:8]
    return '—'


def _to_int_gubkin(value, default=0):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def parse_group_gubkin(prog, is_target):
    """Парсит одну КГ: тип «Основные места в рамках КЦП» -> список общего конкурса.

    Места общего конкурса — из табло, КАК ЕСТЬ (по ТЗ). Строки ниже порога
    (по сумме с ИД) не сохраняются. Если у КГ нет типа «Основные места» —
    общего конкурса нет, возвращаем None.
    """
    types = api_gubkin({'method': 'getEducationTypes', 'contestGroupId': prog['id']}) or []
    main_type = next((t for t in types
                      if 'основные места' in str(t.get('name', '')).lower()), None)
    if main_type is None:
        return None

    rows_raw = api_gubkin({'method': 'get', 'educationTypeId': main_type['id'],
                           'contestGroupId': prog['id']}) or []
    if not rows_raw:
        return None

    # Слоты предметов: ballsBySubjects.priority = номер слота. Подписи — по всем
    # строкам (физика видна, даже если конкретный абитуриент сдавал информатику).
    slot_names = collections.defaultdict(set)
    for row in rows_raw:
        for subject in row.get('ballsBySubjects') or []:
            slot_names[_to_int_gubkin(subject.get('priority'), 0)].add(
                str(subject.get('name') or ''))
    slot_ids = sorted(k for k in slot_names if k > 0)
    slot_labels = {k: subject_label_gubkin(slot_names[k]) for k in slot_ids}
    # Порог ПО КГ (ТЗ): если среди предметов слотов есть физика — пусть даже как
    # альтернатива информатике внутри одного слота — порог 277. Порог 269 только
    # у групп, где ВИ — исключительно информатика.
    all_subject_names = ' '.join(n for names in slot_names.values() for n in names).lower()
    threshold = 277 if 'физик' in all_subject_names else 269

    rows = []
    for row in rows_raw:
        code = str(row.get('fio') or '').strip()
        if not code:
            continue
        total = _to_int_gubkin(row.get('totalBalls'))

        # Балл слота = максимум ненулевых (он и входит в сумму, если сданы оба).
        slot_balls = collections.defaultdict(list)
        for subject in row.get('ballsBySubjects') or []:
            ball = subject.get('ball')
            if ball is None:
                continue
            slot = _to_int_gubkin(subject.get('priority'), 0)
            slot_balls[slot].append(_to_int_gubkin(ball))

        if total < threshold:
            continue

        marks = ', '.join(
            f'{slot_labels[k]}: {max(slot_balls[k])}'
            for k in slot_ids if slot_balls.get(k)
        ) or '—'

        rows.append({
            '№_orig': _to_int_gubkin(row.get('position')),
            'Код': code,
            'Сумма': total,
            'Сумма без ИД': _to_int_gubkin(row.get('entranceBalls')),
            'Оценки': marks,
            'Согласие': 'да' if row.get('enrollmentAgreement') else '',
            'Приоритет': _to_int_gubkin(row.get('priority'), 99),
            'Примечание': str(row.get('benefit') or '').strip(),
        })

    if not rows:
        return None

    df = pd.DataFrame(rows)
    # Дедуп по Коду на всякий случай; сортировка по Сумме (с ИД) по убыванию,
    # тай-брейк по №_orig (порядок сайта) — детерминизм.
    df = (df.sort_values(['Сумма', '№_orig'], ascending=[False, True], kind='stable')
            .drop_duplicates('Код', keep='first')
            .reset_index(drop=True))
    df.insert(0, '№', range(1, len(df) + 1))

    return {
        'df': df,
        'budget': prog['places'],
        'threshold': threshold,
        'form': prog['form'],
        'university': 'РГУ Губкина',
        'competition': prog['display'],
        'is_target': is_target,
        'update_date': prog['live_date'],
    }


# ===== Сбор данных: табло (Москва) + группы API по формам =====
print('Загрузка онлайн-табло РГУ Губкина (г. Москва)...')
live_date_gubkin, kg_map_gubkin = parse_live_gubkin(fetch_live_gubkin())
print(f'Дата табло: {live_date_gubkin}; КГ на табло: {len(kg_map_gubkin)}.')

form_names_gubkin = {f['id']: f['name'] for f in (api_gubkin({'method': 'getForm'}) or [])}
groups_api_gubkin = {}   # short_name без пробелов -> (group, форма)
for form_id in FORM_IDS_GUBKIN:
    faculties = api_gubkin({'method': 'getFaculties', 'educationFormId': form_id}) or []
    if not any(f.get('id') == FACULTY_BAKSPEC_GUBKIN for f in faculties):
        continue
    for group in api_gubkin({'method': 'getGroups', 'educationFormId': form_id,
                             'facultyId': FACULTY_BAKSPEC_GUBKIN}) or []:
        short = str(group.get('short_name', '')).replace(' ', '')
        groups_api_gubkin[short] = (group, form_names_gubkin.get(form_id, str(form_id)))

# Программы для расчёта ВП: КГ есть в табло Москвы И места общ.конкурса > 0
# И группа есть в API бак/спец (очная/очно-заочная/заочная).
all_programs_gubkin = []
for kg, info in kg_map_gubkin.items():
    if info['places_general'] <= 0:
        continue
    if kg not in groups_api_gubkin:
        print(f'  ВНИМАНИЕ: {kg} есть в табло, но не найдена в API — пропуск.')
        continue
    group, form_name = groups_api_gubkin[kg]
    full_name = str(group.get('full_name') or '')
    spec_part = full_name.split(' - ', 1)[1] if ' - ' in full_name else full_name
    profiles = ' | '.join(info['names'])
    display_name = f'{kg}: {spec_part}' + (f' ({profiles})' if profiles else '')
    all_programs_gubkin.append({
        'kg': kg,
        'id': str(group.get('id')),
        'form': form_name,
        'codes': info['codes'],
        'places': info['places_general'],
        'without_exam': _to_int_gubkin(group.get('without_exam')),
        'display': display_name,
        'live_date': live_date_gubkin,
        'nums': tuple(info['nums']),
    })

# Целевые КГ: по точечным кодам ТЗ, в порядке первого упоминания.
target_kgs_gubkin = []
for dotted, name in TARGET_GUBKIN:
    kg_hit = next((p['kg'] for p in all_programs_gubkin if dotted in p['codes']), None)
    if kg_hit is None:
        print(f'  ВНИМАНИЕ: целевая {dotted} «{name[:45]}» не найдена в табло с местами>0.')
        continue
    if kg_hit not in target_kgs_gubkin:
        target_kgs_gubkin.append(kg_hit)
target_ids_gubkin = [p['id'] for kg in target_kgs_gubkin
                     for p in all_programs_gubkin if p['kg'] == kg]
target_names_gubkin = {p['id']: p['display'] for p in all_programs_gubkin
                       if p['kg'] in target_kgs_gubkin}
print(f'Программ (КГ) с местами общего конкурса: {len(all_programs_gubkin)}; '
      f'целевых КГ: {len(target_kgs_gubkin)} (из {len(TARGET_GUBKIN)} строк ТЗ).')

for prog in all_programs_gubkin:
    if prog['id'] in target_ids_gubkin and prog['without_exam']:
        print(f'  ВНИМАНИЕ: у {prog["kg"]} without_exam={prog["without_exam"]} '
              f'(зачисляемые без ВИ) — места табло взяты как есть, проверь у бро.')

# Сигнатура табло: дата + все числовые колонки всех КГ (меняются при обновлении).
programs_signature_gubkin = (live_date_gubkin,
                             tuple(sorted((p['kg'], p['nums']) for p in all_programs_gubkin)))

cached_gubkin = None
if os.path.exists(CACHE_GUBKIN):
    with open(CACHE_GUBKIN, 'rb') as f:
        cached_gubkin = pickle.load(f)

# Страховка по возрасту кэша: на табло Губкина дата БЕЗ времени, а числовые
# колонки — план, квоты и принятые заявления; согласий там нет. После закрытия
# приёма (25.07) счётчики замрут, и в пределах суток кэш вообще не обновится,
# хотя согласия меняются ежедневно. Поэтому старше TTL — перекачиваем всегда.
CACHE_TTL_HOURS_GUBKIN = 3
cache_fresh_gubkin = (
    os.path.exists(CACHE_GUBKIN)
    and (time.time() - os.path.getmtime(CACHE_GUBKIN)) < CACHE_TTL_HOURS_GUBKIN * 3600
)

cache_valid_gubkin = (
    cached_gubkin is not None
    and cache_fresh_gubkin
    and cached_gubkin.get('version') == PARSE_VERSION_GUBKIN
    and cached_gubkin.get('signature') == programs_signature_gubkin
    and cached_gubkin.get('data')
)
if cached_gubkin is not None and not cache_fresh_gubkin:
    print(f'Кэш старше {CACHE_TTL_HOURS_GUBKIN} ч — перекачиваем списки '
          f'(согласия меняются без изменения табло).')

all_gubkin_data = {}
if cache_valid_gubkin:
    all_gubkin_data = cached_gubkin['data']
    print('Табло не изменилось — используем кэш.')
else:
    failed_gubkin = []
    for idx, prog in enumerate(all_programs_gubkin, 1):
        is_target = prog['id'] in target_ids_gubkin
        tag = 'ЦЕЛЕВАЯ' if is_target else 'общая'
        try:
            data = parse_group_gubkin(prog, is_target)
        except Exception as error:
            print(f'  [{idx}/{len(all_programs_gubkin)}] Ошибка {prog["kg"]}: {error}')
            failed_gubkin.append(str(prog['kg']))
            continue
        if data is None:
            continue
        all_gubkin_data[prog['id']] = data
        print(f'  [{idx}/{len(all_programs_gubkin)}] {tag}: {prog["kg"]} '
              f'{prog["codes"][0]}... [{prog["form"]}] мест общ.конкурса={data["budget"]}, '
              f'порог={data["threshold"]}, строк={len(data["df"])}')

    # Кэш пишем ТОЛЬКО при полной загрузке (см. пояснение в блоке МТУСИ).
    if all_gubkin_data and not failed_gubkin:
        with open(CACHE_GUBKIN, 'wb') as f:
            pickle.dump({
                'version': PARSE_VERSION_GUBKIN,
                'signature': programs_signature_gubkin,
                'data': all_gubkin_data,
            }, f)
        print('Данные сохранены в кэш.')
    elif failed_gubkin:
        print(f'  ВНИМАНИЕ: не загрузились группы ({len(failed_gubkin)}): '
              f'{"; ".join(failed_gubkin[:3])} — кэш НЕ сохранён, '
              f'работаем на неполных данных.')

if not all_gubkin_data:
    raise SystemExit('Нет данных ни по одной конкурсной группе РГУ Губкина.')

loaded_targets_gubkin = [pid for pid in target_ids_gubkin if pid in all_gubkin_data]
print(f'Загружено КГ: {len(all_gubkin_data)}; '
      f'целевых с данными: {len(loaded_targets_gubkin)}/{len(target_ids_gubkin)}.')

# ===== Расчёт Основного ВП (отложенная приёмка по ВСЕМ конкурсным группам) =====
# Кандидат-предлагающая deferred acceptance, как в мтуси_блок3.py / политех_блок4.py /
# миит_блок5.py: абитуриент садится на КГ с наивысшим приоритетом, где реально
# проходит по баллам после ухода конкурентов (эффект вытеснения).
prog_rank_gubkin = {}      # id -> {Код: позиция в списке, меньше = сильнее}
prog_budget_gubkin = {}    # id -> места общего конкурса (из табло)
cand_prefs_gubkin = {}     # Код -> id КГ по возрастанию приоритета
for key, data in all_gubkin_data.items():
    df = data['df']
    prog_budget_gubkin[key] = data['budget']
    prog_rank_gubkin[key] = {code: i for i, code in enumerate(df['Код'])}
    for code, priority in zip(df['Код'], df['Приоритет']):
        cand_prefs_gubkin.setdefault(code, []).append((int(priority), key))
for code, pairs in cand_prefs_gubkin.items():
    cand_prefs_gubkin[code] = [key for _, key in sorted(pairs, key=lambda x: x[0])]

admitted_gubkin = {key: {} for key in all_gubkin_data}   # id -> {Код: ранг}
next_choice_gubkin = {code: 0 for code in cand_prefs_gubkin}
pending_gubkin = collections.deque(cand_prefs_gubkin)

while pending_gubkin:
    code = pending_gubkin.popleft()
    prefs = cand_prefs_gubkin[code]
    while next_choice_gubkin[code] < len(prefs):
        key = prefs[next_choice_gubkin[code]]
        budget = prog_budget_gubkin[key]
        if budget <= 0:
            next_choice_gubkin[code] += 1
            continue
        rank = prog_rank_gubkin[key][code]
        held = admitted_gubkin[key]
        if len(held) < budget:
            held[code] = rank
            break
        worst_code = max(held, key=held.get)   # держим слабейшего (наибольший ранг)
        if rank < held[worst_code]:
            del held[worst_code]
            held[code] = rank
            next_choice_gubkin[worst_code] += 1
            pending_gubkin.append(worst_code)   # вытесненный ищет следующий приоритет
            break
        next_choice_gubkin[code] += 1           # здесь не прошёл — следующий приоритет

enrolled_gubkin = {}
for key, data in all_gubkin_data.items():
    admitted_codes = set(admitted_gubkin[key])
    data['df']['Основной ВП'] = (
        data['df']['Код'].isin(admitted_codes).map({True: 'да', False: 'нет'})
    )
    enrolled_gubkin.update((code, True) for code in admitted_codes)

# ===== Свободные места после Основного ВП =====
# «Проходной ВП» своим расчётом получить НЕЛЬЗЯ: после отложенной приёмки у
# программы со свободными местами не остаётся незачисленных кандидатов — тот,
# кого не взяли нигде, дошёл бы до этого места и занял его. Прежний расчёт
# давал «нет» во всех строках и создавал ложное впечатление второй проверки.
# Полезен сам остаток: места не разобраны теми, кто выше порога, значит
# проходной балл программы НИЖЕ нашего порога отбора.
free_places_gubkin = {}
for key, data in all_gubkin_data.items():
    df = data['df']
    df['Проходной ВП'] = 'нет'
    free_places_gubkin[key] = max(
        0, data['budget'] - int((df['Основной ВП'] == 'да').sum()))

# ===== Отдельные таблицы по конкурсным группам (только целевые, порядок ТЗ) =====
display(Markdown('# БЛОК 1: РГУ ГУБКИНА'))
for key in target_ids_gubkin:
    data = all_gubkin_data.get(key)
    if data is None:
        display(Markdown(f'### {target_names_gubkin.get(key, key)} — нет данных'))
        print('-' * 80)
        continue
    df = data['df']
    count_main = (df['Основной ВП'] == 'да').sum()
    count_pass = (df['Проходной ВП'] == 'да').sum()

    display(Markdown(f"### {data['competition']} ({data['form']})"))
    display(Markdown(
        f"**Мест общего конкурса (табло):** {data['budget']}  |  "
        f"**Порог отбора:** {data['threshold']}  |  **Всего строк:** {len(df)}"
    ))
    display(Markdown(
        f'**Основной ВП (да):** {count_main}  |  '
        f'**Мест не занято абитуриентами выше порога:** '
        f'{free_places_gubkin.get(key, 0)} (если больше нуля — проходной балл ниже нашего порога отбора)'
    ))
    display_cols = ['№', '№_orig', 'Код', 'Сумма', 'Сумма без ИД', 'Оценки', 'Согласие',
                    'Приоритет', 'Основной ВП', 'Проходной ВП', 'Примечание']
    display(df[[c for c in display_cols if c in df.columns]].style.hide(axis='index'))
    print('-' * 80)

print(f'\nВремя работы БЛОКА 1 (РГУ Губкина): {time.time() - start_time_gubkin:.1f} сек.')
print('Данные для сводной готовы: all_gubkin_data, target_ids_gubkin — '
      'см. «БЛОК 2: СВОДНАЯ ТАБЛИЦА РГУ ГУБКИНА».')

In [ ]:
# -*- coding: utf-8 -*-
# ===== БЛОК 1: СТАНКИН =====
# Образцы — МТУСИ, Политех, МИИТ, Губкин, МЭИ. Тот же принцип: «Основной ВП» считаем
# по ВСЕМ программам вуза, удовлетворяющим условиям, таблицы — по целевым.
#
# Источники (Bitrix, Selenium не нужен):
#   1. Места: https://priem.stankin.ru/bakalavriatispetsialitet/training_programs/ —
#      таблица программ, колонка «Бюджетные места на 2026 г.» с подколонками
#      Всего/Особая/Отдельная/Целевая квоты. Места общего конкурса =
#      Всего − Особая − Отдельная − Целевая (по ТЗ). Колонка «Предметы»
#      (Р + М + И, Р + М + И/Ф, ...) даёт порог: физика есть -> 277, иначе 269
#      (у всех целевых «Р + М + И» -> 269, подтверждено ТЗ).
#   2. Сверка мест: POST /local/apps/pk.lists/kcp.php с параметрами фильтра ->
#      {"KTSP": N} — места на выбранный конкурс со страницы списков. Проверено
#      17.07.2026: совпадает с расчётом по таблице; при расхождении — ВНИМАНИЕ
#      и берём расчёт по таблице (первичный по ТЗ).
#   3. Списки: страница /bakalavriatispetsialitet/ranked-lists/ грузит грид айфреймом
#      GET https://priem.stankin.ru/gridspisokpostupayushchikh?<параметры формы>
#      (найдено в JS страницы). Параметры: PROPERTY_388=Бюджетная основа,
#      PROPERTY_389=1 - Очная, PROPERTY_394=<код название программы>,
#      PROPERTY_402='-' (Основной бюджет), LIST_TYPE=ranked, EDU_LEVEL=bs,
#      PROPERTY_418=Прием на обучение на бакалавриат/специалитет, apply_filter=Y,
#      PROPERTY_584=ready + пустые PROPERTY_396/710/410. Пагинация — параметр
#      «PAGEN_1=N» ИЛИ «lists_list_elements_91=page-N»: вуз переключает их между
#      собой (22.07.2026 работал только именованный, 24.07.2026 — снова только
#      PAGEN_1), а нерабочий на ЛЮБОЙ номер отдаёт первую страницу, из-за чего
#      списки молча обрезались до 50 строк. Поэтому рабочий параметр определяем
#      на старте опытным путём (detect_page_param_stankin) и переключаемся на
#      лету, если он перестал листать посреди закачки. По 50 строк на страницу,
#      размер берём с первой страницы. Список отсортирован по сумме с ИД по
#      убыванию до конца — страницы качаем, пока последняя строка >= порога.
#
# Колонки грида (17, последняя пустая служебная): №, Уникальный код, Приоритет,
# Высший проходной приоритет, Основной высший приоритет, Согласие, Общежитие,
# Статус заявления, Сумма с ИД, ИД, Сумма без ИД, Профильный предмет, Математика,
# Русский, Преимущ. право ч.9, ч.10. Галочки «✓» переводим в «да» (ТЗ).
# Строки со статусом «Отозвано» в конкурсе не участвуют — исключаем.
#
# «Основной ВП» = сайтовая галочка «Основной высший приоритет» (по ТЗ): сайт
# считает по ВСЕМ конкурсным спискам (включая квоты и платное), т.е. видит уход
# людей на квоты, который нам недоступен — его расчёт полнее нашего. Для контроля
# всё равно считаем свой Основной ВП (deferred acceptance по основным бюджетным
# спискам, как у всех вузов) и печатаем сверку с сайтом (проверено 18.07.2026:
# совпадение ~97%, расхождения объяснимы квотами). Проходной ВП — наш расчёт от
# свободных мест = места − сайтовые «да».
#
# Кэш: строка «Данные обновлены ...» на гриде (одна на все списки) + состав
# программ с местами.

import time
import os
import re
import pickle
import collections
import threading
from concurrent.futures import ThreadPoolExecutor
import requests
import urllib3
import pandas as pd
from IPython.display import display, Markdown

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

start_time_stankin = time.time()

RANKED_URL_STANKIN = 'https://priem.stankin.ru/bakalavriatispetsialitet/ranked-lists/'
PROGRAMS_URL_STANKIN = 'https://priem.stankin.ru/bakalavriatispetsialitet/training_programs/'
GRID_URL_STANKIN = 'https://priem.stankin.ru/gridspisokpostupayushchikh'
KCP_URL_STANKIN = 'https://priem.stankin.ru/local/apps/pk.lists/kcp.php'
HTTP_HEADERS_STANKIN = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'),
    'Accept-Language': 'ru-RU,ru;q=0.9',
    'Referer': RANKED_URL_STANKIN,
}
CACHE_STANKIN = 'stankin_cache.pkl'
# Версия схемы парсинга. Инкрементируй при изменении логики parse_program_stankin —
# иначе при неизменившейся дате из кэша подтянутся старые данные.
# v1: места = Всего − квоты (таблица training_programs) со сверкой kcp.php; грид
#     постранично до порога (сортировка по сумме подтверждена), «Отозвано»
#     исключаем, порог 277/269 по колонке «Предметы», галочки -> «да»,
#     Основной ВП = deferred acceptance + сверка с сайтовой галочкой.
# v2: места дополнительно минус БВИ-олимпиадники (сайтовая галочка стоит, сумма
#     ниже порога, предметные баллы нули — место занято, но порог их скрывает);
#     «Проходной ВП» больше НЕ считается: сайтовая колонка пуста, а свой расчёт
#     давал ложные «да» (свободные места считались по обрезанному порогом списку).
# v3: пагинация переведена на lists_list_elements_91=page-N (сайт перестал
#     понимать PAGEN_1 и отдавал первую страницу на любой номер); обрыв закачки
#     по «страница целиком ниже порога» вместо «последняя строка ниже порога»
#     (порядок строк не всегда убывающий); БВИ считаются по уникальным кодам.
# v4: параметр пагинации больше не константа — определяем опытным путём (сайт
#     24.07.2026 вернулся к PAGEN_1 и перестал понимать именованный параметр,
#     списки снова резались до 50 строк). Плюс: обрыв из-за «залипшей»
#     пагинации теперь ОШИБКА программы (кэш не сохраняется), а не тихий break
#     с печатью — обрезанный список уходил в кэш, сводную и модель шансов.
PARSE_VERSION_STANKIN = 4
MAX_PAGES_STANKIN = 80   # предохранитель пагинации
# Кандидаты в параметр страницы: «PAGEN_1=N» (стандартная навигация Bitrix) и
# «lists_list_elements_91=page-N» (именованная навигация этого грида). Рабочий
# выбирает detect_page_param_stankin: нерабочий не даёт ошибки, а молча отдаёт
# первую страницу, поэтому проверяем именно СМЕНУ первой строки на стр. 2.
PAGE_PARAMS_STANKIN = ('PAGEN_1', 'lists_list_elements_91')
PAGE_PARAM_STANKIN = PAGE_PARAMS_STANKIN[0]   # уточняется на старте

# --- Целевые программы из ТЗ (stankin.txt) — матчим по коду (первый токен опции). ---
TARGET_STANKIN = [
    ('09.03.01', 'Информатика и вычислительная техника'),
    ('09.03.01.01', 'Разработка программных комплексов (ТОП ИТ)'),
    ('09.03.02', 'Информационные системы и технологии'),
    ('09.03.02.01', 'Разработка и внедрение корпоративных информационных систем (ТОП ИТ)'),
    ('09.03.03.01', 'Прикладная информатика (математическое и компьютерное моделирование процессов и систем)'),
    ('09.03.03.02', 'Прикладная информатика (управление данными)'),
    ('09.03.04', 'Программная инженерия'),
]

# Программы качаем в несколько потоков: страница грида весит ~640 КБ и отдаётся
# 20-30 секунд (сайт под нагрузкой перед 27.07), последовательный прогон всех
# программ занимал ~40 минут. Внутри программы страницы по-прежнему идут по
# очереди — следующая нужна только если предыдущая не ушла ниже порога.
STANKIN_WORKERS = 5
# requests.Session на поток: одна общая сессия под многопоточностью роняет
# соединения из пула (ConnectionError/«Connection aborted»).
_thread_local_stankin = threading.local()
_print_lock_stankin = threading.Lock()
_page_param_lock_stankin = threading.Lock()


def session_stankin():
    """Сессия текущего потока (создаётся при первом обращении)."""
    session = getattr(_thread_local_stankin, 'session', None)
    if session is None:
        session = requests.Session()
        session.headers.update(HTTP_HEADERS_STANKIN)
        _thread_local_stankin.session = session
    return session


def log_stankin(message):
    """Печать из потоков без перемешивания строк."""
    with _print_lock_stankin:
        print(message)


# 23.07.2026: сайт СТАНКИНа отваливался по ConnectTimeout. Ждём дольше и
# отдельно ограничиваем установление соединения: мёртвый хост виден за 15 с, а
# не за 120, и остаётся время на повторы.
RETRY_DELAYS_STANKIN = (3, 6, 12, 24, 40, 60)
TIMEOUT_STANKIN = (15, 120)  # (соединение, чтение)


def _retry_stankin(what, url, attempt, retries, error):
    """Пауза перед повтором; True — пробуем ещё, False — попытки кончились."""
    if attempt == retries - 1:
        return False
    pause = RETRY_DELAYS_STANKIN[min(attempt, len(RETRY_DELAYS_STANKIN) - 1)]
    log_stankin(f'  СТАНКИН не ответил на {what} {url} ({type(error).__name__}: {error}). '
                f'Повтор {attempt + 2}/{retries} через {pause} с...')
    time.sleep(pause)
    return True


def fetch_stankin(url, params=None, retries=len(RETRY_DELAYS_STANKIN) + 1):
    """GET с повторами (экспоненциальный бэкофф до ~2.5 мин)."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session_stankin().get(url, params=params,
                                             timeout=TIMEOUT_STANKIN, verify=False)
            response.raise_for_status()
            response.encoding = 'utf-8'
            return response.text
        except Exception as error:
            last_error = error
            if not _retry_stankin('GET', url, attempt, retries, error):
                break
    raise RuntimeError(f'Не удалось загрузить {url}: {last_error}')


def _clean_stankin(html_fragment):
    txt = re.sub(r'<[^>]+>', ' ', html_fragment).replace('&nbsp;', ' ')
    return re.sub(r'\s+', ' ', txt).strip()


def _to_int_stankin(value, default=0):
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return default


def grid_params_stankin(program_option):
    """Параметры фильтра грида/kcp.php для одной программы (основной бюджет, очная)."""
    return {
        'PROPERTY_388': 'Бюджетная основа',
        'PROPERTY_389': '1 - Очная',
        'PROPERTY_394': program_option,
        'PROPERTY_402': '-',
        'PROPERTY_396': '',
        'PROPERTY_710': '',
        'PROPERTY_410': '',
        'PROPERTY_747': '-',
        'apply_filter': 'Y',
        'PROPERTY_584': 'ready',
        'LIST_TYPE': 'ranked',
        'EDU_LEVEL': 'bs',
        'PROPERTY_418': 'Прием на обучение на бакалавриат/специалитет',
    }


def parse_program_options_stankin(ranked_html):
    """Опции селекта «Направление подготовки» (PROPERTY_394) со страницы списков."""
    select_match = re.search(r'<select name="PROPERTY_394".*?</select>', ranked_html, re.S)
    if select_match is None:
        raise RuntimeError('СТАНКИН: не найден селект PROPERTY_394 на странице списков.')
    options = re.findall(r'<option[^>]*value="([^"]+)"', select_match.group(0))
    return [o.strip() for o in options if o.strip()]


def parse_places_stankin(programs_html):
    """training_programs -> {код: {'name', 'form', 'subjects', 'total', 'quotas',
    'places'}} только для очной формы. places = Всего − Особая − Отдельная − Целевая."""
    places = {}
    for tbl in re.findall(r'<table[^>]*>.*?</table>', programs_html, re.S):
        if '09.03.01' not in tbl:
            continue
        for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', tbl, re.S):
            cells = [_clean_stankin(c) for c in re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', tr, re.S)]
            if len(cells) < 15 or not re.match(r'\d{2}\.\d{2}\.\d{2}', cells[0]):
                continue
            if cells[1].strip().lower() != 'очная':
                continue
            code = cells[0].split()[0]
            total = _to_int_stankin(cells[10])
            quotas = sum(_to_int_stankin(cells[i]) for i in (11, 12, 13))
            places[code] = {
                'name': cells[0],
                'form': cells[1],
                'subjects': cells[2],
                'total': total,
                'quotas': quotas,
                'places': total - quotas,
            }
        break
    if not places:
        raise RuntimeError('СТАНКИН: таблица мест не распарсилась (training_programs).')
    return places


def fetch_kcp_stankin(program_option, retries=len(RETRY_DELAYS_STANKIN) + 1):
    """POST kcp.php -> KTSP (места на выбранный конкурс) или None."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session_stankin().post(KCP_URL_STANKIN,
                                              data=grid_params_stankin(program_option),
                                              timeout=TIMEOUT_STANKIN, verify=False)
            response.raise_for_status()
            ktsp = response.json().get('KTSP')
            return None if ktsp is None else _to_int_stankin(ktsp, None)
        except Exception as error:
            last_error = error
            if not _retry_stankin('POST', KCP_URL_STANKIN, attempt, retries, error):
                break
    log_stankin(f'  ВНИМАНИЕ: kcp.php не ответил для «{program_option[:45]}»: {last_error}')
    return None


def parse_grid_rows_stankin(page_html):
    """Строки данных грида: список ячеек (>=16, первая — сквозной №)."""
    tables = re.findall(r'<table[^>]*>.*?</table>', page_html, re.S)
    if not tables:
        return []
    rows = []
    for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', tables[0], re.S):
        cells = [_clean_stankin(c) for c in re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', tr, re.S)]
        if cells and cells[0].isdigit() and len(cells) >= 16:
            rows.append(cells)
    return rows


def _page_value_stankin(page_param, page):
    """Значение параметра страницы: PAGEN_1 ждёт «N», именованная навигация — «page-N»."""
    return str(page) if page_param.startswith('PAGEN') else f'page-{page}'


def fetch_grid_page_stankin(base_params, page_param, page):
    """Одна страница грида указанным параметром пагинации."""
    return fetch_stankin(GRID_URL_STANKIN,
                         params=dict(base_params,
                                     **{page_param: _page_value_stankin(page_param, page)}))


def detect_page_param_stankin(programs):
    """Ищет параметр пагинации, который сайт реально понимает.

    Нерабочий параметр не даёт ошибки — грид отдаёт первую страницу на любой
    номер, и список молча обрезается до 50 строк (так было 22.07.2026 с PAGEN_1
    и 24.07.2026 с lists_list_elements_91). Признак рабочего: на стр. 2 меняется
    сквозной № первой строки. Проверяем на первой программе, у которой первая
    страница заполнена целиком (иначе второй страницы просто нет).
    """
    for prog in programs:
        base = grid_params_stankin(prog['option'])
        first_rows = parse_grid_rows_stankin(
            fetch_grid_page_stankin(base, PAGE_PARAMS_STANKIN[0], 1))
        if len(first_rows) < 50:
            continue
        marker = first_rows[0][0]
        for page_param in PAGE_PARAMS_STANKIN:
            second_rows = parse_grid_rows_stankin(
                fetch_grid_page_stankin(base, page_param, 2))
            if second_rows and second_rows[0][0] != marker:
                print(f'Параметр пагинации грида: «{page_param}» '
                      f'(проверено на {prog["code"]}: стр.2 начинается с № {second_rows[0][0]}).')
                return page_param
        raise RuntimeError(
            'СТАНКИН: ни один известный параметр пагинации не листает грид '
            f'({", ".join(PAGE_PARAMS_STANKIN)}) — списки обрежутся до 50 строк. '
            'Открой страницу списков и посмотри имя параметра в ссылках навигации.')
    print('ВНИМАНИЕ: ни у одной программы нет второй страницы — пагинацию проверить не на чем.')
    return PAGE_PARAMS_STANKIN[0]


def parse_program_stankin(prog, is_target):
    """Качает грид программы постранично до порога, собирает DataFrame.

    Список отсортирован по сумме с ИД по убыванию — останавливаемся, когда
    последняя строка страницы ниже порога. «Отозвано» исключаем (конкурсу не
    участник), №_orig — сквозной номер сайта.
    """
    threshold = 277 if 'ф' in prog['subjects'].lower() else 269
    label = ('Физ.' if re.match(r'р\s*\+\s*ф', prog['subjects'].lower())
             else 'Инф.' if prog['subjects'].lower().rstrip().endswith('и')
             else 'Проф.')

    global PAGE_PARAM_STANKIN

    params = grid_params_stankin(prog['option'])
    rows_raw, update_date, total_site = [], None, None
    page = 1
    page_size = None
    seen_first_orig = set()
    while page <= MAX_PAGES_STANKIN:
        page_html = fetch_grid_page_stankin(params, PAGE_PARAM_STANKIN, page)
        if update_date is None:
            flat = _clean_stankin(re.sub(r'<script.*?</script>', ' ', page_html, flags=re.S))
            date_match = re.search(r'Данные обновлены\s*([\w\d.: ]+?)(?:\s*\.\w|$)', flat)
            update_date = date_match.group(1).strip() if date_match else None
            total_match = re.search(r'Всего:\s*(\d+)', flat)
            total_site = int(total_match.group(1)) if total_match else None
        page_rows = parse_grid_rows_stankin(page_html)
        if not page_rows:
            break
        # Защита от «залипшей» пагинации: если сайт игнорирует параметр страницы,
        # он вернёт ту же первую страницу, и молча получим обрезанный список
        # вместо полного (проверяем по сквозному № сайта). Пробуем второй
        # известный параметр — вуз их периодически меняет местами; если не
        # листает ни один, это ОШИБКА программы: неполный список не должен
        # попасть ни в таблицы, ни в кэш.
        first_orig = page_rows[0][0]
        if first_orig in seen_first_orig:
            switched = False
            for alt_param in PAGE_PARAMS_STANKIN:
                if alt_param == PAGE_PARAM_STANKIN:
                    continue
                alt_rows = parse_grid_rows_stankin(
                    fetch_grid_page_stankin(params, alt_param, page))
                if alt_rows and alt_rows[0][0] not in seen_first_orig:
                    log_stankin(f'  {prog["code"]}: «{PAGE_PARAM_STANKIN}» перестал листать — '
                                f'переключаюсь на «{alt_param}».')
                    with _page_param_lock_stankin:
                        PAGE_PARAM_STANKIN = alt_param
                    page_rows = alt_rows
                    first_orig = page_rows[0][0]
                    switched = True
                    break
            if not switched:
                raise RuntimeError(
                    f'страница {page} повторяет предыдущую — сайт игнорирует все '
                    f'известные параметры пагинации ({", ".join(PAGE_PARAMS_STANKIN)}), '
                    f'список был бы обрезан до {len(rows_raw)} строк')
        seen_first_orig.add(first_orig)
        rows_raw.extend(page_rows)
        # Размер страницы берём с первой страницы, а не константой: сменят его —
        # прежний код оборвал бы закачку на первой же странице.
        if page_size is None:
            page_size = len(page_rows)
        # Обрыв — когда страница ЦЕЛИКОМ ниже порога, а не когда ниже порога её
        # последняя строка. Порядок строк у СТАНКИНа не всегда строго убывающий
        # (сверху идут БВИ с суммой из одних ИД, а под нагрузкой сайт отдавал
        # список и вовсе неотсортированным) — прежнее условие обрывало закачку
        # на первой же странице, и в таблицу попадали 49 строк вместо 237.
        if max(_to_int_stankin(r[8]) for r in page_rows) < threshold:
            break
        if len(page_rows) < page_size:
            break
        page += 1

    # БВИ-олимпиадники стоят первыми строками с суммой = только ИД (предметные
    # баллы нули) и УЖЕ имеют сайтовую галочку «Основной высший приоритет», то
    # есть место занято. Порог их выбрасывает, поэтому места надо уменьшить на
    # их число — иначе конкурс на программе выглядит просторнее, чем есть
    # (проверено: 09.03.04, код 1730575, сумма 10 при предметах 0/0/0).
    # Считаем по УНИКАЛЬНЫМ кодам: строки могут повториться, если сайт отдаст
    # одну и ту же страницу дважды, и тогда вычет из мест был бы кратно завышен.
    bvi_codes = {
        cells[1] for cells in rows_raw
        if cells[7] != 'Отозвано' and '✓' in cells[4]
        and _to_int_stankin(cells[8]) < threshold
        and not any(cells[i].isdigit() and _to_int_stankin(cells[i]) > 0
                    for i in (11, 12, 13))
    }
    bvi_taken = len(bvi_codes)

    rows = []
    for cells in rows_raw:
        total = _to_int_stankin(cells[8])
        if total < threshold:
            continue
        status = cells[7]
        if status == 'Отозвано':
            continue
        marks = ', '.join(
            f'{name}: {cells[i]}'
            for name, i in ((label, 11), ('Мат.', 12), ('Рус.', 13)) if cells[i].isdigit()
        ) or '—'
        note_parts = []
        if '✓' in cells[14] or '✓' in cells[15]:
            note_parts.append('преим. право')
        if status and status != 'Подано':
            note_parts.append(status)
        rows.append({
            '№_orig': _to_int_stankin(cells[0]),
            'Код': cells[1],
            'Сумма': total,
            'Сумма без ИД': _to_int_stankin(cells[10]),
            'Оценки': marks,
            'Согласие': 'да' if '✓' in cells[5] else '',
            'Приоритет': _to_int_stankin(cells[2], 99),
            'Осн.ВП сайт': 'да' if '✓' in cells[4] else 'нет',
            'Примечание': '; '.join(note_parts),
        })

    if not rows:
        return None

    df = pd.DataFrame(rows)
    # Дедуп по Коду на всякий случай; сортировка по Сумме (с ИД) по убыванию,
    # тай-брейк по №_orig (порядок сайта) — детерминизм.
    df = (df.sort_values(['Сумма', '№_orig'], ascending=[False, True], kind='stable')
            .drop_duplicates('Код', keep='first')
            .reset_index(drop=True))
    df.insert(0, '№', range(1, len(df) + 1))

    return {
        'df': df,
        'budget': max(0, prog['places'] - bvi_taken),
        'places_site': prog['places'],
        'bvi_taken': bvi_taken,
        'total_site': total_site,
        'threshold': threshold,
        'form': 'Очная',
        'university': 'СТАНКИН',
        'competition': prog['option'],
        'is_target': is_target,
        'update_date': update_date,
    }


# ===== Сбор данных: опции программ + таблица мест + сверка kcp =====
print('Загрузка страницы списков и таблицы мест СТАНКИНа...')
program_options_stankin = parse_program_options_stankin(fetch_stankin(RANKED_URL_STANKIN))
places_stankin = parse_places_stankin(fetch_stankin(PROGRAMS_URL_STANKIN))
print(f'Опций «Направление подготовки»: {len(program_options_stankin)}; '
      f'очных строк в таблице мест: {len(places_stankin)}.')

all_programs_stankin = []
for option in program_options_stankin:
    code = option.split()[0]
    place_row = places_stankin.get(code)
    if place_row is None:
        print(f'  ВНИМАНИЕ: «{option[:55]}» нет в таблице мест (очная) — пропуск.')
        continue
    if place_row['places'] <= 0:
        print(f'  ВНИМАНИЕ: «{option[:55]}» мест общего конкурса '
              f'{place_row["places"]} — пропуск.')
        continue
    all_programs_stankin.append({
        'code': code,
        'option': option,
        'subjects': place_row['subjects'],
        'places': place_row['places'],
        'total': place_row['total'],
        'quotas': place_row['quotas'],
    })

target_ids_stankin = []
for code, tz_name in TARGET_STANKIN:
    prog_hit = next((p for p in all_programs_stankin if p['code'] == code), None)
    if prog_hit is None:
        print(f'  ВНИМАНИЕ: целевая {code} «{tz_name[:45]}» не найдена среди программ с местами.')
        continue
    if prog_hit['option'] not in target_ids_stankin:
        target_ids_stankin.append(prog_hit['option'])
target_names_stankin = {p['option']: p['option'] for p in all_programs_stankin
                        if p['option'] in target_ids_stankin}
print(f'Программ с местами общего конкурса: {len(all_programs_stankin)}; '
      f'целевых: {len(target_ids_stankin)} (из {len(TARGET_STANKIN)} строк ТЗ).')

# Дата актуальности — с первой страницы грида первой программы (одна на все списки).
first_page_html_stankin = fetch_grid_page_stankin(
    grid_params_stankin(all_programs_stankin[0]['option']), PAGE_PARAM_STANKIN, 1)
flat_stankin = _clean_stankin(re.sub(r'<script.*?</script>', ' ', first_page_html_stankin, flags=re.S))
date_match_stankin = re.search(r'Данные обновлены\s*([\w\d.: ]+?)(?:\s*\.\w|$)', flat_stankin)
update_date_stankin = date_match_stankin.group(1).strip() if date_match_stankin else None
print(f'Данные списков СТАНКИНа: {update_date_stankin}')

programs_signature_stankin = (
    update_date_stankin,
    tuple(sorted((p['option'], p['places']) for p in all_programs_stankin)),
)

cached_stankin = None
if os.path.exists(CACHE_STANKIN):
    with open(CACHE_STANKIN, 'rb') as f:
        cached_stankin = pickle.load(f)

cache_valid_stankin = (
    cached_stankin is not None
    and cached_stankin.get('version') == PARSE_VERSION_STANKIN
    and cached_stankin.get('signature') == programs_signature_stankin
    and cached_stankin.get('data')
)

all_stankin_data = {}
if cache_valid_stankin:
    all_stankin_data = cached_stankin['data']
    print('Дата списков не изменилась — используем кэш.')
else:
    # Проверяем пагинацию ДО закачки — иначе нерабочий параметр тихо обрежет
    # каждый список до первой страницы (50 строк).
    PAGE_PARAM_STANKIN = detect_page_param_stankin(all_programs_stankin)
    failed_stankin = []

    def load_program_stankin(idx, prog):
        """Одна программа целиком: сверка мест + постраничная закачка списка."""
        is_target = prog['option'] in target_ids_stankin
        tag = 'ЦЕЛЕВАЯ' if is_target else 'общая'
        # Сверка мест: kcp.php (страница списков) против таблицы (Всего − квоты).
        ktsp = fetch_kcp_stankin(prog['option'])
        if ktsp is not None and ktsp != prog['places']:
            log_stankin(f'  ВНИМАНИЕ: {prog["code"]} kcp.php даёт {ktsp} мест, таблица '
                        f'{prog["places"]} ({prog["total"]}−{prog["quotas"]}) — берём таблицу (ТЗ).')
        try:
            data = parse_program_stankin(prog, is_target)
        except Exception as error:
            log_stankin(f'  [{idx}/{len(all_programs_stankin)}] Ошибка {prog["code"]}: {error}')
            return prog, None, str(prog['code'])
        if data is not None:
            log_stankin(f'  [{idx}/{len(all_programs_stankin)}] {tag}: {prog["option"][:50]} '
                        f'[{data["form"]}] мест общ.конкурса={data["budget"]} '
                        f'({prog["total"]}−{prog["quotas"]}, kcp={ktsp}), порог={data["threshold"]}, '
                        f'строк={len(data["df"])} (в списке {data["total_site"]})')
        return prog, data, None

    # Целевые идут в очередь первыми: их списки самые длинные (7-9 страниц против
    # 1-2 у общих), и если они попадут в хвост, прогон упрётся в них в одиночку.
    queue_stankin = sorted(enumerate(all_programs_stankin, 1),
                           key=lambda item: item[1]['option'] not in target_ids_stankin)
    print(f'Качаю {len(queue_stankin)} программ в {STANKIN_WORKERS} потоков '
          f'(страница грида отдаётся 20-30 сек)...')
    loaded_stankin = {}
    with ThreadPoolExecutor(max_workers=STANKIN_WORKERS) as pool_stankin:
        for prog, data, failed_code in pool_stankin.map(
                lambda item: load_program_stankin(*item), queue_stankin):
            if failed_code is not None:
                failed_stankin.append(failed_code)
            elif data is not None:
                loaded_stankin[prog['option']] = data
    # Порядок программ — как в таблице мест, а не как отработали потоки.
    for prog in all_programs_stankin:
        if prog['option'] in loaded_stankin:
            all_stankin_data[prog['option']] = loaded_stankin[prog['option']]

    # Кэш пишем ТОЛЬКО при полной загрузке (см. пояснение в блоке МТУСИ).
    if all_stankin_data and not failed_stankin:
        with open(CACHE_STANKIN, 'wb') as f:
            pickle.dump({
                'version': PARSE_VERSION_STANKIN,
                'signature': programs_signature_stankin,
                'data': all_stankin_data,
            }, f)
        print('Данные сохранены в кэш.')
    elif failed_stankin:
        print(f'  ВНИМАНИЕ: не загрузились программы ({len(failed_stankin)}): '
              f'{"; ".join(failed_stankin[:3])} — кэш НЕ сохранён, '
              f'работаем на неполных данных.')

if not all_stankin_data:
    raise SystemExit('Нет данных ни по одной программе СТАНКИНа.')

loaded_targets_stankin = [pid for pid in target_ids_stankin if pid in all_stankin_data]
print(f'Загружено программ: {len(all_stankin_data)}; '
      f'целевых с данными: {len(loaded_targets_stankin)}/{len(target_ids_stankin)}.')

# ===== Контрольный расчёт Основного ВП (отложенная приёмка по ВСЕМ программам) =====
# Кандидат-предлагающая deferred acceptance, как в мтуси_блок3.py ... мэи_блок7.py:
# абитуриент садится на программу с наивысшим приоритетом, где реально проходит
# по баллам после ухода конкурентов (эффект вытеснения). Для СТАНКИНа это
# КОНТРОЛЬНЫЙ расчёт (сверка с сайтом) — в таблицы идёт сайтовая галочка (ТЗ).
prog_rank_stankin = {}      # option -> {Код: позиция в списке, меньше = сильнее}
prog_budget_stankin = {}    # option -> места общего конкурса (Всего − квоты)
cand_prefs_stankin = {}     # Код -> options по возрастанию приоритета
for key, data in all_stankin_data.items():
    df = data['df']
    prog_budget_stankin[key] = data['budget']
    prog_rank_stankin[key] = {code: i for i, code in enumerate(df['Код'])}
    for code, priority in zip(df['Код'], df['Приоритет']):
        cand_prefs_stankin.setdefault(code, []).append((int(priority), key))
for code, pairs in cand_prefs_stankin.items():
    cand_prefs_stankin[code] = [key for _, key in sorted(pairs, key=lambda x: x[0])]

admitted_stankin = {key: {} for key in all_stankin_data}   # option -> {Код: ранг}
next_choice_stankin = {code: 0 for code in cand_prefs_stankin}
pending_stankin = collections.deque(cand_prefs_stankin)

while pending_stankin:
    code = pending_stankin.popleft()
    prefs = cand_prefs_stankin[code]
    while next_choice_stankin[code] < len(prefs):
        key = prefs[next_choice_stankin[code]]
        budget = prog_budget_stankin[key]
        if budget <= 0:
            next_choice_stankin[code] += 1
            continue
        rank = prog_rank_stankin[key][code]
        held = admitted_stankin[key]
        if len(held) < budget:
            held[code] = rank
            break
        worst_code = max(held, key=held.get)   # держим слабейшего (наибольший ранг)
        if rank < held[worst_code]:
            del held[worst_code]
            held[code] = rank
            next_choice_stankin[worst_code] += 1
            pending_stankin.append(worst_code)   # вытесненный ищет следующий приоритет
            break
        next_choice_stankin[code] += 1           # здесь не прошёл — следующий приоритет

# «Основной ВП» в таблицах = сайтовая галочка (ТЗ); наш DA — контрольная колонка.
enrolled_stankin = {}
for key, data in all_stankin_data.items():
    df = data['df']
    admitted_codes = set(admitted_stankin[key])
    df['ВП расч.'] = df['Код'].isin(admitted_codes).map({True: 'да', False: 'нет'})
    df['Основной ВП'] = df['Осн.ВП сайт']
    enrolled_stankin.update((code, True) for code in df.loc[df['Основной ВП'] == 'да', 'Код'])
    site_yes = (df['Основной ВП'] == 'да').sum()
    if site_yes > data['budget']:
        print(f'  ВНИМАНИЕ: {key.split()[0]} сайтовых «да» ({site_yes}) больше мест '
              f'({data["budget"]}) — проверь у бро.')

# ===== Сверка контрольного расчёта с сайтовой галочкой =====
match_stankin = mismatch_stankin = 0
mismatch_examples_stankin = []
for key, data in all_stankin_data.items():
    df = data['df']
    for ours, site, code in zip(df['ВП расч.'], df['Осн.ВП сайт'], df['Код']):
        if ours == site:
            match_stankin += 1
        else:
            mismatch_stankin += 1
            if len(mismatch_examples_stankin) < 8:
                mismatch_examples_stankin.append(f'{code}@{key.split()[0]} расч={ours} сайт={site}')
total_checked_stankin = match_stankin + mismatch_stankin
print(f'\nСверка контрольного DA с сайтовым «Основной высший приоритет»: совпало '
      f'{match_stankin}/{total_checked_stankin} '
      f'({100.0 * match_stankin / max(1, total_checked_stankin):.1f}%), '
      f'расхождений {mismatch_stankin}.')
if mismatch_examples_stankin:
    print('  Примеры расхождений (сайт считает по ВСЕМ спискам, включая квоты и '
          f'платное, мы — по основному бюджету): {"; ".join(mismatch_examples_stankin)}')

# ===== Проходной ВП: у СТАНКИНа его нет =====
# Сайтовая колонка «Высший проходной приоритет» (ячейка 3) пуста у всех строк —
# вуз её не заполняет. Свой расчёт давал ЛОЖНЫЕ «да»: свободные места считались
# как budget − сайтовые «да» в НАШЕМ df, а df обрезан порогом, тогда как сайт
# раздаёт места и тем, кто ниже (на 09.03.02 при 53 местах галочек выше порога
# 21 и ниже порога 32 — ровно 53, свободных нет, а код видел 32 и раздавал их
# незачисленным). Оставляем «нет» и печатаем честный остаток мест в шапке.
free_places_stankin = {}
for key, data in all_stankin_data.items():
    df = data['df']
    df['Проходной ВП'] = 'нет'
    free_places_stankin[key] = max(
        0, data['budget'] - int((df['Основной ВП'] == 'да').sum()))

# ===== Отдельные таблицы по программам (только целевые, порядок ТЗ) =====
display(Markdown('# БЛОК 1: СТАНКИН'))
for key in target_ids_stankin:
    data = all_stankin_data.get(key)
    if data is None:
        display(Markdown(f'### {target_names_stankin.get(key, key)} — нет данных'))
        print('-' * 80)
        continue
    df = data['df']
    count_main = (df['Основной ВП'] == 'да').sum()
    count_pass = (df['Проходной ВП'] == 'да').sum()

    display(Markdown(f"### {data['competition']} ({data['form']})"))
    bvi_note = f" − БВИ {data['bvi_taken']}" if data.get('bvi_taken') else ''
    display(Markdown(
        f"**Мест общего конкурса (Всего − квоты{bvi_note}):** {data['budget']}  |  "
        f"**Порог отбора:** {data['threshold']}  |  **Всего строк:** {len(df)}"
    ))
    display(Markdown(
        f'**Основной ВП (да):** {count_main}  |  '
        f'**Мест не занято абитуриентами выше порога:** {free_places_stankin.get(key, 0)} '
        f'(их займут те, кто ниже порога — сайт «Проходной ВП» не публикует)'
    ))
    display_cols = ['№', '№_orig', 'Код', 'Сумма', 'Сумма без ИД', 'Оценки', 'Согласие',
                    'Приоритет', 'Основной ВП', 'Проходной ВП', 'Примечание']
    display(df[[c for c in display_cols if c in df.columns]].style.hide(axis='index'))
    print('-' * 80)

print(f'\nВремя работы БЛОКА 1 (СТАНКИН): {time.time() - start_time_stankin:.1f} сек.')
print('Данные для сводной готовы: all_stankin_data, target_ids_stankin — '
      'см. «БЛОК 2: СВОДНАЯ ТАБЛИЦА СТАНКИН».')

In [ ]:
# -*- coding: utf-8 -*-
# ===== БЛОК 1: ФИНУНИВЕРСИТЕТ =====
# Образцы — МТУСИ, Политех, МИИТ, Губкин, МЭИ, СТАНКИН, МИРЭА. По ТЗ finka.txt ВП
# НЕ считаем: в списках финки есть сайтовые колонки «Основной высший приоритет» и
# «Высший проходной приоритет» — берём их как есть («Основной ВП»/«Проходной ВП»).
#
# Источники (Selenium не нужен):
#   1. Каталог программ: htmx-выдача фильтра страницы educational-programs/bachelor
#      GET /ajax/educational-programs.php?iblockId=42&block=86718&faculty[]=1&
#          branch[]=10&budgetProgram=true&page=N
#      (faculty 1 = Факультет информационных технологий и анализа больших данных,
#      branch 10 = Москва, budgetProgram = «Программы с наличием бюджетных мест:
#      Да» — фильтры из ТЗ). Карточка education-card: название программы, код
#      направления, «Бюджетных мест N», форма обучения, ссылка на деталку.
#   2. Предметы слотов П1/П2/П3 и порог: страница «Перечень и программы
#      вступительных испытаний» /for-applicants/bachelor/ispitaniya/ — таблица
#      «Приоритетность вступительных испытаний» по кодам направлений. Баллы в
#      списках идут в порядке приоритетности: у всех целевых П1 = Математика,
#      П2 = Русский язык, П3 = Информатика ИЛИ Физика (какой из двух сдал
#      конкретный человек — сайт не раскрывает; слот с физикой подписываем «Физ.»
#      всем, как в остальных вузах, ТЗ).
#      Порядок ПОДТВЕРЖДЁН сверкой множеств баллов слотов со шкалами предметов
#      из данных МЭИ: в П2 есть баллы 81/83/91, существующие только у русского;
#      П3 содержит 79/83/93 (нет у математики) и целиком вписывается в
#      информатику; П1 — не русский. Порог: физика в слотах -> 271, иначе 263.
#      Пороги финки ниже обычных 277/269: ИД здесь дают максимум 4 балла
#      (медаль 3 + значок ГТО 1, ТЗ): 267+4 и 259+4.
#   3. PDF «Количество мест ...» со страницы /for-applicants/bachelor/control/
#      (путь к PDF динамический — ищем ссылку по тексту). В PDF секции
#      «Бакалавриат (<форма> форма обучения, г. Москва)»; секции «... с
#      применением ДОТ», «Специалитет (...)» и филиалы (с «<Город> филиал
#      Финуниверситета») отсекаются по заголовкам. Колонки: всего мест | в т.ч.
#      отдельная квота | в т.ч. особая квота | в т.ч. детализированная целевая
#      квота | наименование целевой организации | договорных мест. У программы
#      может быть несколько целевых организаций — строки-продолжения без кода
#      направления, целевую квоту суммируем. Места общего конкурса =
#      бюджетные места - отдельная - особая - целевая - БВИ-олимпиадники
#      (платные не трогаем, ТЗ); про БВИ — см. п.5.
#   4. Конкурсные списки: applications.php — обёртка над iframe
#      /spiski/listabit.php?itype_list=бкл. Таблица серверная, интерактивные
#      фильтры колонок = GET-параметры (имена = data-field инпутов):
#      facultet_m=<факультет>, facultet=<конкурсная группа>, type_conkurs=Общий
#      конкурс, form_pay=Бюджет, country=РОССИЯ (значения фильтров — ТЗ),
#      page_size=100 (максимум селектора), ordering=-summ_mark,unique_code —
#      сортировка по сумме с ИД по убыванию (проверено: числовая) -> страницы
#      качаем только до порога. Тай-брейк unique_code ОБЯЗАТЕЛЕН: без него
#      серверная сортировка нестабильна на равных суммах и пагинация ТЕРЯЕТ
#      строки (проверено: 4 из 114 пропадали даже при полном обходе).
#      Словари значений фильтров: ?action=get_dict&field=<поле>.
#   5. БВИ-олимпиадники: в списках финки это ОТДЕЛЬНЫЙ вид конкурса
#      «Победители и призеры профильных олимпиад» (в «Общем конкурсе» строк
#      «Без ВИ» нет). Зачисляются на основные места ВНЕ квот, поэтому их
#      сайтовые «Основной высший приоритет» = да вычитаем из мест общего
#      конкурса. Сверено на 10.03.01 очная: бюджетных 80 − квоты 21 − 3 БВИ
#      («да») = 56 — ровно столько «да» сайт отдал общему конкурсу.
#
# Строка списка разбирается ПО ИМЕНАМ ПОЛЕЙ, а не по номерам ячеек: у каждой
# ячейки есть машинный маркер headers="column-<поле>", те же имена стоят в
# id ячеек шапки (<th id="column-<поле>">) и в именах GET-фильтров. Поля:
# number_order (№ по всему списку факультета — НЕ ранг внутри группы),
# unique_code, form_pay, facultet_m, facultet, priority, is_basic_priority,
# is_hight_priority, type_conkurs, bvi, summ_mark («303.00»),
# bak_mark_school_discipline («П1 (О) 100, П2 (Е) 97, ...»; Е=ЕГЭ,
# О=олимпиада, Э=внутр. экзамен; П1..П3 переводим в предметы по таблице
# приоритетности — см. п.2), bak_personal_success (общие ИД),
# bak_personal_target_success (целевые ИД), bak_privilege_right_part_9/_10,
# bak_privilege_right_equality, is_contract, is_agreement, country.
# 29.07.2026 вуз перекроил таблицу: добавил колонку «№» в начало и перенёс
# «Финансирование» из хвоста на 3-е место (19 ячеек -> 20, всё после
# финансирования уехало на +2). Прежний позиционный разбор (код в ячейке 0,
# сумма в ячейке 8) перестал видеть строки, и блок падал с «нет строк выше
# порога» по ВСЕМ группам. Отсюда правило: привязка только к именам полей.
#
# Кэш: в шапке выдачи списка «Обновлено: ДД.ММ.ГГГГ ЧЧ:ММ:СС». Сигнатура: дата
# выдачи первой целевой группы + состав целевых групп с местами и порогами.

import io
import time
import os
import re
import pickle
import requests
import urllib3
import pandas as pd

try:
    from IPython.display import display, Markdown
except ImportError:      # запуск обычным python (регрессия разбора, отладка блока)
    def display(*args, **kwargs):
        for arg in args:
            print(arg)

    def Markdown(text):  # noqa: N802 — имя как в IPython
        return text

try:
    import pdfplumber
except ImportError:
    # Установка «на лету» (чистый Jupyter/Colab). После pip в РАБОТАЮЩЕМ ядре
    # надо сбросить кэш импортёра (importlib.invalidate_caches), а на Windows
    # pip мог поставить пакет в user-site, которого ещё нет в sys.path ядра.
    import subprocess
    import sys
    import importlib
    import site
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pdfplumber'])
    importlib.invalidate_caches()
    try:
        import pdfplumber
    except ImportError:
        if hasattr(site, 'getusersitepackages'):
            site.addsitedir(site.getusersitepackages())
        importlib.invalidate_caches()
        try:
            import pdfplumber
        except ImportError:
            raise SystemExit(
                'pdfplumber установлен, но ядро его не видит. Выполни в отдельной '
                'ячейке:  %pip install pdfplumber  — затем перезапусти ядро '
                '(Kernel -> Restart Kernel) и запусти блок заново.'
            )

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

start_time_fin = time.time()

BASE_FIN = 'https://www.fa.ru'
CATALOG_AJAX_FIN = BASE_FIN + '/ajax/educational-programs.php'
CONTROL_URL_FIN = BASE_FIN + '/for-applicants/bachelor/control/'
LIST_URL_FIN = BASE_FIN + '/spiski/listabit.php'
FACULTY_FIN = 'Факультет информационных технологий и анализа больших данных'
HTTP_HEADERS_FIN = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'),
    'Accept-Language': 'ru-RU,ru;q=0.9',
    'Referer': BASE_FIN + '/for-applicants/bachelor/Rateabit/applications.php',
}
CACHE_FIN = 'finka_cache.pkl'
# Версия схемы парсинга. Инкрементируй при изменении логики разбора — иначе при
# неизменившейся дате из кэша подтянутся старые данные.
# v1: каталог (ФИТиАБД, Москва, бюджет) + PDF КЦП (места общ.конкурса = бюджетные
#     - отдельная - особая - целевая) + списки listabit (Общий конкурс, Бюджет,
#     РОССИЯ, сортировка -summ_mark,unique_code до порога); ВП — сайтовые
#     колонки (ТЗ); порог 271/263 по физике с детальной страницы программы.
# v2: слоты П1/П2/П3 в «Оценках» переведены в предметы (Мат./Рус./Инф./Физ.) по
#     таблице приоритетности ВИ со страницы ispitaniya (метрики проекта, ТЗ
#     finka2); порог тоже считается по этой таблице, деталки программ не нужны.
# v3: (ТЗ finka_zapros) целевые группы — ТОЛЬКО очная форма (очно-заочные и
#     заочные не визуализируем); места общего конкурса дополнительно минус
#     БВИ-олимпиадники с сайтовым «Основной высший приоритет» = да (вид
#     конкурса «Победители и призеры профильных олимпиад» зачисляется на
#     основные места вне квот: 10.03.01 очная — 59 − 3 = 56 сайтовых «да»).
# v4: слот «Инф./Физ.» подписывается «Физ.» (ТЗ: если физика есть среди ВИ, в
#     «Оценках» пишем Физ. всем, даже тем, кто сдавал информатику); порог по
#     программе прежний — 271 там, где физика в ВИ есть, 263 только у программ
#     с исключительно информатикой.
# v5: строки списка разбираются по именам полей (headers="column-<поле>", в
#     фоллбэке — порядок <th id="column-<поле>"> из шапки) вместо номеров
#     ячеек — вуз 29.07.2026 вставил колонку «№» и передвинул «Финансирование»;
#     пустая таблица при живой шапке больше не считается «нет строк выше
#     порога», а поднимает ошибку (группа идёт в неудачные, кэш не пишется).
# v6 (04.08.2026): каталог закрыл кампанию. Фильтр `budgetProgram=true` перестал
#     отдавать что-либо (ответ 159 символов, 0 карточек), а строка «Бюджетных
#     мест» исчезла из карточек вовсе — блок находил 0 целевых из 5 и падал
#     SystemExit'ом «нет ни одной целевой конкурсной группы». Фильтр убран, число
#     мест берётся из PDF КЦП (он живой, разбирается тем же блоком, 51 программа
#     бакалавриата Москвы), квоты вычитаются из него же. Если каталог когда-нибудь
#     вернёт бюджет — снова считаем от каталожного числа, как раньше.
PARSE_VERSION_FIN = 6

THRESHOLD_PHYS_FIN = 271   # есть физика: 267 + 4 балла ИД финки (ТЗ)
THRESHOLD_INF_FIN = 263    # только информатика: 259 + 4 (ТЗ)

# --- Целевые программы из ТЗ (finka.txt): (код направления, название). ---
# «Инженер-разработчик» на сайте — «Инженер-разработчик программного
# обеспечения», матчим по префиксу. Берём ТОЛЬКО очную форму (ТЗ finka_zapros:
# очно-заочные и заочные версии программ не визуализируем — у 09.03.03 есть
# очно-заочная конкурсная группа, она отсекается здесь фильтром формы).
TARGET_FIN = [
    ('10.03.01', 'Безопасность автоматизированных систем в кредитно-финансовой сфере'),
    ('09.03.04', 'Разработка и внедрение информационно - аналитических систем'),
    ('01.03.02', 'Прикладное машинное обучение'),
    ('09.03.03', 'Прикладные информационные системы в экономике и финансах'),
    ('09.03.04', 'Инженер-разработчик'),
]
TARGET_FORM_FIN = 'Очная'

session_fin = requests.Session()
session_fin.headers.update(HTTP_HEADERS_FIN)


# 23.07.2026: сайты вузов в горячую фазу выпадают на минуту-две (502, таймаут
# соединения). Ждём до ~2.5 мин вместо прежних 12 с; connect отдельно от read.
RETRY_DELAYS_FIN = (3, 6, 12, 24, 40, 60)
TIMEOUT_FIN = (15, 90)  # (соединение, чтение)


def _retry_fin(url, attempt, retries, error):
    """Пауза перед повтором; True — пробуем ещё, False — попытки кончились."""
    if attempt == retries - 1:
        return False
    pause = RETRY_DELAYS_FIN[min(attempt, len(RETRY_DELAYS_FIN) - 1)]
    print(f'  Финуниверситет не ответил на {url} ({type(error).__name__}: {error}). '
          f'Повтор {attempt + 2}/{retries} через {pause} с...')
    time.sleep(pause)
    return True


def fetch_fin(url, params=None, retries=len(RETRY_DELAYS_FIN) + 1, binary=False):
    """GET страницы/файла финки с повторами (экспоненциальный бэкофф)."""
    last_error = None
    for attempt in range(retries):
        try:
            response = session_fin.get(url, params=params,
                                       timeout=TIMEOUT_FIN, verify=False)
            response.raise_for_status()
            if binary:
                return response.content
            response.encoding = 'utf-8'
            return response.text
        except Exception as error:
            last_error = error
            if not _retry_fin(url, attempt, retries, error):
                break
    raise RuntimeError(f'Не удалось загрузить {url}: {last_error}')


def _clean_fin(html_fragment):
    txt = re.sub(r'<[^>]+>', ' ', html_fragment).replace('&nbsp;', ' ')
    txt = txt.replace('&mdash;', '—').replace('&quot;', '"').replace('&amp;', '&')
    return re.sub(r'\s+', ' ', txt).strip()


def _norm_prog_fin(name):
    """Нормализация названия программы: регистр, ё, пробелы вокруг дефисов."""
    norm = name.lower().replace('ё', 'е').replace('–', '-').replace('—', '-')
    norm = re.sub(r'\s*-\s*', '-', norm)
    return re.sub(r'\s+', ' ', norm).strip()


def _num_fin(value, default=0):
    try:
        return int(round(float(str(value).replace(',', '.').strip())))
    except (TypeError, ValueError):
        return default


def get_dict_fin(field, retries=len(RETRY_DELAYS_FIN) + 1, **context):
    """Словарь значений фильтра списка (?action=get_dict&field=...).

    С повторами: раньше единственный сетевой сбой здесь валил весь блок —
    запрос шёл напрямую, минуя fetch_fin.
    """
    params = {'itype_list': 'бкл', 'action': 'get_dict', 'field': field, **context}
    last_error = None
    for attempt in range(retries):
        try:
            response = session_fin.get(LIST_URL_FIN, params=params,
                                       timeout=TIMEOUT_FIN, verify=False)
            response.raise_for_status()
            return response.json().get('values', [])
        except Exception as error:
            last_error = error
            if not _retry_fin(f'{LIST_URL_FIN} (get_dict {field})',
                              attempt, retries, error):
                break
    raise RuntimeError(f'Не удалось получить справочник «{field}»: {last_error}')


def fetch_catalog_cards_fin():
    """Карточки каталога: ФИТиАБД, Москва (фильтры ТЗ).

    Возвращает [{'name', 'code', 'budget', 'form', 'href'}], где budget = None,
    если каталог число мест не показывает.

    04.08.2026, после зачисления вуз закрыл кампанию в каталоге сразу с двух
    сторон: фильтр `budgetProgram=true` перестал отдавать хоть что-то (ответ на
    159 символов, ноль карточек), и строка «Бюджетных мест» пропала из карточек
    вовсе. Прежний код требовал и того, и другого, поэтому находил 0 карточек,
    ни одна целевая не сопоставлялась, и блок падал SystemExit'ом. Теперь фильтр
    не шлём, а число мест берём из PDF КЦП — это официальный документ, он живой
    и разбирается тем же блоком.
    """
    cards = []
    seen_pages = set()
    for page in range(1, 11):
        params = [('iblockId', '42'), ('block', '86718'), ('page', str(page)),
                  ('faculty[]', '1'), ('branch[]', '10')]
        html = fetch_fin(CATALOG_AJAX_FIN, params=params)
        chunks = html.split('<div class="education-card">')[1:]
        if not chunks:
            break
        # За последней страницей сайт по кругу отдаёт первую — ловим повтор,
        # иначе список раздувается дублями и уходит 8 лишних запросов.
        page_signature = hash(html)
        if page_signature in seen_pages:
            break
        seen_pages.add(page_signature)
        for chunk in chunks:
            title_m = re.search(r'<h4 class="education-card__title">(.*?)</h4>', chunk, re.S)
            code_m = re.search(r'Направление подготовки.*?\((\d{2}\.\d{2}\.\d{2})\)',
                               chunk, re.S)
            href_m = re.search(r'href="(/for-applicants/educational-programs/[^"]+)"', chunk)
            budget_m = re.search(r'Бюджетных мест.*?ui-links__link">\s*(\d+)', chunk, re.S)
            form_m = re.search(r'Форма обучения.*?ui-links__link">\s*([^<]+)<', chunk, re.S)
            if not (title_m and code_m and form_m):
                continue
            cards.append({
                'name': _clean_fin(title_m.group(1)),
                'code': code_m.group(1),
                'budget': int(budget_m.group(1)) if budget_m else None,
                'form': _clean_fin(form_m.group(1)),
                'href': href_m.group(1) if href_m else None,
            })
    return cards


SUBJECT_LABELS_FIN = {
    'математика': 'Мат.', 'русский': 'Рус.', 'информатика': 'Инф.',
    'физика': 'Физ.', 'обществознание': 'Общ.', 'история': 'Ист.',
    'иностранный': 'Ин.яз.', 'география': 'Геогр.', 'химия': 'Хим.',
    'биология': 'Биол.', 'литература': 'Лит.',
}


def _subject_label_fin(subject):
    low = subject.lower()
    for key, label in SUBJECT_LABELS_FIN.items():
        if key in low:
            return label
    return subject.split()[0][:8] if subject.strip() else '—'


def fetch_subject_slots_fin():
    """Слоты предметов П1/П2/П3 по кодам направлений (страница ispitaniya).

    Таблица «Приоритетность вступительных испытаний»: баллы в конкурсных
    списках идут в этом порядке (подтверждено сверкой шкал баллов с МЭИ).
    Строка с кодом направления открывает направление, число в первой колонке —
    новый слот (колонка ЕГЭ), строка после «или» — альтернатива к последнему слоту.
    Слот, где среди альтернатив есть физика, подписываем «Физ.» всем (ТЗ: в
    таблицах пишем «Физ.», даже если конкретный абитуриент сдавал информатику);
    остальные альтернативы склеиваются через «/».
    Возвращает {код: ['Мат.', 'Рус.', 'Физ.']}.
    """
    html = fetch_fin(BASE_FIN + '/for-applicants/bachelor/ispitaniya/')
    slots_by_code = {}
    current_slots = None
    alternative_next = False
    for table in re.findall(r'<table[^>]*>.*?</table>', html, re.S):
        if 'Приоритетность' not in table:
            continue
        for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', table, re.S):
            cells = [_clean_fin(c) for c in re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', tr, re.S)]
            if not cells:
                continue
            code_m = re.match(r'(\d{2}\.\d{2}\.\d{2})', cells[0])
            if code_m and len(cells) >= 3:
                current_slots = [[_subject_label_fin(cells[2])]]
                slots_by_code[code_m.group(1)] = current_slots
                alternative_next = False
            elif current_slots is not None and re.fullmatch(r'\d', cells[0]) and len(cells) >= 2:
                current_slots.append([_subject_label_fin(cells[1])])
                alternative_next = False
            elif cells[0].lower() == 'или':
                alternative_next = True
            elif current_slots and alternative_next and cells[0]:
                current_slots[-1].append(_subject_label_fin(cells[0]))
                alternative_next = False
    return {code: [('Физ.' if 'Физ.' in slot else '/'.join(slot)) for slot in slots]
            for code, slots in slots_by_code.items()}


def _page_lines_fin(page):
    """Строки текста страницы PDF с координатой top (для привязки заголовков)."""
    if hasattr(page, 'extract_text_lines'):
        return [{'top': line['top'], 'text': line['text']}
                for line in page.extract_text_lines()]
    lines = {}
    for word in page.extract_words():
        lines.setdefault(round(word['top'] / 3), []).append(word)
    result = []
    for key in sorted(lines):
        words = sorted(lines[key], key=lambda w: w['x0'])
        result.append({'top': words[0]['top'], 'text': ' '.join(w['text'] for w in words)})
    return result


def parse_kcp_pdf_fin(pdf_bytes):
    """PDF КЦП -> квоты бакалавриата Москвы (без ДОТ).

    Возвращает {(форма, норм.название программы): {'code', 'total', 'separate',
    'special', 'target', 'orgs'}}. Заголовки секций и таблицы сопоставляются по
    вертикальной координате (секция может смениться посреди страницы и тянуться
    на следующую); строки-продолжения (доп. целевые организации без кода
    направления) прибавляются к целевой квоте последней программы.
    """
    quotas = {}
    section = ''
    last_key = None
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            events = []
            for line in _page_lines_fin(page):
                text = re.sub(r'\s+', ' ', line['text']).strip()
                header_m = re.match(r'^(Бакалавриат|Специалитет|Магистратура)\s*\((.+?)\)',
                                    text)
                if header_m:
                    events.append((line['top'], 'sec',
                                   f'{header_m.group(1)}|{header_m.group(2)}'))
                elif ('филиал Финуниверситета' in text
                      or 'в филиалах Финансового университета' in text):
                    events.append((line['top'], 'sec', 'филиал'))
            for table in page.find_tables():
                events.append((table.bbox[1], 'tbl', table))

            for _, kind, obj in sorted(events, key=lambda e: e[0]):
                if kind == 'sec':
                    section = obj
                    last_key = None
                    continue
                if not (section.startswith('Бакалавриат|') and 'г. Москва' in section
                        and 'ДОТ' not in section):
                    continue
                form = section.split('|')[1].split(' форма')[0].strip().capitalize()
                for row in obj.extract():
                    cells = [re.sub(r'\s+', ' ', (c or '')).strip() for c in row]
                    if len(cells) < 8:
                        continue
                    code_m = re.search(r'\d{2}\.\d{2}\.\d{2}', cells[0])
                    if code_m:
                        key = (form, _norm_prog_fin(cells[1]))
                        quotas[key] = {
                            'code': code_m.group(0),
                            'total': _num_fin(cells[2]),
                            'separate': _num_fin(cells[3]),
                            'special': _num_fin(cells[4]),
                            'target': _num_fin(cells[5]),
                            'orgs': [cells[6]] if cells[6] else [],
                        }
                        last_key = key
                    elif ('Всего' in cells[0]) or ('Всего' in cells[1]):
                        last_key = None
                    elif (last_key and not cells[0] and not cells[1]
                          and re.fullmatch(r'\d+', cells[5] or '')):
                        quotas[last_key]['target'] += int(cells[5])
                        if cells[6]:
                            quotas[last_key]['orgs'].append(cells[6])
    return quotas


# Имена полей таблицы списка (значение атрибута headers="column-<поле>").
COL_CODE_FIN = 'unique_code'                        # уникальный код поступающего
COL_SUM_FIN = 'summ_mark'                           # сумма конкурсных баллов
COL_GRADES_FIN = 'bak_mark_school_discipline'       # «П1 (Е) 100, П2 (Е) 97, ...»
COL_ID_COMMON_FIN = 'bak_personal_success'          # баллы за общие ИД
COL_ID_TARGET_FIN = 'bak_personal_target_success'   # баллы за целевые ИД
COL_PRIORITY_FIN = 'priority'                       # приоритет по этой группе
COL_MAIN_PRIORITY_FIN = 'is_basic_priority'         # Основной высший приоритет
COL_PASS_PRIORITY_FIN = 'is_hight_priority'         # Высший проходной приоритет
COL_NO_EXAM_FIN = 'bvi'                             # Без ВИ
COL_RIGHT9_FIN = 'bak_privilege_right_part_9'       # преим. право ч.9 ст.71
COL_RIGHT10_FIN = 'bak_privilege_right_part_10'     # преим. право ч.10 ст.71
COL_AGREEMENT_FIN = 'is_agreement'                  # согласие на зачисление

CELL_FIELD_RE_FIN = re.compile(
    r'<td[^>]*\bheaders="column-([a-z0-9_]+)"[^>]*>(.*?)</td>', re.S | re.I)
CELL_ANY_RE_FIN = re.compile(r'<td[^>]*>(.*?)</td>', re.S)
TH_FIELD_RE_FIN = re.compile(r'<th[^>]*\bid="column-([a-z0-9_]+)"', re.I)


def list_column_order_fin(html):
    """Порядок полей таблицы по шапке (<th id="column-<поле>">) — для фоллбэка."""
    thead_m = re.search(r'<thead[^>]*>(.*?)</thead>', html, re.S)
    if thead_m is None:
        return []
    order = []
    for field in TH_FIELD_RE_FIN.findall(thead_m.group(1)):
        if field not in order:
            order.append(field)
    return order


def parse_list_rows_fin(html):
    """Строки выдачи списка -> [{поле: значение}] по именам полей.

    Основной путь — атрибут headers у ячейки; если вуз его уберёт, поля берутся
    по позиции из шапки. Служебная мобильная строка («Данные», одна ячейка с
    colspan) отсеивается требованием кода поступающего.
    """
    tbody_m = re.search(r'<tbody[^>]*>(.*?)</tbody>', html, re.S)
    if tbody_m is None:
        return []
    fallback_order = None
    rows = []
    for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', tbody_m.group(1), re.S):
        cells = {field: _clean_fin(value)
                 for field, value in CELL_FIELD_RE_FIN.findall(tr)}
        if not cells:
            if fallback_order is None:
                fallback_order = list_column_order_fin(html)
            values = [_clean_fin(v) for v in CELL_ANY_RE_FIN.findall(tr)]
            if fallback_order and len(values) == len(fallback_order):
                cells = dict(zip(fallback_order, values))
        if not re.fullmatch(r'\d{5,9}', cells.get(COL_CODE_FIN, '')):
            continue
        rows.append(cells)
    return rows


def list_says_no_data_fin(html):
    """True, если сайт сам сообщает «Обновлено: Нет данных» (фильтр пуст).

    Отличает законно пустую выдачу от сломанного разбора: при живых строках,
    которые мы не смогли прочитать, дата обновления в шапке нормальная.
    """
    date_m = re.search(r'Обновлено:\s*<b>([^<]+)</b>', html)
    return date_m is not None and 'нет данных' in date_m.group(1).lower()


def fetch_list_rows_fin(group_name, threshold, slot_labels):
    """Строки общего конкурса конкурсной группы (бюджет, Россия) до порога.

    Список отсортирован сервером по сумме с ИД по убыванию с тай-брейком по
    уникальному коду (ordering=-summ_mark,unique_code — без тай-брейка пагинация
    теряет строки на равных суммах), поэтому страницы качаются, пока минимальная
    сумма страницы не упадёт ниже порога. Возвращает (строки, дата «Обновлено»,
    всего позиций прочитано).
    """
    rows = []
    position = 0
    update_date = None
    for page in range(1, 61):
        params = {
            'itype_list': 'бкл',
            'facultet_m': FACULTY_FIN,
            'facultet': group_name,
            'type_conkurs': 'Общий конкурс',
            'form_pay': 'Бюджет',
            'country': 'РОССИЯ',
            'ordering': '-summ_mark,unique_code',
            'page_size': '100',
            'page': str(page),
        }
        html = fetch_fin(LIST_URL_FIN, params=params)
        if update_date is None:
            date_m = re.search(r'Обновлено:\s*<b>([^<]+)</b>', html)
            update_date = date_m.group(1).strip() if date_m else None
        page_cells = parse_list_rows_fin(html)
        if not page_cells:
            # Тихий ноль недопустим: пустая первая страница значит либо смену
            # разметки/имён полей, либо что фильтр перестал совпадать со
            # словарём. И то и другое молча обнуляет вуз — поднимаем ошибку.
            if page == 1:
                reason = ('фильтр не выбрал ни одной строки (значения '
                          'facultet/type_conkurs/form_pay/country изменились?)'
                          if list_says_no_data_fin(html) else
                          'таблица есть, но ни одна строка не разобрана '
                          '(вуз сменил имена полей column-<поле>?)')
                raise RuntimeError(
                    f'Список «{group_name[:60]}»: {reason}. Проверь вручную: '
                    f'{LIST_URL_FIN}?itype_list=бкл&facultet={group_name}')
            break
        page_sums = []
        for cells in page_cells:
            position += 1
            total = _num_fin(cells.get(COL_SUM_FIN))
            page_sums.append(total)
            if total < threshold:
                continue
            note_parts = []
            if cells.get(COL_NO_EXAM_FIN, '').lower() == 'да':
                note_parts.append('без ВИ')
            if (cells.get(COL_RIGHT9_FIN, '').lower() == 'да'
                    or cells.get(COL_RIGHT10_FIN, '').lower() == 'да'):
                note_parts.append('преим. право')
            marks = (_num_fin(cells.get(COL_ID_COMMON_FIN))
                     + _num_fin(cells.get(COL_ID_TARGET_FIN)))
            # «П1 (О) 100, П2 (Е) 97, П3 (Е) 98» -> «Мат.: 100, Рус.: 97, ...»
            # (метрики проекта; П-слот = позиция в таблице приоритетности ВИ).
            grades = ', '.join(
                f'{slot_labels[int(n) - 1] if int(n) <= len(slot_labels) else f"ВИ{n}"}: {score}'
                for n, score in re.findall(r'П(\d+)\s*\([^)]*\)\s*(\d+)',
                                           cells.get(COL_GRADES_FIN, ''))
            )
            priority = cells.get(COL_PRIORITY_FIN, '')
            main_vp = cells.get(COL_MAIN_PRIORITY_FIN, '').lower()
            pass_vp = cells.get(COL_PASS_PRIORITY_FIN, '').lower()
            rows.append({
                '№_orig': position,
                'Код': cells[COL_CODE_FIN],
                'Сумма': total,
                'Сумма без ИД': total - marks,
                'Оценки': grades or '—',
                'ИД': marks,
                'Согласие': 'да' if cells.get(COL_AGREEMENT_FIN, '').lower() == 'да' else '',
                'Приоритет': int(priority) if priority.isdigit() else 99,
                'Основной ВП': 'да' if main_vp == 'да' else 'нет',
                'Проходной ВП': 'да' if pass_vp == 'да' else 'нет',
                'Примечание': '; '.join(note_parts),
            })
        if min(page_sums) < threshold:
            break
    return rows, update_date, position


def count_bvi_enrolled_fin(group_name, olymp_kind):
    """Основные места группы, занятые БВИ-олимпиадниками (п.5 шапки).

    БВИ идут отдельным видом конкурса и зачисляются на основные места вне
    квот: считаем строки этого вида (Бюджет, страна любая) с сайтовым
    «Основной высший приоритет» = да — ровно столько мест сайт не отдаёт
    общему конкурсу.
    """
    count = 0
    for page in range(1, 11):
        params = {
            'itype_list': 'бкл',
            'facultet_m': FACULTY_FIN,
            'facultet': group_name,
            'type_conkurs': olymp_kind,
            'form_pay': 'Бюджет',
            'country': 'РОССИЯ',   # тот же фильтр, что в основном списке (ТЗ)
            'ordering': '-summ_mark,unique_code',
            'page_size': '100',
            'page': str(page),
        }
        html = fetch_fin(LIST_URL_FIN, params=params)
        page_cells = parse_list_rows_fin(html)
        if not page_cells:
            # Пустой список олимпиадников законен (есть программы без БВИ), но
            # только если это говорит сам сайт. Живая таблица, из которой ничего
            # не разобралось, завысила бы места общего конкурса — это ошибка.
            if page == 1 and not list_says_no_data_fin(html):
                raise RuntimeError(
                    f'Список БВИ «{group_name[:60]}»: таблица есть, но ни одна '
                    f'строка не разобрана (вуз сменил имена полей column-<поле>?).')
            break
        for cells in page_cells:
            if cells.get(COL_MAIN_PRIORITY_FIN, '').lower() == 'да':
                count += 1
        if len(page_cells) < 100:
            break
    return count


# ===== Каталог: целевые программы (карточки с бюджетом, порядок ТЗ) =====
print('Загрузка каталога программ Финуниверситета (ФИТиАБД, Москва, бюджет)...')
all_cards_fin = fetch_catalog_cards_fin()
with_budget_fin = sum(1 for c in all_cards_fin if c['budget'] is not None)
print(f'Карточек в каталоге: {len(all_cards_fin)}; из них с числом бюджетных мест: '
      f'{with_budget_fin}'
      + (' (кампания закрыта — места берём из PDF КЦП).' if not with_budget_fin
         else '.'))
if not all_cards_fin:
    raise RuntimeError(
        'Финуниверситет: каталог не отдал ни одной карточки. Проверь параметры '
        'ajax/educational-programs.php — 04.08.2026 фильтр budgetProgram=true '
        'перестал отдавать что-либо, и его пришлось убрать.')

target_cards_fin = []
for tz_code, tz_name in TARGET_FIN:
    tz_norm = _norm_prog_fin(tz_name)
    hits = [card for card in all_cards_fin
            if card['code'] == tz_code
            and card['form'] == TARGET_FORM_FIN
            and (_norm_prog_fin(card['name']) == tz_norm
                 or _norm_prog_fin(card['name']).startswith(tz_norm))]
    if not hits:
        print(f'  ВНИМАНИЕ: целевая «{tz_name[:55]}» ({tz_code}, '
              f'{TARGET_FORM_FIN.lower()}) не найдена в каталоге.')
        continue
    for card in hits:
        if all(not (card['name'] == t['name'] and card['form'] == t['form'])
               for t in target_cards_fin):
            target_cards_fin.append(card)
print(f'Целевых конкурсных групп (только {TARGET_FORM_FIN.lower()} форма): '
      f'{len(target_cards_fin)} (из {len(TARGET_FIN)} строк ТЗ).')

# ===== PDF КЦП: квоты -> места общего конкурса =====
print('Загрузка PDF с местами и квотами (страница control)...')
control_html_fin = fetch_fin(CONTROL_URL_FIN)
pdf_link_m = re.search(
    r'<a[^>]*href="([^"]+\.pdf)"[^>]*>(?:(?!</a>).)*?Количество мест для приема',
    control_html_fin, re.S | re.I)
if pdf_link_m is None:
    raise RuntimeError('Страница control: не найдена ссылка на PDF «Количество мест...».')
kcp_quotas_fin = parse_kcp_pdf_fin(fetch_fin(BASE_FIN + pdf_link_m.group(1), binary=True))
print(f'PDF разобран: программ бакалавриата Москвы (без ДОТ): {len(kcp_quotas_fin)}.')

# ===== Конкурсные группы списков (словарь фильтра facultet) =====
groups_dict_fin = get_dict_fin('facultet', facultet_m=FACULTY_FIN)

# ===== Вид конкурса БВИ-олимпиадников (едят основные места вне квот, п.5) =====
kinds_dict_fin = get_dict_fin('type_conkurs', facultet_m=FACULTY_FIN)
olymp_kind_fin = next((k for k in kinds_dict_fin if 'олимпиад' in k.lower()), None)
if olymp_kind_fin is None:
    print('ВНИМАНИЕ: вид конкурса олимпиадников (БВИ) не найден в словаре '
          'type_conkurs — вычет БВИ из мест общего конкурса = 0.')

# ===== Слоты предметов П1/П2/П3 по направлениям (страница ispitaniya) =====
subject_slots_fin = fetch_subject_slots_fin()
print(f'Таблица приоритетности ВИ: {len(subject_slots_fin)} направлений.')

# ===== Сбор параметров по каждой целевой группе =====
target_info_fin = []
for card in target_cards_fin:
    display_name = f"{card['name']} ({card['code']}, {card['form'].lower()})"
    # Конкурсная группа: «Направление, Бакалавр, Программа, Форма».
    group_name = None
    for value in groups_dict_fin:
        parts = [p.strip() for p in value.split(',')]
        if len(parts) < 4 or parts[1] != 'Бакалавр' or parts[-1] != card['form']:
            continue
        if _norm_prog_fin(', '.join(parts[2:-1])) == _norm_prog_fin(card['name']):
            group_name = value
            break
    if group_name is None:
        print(f'  ВНИМАНИЕ: {display_name} — конкурсная группа в списках не найдена, пропуск.')
        continue

    quota = kcp_quotas_fin.get((card['form'], _norm_prog_fin(card['name'])))
    if quota is None:
        print(f'  ВНИМАНИЕ: {display_name} — не найдена в PDF КЦП, квоты считаем 0.')
        quota = {'total': card['budget'] or 0, 'separate': 0, 'special': 0,
                 'target': 0, 'orgs': []}
    # Источник числа мест. Каталог его показывает только пока идёт кампания —
    # после зачисления строка «Бюджетных мест» из карточек исчезает (04.08.2026).
    # Тогда единственный живой источник — PDF КЦП, и вычитать квоты надо из него.
    if card['budget'] is None:
        total_places = quota['total']
    else:
        if quota['total'] != card['budget']:
            print(f'  ВНИМАНИЕ: {display_name} — мест в каталоге {card["budget"]}, '
                  f'в PDF {quota["total"]}; квоты вычитаем из каталожных.')
        total_places = card['budget']
    # Кладём обратно в карточку: дальше это число печатается в логе и в шапке
    # таблицы, а `card['budget']` может быть None (каталог кампанию закрыл).
    card['total_places'] = total_places
    card['places_source'] = 'каталог' if card['budget'] is not None else 'PDF КЦП'
    places = total_places - quota['separate'] - quota['special'] - quota['target']
    if places < 0:
        print(f'  ВНИМАНИЕ: {display_name} — квоты больше бюджетных мест, считаем 0.')
        places = 0
    # Сетевой сбой здесь не должен ронять весь блок до записи кэша — считаем 0
    # и явно предупреждаем (места окажутся завышены на число зачисленных БВИ).
    try:
        bvi_taken = (count_bvi_enrolled_fin(group_name, olymp_kind_fin)
                     if olymp_kind_fin else 0)
    except Exception as error:
        print(f'  ВНИМАНИЕ: {display_name} — не удалось прочитать список БВИ '
              f'({str(error)[:60]}), вычет БВИ = 0, места могут быть завышены.')
        bvi_taken = 0
    if bvi_taken > places:
        print(f'  ВНИМАНИЕ: {display_name} — БВИ «да» ({bvi_taken}) больше мест '
              f'общего конкурса ({places}), считаем 0.')
    places = max(places - bvi_taken, 0)

    slots = subject_slots_fin.get(card['code'])
    if slots is None:
        print(f'  ВНИМАНИЕ: {display_name} — направления {card["code"]} нет в таблице '
              f'приоритетности ВИ, слоты ВИ1..ВИ3, порог {THRESHOLD_PHYS_FIN}.')
        slots = []
    physics = any('Физ.' in slot for slot in slots) if slots else True
    threshold = THRESHOLD_PHYS_FIN if physics else THRESHOLD_INF_FIN
    target_info_fin.append({
        'card': card,
        'display': display_name,
        'group': group_name,
        'places': places,
        'bvi_taken': bvi_taken,
        'quota': quota,
        'threshold': threshold,
        'slots': slots,
    })

if not target_info_fin:
    raise SystemExit('Нет ни одной целевой конкурсной группы Финуниверситета.')

# ===== Кэш: дата «Обновлено» первой группы + состав групп/мест/порогов =====
probe_html_fin = fetch_fin(LIST_URL_FIN, params={
    'itype_list': 'бкл', 'facultet_m': FACULTY_FIN,
    'facultet': target_info_fin[0]['group'], 'type_conkurs': 'Общий конкурс',
    'form_pay': 'Бюджет', 'country': 'РОССИЯ', 'page_size': '10', 'page': '1',
})
probe_date_m = re.search(r'Обновлено:\s*<b>([^<]+)</b>', probe_html_fin)
update_date_fin = probe_date_m.group(1).strip() if probe_date_m else None
print(f'Данные списков Финуниверситета на: {update_date_fin}')

signature_fin = (update_date_fin,
                 tuple((info['group'], info['places'], info['threshold'],
                        tuple(info['slots'])) for info in target_info_fin))

cached_fin = None
if os.path.exists(CACHE_FIN):
    with open(CACHE_FIN, 'rb') as f:
        cached_fin = pickle.load(f)

cache_valid_fin = (
    cached_fin is not None
    and cached_fin.get('version') == PARSE_VERSION_FIN
    and cached_fin.get('signature') == signature_fin
    and cached_fin.get('data')
)

all_fin_data = {}
if cache_valid_fin:
    all_fin_data = cached_fin['data']
    print('Дата списков не изменилась — используем кэш.')
else:
    failed_fin = []
    for idx, info in enumerate(target_info_fin, 1):
        card, quota = info['card'], info['quota']
        try:
            rows, group_date, total_positions = fetch_list_rows_fin(
                info['group'], info['threshold'], info['slots'])
        except Exception as error:
            print(f'  [{idx}/{len(target_info_fin)}] Ошибка {info["display"][:45]}: {error}')
            failed_fin.append(info['display'][:45])
            continue
        if not rows:
            # Число прочитанных позиций отличает «все ниже порога» от «список
            # прочитан пустым»: в июле 2026 блок молчал именно так, когда вуз
            # перекроил колонки и разбор перестал видеть строки.
            print(f'  [{idx}/{len(target_info_fin)}] {info["display"][:45]} — '
                  f'нет строк выше порога {info["threshold"]} '
                  f'(позиций списка прочитано {total_positions}), пропуск.')
            continue
        df = pd.DataFrame(rows)
        # Дедуп по Коду (страховка от сдвига пагинации при обновлении данных);
        # сортировка по Сумме (с ИД) по убыванию, тай-брейк по №_orig.
        df = (df.sort_values(['Сумма', '№_orig'], ascending=[False, True], kind='stable')
                .drop_duplicates('Код', keep='first')
                .reset_index(drop=True))
        df.insert(0, '№', range(1, len(df) + 1))
        # Самопроверка мест: сайтовое «Основной высший приоритет» = да стоит
        # ровно у тех, кого вуз считает проходящими сюда, то есть их число —
        # это места общего конкурса в расчёте самого сайта. Сверяем со своим
        # расчётом (каталог − квоты − БВИ), но только когда проходной балл
        # программы выше нашего порога: иначе часть «да» осталась за обрезкой
        # и сравнивать не с чем. Расхождение обычно значит, что квота
        # разыграна не полностью или зачисленный БВИ уже снят со списка.
        main_yes = df[df['Основной ВП'] == 'да']
        if (len(main_yes) and main_yes['Сумма'].min() > info['threshold']
                and len(main_yes) != info['places']):
            print(f'  ВНИМАНИЕ: {info["display"][:45]} — мест по расчёту '
                  f'{info["places"]}, сайт отдал общему конкурсу {len(main_yes)} '
                  f'(разница {info["places"] - len(main_yes):+d}): проверь квоты '
                  f'в PDF КЦП и уход БВИ из списка.')
        quota_str = (f'отдельная {quota["separate"]}; особая {quota["special"]}; '
                     f'целевая {quota["target"]}')
        all_fin_data[info['group']] = {
            'df': df,
            'budget': info['places'],
            'catalog_places': card['total_places'],
            'places_source': card['places_source'],
            'quota_breakdown': quota_str,
            'quota_orgs': '; '.join(quota['orgs']) if quota['orgs'] else '',
            'bvi_taken': info['bvi_taken'],
            'threshold': info['threshold'],
            'form': card['form'],
            'university': 'Финуниверситет',
            'competition': info['display'],
            'is_target': True,
            'update_date': group_date,
        }
        print(f'  [{idx}/{len(target_info_fin)}] {info["display"][:50]} '
              f'мест общ.конкурса={info["places"]} (бюджетных {card["total_places"]} '
              f'по данным «{card["places_source"]}» - '
              f'[{quota_str}] - БВИ {info["bvi_taken"]}), порог={info["threshold"]}, '
              f'предметы=[{", ".join(info["slots"]) or "ВИ1..ВИ3"}], '
              f'строк={len(df)} (позиций списка прочитано {total_positions})')

    # Кэш пишем ТОЛЬКО при полной загрузке (см. пояснение в блоке МТУСИ).
    if all_fin_data and not failed_fin:
        with open(CACHE_FIN, 'wb') as f:
            pickle.dump({
                'version': PARSE_VERSION_FIN,
                'signature': signature_fin,
                'data': all_fin_data,
            }, f)
        print('Данные сохранены в кэш.')
    elif failed_fin:
        print(f'  ВНИМАНИЕ: не загрузились группы ({len(failed_fin)}): '
              f'{"; ".join(failed_fin[:3])} — кэш НЕ сохранён, '
              f'работаем на неполных данных.')

if not all_fin_data:
    raise SystemExit(
        'Нет данных ни по одной конкурсной группе Финуниверситета. Если выше по '
        'всем группам «позиций списка прочитано 0» — вуз снова перекроил таблицу '
        f'списков или переименовал значения фильтров ({LIST_URL_FIN}): разбор '
        'идёт по именам полей headers="column-<поле>", см. parse_list_rows_fin.')

target_ids_fin = [info['group'] for info in target_info_fin if info['group'] in all_fin_data]
target_names_fin = {info['group']: info['display'] for info in target_info_fin}
print(f'Загружено целевых групп: {len(all_fin_data)}/{len(target_info_fin)}.')

# ===== Отдельные таблицы по конкурсным группам (порядок ТЗ, очная перед о-з) =====
# ВП не считаем — сайтовые колонки «Основной высший приоритет» / «Высший
# проходной приоритет» уже разобраны в «Основной ВП» / «Проходной ВП» (ТЗ).
display(Markdown('# БЛОК 1: ФИНУНИВЕРСИТЕТ'))
for key in target_ids_fin:
    data = all_fin_data[key]
    df = data['df']
    count_main = (df['Основной ВП'] == 'да').sum()
    count_pass = (df['Проходной ВП'] == 'да').sum()

    display(Markdown(f"### {data['competition']}"))
    orgs_note = f"  |  Целевые организации: {data['quota_orgs']}" if data['quota_orgs'] else ''
    bvi_note = f" − БВИ {data['bvi_taken']}" if data['bvi_taken'] else ''
    display(Markdown(
        f"**Мест общего конкурса:** {data['budget']} "
        f"(бюджетных {data['catalog_places']} по данным «{data.get('places_source', '—')}» "
        f"− квоты [{data['quota_breakdown']}]"
        f"{bvi_note})  |  "
        f"**Порог отбора:** {data['threshold']}  |  **Всего строк:** {len(df)}{orgs_note}"
    ))
    display(Markdown(
        f'**Основной ВП (да):** {count_main}  |  **Проходной ВП (да):** {count_pass}  |  '
        f'**Сумма (да):** {count_main + count_pass}'
    ))
    display_cols = ['№', '№_orig', 'Код', 'Сумма', 'Сумма без ИД', 'Оценки', 'ИД',
                    'Согласие', 'Приоритет', 'Основной ВП', 'Проходной ВП', 'Примечание']
    display(df[[c for c in display_cols if c in df.columns]].style.hide(axis='index'))
    print('-' * 80)

print(f'\nВремя работы БЛОКА 1 (Финуниверситет): {time.time() - start_time_fin:.1f} сек.')
print('Данные для сводной готовы: all_fin_data, target_ids_fin — '
      'см. «БЛОК 2: СВОДНАЯ ТАБЛИЦА ФИНУНИВЕРСИТЕТ».')

In [ ]:
# -*- coding: utf-8 -*-
# ===== БЛОК: СОГЛАСИЯ ВНЕШНИХ (НЕЦЕЛЕВЫХ) ВУЗОВ =====
# Запускать ПЕРЕД блоком «ФИНАЛЬНАЯ СТАТИСТИКА». Собирает по внешним московским
# вузам с бюджетным приёмом множества кодов ЕПГУ абитуриентов, подавших СОГЛАСИЕ
# на зачисление (бюджет, бакалавриат/специалитет, головной кампус) ->
# external_consents_by_vuz = {вуз: set(кодов)}. Код ЕПГУ — сквозной, один и тот же
# во всех вузах; шаг 3 финальной статистики удаляет из наших вузов тех, кто отдал
# согласие в чужой вуз.
#
# Каждый чекер check_<slug>() -> set(str цифр). Пути/рецепты выяснены разведкой
# 18-19.07.2026 и проверены живыми запросами. Источники — открытые публичные
# конкурсные списки (JSON API / HTML-таблицы / Excel / PDF), без Selenium.
# Реестр EXTERNAL_CHECKERS расширяемый; недоступные программно вузы — в
# UNSUPPORTED_EXTERNAL с причиной. Сбор кэшируется (external_consents.pkl),
# сетевые сбои деградируют на последнее удачное значение.
#
# ВАЖНО про совпадение кодов: наши целевые вузы идентифицируют абитуриента кодом
# ЕПГУ (6-7 цифр). Вузы, которые в списках дают только СНИЛС (11 цифр, напр.
# Университет Дубна) или иной внутренний ID, в пересечение почти не попадут — их
# согласия по факту не матчатся; это нормально (шаг 3 просто ничего не удалит).

import io
import os
import re
import time
import json
import pickle
import base64
import zipfile
import threading
import urllib.parse
from concurrent.futures import ThreadPoolExecutor

import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

start_time_ext = time.time()


def _data_path(name):
    """Путь к файлу данных (кэш, cookie): рядом со скриптом, в текущей папке или
    на уровень выше — берём первый существующий.

    Блок кладут в подпапку проекта, а кэш и mephi_cookie.txt лежат в корне.
    Раньше искали только по относительному имени, то есть в текущей папке: при
    запуске из подпапки кэш «пропадал», и вуз с сетевой ошибкой (МИФИ 23.07.2026)
    выпадал из сбора совсем, вместо деградации на прошлое значение.
    """
    here = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() \
        else os.getcwd()
    candidates = [os.path.join(here, name),
                  os.path.join(os.getcwd(), name),
                  os.path.join(os.path.dirname(here), name)]
    for path in candidates:
        if os.path.exists(path):
            return path
    return candidates[0]


CACHE_EXT = _data_path('external_consents.pkl')
# 3 часа: в горячую фазу (согласия подают и отзывают до самого зачисления)
# шестичасовой кэш успевал отстать от реальности на полдня.
MAX_AGE_HOURS_EXT = 3
UA = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
      '(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36')
CHECK = '\u2713'   # ✓


def _ensure_module(import_name, pip_name, alt_import=None):
    """Импортирует модуль, при отсутствии ставит его через pip и повторяет.

    Четыре чекера читают не HTML, а файлы: МГТУ ГА — .xls (xlrd), МГТУ им. Баумана
    — PDF (fitz из пакета PyMuPDF), МГГЭУ — PDF (pdfplumber), РГУНХ — .xlsx
    (openpyxl). Без пакета чекер падал с «No module named ...», и вуз молча
    выпадал из сбора (адреса и пути при этом верные — дело только в окружении).

    Грабля Jupyter: после pip в РАБОТАЮЩЕМ ядре надо сбросить кэш импортёра, а на
    Windows пакет мог уехать в user-site, которого ещё нет в sys.path ядра.
    """
    import importlib
    try:
        return importlib.import_module(import_name)
    except ImportError:
        pass
    import subprocess
    import sys
    print(f'    ставлю недостающий пакет {pip_name}...')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
    except Exception as error:
        raise RuntimeError(
            f'не удалось поставить {pip_name} ({str(error)[:60]}); выполни вручную '
            f'в отдельной ячейке: %pip install {pip_name}')
    names = [n for n in (import_name, alt_import) if n]
    for attempt in range(2):
        if attempt:
            import site
            if hasattr(site, 'getusersitepackages'):
                site.addsitedir(site.getusersitepackages())
        importlib.invalidate_caches()
        for name in names:
            try:
                return importlib.import_module(name)
            except ImportError:
                continue
    raise RuntimeError(
        f'{pip_name} установлен, но ядро его не видит. Выполни в отдельной ячейке: '
        f'%pip install {pip_name} — затем перезапусти ядро (Kernel -> Restart Kernel).')


def _session(accept_json=False):
    s = requests.Session()
    hdr = {'User-Agent': UA, 'Accept-Language': 'ru-RU,ru;q=0.9'}
    if accept_json:
        hdr['Accept'] = 'application/json'
    s.headers.update(hdr)
    s.verify = False
    s.trust_env = False
    return s


def _digits(v):
    return re.sub(r'\D', '', str(v or ''))


def _strip(html):
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', html)).replace('&nbsp;', ' ').strip()


# 23.07.2026: сайты выпадают на минуту-две (502, таймаут соединения). Прежних
# 3 попыток по 2/4/6 с (12 секунд) на это не хватало. Здесь бэкофф короче, чем
# в блоках вузов: вузов 23, и каждый лежащий не должен тормозить сбор надолго.
RETRY_DELAYS_EXT = (3, 6, 12, 24)
CONNECT_TIMEOUT_EXT = 15  # мёртвый хост виден за 15 с, а не за полный таймаут


def _timeout_ext(value):
    """(соединение, чтение) — connect отдельно, чтение как просил вызывающий."""
    if isinstance(value, tuple):
        return value
    return (CONNECT_TIMEOUT_EXT, value)


def _retry_ext(what, url, attempt, retries, error):
    """Пауза перед повтором; True — пробуем ещё, False — попытки кончились."""
    if attempt == retries - 1:
        return False
    pause = RETRY_DELAYS_EXT[min(attempt, len(RETRY_DELAYS_EXT) - 1)]
    print('    %s %s не ответил (%s: %s). Повтор %d/%d через %d с...'
          % (what, url[:70], type(error).__name__, error, attempt + 2, retries, pause))
    time.sleep(pause)
    return True


def _get(session, url, retries=len(RETRY_DELAYS_EXT) + 1, **kw):
    # timeout извлекаем ДО цикла: раньше kw.pop внутри цикла терял кастомное
    # значение на повторе и молча откатывал его к 120 с.
    timeout = _timeout_ext(kw.pop('timeout', 120))
    last = None
    for a in range(retries):
        try:
            r = session.get(url, timeout=timeout, **kw)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            if not _retry_ext('GET', url, a, retries, e):
                break
    raise RuntimeError('GET %s: %s' % (url, last))


def _looks_truncated(response):
    """Тело короче объявленного Content-Length — ответ оборвался на полпути.

    Именно так рвётся Плеханов: статус 200, заголовки на месте, а JSON обрывается
    в произвольном месте. Проверять стоит до разбора: сообщение «битый JSON» без
    этой проверки не отличает обрыв канала от смены формата ответа.
    """
    declared = response.headers.get('Content-Length')
    if not declared or response.headers.get('Content-Encoding'):
        return False
    try:
        return len(response.content) < int(declared)
    except (TypeError, ValueError):
        return False


def _rows_missing(response, parsed):
    """Строк в теле меньше, чем объявил Content-Range, — ответ неполный.

    PostgREST (РЭУ) к каждому ответу даёт `Content-Range: 0-159/*` — номера первой
    и последней отданной строки, то есть их точное количество. Само тело идёт
    gzip + chunked, без Content-Length, поэтому обрыв не видят ни requests
    (обрезанный gzip докачивать нечем, остаток декодируется молча), ни
    _looks_truncated. Сверка со счётчиком ловит и обрыв, и подмену ответа пустым
    списком: молча неполный список согласий хуже ошибки — из него абитуриент
    выходит «не подавшим согласие» и остаётся в наших списках.
    """
    header = (response.headers.get('Content-Range') or '').strip()
    match = re.fullmatch(r'(?:items\s+)?(\d+)-(\d+)/(?:\d+|\*)', header)
    if not match or not isinstance(parsed, list):
        return None
    declared = int(match.group(2)) - int(match.group(1)) + 1
    if len(parsed) >= declared:
        return None
    return ('строк %d, а Content-Range обещал %d (ответ неполный)'
            % (len(parsed), declared))


CAPTCHA_MARKERS_EXT = ('не робот', 'not a robot', 'captcha', 'каптча',
                       'введите символы', 'проверка безопасности')


def _looks_like_captcha(response):
    """Ответ — страница проверки «вы не робот», а не данные.

    Отдельная проверка нужна потому, что такая страница приходит со статусом 200
    и телом HTML: разбор JSON падает с «Expecting value: line 1 column 1», и в
    логе это выглядит как «битое тело» — то есть как временный сбой, который
    лечится повтором. 04.08.2026 РЭУ им. Плеханова закрыл капчей весь портал
    abitrating.rea.ru, включая корень, и чекер пять раз подряд ждал впустую 45
    секунд, а потом печатал диагноз, уводящий не туда. Капча повтором не лечится:
    честнее сказать это сразу и уйти на кэш.
    """
    ctype = (response.headers.get('Content-Type') or '').lower()
    if 'json' in ctype:
        return False
    head = response.text[:4000].lower()
    if '<html' not in head and '<!doctype' not in head:
        return False
    return any(marker in head for marker in CAPTCHA_MARKERS_EXT)


def _get_json(session, url, retries=len(RETRY_DELAYS_EXT) + 1, **kw):
    """GET + разбор JSON с повтором на битом/оборванном/неполном теле.

    Сервер иногда отдаёт обрезанный ответ со статусом 200 (Плеханов 22.07.2026:
    «Expecting value: line 2 column 1 (char 2)» — тело оборвалось сразу после
    открывающей скобки; 28-29.07.2026 то же повторилось на /all_competitive_group_
    stats). Одна такая случайность роняла чекер целиком, и вуз выпадал из сбора.
    Повторяем с тем же бэкоффом, что и сетевые сбои. Сетевые сбои ловит _get.
    """
    last_error, body, captcha = None, '', False
    for attempt in range(retries):
        request_kw = dict(kw)
        if attempt:
            # Повтор с теми же заголовками 29.07.2026 пять раз подряд принёс тот же
            # обрыв — так себя ведёт закэшированный шлюзом битый ответ. Просим
            # свежее и без сжатия: тогда тело идёт с Content-Length и обрыв виден
            # сразу, а не превращается в «битый JSON» непонятного происхождения.
            request_kw['headers'] = dict(kw.get('headers') or {},
                                         **{'Accept-Encoding': 'identity',
                                            'Cache-Control': 'no-cache'})
        response = _get(session, url, **request_kw)
        if _looks_like_captcha(response):
            # Капча у РЭУ оказалась ВРЕМЕННОЙ: в одном прогоне 04.08.2026 она
            # висела на всех пяти попытках, а через час тот же чекер собрал 2009
            # согласий за 3 секунды. Значит она включается под нагрузкой, и сдаваться
            # на первой же встрече нельзя — повторяем, как при сетевом сбое. Но
            # называем вещи своими именами: раньше это печаталось как «битое тело»
            # и уводило искать обрыв gzip там, где стоит анти-бот.
            captcha = True
            last_error = 'портал показывает капчу («вы не робот»)'
            body = response.text[:80]
        elif _looks_truncated(response):
            last_error = 'тело короче Content-Length (ответ оборван)'
            body = response.text[:80]
        else:
            try:
                parsed = response.json()
            except ValueError as error:
                last_error, body = error, response.text[:80]
            else:
                incomplete = _rows_missing(response, parsed)
                if not incomplete:
                    return parsed
                last_error, body = incomplete, response.text[:80]
        if attempt == retries - 1:
            break
        pause = RETRY_DELAYS_EXT[min(attempt, len(RETRY_DELAYS_EXT) - 1)]
        print('    %s от %s (%s). Повтор %d/%d через %d с...'
              % ('капча' if captcha else 'битое тело', url[:70], last_error,
                 attempt + 2, retries, pause))
        time.sleep(pause)
        captcha = False          # следующая попытка судится по себе, а не по прошлой
    if captcha:
        raise RuntimeError(
            'портал закрыт капчей («вы не робот»): %s. Данные недоступны, пока она '
            'висит; если держится постоянно — нужен ручной путь, как у МИФИ.'
            % url[:70])
    raise RuntimeError('битый JSON от %s: %s (начало тела: %r)'
                       % (url, last_error, body))


def _post(session, url, retries=len(RETRY_DELAYS_EXT) + 1, **kw):
    timeout = _timeout_ext(kw.pop('timeout', 120))
    last = None
    for a in range(retries):
        try:
            r = session.post(url, timeout=timeout, **kw)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            if not _retry_ext('POST', url, a, retries, e):
                break
    raise RuntimeError('POST %s: %s' % (url, last))


def _rows_cells(html):
    out = []
    for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', html, re.S | re.I):
        cells = [_strip(c) for c in re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', tr, re.S | re.I)]
        if cells:
            out.append(cells)
    return out


# ===== Чекеры: JSON API =====

# Размер страницы ВШЭ. size=5000 их API держит только на мелких группах: на
# крупных (от ~1700 заявлений) он отвечает 500 Internal Server Error, и 23.07.2026
# так выпали 12 самых больших групп — как раз те, где согласий больше всего.
# Тысяча строк за запрос проходит стабильно, дальше листаем страницами.
HSE_PAGE_SIZE = 1000


def _hse_applicants(session, setid, ptid):
    """Все заявления конкурсной группы, страницами по HSE_PAGE_SIZE."""
    out, page = [], 0
    while True:
        payload = _get_json(session, 'https://pk.hse.ru/admissions/api/applicant', params={
            'sort': 'index_number_in_comp_list', 'level': 'BAK',
            'setOfCompetitiveGroupId': setid, 'placeType': ptid,
            'page': page, 'size': HSE_PAGE_SIZE})
        content = payload.get('content', [])
        out.extend(content)
        total_pages = payload.get('totalPages') or 1
        page += 1
        if not content or page >= total_pages:
            return out


def check_hse():
    s = _session(accept_json=True)
    codes = set()
    cat = _get_json(s, 'https://pk.hse.ru/admissions/api/competitve-group')
    msk = next((f for f in cat['filials'] if f['name'] == 'Москва'), None)
    groups = []
    for td in (msk or {}).get('trainingDirections', []):
        for ep in td.get('educationPrograms', []):
            if ep.get('educationLevel', {}).get('name') not in ('Бакалавриат', 'Специалитет'):
                continue
            for cg in ep.get('competitiveGroups', []):
                if cg.get('placeType', {}).get('code') != 'Б':
                    continue
                if not re.search(r'\(О\s', cg.get('name', '')):
                    continue
                groups.append((cg['setOfCompetitiveGroup']['id'], cg['placeType']['id']))
    for setid, ptid in set(groups):
        try:
            for a in _hse_applicants(s, setid, ptid):
                if a.get('isConcertToEnrollment'):
                    code = _digits(a.get('idEpgu'))
                    if code:
                        codes.add(code)
        except Exception as e:
            print(f'    hse: группа {setid} пропущена ({e})')
    return codes


# ===== Университет Дубна =====


def check_dubna():
    s = _session(accept_json=True)
    base = 'https://1c-api.uni-dubna.ru/'
    ct = _get_json(s, base + 'v1/api/admission/applicants/concurrency_types/'
                   'GetBachelorsСoncurrencyTypes')
    def _names(items):
        return {(x.get('name') if isinstance(x, dict) else x) for x in (items or [])}
    targets = {c['id'] for c in ct
               if 'Очная' in _names(c.get('study_types'))
               and 'Бакалавриат' in _names(c.get('degrees'))
               and c.get('budget_level') == 'Федеральный бюджет'}
    data = _get_json(s, base + 'v1/api/admission/applicants/GetAllBachelors')
    codes = set()
    for a in data:
        for sp in a.get('specs', []):
            if sp.get('sponsorship_type', {}).get('name') != 'Бюджетная основа':
                continue
            if sp.get('concurrency_type', {}).get('structureDep') != 'Университет Дубна':
                continue
            if sp.get('concurrency_type', {}).get('id') not in targets:
                continue
            if (sp.get('status') or {}).get('name') == 'Отозвано':
                continue
            if sp.get('enroll_accepted'):
                code = _digits(a.get('snils') or a.get('code1C'))
                if code:
                    codes.add(code)
    return codes


# ===== РЭУ Плеханова =====


def check_rea():
    s = _session(accept_json=True)
    # Просим ответы без сжатия. У них два слоя перед PostgREST (Kong 3.9 -> nginx,
    # признак — `Vary: Accept-Encoding, Accept-Encoding` в ответе), и время от
    # времени наружу уходит обрезанный gzip со статусом 200: requests молча
    # декодирует остаток, JSON рвётся на первой же строке, и весь вуз падал в кэш
    # (22.07 и 29.07.2026). На identity тот же ответ приходит с Content-Length —
    # обрыв виден проверкой, а не гаданием по тексту ошибки.
    s.headers['Accept-Encoding'] = 'identity'
    base = 'https://abitrating.rea.ru/rest/v1'
    key = ('eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlIjoiYW5vbiIsImlzcyI6InN1'
           'cGFiYXNlIiwiaWF0IjoxNzgwNjQxOTU0LCJleHAiOjIwOTYwMDE5NTR9.'
           'HK1E0UwpIPbIHK-C1HtCjoiszflge1Ul8gfD7DPicXQ')
    s.headers.update({'apikey': key, 'Authorization': f'Bearer {key}'})
    branch = 'ФГБОУ ВО "РЭУ им. Г.В. Плеханова"'
    cat = _get_json(s, base + '/all_competitive_group_stats', params={
        'select': 'competitive_group_id',
        'branch_name': f'eq.{branch}',
        'educational_level_name': 'in.(Бакалавриат,Специалитет)',
        'education_form_name': 'eq.Очная',
        'place_type_name': 'in.("Общий конкурс","Особая квота","Отдельная квота",'
                           '"Целевая детализированная квота")',
    })
    if not cat:
        raise RuntimeError('РЭУ: каталог конкурсных групп пуст — сменился формат '
                          'справочника all_competitive_group_stats')
    codes = set()
    failed = []
    for row in cat:
        cgid = row['competitive_group_id']
        try:
            ent = _get_json(s, base + '/entrants', params={
                'select': 'unique_code_profile,agreement',
                'competitive_group_id': f'eq.{cgid}'})
            for e in ent:
                if e.get('agreement'):
                    code = _digits(e.get('unique_code_profile'))
                    if code:
                        codes.add(code)
        except Exception as e:
            failed.append(cgid)
            print(f'    rea: группа {cgid} пропущена ({e})')
    # Пропущенные группы раньше просто печатались, а неполный набор уходил дальше
    # как полный: коды из выпавших групп считались «не подавшими согласие», и мы
    # оставляли их в своих списках. Небольшую потерю (1-2 группы из 154) терпим,
    # заметную — считаем отказом, чтобы сбор честно деградировал на кэш.
    if failed and len(failed) > max(2, len(cat) // 20):
        raise RuntimeError('РЭУ: не отдал %d конкурсных групп из %d — данные неполные'
                           % (len(failed), len(cat)))
    if failed:
        print(f'    ВНИМАНИЕ rea: пропущено групп {len(failed)} из {len(cat)} — '
              f'согласия из них не учтены.')
    return codes


# ===== РХТУ Менделеева =====


def check_muctr():
    s = _session(accept_json=True)
    payload = _get_json(s, 'https://rating.muctr.ru/api/file/', params={
        'organization': 'muctr', 'admission': 'bachelor', 'category': 4},
        timeout=180)
    codes = set()
    for g in payload.get('groups', []):
        if g.get('КонкурснаяГруппаФормаОбучения') != 'Очная':
            continue
        if g.get('ОснованиеПоступления') != 'Бюджетная основа':
            continue
        if g.get('bvi_only'):
            continue
        for e in g.get('Incoming', []):
            if e.get('Состояние') == 'Отозвано':
                continue
            if str(e.get('СогласиеНаЗачисление')) == 'Да':
                code = _digits(e.get('УникальныйКод'))
                if code:
                    codes.add(code)
    return codes


# ===== РГАУ-МСХА Тимирязева =====


# У Тимирязевки ДВА вида списков на одну и ту же программу, и колонка «Согласие на
# зачисление» есть в обоих:
#   kind=konkurs — короткий ранжированный список по «номеру предложения» (13-35 строк);
#   kind=spisok  — полный список подавших заявления (сотни строк).
# Читали только konkurs — и получали 11 согласий на весь вуз при 65 тысячах строк в
# spisok, где их 1473 (проверено 23.07.2026). Берём оба: konkurs добавляет единичные
# коды, которых в spisok уже нет (зачисленные), стоит это лишних 20 секунд.
TIMACAD_KINDS = ('spisok', 'konkurs')


def _timacad_codes(s, base, kind):
    meta = _get_json(s, base + '/api/admission/lists/', params={
        'year': 2026, 'kind': kind}, timeout=60)
    groups = [m for m in meta.get('results', [])
              if m.get('level') in ('bak', 'spec') and m.get('form') == 'ofo'
              and m.get('basis') in ('budget', 'target') and not m.get('is_filial')]
    codes = set()
    for g in groups:
        try:
            html = _get(s, base + f'/api/admission/lists/{g["id"]}/html/',
                        timeout=60).content.decode('utf-8', 'replace')
        except Exception as e:
            print(f'    timacad: {kind} {g.get("id")} пропущен ({e})')
            continue
        body = re.sub(r'<style.*?</style>', '', html, flags=re.S | re.I)
        body = re.sub(r'<script.*?</script>', '', body, flags=re.S | re.I)
        rows = []
        for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', body, re.S | re.I):
            cells = [re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', '', c)).replace('&nbsp;', ' ').strip()
                     for c in re.findall(r'<td[^>]*>(.*?)</td>', tr, re.S | re.I)]
            if cells:
                rows.append(cells)
        hdr = next((r for r in rows if any('Уникальный код' in c for c in r)), None)
        if hdr is None:
            continue
        ki = hdr.index('Уникальный код')
        ci = next((i for i, c in enumerate(hdr) if 'Согласие на зачисление' in c), None)
        if ci is None:
            continue
        for r in rows:
            if len(r) <= max(ki, ci) or not r[ki]:
                continue
            if not _digits(r[ki]) or not r[ki].replace('-', '').replace(' ', '').isdigit():
                continue
            if '✓' in r[ci] or 'да' in r[ci].lower():
                codes.add(_digits(r[ki]))
    return codes


def check_timacad():
    s = _session()
    s.headers.update({'Referer': 'https://www.timacad.ru/'})
    base = 'https://new.timacad.ru'
    codes = set()
    for kind in TIMACAD_KINDS:
        codes |= _timacad_codes(s, base, kind)
    return codes


# ===== МАДИ (каталог pk.madi.ru -> Госуслуги) =====


def check_madi():
    s = _session(accept_json=True)
    r = _get(s, 'https://pk.madi.ru/4608-spiski-abiturientov-postupayuschih-'
             'na-programmy-bakalavriata.html')
    r.encoding = r.apparent_encoding
    html = r.text
    m = re.search(r'var\s+data\s*=\s*(\{.*?\});', html, re.S)
    comp_ids = set()
    if m:
        # t1 = бюджет очная; парсим competitionId из links.general/special/separate/target
        block = m.group(1)
        t1 = re.search(r't1\s*:\s*(\[.*?\])\s*,\s*t2\s*:', block, re.S)
        raw = t1.group(1) if t1 else block
        for cid in re.findall(r'/applicants/(\d+)', raw):
            comp_ids.add(cid)
    codes = set()
    for cid in comp_ids:
        try:
            data = _get_json(s, f'https://www.gosuslugi.ru/api/university-applicant-list/'
                             f'v1/public/2026/competition/{cid}/applicants')
            for a in data.get('applicants', []):
                if str(a.get('statusName', '')).startswith(('Конкурсная группа исключена',
                                                            'Вуз отклонил')):
                    continue
                if str(a.get('consent')) in ('ONLINE', 'OFFLINE'):
                    code = _digits(a.get('idApplication'))
                    if code:
                        codes.add(code)
        except Exception as e:
            print(f'    madi: competition {cid} пропущен ({e})')
    return codes


# ===== ГУЗ (1С:Элемент облако) =====


def check_miet():
    s = _session(accept_json=True)
    r = _get(s, 'https://abit.miet.ru/bak-submitted/index.php')
    r.encoding = 'utf-8'
    codes = set()
    cgs = {}
    for m in re.finditer(r'list\.php\?[^"]*?cg=(\d{9})"\s*>(.*?)</a>', r.text, re.S):
        cg, label = m.group(1), re.sub(r'<[^>]+>', '', m.group(2)).strip()
        if label.endswith('Бюджет'):
            cgs[cg] = label
    for cg in cgs:
        try:
            data = _get_json(s, f'https://abit.miet.ru/data/public/bak/basic/submitted/'
                             f'{cg}.json')
            cols = [c.get('title') for c in data.get('parametrs', {}).get('columns', [])]
            i_code = cols.index('Уникальный код поступающего') if \
                'Уникальный код поступающего' in cols else 4
            i_cons = cols.index('Согласие на зачисление') if \
                'Согласие на зачисление' in cols else 5
            for row in data.get('applications', []):
                if str(row[i_cons]) == '+':
                    code = _digits(row[i_code])
                    if code:
                        codes.add(code)
        except Exception as e:
            print(f'    miet: группа {cg} пропущена ({e})')
    return codes


# ===== Чекеры: HTML =====

def check_mipt():
    s = _session()
    idx = None
    # 2026_list у МФТИ отдаёт 404 (кампания живёт по адресу 2025_list) — на заведомо
    # запасной адрес одна попытка, иначе три повтора по 2/4/6 с уходят впустую.
    for u in ('https://pk.mipt.ru/bachelor/2026_list/', 'https://pk.mipt.ru/bachelor/2025_list/'):
        try:
            idx = _get(s, u, retries=1)
            break
        except Exception:
            continue
    if idx is None:
        raise RuntimeError('МФТИ: индекс списков не открылся')
    codes = set()
    hrefs = set(re.findall(r'href="(https://priem\.mipt\.ru/applications_v2/[^"]+)"', idx.text))
    budget = []
    for href in hrefs:
        b64 = href.rsplit('/', 1)[-1]
        try:
            name = base64.b64decode(b64).decode('utf-8')
        except Exception:
            continue
        if name.endswith('_Byudzhet_Na obshchikh osnovaniyakh.html'):
            budget.append(href)
    for href in budget:
        try:
            html = _get(s, href).content.decode('utf-8', 'replace')
        except Exception as e:
            print(f'    mipt: список пропущен ({e})')
            continue
        # Шапка вынесена в <th> вне <tr> — индексы берём из упорядоченного списка
        # <th> (он 1:1 совпадает с ячейками строк данных).
        ths = [_strip(t) for t in re.findall(r'<th[^>]*>(.*?)</th>', html, re.S | re.I)]
        ki = next((i for i, t in enumerate(ths) if 'никальны' in t.lower()), None)
        ci = next((i for i, t in enumerate(ths) if 'оглас' in t.lower()), None)
        if ki is None or ci is None:
            # Пустой список у них отдаётся заглушкой: тело шапки не подставлено
            # («table_header»), tbody пуст. Это не сломанный разбор, а «никто не
            # подал» — отделяем от смены вёрстки, чтобы не искать несуществующую
            # ошибку (23.07.2026: так выглядела программа «Проектирование и
            # разработка комплексных бизнес-приложений»).
            name = base64.b64decode(href.rsplit('/', 1)[-1]).decode('utf-8', 'replace')
            reason = 'список пуст' if 'table_header' in html else 'колонки не найдены'
            print(f'    mipt: {reason} — {name.rsplit("/", 1)[-1][:60]}')
            continue
        for r in _rows_cells(html):
            if len(r) <= max(ki, ci) or not re.fullmatch(r'\d{5,9}', _digits(r[ki])):
                continue
            if CHECK in r[ci]:
                codes.add(_digits(r[ki]))
    return codes


# ===== МГУ Ломоносова =====


def check_msu():
    s = _session()
    idx = _get(s, 'https://cpk.msu.ru/submitted/bachelor')
    idx.encoding = 'utf-8'
    deps = set()
    for m in re.finditer(r'href="(/submitted/bachelor/dep_(\d+))"[^>]*>(.*?)</a>', idx.text, re.S):
        name = _strip(m.group(3))
        if 'Филиал' in name or 'филиал' in name:
            continue
        deps.add(m.group(1))
    codes = set()
    for path in deps:
        try:
            r = _get(s, 'https://cpk.msu.ru' + path, timeout=180)
            r.encoding = 'utf-8'
            codes |= _msu_parse_dep(r.text)
        except Exception as e:
            print(f'    msu: {path} пропущен ({e})')
            continue
    return codes


def _msu_parse_dep(html):
    codes = set()
    for block in re.split(r'<div class="submitted-concourse">', html)[1:]:
        head = re.search(r'submitted-concourse-title[^>]*>(.*?)</h4>', block, re.S)
        category = _strip(head.group(1)) if head else ''
        passport = re.search(r'submitted-passport-main">(.*?)</div>\s*<div>(.*?)</div>',
                             block, re.S)
        form = _strip(passport.group(2)) if passport else ''
        if 'Основные места в рамках КЦП' not in category:
            continue
        if 'очная форма' not in form.lower() or 'очно-заочная' in form.lower():
            continue
        for r in _rows_cells(block):
            code_cell = next((c for c in r if re.fullmatch(r'\d{5,9}', _digits(c))
                              and len(_digits(c)) >= 5), None)
            if not code_cell:
                continue
            # колонка согласия «Наличие согласия…» = предпоследняя (Да/Нет), затем статус
            yn = [i for i, c in enumerate(r) if c.strip() in ('Да', 'Нет')]
            if yn and r[yn[-1]].strip() == 'Да':
                codes.add(_digits(code_cell))
    return codes


# ===== МИСиС =====


# Хабы списков МИСиС. 04.08.2026 вуз развёл их по двум страницам: на старой
# «Списки подавших заявления» остались ТОЛЬКО платные (16 списков, все с токеном
# COMM), а бюджетные уехали на «Конкурсные списки и списки зачисленных» — там 88
# списков, из них 51 бюджетный и 21 целевой. Прежний чекер знал только старый хаб
# и фильтр «-BUDJ- и -OKM-», поэтому находил ноль и уводил вуз на кэш 88-часовой
# давности. Читаем оба: формат ссылок у них разный («./list/?id=» на новом против
# «list-p/?id=» на старом), поэтому ищем любую ссылку с «?id=».
MISIS_HUBS = (
    'https://misis.ru/applicants/admission/progress/'
    'baccalaureate-and-specialties/list-of-applicants/',
    'https://misis.ru/applicants/admission/progress/'
    'baccalaureate-and-specialties/spiskipodavshihzayavleniya/',
)
# Бюджет и целевая квота: уход туда выводит человека из наших списков. COMM
# (места по договорам) не берём — согласие на платное бюджетный конкурс не
# закрывает, тот же вывод, что по РГУНХ 30.07.2026.
MISIS_BUDGET_TOKENS = ('-BUDJ-', '-CELEV-')


def _misis_codes(html):
    """Коды с согласием из одного списка МИСиС.

    Колонки ищем ПО НАЗВАНИЮ в шапке: 04.08.2026 в таблице стало 23 заголовка,
    «Согласие на зачисление» — седьмой по счёту, а прежний разбор брал согласие
    как третью ячейку с конца и на новой раскладке попадал в пустой хвост.
    Строк в теле на одну ячейку меньше, чем заголовков (недостающая — в конце),
    поэтому отсчёт от начала верен, а от конца — нет.
    """
    body = re.search(r'<tbody[^>]*>(.*?)</tbody>', html, re.S | re.I)
    if not body:
        return set()
    head = re.search(r'<thead[^>]*>(.*?)</thead>', html, re.S | re.I)
    idx_code = idx_consent = idx_status = None
    if head:
        titles = [_strip(c).lower() for c in
                  re.findall(r'<t[dh][^>]*>(.*?)</t[dh]>', head.group(1), re.S | re.I)]
        for i, title in enumerate(titles):
            if idx_code is None and 'уникальный код' in title:
                idx_code = i
            elif idx_consent is None and title.startswith('согласие'):
                idx_consent = i
            elif idx_status is None and title.startswith('статус'):
                idx_status = i

    def cell(cells, index, fallback):
        if index is not None and index < len(cells):
            return cells[index]
        return cells[fallback] if abs(fallback) <= len(cells) else ''

    codes = set()
    for cells in _rows_cells(body.group(1)):
        if len(cells) < 4:
            continue
        code = _digits(cell(cells, idx_code, 1))
        if not re.fullmatch(r'\d{5,9}', code):
            continue
        status = cell(cells, idx_status, 2)
        if 'Отозвано' in status:
            continue
        # «Включено в приказ на зачисление» — человек уже зачислен, ушёл
        # окончательно; это сильнее согласия и учитывается независимо от галочки.
        if 'приказ' in status.lower():
            codes.add(code)
            continue
        if cell(cells, idx_consent, -3).strip() in ('+', 'да', 'Да'):
            codes.add(code)
    return codes


def check_misis():
    s = _session()
    lists = {}
    for hub in MISIS_HUBS:
        try:
            page = _get(s, hub)
        except Exception as error:
            print(f'    misis: хаб {hub[-28:]} не открылся ({error})')
            continue
        page.encoding = 'utf-8'
        for href in re.findall(r'href=["\']([^"\']*\?id=[^"\']+)["\']', page.text):
            gid = href.split('?id=', 1)[1]
            if any(token in gid for token in MISIS_BUDGET_TOKENS):
                lists.setdefault(gid, urllib.parse.urljoin(hub, href))
    if not lists:
        raise RuntimeError('МИСиС: ни на одном из двух хабов нет бюджетных списков — '
                           'сменилась вёрстка или вуз убрал списки')

    codes, failed = set(), []
    for gid, url in sorted(lists.items()):
        try:
            r = _get(s, url, timeout=120)
            r.encoding = 'utf-8'
        except Exception as error:
            failed.append(gid)
            print(f'    misis: {gid} пропущен ({error})')
            continue
        codes.update(_misis_codes(r.text))
    # Неполный набор уходит дальше как полный, и коды из выпавших списков считаются
    # «не подавшими согласие» — значит остаются в наших списках и завышают ВП.
    # Небольшую потерю терпим, заметную считаем отказом (как у РЭУ).
    if failed and len(failed) > max(2, len(lists) // 20):
        raise RuntimeError('МИСиС: не отдал %d списков из %d — данные неполные'
                           % (len(failed), len(lists)))
    if failed:
        print(f'    ВНИМАНИЕ misis: пропущено списков {len(failed)} из {len(lists)}.')
    print(f'    misis: списков {len(lists)} (бюджет и целевая квота)')
    return codes


# ===== МАИ =====


def check_mai():
    s = _session()
    root = _get(s, 'https://priem.mai.ru/rating/')
    root.encoding = 'utf-8'
    m = re.search(r'value="(p\d{14}_1)"', root.text)
    if not m:
        raise RuntimeError('МАИ: не найден префикс снапшота')
    prefix = m.group(1)
    base = 'https://public.mai.ru/priem/rating/data/'

    def opts(key):
        r = _get(s, base + key + '.html')
        r.encoding = 'utf-8'
        return re.findall(r'<option value="([^"]+)"', r.text)
    codes = set()
    # p..._1 -> уровни, берём _l1; -> _p1 бюджет; -> _f1 очная; -> _sN группы
    lvl = f'{prefix}_l1'
    pay = f'{lvl}_p1'
    form = f'{pay}_f1'
    try:
        groups = [o for o in opts(form) if re.search(r'_f1_s\d+$', o)]
    except Exception as e:
        raise RuntimeError(f'МАИ: каскад не прошёл ({e})')
    for g in groups:
        try:
            r = _get(s, base + g + '.html', timeout=120)
            r.encoding = 'utf-8'
            html = r.text
        except Exception as e:
            print(f'    mai: {g} пропущен ({e})')
            continue
        # берём последнюю секцию «общий конкурс»
        idx = html.rfind('общему конкурсу')
        segment = html[idx:] if idx >= 0 else html
        for tbl in re.findall(r'<table[^>]*>(.*?)</table>', segment, re.S | re.I):
            rows = _rows_cells(tbl)
            hdr = next((r for r in rows if any('оглас' in c.lower() for c in r)), None)
            if not hdr:
                continue
            ci = next((i for i, c in enumerate(hdr) if 'оглас' in c.lower()), None)
            ki = next((i for i, c in enumerate(hdr) if 'УКП' in c or 'никальн' in c.lower()), 1)
            for r in rows:
                if len(r) <= max(ci or 0, ki) or not re.fullmatch(r'\d{5,}', _digits(r[ki])):
                    continue
                if ci is not None and CHECK in r[ci]:
                    codes.add(_digits(r[ki]))
    return codes


# ===== МГСУ =====


def check_mgsu():
    s = _session()
    cat = _get(s, 'https://mgsu.ru/2026/ks/filter/')
    cat.encoding = 'utf-8'
    m = re.search(r'const data = (\[.*?\]);', cat.text, re.S)
    if not m:
        raise RuntimeError('МГСУ: каталог не найден')
    import json
    data = json.loads(m.group(1))
    codes = set()
    for row in data:
        if row.get('organization') != 'МГСУ':
            continue
        if row.get('education_level') != 'Бакалавриат/специалитет':
            continue
        if row.get('form') != 'Очная':
            continue
        if not str(row.get('place_type', '')).startswith('Бюджет'):
            continue
        url = 'https://mgsu.ru' + row['list_url'] + urllib.parse.quote(row['file'])
        try:
            r = _get(s, url, timeout=120)
            r.encoding = 'utf-8'
            html = r.text
        except Exception as e:
            print(f'    mgsu: {row.get("file","?")[:30]} пропущен ({e})')
            continue
        rows = _rows_cells(html)
        hdr = next((r for r in rows if any('никальн' in c.lower() for c in r)
                    and any('оглас' in c.lower() for c in r)), None)
        if not hdr:
            continue
        ki = next(i for i, c in enumerate(hdr) if 'никальн' in c.lower())
        ci = next(i for i, c in enumerate(hdr) if 'оглас' in c.lower())
        for r in rows:
            if len(r) <= max(ki, ci):
                continue
            code = _digits(r[ki])
            if not re.fullmatch(r'\d{5,9}', code):
                continue
            if CHECK in r[ci]:
                codes.add(code)
    return codes


# ===== Чекеры: HTML/Excel/PDF (партия 2) =====

# 28.07.2026 МГТУ ГА перенёс конкурсные списки: со страницы «Списки лиц, подавших
# документы» бюджетные выгрузки убрали, там остались только «... бак, спец ДОГОВОР»
# и магистратура. Прежний чекер искал ссылку по слову «бак» — и цеплял ДОГОВОР, где
# секции «Основные места ... ОФО» нет вовсе. Разбор молча давал 0 согласий, а сбор
# списывал это на смену вёрстки и деградировал на кэш («тихий ноль», ТЗ этап 2 п.1).
# Списки теперь на /spiski, четырьмя файлами: целевая квота, особая квота, отдельная
# квота и основные места. Берём ВСЕ четыре: для нас важен сам факт согласия в этом
# вузе, а не конкурс, по которому человек идёт. Только по «основным местам» это уже
# 2556 согласий против прежних 330.
MSTUCA_LISTS_URL = ('https://www.mstuca.ru/applicants/the_admissions_committee/spiski')


def _mstuca_codes(sheet):
    """Коды с согласием из одной книги.

    Книга разбита на секции: «Конкурсная группа - ...», под ней параметры, затем
    шапка таблицы и строки. Набор параметров у файлов РАЗНЫЙ: в «основных местах»
    есть «Форма обучения» и «Уровень подготовки», в квотных — только «Основание
    поступления» и «Всего мест». Поэтому по уровню и основанию не фильтруем: в
    этих четырёх файлах и так только бюджет бакалавриата/специалитета (платное
    лежит отдельным файлом «ДОГОВОР», магистратура — файлом «МАГ»). Магистратуру
    всё же отсекаем, если вуз однажды сольёт выгрузки в одну.

    Колонку согласия берём по шапке, а не по номеру: рядом стоят «Преимущественное
    право» (две штуки), «Заключение ВЛЭК» и «Участвует в конкурсе» — все помечены
    той же галочкой, и промах на одну колонку даёт правдоподобный, но чужой ответ.
    """
    codes = set()
    skip_section = False
    code_col = consent_col = None
    for r in range(sheet.nrows):
        row = [str(sheet.cell_value(r, c)).strip() for c in range(sheet.ncols)]
        head = row[0] if row else ''
        if head.startswith('Конкурсная группа'):
            skip_section = False
            code_col = consent_col = None
            continue
        if head.startswith('Уровень подготовки'):
            skip_section = 'Магистратура' in head
            continue
        if consent_col is None and 'Согласие на зачисление' in ' '.join(row):
            consent_col = next((c for c, v in enumerate(row)
                                if v.startswith('Согласие')), None)
            code_col = next((c for c, v in enumerate(row)
                             if 'Уникальный код' in v), 1)
            continue
        if skip_section or consent_col is None:
            continue
        code = _digits(row[code_col]) if len(row) > code_col else ''
        if re.fullmatch(r'\d{6,9}', code) and len(row) > consent_col \
                and CHECK in row[consent_col]:
            codes.add(code)
    return codes


def check_mstuca():
    xlrd = _ensure_module('xlrd', 'xlrd')
    s = _session()
    page = _get(s, MSTUCA_LISTS_URL)
    page.encoding = 'utf-8'
    links = []
    for href, text in re.findall(r'<a[^>]*href="([^"]+\.xlsx?)"[^>]*>(.*?)</a>',
                                 page.text, re.S | re.I):
        name = _strip(text)
        if 'конкурсный список' in name.lower():
            links.append((urllib.parse.urljoin(
                MSTUCA_LISTS_URL, urllib.parse.quote(href, safe='/:%')), name))
    if not links:
        raise RuntimeError('МГТУ ГА: на %s не найдено ни одного «Конкурсного списка» '
                           '— сменилась вёрстка раздела' % MSTUCA_LISTS_URL)
    codes = set()
    for url, name in links:
        content = _get(s, url, timeout=300).content
        book = xlrd.open_workbook(file_contents=content)
        found = set()
        for index in range(book.nsheets):
            found |= _mstuca_codes(book.sheet_by_index(index))
        print(f'    mstuca: {name[:52]} -> {len(found)}')
        codes |= found
    # Пустой результат по всем четырём файлам — это не «никто не подал согласие»,
    # а поломка разбора. Отдаём ошибку, чтобы сбор честно ушёл в кэш.
    if not codes:
        raise RuntimeError('МГТУ ГА: файлы скачаны (%d), но согласий не найдено — '
                           'сменился формат выгрузки' % len(links))
    return codes


# ===== РГУНХ Вернадского (Nextcloud: несколько публичных шар) =====

# Вуз кладёт выгрузки в публичные шары своего Nextcloud, и 29.07.2026 развёл их
# по ДВУМ: «Списки поступающих и результаты ВИ» (в ней остались только платные
# программы) и «Конкурсные списки» (бюджет). Прежний код брал первую найденную
# ссылку — попадал в платную шару и отдавал ноль согласий. Берём все шары со
# страницы бакалавриата, а нужное отбираем по содержимому файлов.
RGUNH_PAGE = 'https://rgunh.ru/abitur/bachelor'
# Платное распознаём по имени файла или папки: внутри выгрузки признака
# «бюджет/платно» нет вовсе (колонка о договоре есть, но у платных списков она
# почти вся «НЕТ» — по ней финансирование не определить).
RGUNH_PAID_WORDS = ('платн', 'внебюджет', 'договор', 'контракт', 'возмещ')
# Со страницы бакалавриата приходят шары бакалавриата и специалитета, но если вуз
# однажды сольёт всё в одну — чужие уровни не должны попасть в наш набор.
RGUNH_SKIP_LEVELS = ('магистр', 'аспирант', 'ординатур', 'спо ', 'среднее проф')
# Пустое значение колонки согласия. Всё остальное (ONLINE, OFFLINE и любая иная
# отметка, которую вуз придумает) считаем согласием — иначе новая формулировка
# молча превратится в «согласий нет».
RGUNH_NO_CONSENT = ('', 'NONE', 'НЕТ', 'НЕТ ДАННЫХ', 'NO', '-', '0')


def _rgunh_tokens(session):
    """Токены всех публичных шар, на которые ссылается страница бакалавриата."""
    page = _get(session, RGUNH_PAGE)
    page.encoding = 'utf-8'
    tokens = list(dict.fromkeys(re.findall(r'nextcloud\.rgunh\.ru/s/(\w+)', page.text)))
    if not tokens:
        raise RuntimeError('на %s не найдено ни одной ссылки nextcloud — вуз сменил '
                           'способ публикации списков' % RGUNH_PAGE)
    return tokens


def _rgunh_files(session, token):
    """{путь в шаре: содержимое xlsx} — вся шара одним запросом.

    Публичный WebDAV у них шатает: PROPFIND по шаре «Конкурсные списки» отвечает
    то 207, то 412, а скачивание файла по webdav-пути отдаёт JSON с ошибкой
    вместо xlsx. Zip всей папки (`/s/<token>/download`) отработал одинаково на
    обеих шарах — и это один запрос вместо семидесяти.
    """
    response = _get(session, 'https://nextcloud.rgunh.ru/s/%s/download' % token,
                    timeout=(CONNECT_TIMEOUT_EXT, 300))
    if response.content[:2] != b'PK':
        # Часть ссылок на nextcloud ведёт не на списки, а на картинку или PDF.
        return {}
    try:
        archive = zipfile.ZipFile(io.BytesIO(response.content))
    except zipfile.BadZipFile:
        raise RuntimeError('шара %s: тело не разбирается как архив' % token)
    inside = [n for n in archive.namelist()
              if n.lower().endswith(('.xlsx', '.xls')) and not n.endswith('/')]
    if inside:
        return {n: archive.read(n) for n in inside}
    if '[Content_Types].xml' in archive.namelist():
        return {token + '.xlsx': response.content}   # шара на один файл, не на папку
    return {}


def _rgunh_form(name):
    """Форма обучения из имени файла или папки.

    Имена у двух шар разные: `...-Бакалавриат-Очная-Платное.xlsx` в одной и
    `06.03.01 Биология очно общий_гот.xlsx` в другой, плюс `ОЗ` для
    очно-заочной. Порядок проверок важен: «очно-заочная» обязана распознаться
    раньше, чем «очная», иначе заочники попадут в наш набор.
    """
    low = name.lower().replace('ё', 'е')
    if 'очно-заочн' in low or 'очно заочн' in low or re.search(r'(?:^|[\s_(-])оз(?:[\s_)-]|$)', low):
        return 'очно-заочная'
    if 'заочн' in low:
        return 'заочная'
    if 'очн' in low:
        return 'очная'
    return ''


def _rgunh_cells(row):
    """Поля строки, даже если файл — CSV, сохранённый как xlsx.

    Пять файлов из 33 в шаре вуза лежат именно так: вся строка сидит в одной
    ячейке с разделителем «;», а лишняя вторая ячейка появилась потому, что
    импорт разрезал текст по запятой внутри названия колонки («Идентификатор
    поступающего (в вузе, присвоенного…»). Склейка через запятую возвращает
    исходную строку, дальше делим по «;». Без этого пять файлов выглядели как
    «нет колонки согласия» и молча выпадали.
    """
    cells = ['' if value is None else str(value) for value in row]
    if len(cells) < 5 and any(';' in cell for cell in cells):
        return [part.strip() for part in ','.join(cells).split(';')]
    return cells


def _rgunh_consents(content, openpyxl):
    """(коды с согласием, была ли колонка согласия) для одного файла выгрузки.

    Колонку ищем по названию, а не по номеру: у «списков подавших» 34 колонки с
    русской шапкой («Наличие согласия на зачисление»), у «конкурсных списков» —
    41 с английской (idCompetitions … PaidContract) и согласия нет вовсе. Второй
    случай не ошибка разбора, поэтому он возвращается флагом, а не исключением.
    """
    book = openpyxl.load_workbook(io.BytesIO(content), read_only=True, data_only=True)
    sheet = book[book.sheetnames[0]]
    rows = sheet.iter_rows(values_only=True)
    header = _rgunh_cells(next(rows, ()))
    consent_index = next((i for i, h in enumerate(header)
                          if 'огласи' in h and 'пособ' not in h.lower()
                          and not h.lower().startswith('дата')), None)
    if consent_index is None:
        return set(), False
    code_index = next((i for i, h in enumerate(header)
                       if 'никальн' in h or 'uniquecode' in h.lower().replace(' ', '')), None)
    if code_index is None:
        raise RuntimeError('в шапке нет колонки уникального кода поступающего')
    codes = set()
    for row in rows:
        cells = _rgunh_cells(row)
        if max(code_index, consent_index) >= len(cells):
            continue
        if cells[consent_index].strip().upper() in RGUNH_NO_CONSENT:
            continue
        code = _digits(cells[code_index])
        if re.fullmatch(r'\d{6,7}', code):
            codes.add(code)
    return codes, True


# Страницы уровней, где вуз держит раздел «Информация о зачислении (Приказы о
# зачислении)». Приём закрыт 29.07.2026, и приказ — признак сильнее согласия:
# зачисленный ушёл окончательно. Пока в разделе лежит только шаблон-заглушка
# (/vikon/abitur/file_stubs/...), но как только приказы издадут, чекер возьмёт
# коды оттуда, без правок.
RGUNH_ORDER_PAGES = ('https://rgunh.ru/abitur/bachelor', 'https://rgunh.ru/abitur/special')
RGUNH_ORDER_STUB = '/file_stubs/'


def _rgunh_order_links(session):
    """Ссылки на файлы приказов о зачислении на бюджетные места."""
    links = {}
    for url in RGUNH_ORDER_PAGES:
        try:
            page = _get(session, url)
        except Exception as error:
            print('    rgunh: %s не открылась (%s)' % (url, str(error)[:50]))
            continue
        page.encoding = 'utf-8'
        start = page.text.find('о зачислении')
        if start < 0:
            continue
        # Раздел идёт одним блоком: сначала бюджетные приказы, дальше платные.
        # Берём кусок от заголовка до слов о договорах, чтобы не захватить платку.
        chunk = page.text[start:start + 6000]
        paid = re.search(r'по договор|с оплатой|платн', chunk, re.I)
        if paid:
            chunk = chunk[:paid.start()]
        for href in re.findall(r'href="([^"]+\.(?:xlsx|xls|csv|pdf))"', chunk, re.I):
            if RGUNH_ORDER_STUB in href:
                continue          # шаблон «Prikaz_zach_1.doc», а не сам приказ
            links[urllib.parse.urljoin(url, href)] = url
    return list(links)


def _rgunh_codes_from_table(content, openpyxl):
    """Коды ЕПГУ из приказа-таблицы: только из колонки, названной кодом.

    Брать «любое число из 6-7 цифр» нельзя: в приказе есть номера приказов, даты
    и номера мест, и каждый ложный код — это абитуриент, вычеркнутый из наших
    списков зря (наши шансы вырастут на пустом месте). Поэтому колонка ищется по
    названию, а если её нет, файл честно возвращает пусто.
    """
    book = openpyxl.load_workbook(io.BytesIO(content), read_only=True, data_only=True)
    codes = set()
    for name in book.sheetnames:
        rows = book[name].iter_rows(values_only=True)
        header = _rgunh_cells(next(rows, ()))
        index = next((i for i, h in enumerate(header)
                      if 'никальн' in h or 'ЕПГУ' in h or 'епгу' in h.lower()), None)
        if index is None:
            continue
        for row in rows:
            cells = _rgunh_cells(row)
            if index < len(cells):
                code = _digits(cells[index])
                if re.fullmatch(r'\d{6,7}', code):
                    codes.add(code)
    return codes


def _rgunh_orders(session, openpyxl):
    """Коды зачисленных из приказов, если приказы уже изданы."""
    codes = set()
    links = _rgunh_order_links(session)
    for url in links:
        try:
            content = _get(session, url, timeout=(CONNECT_TIMEOUT_EXT, 180)).content
        except Exception as error:
            print('    rgunh: приказ %s не скачался (%s)' % (url[-40:], str(error)[:50]))
            continue
        if content[:2] == b'PK':
            found = _rgunh_codes_from_table(content, openpyxl)
        else:
            # PDF: коды берём только если в приказе есть подписанная колонка —
            # иначе там ФИО, и сопоставить с нашими кодами ЕПГУ нечем.
            found = set()
        if found:
            print('    rgunh: приказ %s — кодов %d' % (url.rsplit('/', 1)[-1][:40], len(found)))
        codes |= found
    return codes, len(links)


def check_rgunh():
    openpyxl = _ensure_module('openpyxl', 'openpyxl')
    s = _session()
    tokens = _rgunh_tokens(s)
    codes, taken, skipped_paid, skipped_form, without_column = set(), 0, 0, 0, []
    total = 0
    for token in tokens:
        try:
            files = _rgunh_files(s, token)
        except Exception as error:
            print('    rgunh: шара %s пропущена (%s)' % (token, str(error)[:60]))
            continue
        total += len(files)
        for path, content in files.items():
            low = path.lower().replace('ё', 'е')
            if any(word in low for word in RGUNH_PAID_WORDS):
                skipped_paid += 1
                continue
            if any(word in low for word in RGUNH_SKIP_LEVELS):
                continue
            if _rgunh_form(path) != 'очная':
                skipped_form += 1
                continue
            name = path.rsplit('/', 1)[-1]
            try:
                found, has_column = _rgunh_consents(content, openpyxl)
            except Exception as error:
                print('    rgunh: %s пропущен (%s)' % (name[:40], str(error)[:60]))
                continue
            if not has_column:
                without_column.append(name)
                continue
            taken += 1
            codes |= found
    # Согласий в выгрузках может не быть вовсе (так с 29.07.2026: бюджет выложен
    # как «конкурсные списки», где колонки согласия нет по формату). Тогда идём
    # за приказами о зачислении: приём закрыт, и зачисление — признак сильнее
    # согласия. Приказы добираем всегда, а не только при пустом наборе: после
    # издания приказов согласия перестают обновляться.
    orders, order_files = _rgunh_orders(s, openpyxl)
    codes |= orders
    # Ни согласий, ни приказов — это не сбой разбора, а состояние публикации.
    # Сообщение должно называть причину с цифрами, иначе в следующий раз всё
    # уйдёт на разведку заново; сбор при этом честно деградирует на кэш.
    if not taken and not orders:
        raise RuntimeError(
            'бюджетных списков с колонкой согласия нет ни в одной из %d шар '
            '(файлов %d: платных %d, не очная форма %d, бюджетных без колонки '
            'согласия %d), приказов о зачислении тоже нет (файлов в разделе %d). '
            'Вуз публикует бюджет как конкурсные списки — в них согласие не '
            'выгружается. Проверить глазами: %s'
            % (len(tokens), total, skipped_paid, skipped_form,
               len(without_column), order_files, RGUNH_PAGE))
    if orders:
        print('    rgunh: приказы о зачислении дали %d кодов' % len(orders))
    if without_column:
        print('    rgunh: файлов без колонки согласия %d (конкурсные списки, '
              'согласий в них нет)' % len(without_column))
    return codes


# ===== МГГЭУ / РГУ СоцТех (PDF, pdfplumber) =====


def check_mggeu():
    pdfplumber = _ensure_module('pdfplumber', 'pdfplumber')
    s = _session()
    page = _get(s, 'https://rgust.ru/postuplenie/podavshie/')
    page.encoding = 'utf-8'
    m = re.search(r'БАКАЛАВРИАТ.*?href="(/upload/[^"]+\.pdf)"', page.text, re.S)
    if not m:
        raise RuntimeError('МГГЭУ: не найден PDF бакалавриата')
    content = _get(s, 'https://rgust.ru' + m.group(1)).content
    codes = set()
    ci = None
    take = False
    with pdfplumber.open(io.BytesIO(content)) as pdf:
        for page_obj in pdf.pages:
            text = page_obj.extract_text() or ''
            gm = re.search(r'Конкурсная группа - (.+)', text)
            if gm:
                name = gm.group(1)
                take = '_Очная_' in name and '_Федеральный' in name and '_Б_' in name
                ci = None
            for table in page_obj.extract_tables():
                for row in table:
                    cells = [(c or '').replace('\n', ' ').strip() for c in row]
                    if ci is None and any('Согласие' in c for c in cells):
                        ci = next((i for i, c in enumerate(cells) if 'Согласие' in c), None)
                        continue
                    if not take or ci is None:
                        continue
                    if len(cells) > max(1, ci) and cells[0].isdigit():
                        code = _digits(cells[1])
                        if re.fullmatch(r'\d{6,7}', code) and CHECK in cells[ci]:
                            codes.add(code)
    return codes


# ===== МГТУ Баумана (PDF, fitz) =====


def _bmstu_from_meta(s, meta_url):
    # fitz — модуль пакета PyMuPDF (ставить надо именно PyMuPDF: пакет с именем
    # «fitz» на PyPI — чужой и не тот). В свежих версиях модуль зовётся pymupdf,
    # а fitz оставлен совместимости ради — поэтому alt_import.
    fitz = _ensure_module('fitz', 'PyMuPDF', alt_import='pymupdf')
    codes = set()
    meta = _get_json(s, meta_url)
    base = meta_url.rsplit('/', 1)[0]
    for it in meta.get('list', []):
        title = it.get('title', '')
        if 'ГУИМЦ' in title:
            continue
        pdf_url = urllib.parse.urljoin(base + '/', it['file'].split('/')[-1])
        try:
            content = _get(s, pdf_url).content
            doc = fitz.open(stream=content, filetype='pdf')
        except Exception as e:
            print(f'    bmstu: {title[:30]} пропущен ({e})')
            continue
        lines = []
        for pg in doc:
            lines.extend(l.strip() for l in pg.get_text().split('\n'))
        doc.close()
        for i, ln in enumerate(lines):
            if not re.fullmatch(r'\d{6,7}', ln):
                continue
            # следом рег.номер вида И2261
            if i + 1 < len(lines) and re.fullmatch(r'[А-ЯЁA-Z]{1,3}\d{3,5}', lines[i + 1]):
                # ищем тройку Да/Нет + приоритет + прогноз
                for j in range(i + 2, min(i + 42, len(lines) - 2)):
                    if (lines[j] in ('Да', 'Нет')
                            and re.fullmatch(r'\d{1,2}', lines[j + 1])
                            and lines[j + 2][:2].lower() in ('да', 'не')):
                        if lines[j] == 'Да':
                            codes.add(ln)
                        break
    return codes


def check_bmstu():
    s = _session()
    return _bmstu_from_meta(
        s, 'https://priem.bmstu.ru/lists/upload/enrollees/first/MGTU-1/meta.json')


def check_bmstu_mf():
    s = _session()
    # Мытищинский филиал: индекс lists.json -> ветка MF-1 (бюджет)
    idx = _get_json(s, 'https://priem.bmstu.ru/lists/lists.json')
    meta_url = None
    for item in idx if isinstance(idx, list) else []:
        for ch in item.get('children', []) if isinstance(item, dict) else []:
            data = str(ch.get('data', ''))
            if 'MF-1/meta.json' in data:
                meta_url = urllib.parse.urljoin('https://priem.bmstu.ru', data)
    if not meta_url:
        meta_url = 'https://priem.bmstu.ru/lists/upload/enrollees/first/MF-1/meta.json'
    return _bmstu_from_meta(s, meta_url)


# ===== МИФИ (org.mephi.ru; геоблок 451 вне РФ + капча на самих списках) =====

# С 22.07.2026 МИФИ закрыл страницы списков капчей: индекс (/pupil-rating/index/osp/1)
# отдаётся свободно, а /pupil-rating/get-rating/... возвращает форму «введите символы
# с картинки» — таблицы в ответе нет, и разбор давал 0. Утром 22-23.07 выручал POST на
# тот же адрес (штатный сабмит формы), им и собрали 1290 согласий.
#
# Проверка 23.07.2026, 18:40 — POST закрыли тоже, и капча теперь НА КАЖДЫЙ СПИСОК:
# решаешь её для одного (POST /register/check-captcha -> POST страницы), получаешь
# ровно этот список, соседний снова просит картинку. Списков 319, то есть ручной путь
# «прошёл капчу — собрал всё» больше не работает: одна пройденная капча = один список.
# Распознавать картинки мы не будем.
#
# Что делает чекер: пробует POST, затем GET с ручным PHPSESSID (если МИФИ откатит
# защиту — оба пути снова заработают без правок), а при капче честно падает, и сбор
# деградирует на кэш с печатью его возраста. Ручной PHPSESSID:
#   1. Открой https://org.mephi.ru/pupil-rating/index/osp/1, ткни список, введи символы.
#   2. F12 -> Application/Хранилище -> Cookies -> org.mephi.ru -> PHPSESSID.
#   3. Значение в файл mephi_cookie.txt рядом со скриптом, в текущую папку или в корень
#      проекта (или в переменную окружения MEPHI_PHPSESSID). Проверено 23.07: пока
#      капча висит на каждом списке, этот путь отдаёт только тот список, который ты
#      открыл руками, — то есть спасает лишь при откате защиты на прежний режим.
#
# Если МИФИ нужен свежим, а не из кэша — есть ручной режим: MEPHI_MANUAL=1. Тогда на
# каждый список чекер сохраняет картинку капчи, открывает её и ждёт, что ты введёшь
# символы в консоль (пустой ввод — пропустить список, Ctrl+C — закончить и оставить
# собранное). Списков после фильтров 142, и они идут в порядке «сначала общий конкурс,
# потом квоты и целевые», так что первые ~30 вводов закрывают основную массу согласий.
# Автоматический прогон этим режимом не трогается: без переменной всё как было.
MEPHI_COOKIE_FILE = _data_path('mephi_cookie.txt')
MEPHI_CAPTCHA_FILE = os.path.join(
    os.path.dirname(os.path.abspath(MEPHI_COOKIE_FILE)), 'mephi_captcha.png')
MEPHI_MANUAL = os.environ.get('MEPHI_MANUAL', '').strip().lower() \
    not in ('', '0', 'no', 'false')
# Пауза между списками: их триста с лишним, ломиться без передышки незачем.
MEPHI_DELAY_SEC = 0.2


def _mephi_session_cookie():
    """PHPSESSID из mephi_cookie.txt или MEPHI_PHPSESSID (можно строкой целиком)."""
    value = os.environ.get('MEPHI_PHPSESSID', '').strip()
    if not value and os.path.exists(MEPHI_COOKIE_FILE):
        with io.open(MEPHI_COOKIE_FILE, encoding='utf-8', errors='replace') as f:
            value = f.read().strip()
    match = re.search(r'PHPSESSID\s*=\s*([A-Za-z0-9]+)', value)
    if match:
        return match.group(1)
    return value if re.fullmatch(r'[A-Za-z0-9]{10,64}', value) else ''


def _mephi_is_captcha(html):
    """Страница-заглушка с капчей вместо списка (форма id=captchaform)."""
    lowered = html.lower()
    return 'captchaform' in lowered or 'captcha[input]' in lowered


def _mephi_captcha_error():
    return RuntimeError(
        'МИФИ: капча и на POST, и на GET — с 23.07.2026 она висит на каждом списке '
        '(319 шт.), программно списки не берутся; ручной PHPSESSID в %s помогает '
        'только если МИФИ откатит защиту' % os.path.basename(MEPHI_COOKIE_FILE))


def _mephi_list_html(session, eid):
    """HTML списка: сначала POST (в прежнем режиме он проходил без капчи),
    при капче — GET с ручным cookie."""
    url = f'https://org.mephi.ru/pupil-rating/get-rating/entity/{eid}/original/no'
    html = _post(session, url).content.decode('utf-8', 'replace')
    if not _mephi_is_captcha(html):
        return html
    return _get(session, url).content.decode('utf-8', 'replace')


def _mephi_solve_captcha(session, eid, html, label=''):
    """Ручной ввод капчи для одного списка (MEPHI_MANUAL=1) -> HTML списка или None.

    Порядок ровно тот же, что у их страницы: POST /register/check-captcha (сервер
    отвечает true/false), и только при true — POST самой формы, который отдаёт
    таблицу. Символы вводит человек; распознавания тут нет.
    """
    url = f'https://org.mephi.ru/pupil-rating/get-rating/entity/{eid}/original/no'
    for attempt in range(3):
        found = re.search(r'name="captcha\[id\]"[^>]*value="([^"]+)"', html)
        if not found:
            return None
        captcha_id = found.group(1)
        picture = _get(session, f'https://org.mephi.ru/images/captcha/{captcha_id}.png')
        with open(MEPHI_CAPTCHA_FILE, 'wb') as f:
            f.write(picture.content)
        try:
            os.startfile(MEPHI_CAPTCHA_FILE)   # Windows: откроется просмотрщиком
        except Exception:
            pass
        answer = input(f'    капча для «{label[:60]}» ({MEPHI_CAPTCHA_FILE}), '
                       f'пусто — пропустить: ').strip()
        if not answer:
            return None
        check = _post(session, 'https://org.mephi.ru/register/check-captcha',
                      data={'captcha': answer, 'code': captcha_id},
                      headers={'X-Requested-With': 'XMLHttpRequest'})
        if 'false' in check.text.lower():
            print('    не подошло, картинка обновлена')
            html = _get(session, url).content.decode('utf-8', 'replace')
            continue
        page = _post(session, url, data={'captcha[id]': captcha_id,
                                         'captcha[input]': answer})
        page_html = page.content.decode('utf-8', 'replace')
        return None if _mephi_is_captcha(page_html) else page_html
    return None


def _mephi_sort_key(item):
    """Сначала общий конкурс, потом квоты и целевые: в ручном режиме первые вводы
    должны закрывать списки, где согласий больше всего."""
    name = item[1]
    return (1 if ('квота' in name or 'целев' in name) else 0, name)


def check_mephi():
    s = _session()
    s.headers['Accept-Language'] = 'ru-RU,ru;q=0.9'
    cookie = _mephi_session_cookie()
    if cookie:
        s.cookies.set('PHPSESSID', cookie, domain='org.mephi.ru', path='/')
    idx = _get(s, 'https://org.mephi.ru/pupil-rating/index/osp/1').content.decode('utf-8', 'replace')
    if _mephi_is_captcha(idx):
        raise _mephi_captcha_error()
    if not idx.strip() or '<table' not in idx.lower():
        raise RuntimeError('МИФИ: пустой ответ (вероятно геоблок 451 — нужен РФ IP)')
    entities = []
    for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', idx, re.S | re.I):
        first = re.search(r'<td[^>]*>(.*?)</td>', tr, re.S | re.I)
        eid = re.search(r'entity/(\d+)/original/(?:yes|no)', tr)
        if not (first and eid):
            continue
        name = _strip(first.group(1)).lower()
        if 'магистр' in name or 'аспирант' in name:
            continue
        if not ('бакалавр' in name or 'специалит' in name or ',спец,' in name or ' спец,' in name):
            continue
        if 'очно-заочн' in name or 'заочн' in name and 'очная' not in name:
            continue
        if any(w in name for w in ('платн', 'договор', 'контракт', 'полное возмещ')):
            continue
        entities.append((eid.group(1), name))
    unique = sorted({e: n for e, n in entities}.items(), key=_mephi_sort_key)
    if MEPHI_MANUAL:
        print(f'    mephi: ручной режим, списков {len(unique)}; пустой ввод — пропуск, '
              f'Ctrl+C — закончить и оставить собранное')
    codes = set()
    captcha_hits = 0
    solved = 0
    for eid, name in unique:
        try:
            html = _mephi_list_html(s, eid)
        except Exception as e:
            print(f'    mephi: entity {eid} пропущен ({e})')
            continue
        time.sleep(MEPHI_DELAY_SEC)
        # Капча вместо таблицы: три подряд — дальше ходить бессмысленно (ни POST,
        # ни ручной cookie не проходят, а 300 запросов впустую только злят антибот).
        # В ручном режиме вместо этого спрашиваем символы у человека.
        if _mephi_is_captcha(html):
            if MEPHI_MANUAL:
                try:
                    html = _mephi_solve_captcha(s, eid, html, name)
                except KeyboardInterrupt:
                    print(f'\n    mephi: остановлено вручную, разобрано списков {solved}')
                    break
                if not html:
                    continue
                solved += 1
            else:
                captcha_hits += 1
                if captcha_hits >= 3:
                    raise _mephi_captcha_error()
                continue
        # Колонки ищем по строке данных, а не по фиксированным индексам: в шапке
        # МИФИ есть закомментированная ячейка («--> <!--»), из-за которой индексы
        # <th> и <td> разъезжаются, а вёрстка уже сдвигалась на 1 (19.07.2026 код
        # уехал с tds[3] на tds[2], статус согласия — с tds[8] на tds[9], и чекер
        # молча отдавал 0). Код — первая ячейка ЦЕЛИКОМ из 6-9 цифр после номера
        # строки; сравниваем текст ячейки, а не её цифры: рядом стоит «№ дела» вида
        # «НИЯУ МИФИ-12657», и по одним цифрам он пролезет в коды. Статус — ячейка
        # со словом «согласие» («Согласие подано» / «Нет согласия»).
        for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', html, re.S | re.I):
            tds = [_strip(c) for c in re.findall(r'<td[^>]*>(.*?)</td>', tr, re.S | re.I)]
            if len(tds) < 9 or not re.match(r'^\d+$', tds[0]):
                continue
            status = next((c for c in tds if 'соглас' in c.lower()), '')
            if 'согласие подано' not in status.lower():
                continue
            code = next((c for c in tds[1:6] if re.fullmatch(r'\d{6,9}', c)), '')
            if code:
                codes.add(code)
    if not codes:
        if captcha_hits or MEPHI_MANUAL:
            raise _mephi_captcha_error()
        raise RuntimeError('МИФИ: разобрано 0 согласий — вероятно сменилась вёрстка '
                           'списков (проверить колонки кода и статуса)')
    return codes


# ===== МИИГАиК =====


def _miigaik_collect(s, is_filial):
    codes = set()
    for lvl in (1, 2):
        try:
            opts = _get(s, f'https://abiturient.miigaik.ru/specialities?'
                        f'edu_level={lvl}&is_filial={is_filial}').text
        except Exception:
            continue
        spec_ids = re.findall(r'<option value=(\d+)>', opts)
        for sid in spec_ids:
            for fin in (1, 3, 4, 5):
                try:
                    html = _get(s, f'https://abiturient.miigaik.ru/abiturients?'
                                f'is_filial={is_filial}&edu_level={lvl}&speciality={sid}'
                                f'&edu_form=1&finance={fin}&condition__not_in=2',
                                timeout=90).text
                except Exception:
                    continue
                if 'не найдена' in html:
                    continue
                rows = _rows_cells(html)
                hdr = next((r for r in rows if any('оглас' in c.lower() for c in r)), None)
                if not hdr:
                    continue
                ci = next((i for i, c in enumerate(hdr) if 'оглас' in c.lower()), None)
                ki = next((i for i, c in enumerate(hdr) if c.strip().lower() == 'код'), 1)
                for r in rows:
                    if len(r) <= max(ci or 0, ki):
                        continue
                    code = _digits(r[ki])
                    if re.fullmatch(r'\d{6,7}', code) and r[ci].strip().lower() == 'да':
                        codes.add(code)
    return codes


def check_miigaik():
    return _miigaik_collect(_session(), 0)


def check_unitech():
    return _miigaik_collect(_session(), 1)


# ===== РГУ Косыгина =====


def check_rguk():
    s = _session()
    lvl = _get(s, 'https://lists.rguk.ru/level/2026/6/1').text
    plan_m = re.search(r'/plan/2026/6/(\d+)/b/o', lvl)
    fac = plan_m.group(1) if plan_m else '18'
    plan = _get(s, f'https://lists.rguk.ru/plan/2026/6/{fac}/b/o').text
    ids = set(re.findall(rf'/list/2026/6/{fac}/(\d+)/b/o/b', plan))
    codes = set()
    for gid in ids:
        try:
            html = _get(s, f'https://lists.rguk.ru/list/2026/6/{fac}/{gid}/b/o/b',
                        timeout=90).text
        except Exception as e:
            print(f'    rguk: {gid} пропущен ({e})')
            continue
        for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', html, re.S | re.I):
            flat = _strip(tr)
            cm = re.search(r'Уникальный код поступающего:\s*([\d\- ]{5,20})', flat)
            sm = re.search(r'Наличие Согласия на зачисление:\s*(Да|Нет)', flat)
            if cm and sm and sm.group(1) == 'Да':
                code = _digits(cm.group(1))
                if re.fullmatch(r'\d{6,7}', code):
                    codes.add(code)
    return codes


# ===== РУДН =====


def check_rudn():
    s = _session()
    cat = _get(s, 'https://admission.rudn.ru/undergraduate/competition_list/').content.decode(
        'utf-8', 'replace')
    # секции; исключаем Сочинский филиал
    ids = []
    parts = re.split(r'competition-section__title', cat)
    for part in parts[1:]:
        title = _strip(part[:200])
        if 'Сочинск' in title:
            continue
        # fulltime-панель до parttime/distance
        ft = re.search(r'data-panel="fulltime"(.*?)(?:data-panel="(?:parttime|distance)"|$)',
                       part, re.S)
        zone = ft.group(1) if ft else ''
        for m in re.finditer(r'href="(/undergraduate/competition_list/(\d+)/)"[^>]*>([^<]*)</a>',
                             zone):
            if 'Общий конкурс' in m.group(3):
                ids.append(m.group(2))
    codes = set()
    for cid in set(ids):
        page = 1
        while page <= 40:
            url = (f'https://admission.rudn.ru/undergraduate/competition_list/{cid}/'
                   if page == 1 else
                   f'https://admission.rudn.ru/undergraduate/competition_list/{cid}/'
                   f'?LEVEL_CODE=undergraduate&ELEMENT_ID={cid}&PAGEN_1={page}')
            try:
                html = _get(s, url, timeout=90).content.decode('utf-8', 'replace')
            except Exception:
                break
            got = False
            for tr in re.findall(r'<tr class="applicants-table__row[^"]*"[^>]*>(.*?)</tr>',
                                 html, re.S | re.I):
                tds = [_strip(c) for c in re.findall(r'<td[^>]*>(.*?)</td>', tr, re.S | re.I)]
                if len(tds) < 13:
                    continue
                code = _digits(tds[2])
                if re.fullmatch(r'\d{6,7}', code):
                    got = True
                    if tds[12].strip() == 'Да':
                        codes.add(code)
            if page == 1:
                pages = [int(x) for x in re.findall(r'PAGEN_1=(\d+)', html)]
                maxp = max(pages) if pages else 1
            if page >= maxp or not got:
                break
            page += 1
    return codes


# ===== МПГУ =====


# 29.07.2026 МПГУ сменил вёрстку карточки направления, и чекер отдал 0 согласий
# (вместо 2274 днём раньше) — снова «тихий ноль». Ячейку формы обучения дополнили
# блоком «<p>Всего мест: 25, из них: отдельная квота - 3...</p>», а прежний шаблон
# `__form">\s*([^<]+?)\s*</td>` требовал, чтобы между началом ячейки и её концом не
# было ни одного тега. Ячейки с этим блоком перестали совпадать, и до ссылок на
# списки разбор просто не доходил. Вторая мина того же шаблона: `(.*?)</tr>`
# обрывался на первом `</tr>`, поэтому из четырёх ссылок строки видел только одну.
# Теперь идём по строкам таблицы: форма — текст ячейки ДО первого тега, ссылки —
# все, что есть в строке. Заодно берём не только «Бюджетные места», но и квоты
# (для нас важен факт согласия в вузе, а не конкурс), кроме платных.
MPGU_SKIP_PILLS = ('платн', 'договор', 'контракт', 'возмещ')

# 31.07.2026: сбор получил 404 на ЖИВОЙ маршрут — пять попыток подряд за 45 секунд,
# вуз ушёл на кэш 61-часовой давности. Через двадцать минут тот же самый адрес
# отдал 200 и 2909 согласий, ничего не меняя. Значит их Yii-фронтенд под нагрузкой
# отвечает 404 вместо 5xx, и 404 здесь — не «страница переехала», а «сервер сейчас
# не может». Поэтому на стартовой странице попыток больше (паузы 3-6-12-24-24…,
# суммарно около двух минут): это единственный запрос, без которого чекер вообще
# не начинается, а его цена — время только тогда, когда вуз действительно лежит.
MPGU_BASE = 'https://epk25.mpgu.su'
MPGU_START_RETRIES = 8
MPGU_LEVEL_DEFAULT = 'group_first_higher_education'


def _mpgu_level(session):
    """Код уровня «базовое высшее образование и специалитет» со страницы списков.

    Раньше код уровня был вшит в адрес. Он у вуза уже менялся по смыслу (сейчас
    их четыре: СПО, базовое высшее+специалитет, специализированное+магистратура,
    аспирантура), и переименование выглядело бы как «вуз упал» — с 404 на старте и
    молчаливым уходом на кэш. Читаем со страницы, дефолт оставляем запасным.
    """
    try:
        page = _get(session, MPGU_BASE + '/competitive-list').text
    except Exception as error:
        print('    mpgu: страница уровней недоступна (%s), беру уровень по умолчанию.'
              % str(error)[:60])
        return MPGU_LEVEL_DEFAULT
    levels = re.findall(r'href="[^"]*educationLevel=([a-z0-9_]+)"[^>]*>(.*?)</a>',
                        page, re.S | re.I)
    for level, label in levels:
        if 'базовое высшее' in _strip(label).lower():
            return level
    for level, label in levels:
        if 'специалитет' in _strip(label).lower():
            return level
    if levels:
        print('    mpgu: уровня «базовое высшее» нет среди %d ссылок (%s) — беру '
              'уровень по умолчанию.'
              % (len(levels), ', '.join(l for l, _ in levels)[:70]))
    return MPGU_LEVEL_DEFAULT


def check_mpgu():
    s = _session()
    base = MPGU_BASE
    level = _mpgu_level(s)
    struct = _get(s, base + '/competitive-list/structural?educationLevel=' + level,
                  retries=MPGU_START_RETRIES).text
    units = sorted(set(re.findall(r'structuralUnit=(\d+)', struct.replace('&amp;', '&'))))
    if not units:
        raise RuntimeError('МПГУ: не найдено ни одной структурной единицы')
    codes = set()
    lists_seen = 0
    for u in units:
        try:
            dpage = _get(s, base + '/competitive-list/direction?'
                         f'educationLevel={level}&university=main_university'
                         f'&structuralUnit={u}').text
        except Exception as error:
            print(f'    mpgu: подразделение {u} пропущено ({error})')
            continue
        for row in re.findall(r'<tr[^>]*>(.*?)</tr>', dpage, re.S | re.I):
            cell = re.search(r'__form"[^>]*>(.*?)</td>', row, re.S | re.I)
            if not cell:
                continue
            # Текст формы — до первого тега: дальше идёт блок «Всего мест».
            form = cell.group(1).split('<')[0]
            if 'Очная форма' != _strip(form):
                continue
            for code_id, label in re.findall(
                    r'href="/competitive-list/view\?code=([^"]+)"[^>]*>([^<]+)</a>', row):
                if any(word in label.lower() for word in MPGU_SKIP_PILLS):
                    continue
                try:
                    html = _get(s, base + f'/competitive-list/view?code={code_id}',
                                timeout=90).text
                except Exception as error:
                    print(f'    mpgu: список {code_id} пропущен ({error})')
                    continue
                lists_seen += 1
                for tr in re.findall(r'<TR[^>]*>(.*?)</TR>', html, re.S | re.I):
                    tds = [_strip(c) for c in
                           re.findall(r'<TD[^>]*>(.*?)</TD>', tr, re.S | re.I)]
                    if len(tds) < 15 or not tds[0].isdigit():
                        continue
                    code = _digits(tds[1])
                    if re.fullmatch(r'\d{6,9}', code) and CHECK in tds[2]:
                        codes.add(code)
    if not lists_seen:
        raise RuntimeError('МПГУ: ни одного конкурсного списка не открыто — '
                           'сменилась вёрстка карточки направления')
    return codes


# ===== Финуниверситет =====
# Финка была нашим целевым вузом (блок 10), и её согласия брались из таблиц блока.
# Там срез узкий по устройству: 5 программ ТЗ, только «Общий конкурс» и только
# строки выше порога 271/263 — то есть одна таблица вместо вуза. Уход по квоте, на
# любую другую программу или с баллом ниже порога так не виден вовсе. С 29.07.2026
# вуз внешний, поэтому списки читаются целиком, отдельным чекером.
#
# Источник тот же, что у блока 10 — /spiski/listabit.php, но без фильтра конкурсной
# группы и без обрезки по порогу:
#   * form_pay=Бюджет   — платное согласие уходом не считаем (человек остаётся в
#                         бюджетном конкурсе, см. тот же вывод по РГУНХ);
#   * is_agreement=true — булев фильтр колонки согласия. Значение именно «true»:
#                         «да»/«Да»/«1» сервер молча игнорирует и отдаёт всех
#                         подряд (проверено 31.07.2026), поэтому колонка ещё и
#                         перепроверяется при разборе;
#   * page_size=100     — максимум селектора, на больших значениях сервер молча
#                         возвращается к 20 строкам на страницу.
# Сколько строк в выдаче, вуз печатает сам: window.APP_CONFIG = { ..., total: N }.
# Это единственный внешний контроль полноты — по нему сверяется каждый срез.
#
# Почему обход идёт по факультетам, а не одной сплошной пагинацией: параметр
# сортировки новый движок игнорирует, порядок строк на равных ключах между
# запросами плывёт, и сплошной обход 366 страниц 31.07.2026 дал 36 487 строк из
# 36 599 при 1058 повторах — 112 человек молча пропали бы. В разрезе факультета
# выдача мельче и порядок устойчив: все 37 срезов сошлись со своим total строка
# в строку. Строки с «No value» вместо кода (СПО без кода ЕПГУ) законны — они
# считаются прочитанными, но кода не дают.
FINKA_LIST_URL = 'https://www.fa.ru/spiski/listabit.php'
FINKA_REFERER = 'https://www.fa.ru/for-applicants/bachelor/Rateabit/applications.php'
FINKA_PAGE_SIZE = 100
FINKA_WORKERS = 5          # срезы качаются параллельно, страницы среза — подряд
# Предохранитель на случай, если вуз выключит фильтр согласия: весь бюджетный
# список — это 2170 страниц (216 977 строк на 31.07.2026), качать столько внутри
# сбора по 23 вузам нельзя. Ниже порога добираем сами, выше — честная ошибка.
FINKA_MAX_PAGES = 700
FINKA_BASE_PARAMS = {'form_pay': 'Бюджет', 'is_agreement': 'true',
                     'page_size': str(FINKA_PAGE_SIZE)}

_FINKA_CELL_RE = re.compile(
    r'<td[^>]*\bheaders="column-([a-z0-9_]+)"[^>]*>(.*?)</td>', re.S | re.I)
_FINKA_TH_RE = re.compile(r'<th[^>]*\bid="column-([a-z0-9_]+)"', re.I)
_FINKA_TD_RE = re.compile(r'<td[^>]*>(.*?)</td>', re.S)
_FINKA_LOCAL = threading.local()


def _finka_session():
    """Сессия текущего потока (requests.Session не рассчитан на общий доступ)."""
    session = getattr(_FINKA_LOCAL, 'session', None)
    if session is None:
        session = _session()
        session.headers['Referer'] = FINKA_REFERER
        _FINKA_LOCAL.session = session
    return session


def _finka_total(html):
    """Сколько строк в выдаче по мнению самого сайта (window.APP_CONFIG.total)."""
    match = re.search(r'\btotal:\s*(\d+)', html)
    return int(match.group(1)) if match else None


def _finka_header_order(html):
    """Порядок полей по шапке (<th id="column-<поле>">) — фоллбэк разбора."""
    head = re.search(r'<thead[^>]*>(.*?)</thead>', html, re.S)
    order = []
    for field in (_FINKA_TH_RE.findall(head.group(1)) if head else []):
        if field not in order:
            order.append(field)
    return order


def _finka_rows(html):
    """Строки таблицы -> [{поле: значение}] по именам полей.

    Разбор привязан к машинному имени ячейки (headers="column-<поле>"), а не к её
    номеру: 29.07.2026 вуз вставил колонку «№» и передвинул «Финансирование», чем
    сломал позиционный разбор в блоке 10. Если вуз уберёт и headers, поля берутся
    по позиции из шапки. Служебная мобильная строка (одна ячейка с colspan, по
    штуке на страницу) ячеек с именами не имеет и отсеивается сама.
    """
    body = re.search(r'<tbody[^>]*>(.*?)</tbody>', html, re.S)
    if body is None:
        return []
    order, rows = None, []
    for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', body.group(1), re.S):
        cells = {field: _strip(value) for field, value in _FINKA_CELL_RE.findall(tr)}
        if not cells:
            if order is None:
                order = _finka_header_order(html)
            values = [_strip(value) for value in _FINKA_TD_RE.findall(tr)]
            if order and len(values) == len(order):
                cells = dict(zip(order, values))
        if 'unique_code' in cells:
            rows.append(cells)
    return rows


def _finka_codes(rows):
    """Коды с согласием -> (коды, сколько строк отсеяно по колонке согласия).

    Колонка перепроверяется всегда: выключи вуз параметр is_agreement — сервер
    молча отдаст весь бюджетный список, и в «ушедших» оказались бы все подряд.
    """
    codes, dropped = set(), 0
    for row in rows:
        if row.get('is_agreement', '').strip().lower() != 'да':
            dropped += 1
            continue
        code = row.get('unique_code', '').strip()
        if re.fullmatch(r'\d{5,9}', code):
            codes.add(code)
    return codes, dropped


def _finka_page(slice_params, page, base=None):
    """Одна страница выдачи (срез задаётся slice_params, обычно facultet_m)."""
    params = dict(base or FINKA_BASE_PARAMS, page=str(page), **slice_params)
    response = _get(_finka_session(), FINKA_LIST_URL, params=params, timeout=(15, 120))
    response.encoding = 'utf-8'
    return response.text


def _finka_scan(slice_params, base=None):
    """Все строки среза -> (строки, total среза по версии сайта).

    Страницы среза читаются подряд одной сессией: параллельный обход внутри
    среза сервер отдаёт вразнобой и строки на равных ключах теряются.
    """
    html = _finka_page(slice_params, 1, base)
    total = _finka_total(html)
    if total is None:
        raise RuntimeError('в выдаче нет счётчика строк (window.APP_CONFIG.total) — '
                           'сменился движок списков')
    rows = _finka_rows(html)
    pages = min((total + FINKA_PAGE_SIZE - 1) // FINKA_PAGE_SIZE, FINKA_MAX_PAGES)
    for page in range(2, pages + 1):
        rows += _finka_rows(_finka_page(slice_params, page, base))
    return rows, total


def _finka_slice_name(slice_params):
    """Человекочитаемое имя среза для сообщений."""
    values = [str(v) for v in slice_params.values() if v]
    return values[0][:40] if values else 'вуз целиком'


def _finka_slices():
    """Срезы обхода: по факультету, а если словарь недоступен — вуз целиком."""
    try:
        response = _get(_finka_session(), FINKA_LIST_URL,
                        params={'itype_list': 'бкл', 'action': 'get_dict',
                                'field': 'facultet_m'})
        values = [v for v in response.json().get('values', []) if v]
    except Exception as error:
        print('    финка: словарь факультетов недоступен (%s), идём сплошным '
              'обходом — возможны потери строк.' % str(error)[:60])
        return [{}]
    return [{'facultet_m': value} for value in values] or [{}]


def check_finka():
    codes = set()
    # Фильтр согласия обязан сужать выдачу: если вуз его выключит, сплошной
    # список бюджета — это тысячи страниц, и молча качать их нельзя.
    filtered_total = _finka_total(_finka_page({}, 1))
    raw_total = _finka_total(_finka_page({}, 1, base={
        'form_pay': 'Бюджет', 'page_size': str(FINKA_PAGE_SIZE)}))
    if filtered_total is None or raw_total is None:
        raise RuntimeError('Финуниверситет: в выдаче нет счётчика строк '
                           '(window.APP_CONFIG.total) — сменился движок списков')
    if filtered_total >= raw_total:
        if raw_total > FINKA_MAX_PAGES * FINKA_PAGE_SIZE:
            raise RuntimeError(
                'Финуниверситет: фильтр согласия (is_agreement=true) больше не '
                'сужает выдачу (%d строк из %d), полный обход слишком велик — '
                'проверь имя параметра на %s' % (filtered_total, raw_total,
                                                 FINKA_LIST_URL))
        print('    финка: фильтр is_agreement не сужает выдачу (%d из %d) — читаем '
              'списки целиком и отбираем по колонке.' % (filtered_total, raw_total))

    slices = _finka_slices()
    print('    финка: %d строк с согласием в выдаче, срезов %d.'
          % (filtered_total, len(slices)))

    def scan_safe(slice_params):
        try:
            return _finka_scan(slice_params) + (None,)
        except Exception as error:
            return [], 0, error

    with ThreadPoolExecutor(max_workers=FINKA_WORKERS) as pool:
        results = list(pool.map(scan_safe, slices))

    # Срез, отдавший 0 строк или недобравший до своего total, повторяем один раз:
    # 31.07.2026 один филиал на первом заходе ответил total=0 при живых данных —
    # без повтора 595 строк ушли бы в тишину.
    read_rows, dropped_rows, failed = 0, 0, []
    for index, (rows, total, error) in enumerate(results):
        if error is not None or len(rows) < total or not total:
            rows, total, error = scan_safe(slices[index])
        if error is not None:
            failed.append('%s (%s)' % (_finka_slice_name(slices[index]), str(error)[:50]))
            continue
        if len(rows) < total:
            print('    финка: срез «%s» отдал %d строк из %d.'
                  % (_finka_slice_name(slices[index]), len(rows), total))
        read_rows += len(rows)
        slice_codes, dropped = _finka_codes(rows)
        dropped_rows += dropped
        codes |= slice_codes

    if failed:
        print('    финка: срезов не прочитано %d: %s' % (len(failed), '; '.join(failed[:3])))
    # Полнота: тихая потеря части списка опаснее ошибки — из недочитанного среза
    # абитуриент выходит «не подавшим согласие» и остаётся в наших списках.
    if read_rows < filtered_total * 0.97:
        raise RuntimeError('Финуниверситет: прочитано %d строк из %d (%.1f%%) — '
                           'выдача читается неполно, результат ненадёжен'
                           % (read_rows, filtered_total,
                              100.0 * read_rows / max(filtered_total, 1)))
    if dropped_rows:
        print('    финка: строк без согласия в выдаче %d (фильтр сайта их пропустил, '
              'отсеяны по колонке).' % dropped_rows)
    if not codes:
        raise RuntimeError('Финуниверситет: прочитано %d строк, но ни одного кода — '
                           'сменились имена полей column-<поле>' % read_rows)
    return codes


# ===== Реестр чекеров и неподдерживаемых вузов =====
EXTERNAL_CHECKERS = {
    'ВШЭ': check_hse,
    'РЭУ им. Плеханова': check_rea,
    'МАДИ': check_madi,
    'МИЭТ': check_miet,
    'РХТУ им. Менделеева': check_muctr,
    'РГАУ-МСХА им. Тимирязева': check_timacad,
    'Университет Дубна': check_dubna,           # коды-СНИЛС: почти не матчатся
    'МФТИ': check_mipt,
    'МГУ им. Ломоносова': check_msu,
    'МИСиС': check_misis,
    'МАИ': check_mai,
    'НИУ МГСУ': check_mgsu,
    'МГТУ ГА': check_mstuca,
    'МГГЭУ (РГУ СоцТех)': check_mggeu,
    'МИИГАиК': check_miigaik,
    'ТУ им. Леонова (Королёв)': check_unitech,
    'РГУ им. Косыгина': check_rguk,
    'РУДН': check_rudn,
    'МПГУ': check_mpgu,
    'МГТУ им. Баумана': check_bmstu,
    'Мытищинский филиал МГТУ им. Баумана': check_bmstu_mf,
    'РГУНХ им. Вернадского': check_rgunh,        # Nextcloud WebDAV — проверить локально
    'Финуниверситет': check_finka,   # бывший наш вуз: списки целиком, а не 5 программ
    'НИЯУ МИФИ': check_mephi,      # РФ IP + PHPSESSID из браузера (капча, см. выше)
}

# Вузы без программного пути сбора (или требующие обхода защиты) — их согласия
# не проверяются, что лишь снижает полноту шага 3 (не ломает результат).
UNSUPPORTED_EXTERNAL = {
    'ГУУ': 'нет публичных списков в машиночитаемом виде',
    'ГУЗ': '1С:Элемент, требует подделки служебного заголовка версии приложения',
    'МГУТУ им. Разумовского': 'движок отчётов FastReport (двухшаговый internal API)',
    'МГУПП (Росбиотех)': 'движок отчётов FastReport (двухшаговый internal API)',
    'МГРИ': 'движок отчётов FastReport (двухшаговый internal API)',
    'РАНХиГС': 'антибот-JS-челлендж Bitrix (требует эмуляции __jhash_)',
    'Сеченовский университет': '5 строк на страницу — тысячи запросов, непрактично',
    'РГГУ': 'каскад из ~130 POST-запросов фильтра (можно добавить при необходимости)',
    'РГСУ': 'wizard из ~170 POST-запросов (можно добавить при необходимости)',
    'Агрегаторы (admlist и др.)': 'живых агрегаторов с выгрузкой согласий не найдено',
}


def collect_external_consents(force=False):
    """Собирает согласия по всем вузам реестра, с кэшем и деградацией."""
    cached = {}
    if os.path.exists(CACHE_EXT):
        with open(CACHE_EXT, 'rb') as f:
            cached = pickle.load(f)
    results = {}
    now = time.time()
    for vuz, checker in EXTERNAL_CHECKERS.items():
        entry = cached.get(vuz)
        if not force and entry and now - entry['ts'] < MAX_AGE_HOURS_EXT * 3600:
            results[vuz] = entry['codes']
            print('  %s: из кэша (%d согласий, %.1f ч назад).'
                  % (vuz, len(entry['codes']), (now - entry['ts']) / 3600))
            continue
        try:
            t0 = time.time()
            codes = {c for c in map(str, checker()) if c}
            # Защита от «тихого нуля»: смена вёрстки не роняет чекер, он просто
            # возвращает пустой/куцый набор, и шаг 3 молча недоочищает списки.
            # Согласия по ходу кампании только накапливаются, поэтому падение
            # больше чем вдвое против прошлого сбора — сигнал сломанного разбора
            # (кейс МИФИ 19.07.2026: вёрстка уехала на колонку, стало 0 из ~1500).
            previous = len(entry['codes']) if entry else 0
            if codes and previous >= 20 and len(codes) < previous // 2:
                print('  ВНИМАНИЕ: %s — согласий резко меньше, чем в прошлый раз '
                      '(%d против %d): проверь разбор, взято НОВОЕ значение.'
                      % (vuz, len(codes), previous))
            if not codes:
                raise RuntimeError('чекер вернул 0 согласий (вероятно сменилась вёрстка)')
            results[vuz] = codes
            cached[vuz] = {'codes': codes, 'ts': now}
            print('  %s: %d согласий (%.0f сек).' % (vuz, len(codes), time.time() - t0))
        except Exception as error:
            if entry:
                results[vuz] = entry['codes']
                # Возраст кэша важен: согласия меняются до самого зачисления, и
                # «взято из кэша» суточной давности — уже не факт, а оценка.
                print('  ВНИМАНИЕ: %s — ошибка (%s), взято из кэша (%d согласий, '
                      '%.1f ч назад).'
                      % (vuz, str(error)[:70], len(entry['codes']),
                         (now - entry['ts']) / 3600))
            else:
                print('  ВНИМАНИЕ: %s — ошибка (%s), не проверен.'
                      % (vuz, str(error)[:70]))
    with open(CACHE_EXT, 'wb') as f:
        pickle.dump(cached, f)
    return results


print('Сбор согласий внешних вузов (%d в реестре)...' % len(EXTERNAL_CHECKERS))
external_consents_by_vuz = collect_external_consents()

total_codes_ext = set().union(*external_consents_by_vuz.values()) \
    if external_consents_by_vuz else set()
print('\nИтого: %d вузов проверено, %d уникальных кодов с согласием.'
      % (len(external_consents_by_vuz), len(total_codes_ext)))
if UNSUPPORTED_EXTERNAL:
    print('Не проверяются (%d):' % len(UNSUPPORTED_EXTERNAL))
    for vuz, reason in UNSUPPORTED_EXTERNAL.items():
        print('  %s: %s' % (vuz, reason))
print('Время работы блока: %.1f сек.' % (time.time() - start_time_ext))

In [ ]:
# -*- coding: utf-8 -*-
# ===== БЛОК: ФИНАЛЬНАЯ СТАТИСТИКА (очистка списков + пересчёт ВП) =====
# Запускать ПОСЛЕ всех блоков вузов (нужны all_<вуз>_data и target_*_<вуз>).
# Если перед этим выполнен «БЛОК: СОГЛАСИЯ ВНЕШНИХ ВУЗОВ» (внешние_согласия.py),
# используется и проверка согласий в нецелевых вузах (external_consents_by_vuz);
# без него шаг 3 пропускается с предупреждением.
#
# Оригинальные all_<вуз>_data НЕ изменяются — вся очистка идёт в глубокой копии
# final_data (по ТЗ zapros.txt).
#
# Шаги очистки (ТЗ):
#   1. «Забрал документы» и т.п. в колонках Статус/Примечание -> код удаляется из
#      ВСЕХ программ этого вуза. (Фактически встречается только у МЭИ: «Забрал
#      документы»; у МИРЭА isActive=0 и у СТАНКИНа «Отозвано» уже отфильтрованы
#      при парсинге, у Политеха/МИИТ статусы ухода в списках не встречаются.)
#   2. Согласие='да' в вузе V -> код остаётся в V (во всех его программах),
#      из всех остальных НАШИХ вузов удаляется. Согласие в двух наших вузах
#      одновременно (рассинхрон дат выгрузок) -> предупреждение, код остаётся в
#      каждом вузе с согласием.
#   3. Коды без согласия в наших вузах ищутся в согласиях НЕЦЕЛЕВЫХ вузов
#      (external_consents_by_vuz из блока внешних согласий): найдено -> код
#      удаляется из всех наших вузов.
#   4. После удалений строки смещаются вверх (№ пересчитывается, №_orig остаётся
#      следом позиции в списке вуза), ВП пересчитывается по каждому вузу отдельно:
#      «Основной ВП» = deferred acceptance по ВСЕМ программам вуза на очищенных
#      списках (места = budget_da у МИРЭА (план − активные БВИ), иначе budget),
#      «Проходной ВП» = наш расчёт по свободным местам. Для вузов с сайтовым ВП
#      (МИРЭА/СТАНКИН/Финка) сайтовые колонки пересчитать нельзя (сайт считал по
#      полным спискам) — после очистки ВП везде РАСЧЁТНЫЙ, единым алгоритмом.
#   5. Строки сортируются по убыванию: сначала «Сумма», затем «Сумма без ИД»
#      (SORT_COLS_FINAL). Сортировка идёт ПОСЛЕ расчёта ВП: ранг конкурса задаёт
#      вуз (у МТУСИ/МИИТ строки БВИ с суммой 0 стоят вверху и держат места, у
#      Финки при равной сумме свой тай-брейк), пересортировка до расчёта сдвинула
#      бы места и исказила «Основной ВП». № — номер строки в таблице после
#      сортировки, №_orig — позиция в списке вуза.
#
# Исключения: коды из KEEP_CODES_FINAL не удаляются ни на одном шаге очистки —
# остаются во всех вузах и таблицах, где есть в исходных списках, независимо от
# согласия (в наших вузах и во внешних) и статуса.
#
# Вывод: те же таблицы и шапки, что в блоках вузов, только целевые программы;
# между вузами разделитель '='*70 и пустая строка. Результат сохраняется в
# final_cleaned.pkl и в Excel «финальная_статистика.xlsx» (лист = вуз).

import collections
import glob
import hashlib
import os
import pickle
import quopri
import re
import sys
import time

import pandas as pd

try:
    from IPython.display import display, Markdown
except ImportError:      # запуск обычным python (проверка МИФИ, отладка блока)
    def display(*args, **kwargs):
        for arg in args:
            print(arg)

    def Markdown(text):  # noqa: N802 — имя как в IPython
        return text

start_time_final = time.time()

# --- Источники: (вуз, словарь данных, список ключей целевых программ). ---
# С 28.07.2026 (ТЗ «этап 2», п.2) документы поданы ровно в пять вузов, сменить их
# уже нельзя. Губкин, СТАНКИН и Финуниверситет остаются в конвейере, но переходят
# в разряд ВНЕШНИХ: таблицы по ним больше не нужны, а их согласия нужны — по ним мы
# узнаём, что абитуриент из наших списков ушёл. Блоки этих вузов не переписываем:
# они уже качают списки и колонку «Согласие», меняется только роль их данных здесь.
VUZ_SOURCES_FINAL = [
    ('МИРЭА', 'all_mirea_data', 'target_ids_mirea'),
    ('МИИТ', 'all_miit_data', 'target_ids_miit'),
    ('МТУСИ', 'all_mtuci_data', 'target_programs_mtuci'),
    ('МЭИ', 'all_mei_data', 'target_ids_mei'),
    ('Политех', 'all_polytech_data', 'target_keys_polytech'),
]

# Вузы, чьи блоки выполняются, но которые считаются внешними: (вуз, имя словаря).
# Их согласия подмешиваются к согласиям внешних вузов на шаге 3 — но только если
# по этому вузу НЕ отработал чекер из блока внешних согласий (см. ниже).
#
# ВАЖНО про полноту: блок вуза хранит только строки ВЫШЕ нашего порога (277/269) и
# только общий конкурс целевых программ. Значит согласие абитуриента, который
# прошёл туда по квоте, на другую программу или с баллом ниже порога, здесь не
# видно — это одна таблица вместо вуза. Полный путь — чекер в «БЛОКЕ: СОГЛАСИЯ
# ВНЕШНИХ ВУЗОВ», там списки берутся целиком: у Финуниверситета такой чекер
# появился 31.07.2026 и дал 5566 кодов против 109 из таблиц блока 10.
#
# Почему чекер имеет приоритет, а не дополняет. Блок вуза работает на своём кэше и
# может отставать: 31.07.2026 в кэше блока 10 (от 29.07, 15:40) 8 кодов стояли с
# согласием, а в живых списках согласие уже отозвано. Подмешать их — значит зря
# вычеркнуть 8 человек из НАШИХ списков. Свежий полный набор от чекера сильнее.
EXTERNAL_FROM_BLOCKS_FINAL = [
    ('Губкин', 'all_gubkin_data'),
    ('СТАНКИН', 'all_stankin_data'),
    ('Финуниверситет', 'all_fin_data'),
]

# --- Согласия МИФИ из сохранённых вручную файлов (ТЗ «этап 2», п.4) ---
# МИФИ с 23.07.2026 закрыл КАЖДЫЙ из 319 списков капчей, программно они не берутся
# (проверено ещё раз 29.07.2026: и GET, и POST отдают форму с картинкой). «Сохранить
# как» из браузера тоже приносит капчу — страница перезапрашивается без сессии. Что
# работает: открыть список, ввести символы, дождаться таблицы и СКОПИРОВАТЬ её
# мышью (Ctrl+C -> Ctrl+V в блокнот/Word/Excel). Поэтому читаем ЛЮБОЙ из форматов:
#   * текстовая копипаста (.txt/.tsv/.csv, ячейки разделены табами) — основной путь;
#   * сохранённая страница (.html/.htm/.mhtml) — если капчу удалось обойти;
#   * .xlsx/.xls — если вставили в Excel.
# Формат определяется ПО СОДЕРЖИМОМУ, а не по расширению: присланный 29.07.2026
# «14.03.01 и 02.html» — это тот же текст копипасты, просто сохранённый с другим
# расширением (побайтово равен .txt), и разбор по тегам дал бы на нём ноль.
MEPHI_DIRS_FINAL = [
    'mifi_sogl',
    'согласия мифи',
    os.path.join('I:', os.sep, 'MyJupProject', 'mifi_sogl'),
    os.path.join('I:', os.sep, 'MyJupProject', 'согласия мифи'),
    os.path.join('D:', os.sep, 'Tanya', 'mifi_sogl'),
    os.path.join('D:', os.sep, 'Tanya', 'согласия мифи'),
]
MEPHI_HTML_DIRS_FINAL = MEPHI_DIRS_FINAL     # старое имя, чтобы не ломать ссылки
MEPHI_EXTS_FINAL = ('.txt', '.tsv', '.csv', '.htm', '.html', '.mhtml', '.mht',
                    '.xlsx', '.xls')
# Форматы, которые мы НЕ разбираем. Молча пропустить их — тот же тихий ноль: бро
# будет считать, что файлы учтены. Поэтому про них пишем в отчёт отдельной строкой.
MEPHI_UNSUPPORTED_EXTS_FINAL = ('.docx', '.doc', '.rtf', '.odt', '.pdf', '.png',
                                '.jpg', '.jpeg')
# Файлы, к которым относится сам запрос, а не списки — читать их бессмысленно.
MEPHI_SKIP_NAMES_FINAL = ('запрос', 'ответ', 'readme', 'changelog', 'session_state')
# Метки в имени файла, разрешающие взять ВСЕ коды, когда колонки статуса в файле
# нет вовсе (список со вкладки «по согласиям» — там статус не печатают).
MEPHI_CONSENT_NAME_MARKERS_FINAL = ('соглас', 'sogl', 'original', 'yes', 'подан')

# Маркеры ухода из конкурса в Статус/Примечание (регистронезависимо).
WITHDRAW_MARKERS_FINAL = ('забрал документы', 'отозв', 'выбыл')

# Коды, которые очистка не трогает ни на одном шаге (остаются во всех вузах).
KEEP_CODES_FINAL = {'2033142'}

# Сортировка строк в таблицах: по убыванию, в этом порядке приоритета.
SORT_COLS_FINAL = ['Сумма', 'Сумма без ИД']

PICKLE_FINAL = 'final_cleaned.pkl'
EXCEL_FINAL = 'финальная_статистика.xlsx'
SEPARATOR_FINAL = '=' * 70


def _strip_html_final(fragment):
    """Текст ячейки без тегов и &nbsp;."""
    text = re.sub(r'<[^>]+>', ' ', fragment).replace('&nbsp;', ' ')
    return re.sub(r'\s+', ' ', text).strip()


def _mephi_dirs_final():
    """Все существующие папки со списками МИФИ (env MEPHI_DIR идёт первой).

    Раньше бралась ПЕРВАЯ существующая папка, и вторая с новыми файлами молча
    игнорировалась — классический «тихий ноль» на ровном месте. Теперь читаем все,
    дубли между ними отсеиваются по хешу файла.
    """
    dirs, seen = [], set()
    env_dir = os.environ.get('MEPHI_DIR', '').strip().strip('"')
    for path in ([env_dir] if env_dir else []) + MEPHI_DIRS_FINAL:
        if not path or not os.path.isdir(path):
            continue
        key = os.path.normcase(os.path.abspath(path))
        if key in seen:
            continue
        seen.add(key)
        dirs.append(path)
    return dirs


def _decode_bytes_final(raw):
    """Байты файла -> текст с определением кодировки.

    Кодировку берём из <meta charset>, а если её нет — пробуем UTF-8 и
    откатываемся на 1251 (браузер сохраняет в UTF-8, копипаста через Word и старые
    сборки дают windows-1251). Это не косметика: при неверной кодировке слово
    «согласие» в ячейке не находится, файл выглядит как список «по согласиям», и в
    уходы попал бы ВЕСЬ список подавших документы.

    «Веб-страница, один файл» (.mhtml) — MIME-контейнер, тело закодировано
    quoted-printable: без раскодирования кириллица в нём выглядит как «=D0=9E».
    """
    if raw[:3] == b'\xef\xbb\xbf':          # BOM от блокнота (UTF-8)
        raw = raw[3:]
    # Блокнот умеет сохранять и в «Юникод» — это UTF-16 с BOM. Без этой ветки такой
    # файл декодируется в мусор, кириллица «не читается» и файл выпадает из сбора.
    if raw[:2] in (b'\xff\xfe', b'\xfe\xff'):
        try:
            return raw.decode('utf-16')
        except UnicodeDecodeError:
            pass
    if re.search(rb'Content-Transfer-Encoding:\s*quoted-printable', raw[:8192], re.I):
        try:
            raw = quopri.decodestring(raw)
        except Exception:                    # noqa: BLE001 — битый контейнер не фатален
            pass
    match = re.search(rb'charset=["\']?\s*([\w-]+)', raw[:4096], re.I)
    encodings = []
    if match:
        encodings.append(match.group(1).decode('ascii', 'replace'))
    encodings += ['utf-8', 'cp1251']
    for encoding in encodings:
        try:
            return raw.decode(encoding)
        except (UnicodeDecodeError, LookupError):
            continue
    return raw.decode('utf-8', 'replace')


def _read_html_final(path):
    """Файл -> текст (оставлено под старым именем, используется в проверках)."""
    with open(path, 'rb') as handle:
        return _decode_bytes_final(handle.read())


def _mephi_code_from_cells_final(cells):
    """Код ЕПГУ строки: ячейка ЦЕЛИКОМ из 6-9 цифр.

    Рядом стоит «№ дела» вида «НИЯУ МИФИ-12657» — по одним цифрам он пролез бы в
    коды, поэтому ячейка должна совпасть целиком. Баллы (3 цифры), ИД и приоритет
    (1-2 цифры) под правило не подходят.
    """
    for cell in cells[1:8]:
        if re.fullmatch(r'\d{6,9}', cell):
            return cell
    return ''


def _mephi_status_yes_final(status):
    """Статус согласия -> True/False. «Не подано» проверяем ПЕРВЫМ.

    «Согласие не подано» содержит слово «подано», и наивная проверка на вхождение
    записала бы отказника в ушедшие, а это удаление живого конкурента из наших
    списков.
    """
    text = status.lower().strip()
    if 'не подан' in text or 'отозв' in text or 'отказ' in text or text == 'нет':
        return False
    return 'подан' in text or text in ('да', 'есть', 'согласие', '+')


def _mephi_rows_stats_final(rows):
    """Строки-ячейки -> (коды с согласием, все коды, строк, строк со статусом)."""
    yes_codes, all_codes, rows_seen, with_status = set(), set(), 0, 0
    for cells in rows:
        if len(cells) < 3 or not re.fullmatch(r'\d{1,5}', cells[0]):
            continue
        code = _mephi_code_from_cells_final(cells)
        if not code:
            continue
        rows_seen += 1
        all_codes.add(code)
        status = next((c for c in cells if 'соглас' in c.lower()), None)
        if status is None:
            continue
        with_status += 1
        if _mephi_status_yes_final(status):
            yes_codes.add(code)
    return yes_codes, all_codes, rows_seen, with_status


def _mephi_rows_from_html_final(html):
    """Таблица сохранённой страницы -> строки ячеек.

    Колонки ищем по строке данных, а не по номерам: в шапке МИФИ есть
    закомментированная ячейка, из-за которой <th> и <td> разъезжаются, и вёрстка
    уже сдвигалась (19.07.2026).
    """
    rows = []
    for tr in re.findall(r'<tr[^>]*>(.*?)</tr>', html, re.S | re.I):
        rows.append([_strip_html_final(c)
                     for c in re.findall(r'<td[^>]*>(.*?)</td>', tr, re.S | re.I)])
    return rows


def _mephi_rows_from_text_final(text):
    """Копипаста таблицы (Ctrl+C -> Ctrl+V) -> строки ячеек.

    Ячейки разделены табами (в Word/Excel — тоже). Одна запись занимает несколько
    физических строк, когда внутри ячейки есть перенос: у олимпиадников это
    «Олимпиада: ...\\n2 степень\\nИД: 10». Начало записи узнаём по содержимому
    (первая ячейка — номер строки, дальше ячейка-код), продолжения подклеиваем к
    предыдущей записи. Привязываться к номеру строки или к переносам нельзя —
    ровно на такой привязке Политех терял строки.
    """
    lines = [line.rstrip('\r') for line in text.split('\n')]
    tabs = sum(line.count('\t') for line in lines)
    semicolons = sum(line.count(';') for line in lines)
    separator = '\t' if tabs else (';' if semicolons else None)
    if separator is None or max(tabs, semicolons) < 3:
        return []
    rows = []
    for line in lines:
        if not line.strip():
            continue
        cells = [cell.strip() for cell in line.split(separator)]
        starts_record = (len(cells) >= 3 and re.fullmatch(r'\d{1,5}', cells[0])
                         and _mephi_code_from_cells_final(cells))
        if starts_record:
            rows.append(cells)
        elif rows:
            rows[-1].extend(cells)
    return rows


def _mephi_stats_from_segments_final(text):
    """Запасной разбор: текст без табов режем по «№ дела» МИФИ.

    Так выглядит копипаста, если табы по дороге съел редактор или PDF. Якорь —
    «НИЯУ МИФИ-12657»: он есть в каждой строке списка, и всё до следующего якоря —
    одна запись. Код ищем в теле записи без самого якоря.
    """
    anchor = re.compile(r'НИЯУ\s*МИФИ\s*[-–—]\s*\d+', re.I)
    marks = [m.start() for m in anchor.finditer(text)]
    yes_codes, all_codes, rows_seen, with_status = set(), set(), 0, 0
    for i, start in enumerate(marks):
        end = marks[i + 1] if i + 1 < len(marks) else len(text)
        segment = text[start:end]
        body = anchor.sub(' ', segment)
        match = re.search(r'(?<!\d)(\d{6,9})(?!\d)', body)
        if not match:
            continue
        rows_seen += 1
        all_codes.add(match.group(1))
        status = re.search(r'соглас\w*[^\n\t;]{0,40}', body, re.I)
        if not status:
            continue
        with_status += 1
        if _mephi_status_yes_final(status.group(0)):
            yes_codes.add(match.group(1))
    return yes_codes, all_codes, rows_seen, with_status


def _mephi_rows_from_excel_final(path):
    """Вставка таблицы в Excel -> строки ячеек (все листы книги)."""
    rows = []
    sheets = pd.read_excel(path, sheet_name=None, header=None, dtype=str)
    for frame in sheets.values():
        for values in frame.fillna('').values.tolist():
            cells = []
            for value in values:
                cell = str(value).strip()
                # Excel возвращает числовые коды как «1289813.0» — хвост убираем,
                # иначе ячейка перестаёт быть «целиком 6-9 цифр» и код теряется.
                cells.append(re.sub(r'\.0$', '', cell) if re.fullmatch(
                    r'\d+\.0', cell) else cell)
            rows.append(cells)
    return rows


def _mephi_files_final(dirs):
    """Файлы-кандидаты из папок (рекурсивно) -> (список путей, неподдержанные)."""
    files, skipped, seen = [], [], set()
    for folder in dirs:
        for path in sorted(glob.glob(os.path.join(folder, '**', '*'), recursive=True)):
            if not os.path.isfile(path):
                continue
            name = os.path.basename(path)
            extension = os.path.splitext(name)[1].lower()
            if any(skip in name.lower() for skip in MEPHI_SKIP_NAMES_FINAL):
                continue
            key = os.path.normcase(os.path.abspath(path))
            if key in seen:
                continue
            seen.add(key)
            if extension in MEPHI_UNSUPPORTED_EXTS_FINAL:
                skipped.append(name)
            elif extension in MEPHI_EXTS_FINAL:
                files.append(path)
    return files, skipped


def collect_mephi_consents_final():
    """Согласия МИФИ из сохранённых вручную файлов -> (коды, отчёт по файлам)."""
    dirs = _mephi_dirs_final()
    if not dirs:
        return set(), ['папка со списками МИФИ не найдена (искали: %s)'
                       % '; '.join(MEPHI_DIRS_FINAL)]
    files, unsupported = _mephi_files_final(dirs)
    unsupported_note = (
        'ВНИМАНИЕ: %d файл(ов) в неподдерживаемом формате НЕ учтены: %s. Сохрани '
        'их как .txt (Ctrl+C из браузера -> блокнот -> «Сохранить как», кодировка '
        'UTF-8) или как .xlsx.' % (len(unsupported), ', '.join(unsupported[:6]))
        if unsupported else '')
    if not files:
        message = ['в папках (%s) нет файлов со списками (%s)'
                   % ('; '.join(dirs), ', '.join(MEPHI_EXTS_FINAL))]
        if unsupported_note:
            message.append(unsupported_note)
        return set(), message
    codes, report, captcha_files = set(), [], []
    report.append('папки: %s; файлов-кандидатов %d' % ('; '.join(dirs), len(files)))
    if unsupported_note:
        report.append(unsupported_note)
    seen_digests, taken_files = {}, 0
    for path in files:
        name = os.path.basename(path)
        try:
            with open(path, 'rb') as handle:
                raw = handle.read()
        except OSError as error:
            report.append('%s — не прочитан (%s)' % (name, error))
            continue
        digest = hashlib.md5(raw).hexdigest()
        if digest in seen_digests:
            report.append('%s — побайтовая копия «%s», пропущен'
                          % (name, seen_digests[digest]))
            continue
        seen_digests[digest] = name

        extension = os.path.splitext(name)[1].lower()
        text = ''
        if extension in ('.xlsx', '.xls'):
            try:
                rows = _mephi_rows_from_excel_final(path)
            except Exception as error:      # noqa: BLE001 — битую книгу просто назовём
                report.append('%s — Excel не прочитан (%s)' % (name, error))
                continue
            mode = 'excel'
            yes_codes, all_codes, rows_seen, with_status = _mephi_rows_stats_final(rows)
        else:
            text = _decode_bytes_final(raw)
            # Без читаемой кириллицы отличить «список по согласиям» от «списка всех
            # подавших» нельзя, а цена ошибки — удаление из наших списков всех, кто
            # просто подал в МИФИ документы. Такой файл не берём вовсе.
            if not re.search(r'[а-яА-Я]{3}', text):
                report.append('%s — кириллица не читается (файл в другой кодировке), '
                              'файл пропущен' % name)
                continue
            # Формат — по содержимому, а не по расширению: присланная копипаста
            # лежала в файле с расширением .html, но тегов в ней нет.
            if re.search(r'<t[rd]\b|<table\b', text, re.I):
                mode = 'html'
                rows = _mephi_rows_from_html_final(text)
            else:
                mode = 'текст'
                rows = _mephi_rows_from_text_final(text)
            yes_codes, all_codes, rows_seen, with_status = _mephi_rows_stats_final(rows)
            if not rows_seen and mode == 'текст':
                mode = 'текст без табов'
                yes_codes, all_codes, rows_seen, with_status = \
                    _mephi_stats_from_segments_final(text)

        # Страница капчи выглядит как обычный сохранённый файл: те же стили и футер
        # МИФИ, только вместо таблицы — форма «введите символы». Отличаем по
        # отсутствию строк с кодом при наличии капчи в разметке (29.07.2026 так были
        # сохранены 15 файлов первой попытки и 3 файла второй — по 11-12 КБ, 0 таблиц).
        if not rows_seen and re.search(r'captcha|введите символы', text, re.I):
            captcha_files.append(name)
            continue
        if not rows_seen:
            report.append('%s — строк с кодом не найдено (не список?), файл пропущен'
                          % name)
            continue

        if with_status:
            taken = yes_codes
            note = ''
        elif any(marker in name.lower()
                 for marker in MEPHI_CONSENT_NAME_MARKERS_FINAL):
            # Вкладка «по согласиям» статуса не печатает — там все строки уже с
            # согласием. Разрешаем это только когда метка есть в имени файла.
            taken = all_codes
            note = ' (статуса в файле нет, имя помечено как «согласия» — взяты все)'
        else:
            # Молча взять все — значит объявить ушедшими всех, кто просто подал
            # документы, и вычистить их из наших списков. Дешевле пропустить файл.
            report.append(
                '%s — колонки «Согласие» в файле нет, а по содержимому это может '
                'быть список ВСЕХ подавших: файл пропущен. Если это точно выгрузка '
                'по согласиям — добавь в имя файла слово «согласия».' % name)
            continue
        report.append('%s — записей %d, согласий %d (%s)%s'
                      % (name, rows_seen, len(taken), mode, note))
        codes |= taken
        taken_files += 1

    if captcha_files:
        report.append(
            'ВНИМАНИЕ: %d файл(ов) — страница КАПЧИ, а не список: %s. Так выходит, '
            'если сохранять страницу через Ctrl+S: браузер перекачивает её заново и '
            'снова получает капчу. Рабочий способ — ввести символы, дождаться '
            'таблицы, выделить её мышью, Ctrl+C и вставить в блокнот (сохранить '
            '.txt в UTF-8) или в Excel.' % (len(captcha_files), ', '.join(
                captcha_files[:6])))
    report.append('принято файлов %d из %d, уникальных кодов с согласием %d'
                  % (taken_files, len(files), len(codes)))
    return codes, report


# Быстрая проверка присланных файлов без полного конвейера:
#   python финал_статистика.py --мифи       (или --mephi, или MEPHI_DIR=путь)
# В Jupyter не срабатывает: там в sys.argv аргументов ядра такого нет.
if {'--мифи', '--mephi'} & set(sys.argv[1:]):
    _codes_cli, _report_cli = collect_mephi_consents_final()
    for _line in _report_cli:
        print('МИФИ: %s' % _line)
    print('\nИТОГО кодов с согласием: %d' % len(_codes_cli))
    if _codes_cli:
        print('первые 20: %s' % ', '.join(sorted(_codes_cli)[:20]))
    raise SystemExit(0 if _codes_cli else 1)


def _copy_vuz_data_final(all_data):
    """Глубокая копия данных вуза (df копируются, оригиналы не трогаем)."""
    copied = {}
    for key, data in all_data.items():
        entry = dict(data)
        entry['df'] = data['df'].copy()
        copied[key] = entry
    return copied


# ===== Сбор исходных данных =====
final_data = {}          # вуз -> {ключ программы: data с копией df}
final_targets = {}       # вуз -> список ключей целевых программ
rows_before_final = {}   # вуз -> строк до очистки
for vuz, data_name, targets_name in VUZ_SOURCES_FINAL:
    if data_name not in globals() or targets_name not in globals():
        print(f'ВНИМАНИЕ: {vuz} — нет {data_name}/{targets_name} '
              f'(блок вуза не выполнен), вуз пропущен.')
        continue
    final_data[vuz] = _copy_vuz_data_final(globals()[data_name])
    final_targets[vuz] = [k for k in globals()[targets_name] if k in final_data[vuz]]
    rows_before_final[vuz] = sum(len(d['df']) for d in final_data[vuz].values())

if not final_data:
    raise SystemExit('Нет данных ни по одному вузу — выполни блоки вузов.')
print(f'Вузов в финальной статистике: {len(final_data)}; '
      f'строк всего: {sum(rows_before_final.values())}.')

# Неполный состав портит очистку: согласия выпавшего вуза не участвуют в шаге 2,
# и ушедшие туда коды остаются в списках остальных, завышая Основной ВП. Поэтому
# результат такого прогона пишем в ОТДЕЛЬНЫЕ файлы, не затирая полную выгрузку.
partial_run_final = len(final_data) < len(VUZ_SOURCES_FINAL)
if partial_run_final:
    missing_vuzes_final = [v for v, _, _ in VUZ_SOURCES_FINAL if v not in final_data]
    PICKLE_FINAL = PICKLE_FINAL.replace('.pkl', '_partial.pkl')
    EXCEL_FINAL = EXCEL_FINAL.replace('.xlsx', '_partial.xlsx')
    print(f'ВНИМАНИЕ: прогон НЕПОЛНЫЙ, нет вузов: {", ".join(missing_vuzes_final)}. '
          f'Результат сохраняем в «{PICKLE_FINAL}» / «{EXCEL_FINAL}», полная '
          f'выгрузка не перезаписывается. Коды, ушедшие в недостающие вузы, '
          f'останутся в списках остальных и завысят Основной ВП.')

# Пробный прогон (проверка правки, отладка на старых данных) — FINAL_DRY_RUN=1.
# Полный состав вузов иначе перезаписывает боевые final_cleaned.pkl и Excel: так
# 29.07.2026 тестовый прогон на данных из самого final_cleaned.pkl затёр снимок
# 23.07. Файл со снимком у нас один, второй копии нет — предохранитель дешевле.
dry_run_final = os.environ.get('FINAL_DRY_RUN', '').strip().lower() \
    not in ('', '0', 'нет', 'false')
if dry_run_final:
    PICKLE_FINAL = PICKLE_FINAL.replace('.pkl', '_dryrun.pkl')
    EXCEL_FINAL = EXCEL_FINAL.replace('.xlsx', '_dryrun.xlsx')
    print(f'FINAL_DRY_RUN=1: пробный прогон, пишем в «{PICKLE_FINAL}» / '
          f'«{EXCEL_FINAL}», боевые файлы не трогаем.')

cleanup_log_final = []   # (вуз, категория, кодов, строк)


def _sort_rows_final(df):
    """Сортировка по убыванию «Сумма», затем «Сумма без ИД» (стабильная).

    Ключи приводятся к числам отдельно от самих колонок: значения в таблице не
    меняются, нечисловые/пустые уходят вниз. Стабильность важна — при равных
    суммах сохраняется исходный порядок вуза (его тай-брейк по предметам).
    """
    sort_cols = [c for c in SORT_COLS_FINAL if c in df.columns]
    if not sort_cols:
        return df.reset_index(drop=True)
    keys = {f'_sort_key_{i}': pd.to_numeric(df[col], errors='coerce')
            for i, col in enumerate(sort_cols)}
    key_names = list(keys)
    return (df.assign(**keys)
              .sort_values(key_names, ascending=[False] * len(key_names),
                           kind='mergesort', na_position='last')
              .drop(columns=key_names)
              .reset_index(drop=True))


def _drop_codes_final(vuz, codes, reason):
    """Удалить коды из всех программ вуза, вернуть число удалённых строк.

    Коды из KEEP_CODES_FINAL не удаляются никогда (исключение по ТЗ).
    """
    codes = {str(code) for code in codes} - KEEP_CODES_FINAL
    if not codes:
        return 0
    removed_rows = 0
    for data in final_data[vuz].values():
        df = data['df']
        mask = df['Код'].astype(str).isin(codes)
        removed_rows += int(mask.sum())
        if mask.any():
            data['df'] = df[~mask].reset_index(drop=True)
    if removed_rows:
        cleanup_log_final.append((vuz, reason, len(codes), removed_rows))
    return removed_rows


# ===== Шаг 1: забравшие документы (Статус/Примечание) =====
print('\nШаг 1: уходы из конкурса (Статус/Примечание)...')
for vuz, vuz_data in final_data.items():
    withdrawn = set()
    for data in vuz_data.values():
        df = data['df']
        for col in ('Статус', 'Примечание'):
            if col not in df.columns:
                continue
            lowered = df[col].astype(str).str.lower()
            mask = lowered.apply(lambda s: any(m in s for m in WITHDRAW_MARKERS_FINAL))
            withdrawn.update(df.loc[mask, 'Код'].astype(str))
    removed = _drop_codes_final(vuz, withdrawn, 'забрал документы')
    if withdrawn:
        print(f'  {vuz}: забрали документы {len(withdrawn)} кодов, '
              f'удалено {removed} строк (из всех программ вуза).')

# ===== Шаг 2: согласия в наших вузах =====
print('\nШаг 2: согласия внутри наших вузов...')
consent_vuzes_final = collections.defaultdict(set)   # код -> {вузы с согласием}
no_consent_col_final = collections.Counter()
for vuz, vuz_data in final_data.items():
    for data in vuz_data.values():
        df = data['df']
        # Колонки может не быть: после зачисления вузы начали перекраивать
        # таблицы, а РГУНХ им. Вернадского 30.07 убрал согласие из бюджетных
        # выгрузок вовсе. В шагах 1 и 3 такая проверка есть, в шаге 2 её не было
        # — и весь блок падал бы с KeyError на середине, не сохранив ни pkl, ни
        # Excel. Программу без колонки пропускаем и обязательно шумим: молча
        # пропущенное согласие = конкурент, который остаётся в наших списках.
        if 'Согласие' not in df.columns:
            no_consent_col_final[vuz] += 1
            continue
        for code in df.loc[df['Согласие'] == 'да', 'Код'].astype(str):
            consent_vuzes_final[code].add(vuz)

if no_consent_col_final:
    print('  ВНИМАНИЕ: колонки «Согласие» нет у программ: ' + '; '.join(
        f'{vuz} — {count}' for vuz, count in no_consent_col_final.items()) +
        '. Согласия по ним не учтены, Основной ВП будет завышен.')

conflicts_final = {c: v for c, v in consent_vuzes_final.items() if len(v) > 1}
if conflicts_final:
    print(f'  ВНИМАНИЕ: у {len(conflicts_final)} кодов согласие сразу в нескольких '
          f'наших вузах (рассинхрон дат выгрузок) — оставлены в каждом вузе с '
          f'согласием: ' + '; '.join(
              f'{c}: {sorted(v)}' for c, v in list(conflicts_final.items())[:5]))

for vuz in final_data:
    to_drop = {code for code, vuzes in consent_vuzes_final.items() if vuz not in vuzes}
    removed = _drop_codes_final(vuz, to_drop, 'согласие в другом нашем вузе')
    if removed:
        print(f'  {vuz}: удалено {removed} строк — согласие в другом нашем вузе.')
print(f'  Кодов с согласием в наших вузах: {len(consent_vuzes_final)}.')

# ===== Шаг 3: согласия во внешних (нецелевых) вузах =====
print('\nШаг 3: согласия во внешних вузах...')
external_map = dict(globals().get('external_consents_by_vuz') or {})

# Губкин / СТАНКИН / Финуниверситет: блоки выполнены, но вузы теперь внешние —
# берём из их таблиц коды с согласием и добавляем к внешним источникам. Если по
# вузу отработал чекер блока внешних согласий, таблицы блока не трогаем: чекер
# читает списки целиком и свежее (в кэше блока согласие может быть уже отозвано).
for ext_vuz, data_name in EXTERNAL_FROM_BLOCKS_FINAL:
    if external_map.get(ext_vuz):
        print(f'  {ext_vuz} (теперь внешний): согласий {len(external_map[ext_vuz])} '
              f'из блока внешних согласий — таблицы блока вуза не подмешиваем.')
        continue
    block = globals().get(data_name)
    if not block:
        print(f'  ВНИМАНИЕ: {ext_vuz} — нет ни чекера в блоке внешних согласий, ни '
              f'{data_name}; ушедшие туда коды останутся в наших списках и завысят '
              f'Основной ВП.')
        continue
    ext_codes = set()
    for data in block.values():
        df = data.get('df')
        if df is None or 'Согласие' not in df.columns:
            continue
        ext_codes.update(df.loc[df['Согласие'] == 'да', 'Код'].astype(str))
    external_map[ext_vuz] = external_map.get(ext_vuz, set()) | ext_codes
    print(f'  {ext_vuz} (теперь внешний): согласий {len(ext_codes)}.')

# МИФИ: списки закрыты капчей, читаем сохранённые вручную файлы из папки
# (копипаста таблицы в .txt/.xlsx, сохранённый .html — формат определяется сам).
mephi_codes_final, mephi_report_final = collect_mephi_consents_final()
for line in mephi_report_final:
    print(f'  МИФИ: {line}')
if mephi_codes_final:
    external_map['НИЯУ МИФИ (из файлов)'] = (
        external_map.get('НИЯУ МИФИ (из файлов)', set()) | mephi_codes_final)
    print(f'  МИФИ: всего уникальных кодов с согласием {len(mephi_codes_final)}.')
else:
    print('  ВНИМАНИЕ: согласий МИФИ нет — уходы туда не будут учтены.')

if external_map:
    remaining_codes = set()
    for vuz_data in final_data.values():
        for data in vuz_data.values():
            remaining_codes.update(data['df']['Код'].astype(str))
    remaining_codes -= set(consent_vuzes_final)
    gone_by_ext = {}
    for ext_vuz, ext_codes in sorted(external_map.items()):
        hit = remaining_codes & set(map(str, ext_codes))
        if hit:
            gone_by_ext[ext_vuz] = hit
    gone_codes = set().union(*gone_by_ext.values()) if gone_by_ext else set()
    for vuz in final_data:
        _drop_codes_final(vuz, gone_codes, 'согласие во внешнем вузе')
    print(f'  Проверено внешних вузов: {len(external_map)}; наших кодов ушло к ним: '
          f'{len(gone_codes)}.')
    for ext_vuz, hit in sorted(gone_by_ext.items(), key=lambda x: -len(x[1])):
        print(f'    {ext_vuz}: {len(hit)}')
    # Вуз, отдавший пустой набор согласий, выглядит как «туда никто не ушёл» — это
    # главный класс дефекта сбора («тихий ноль»), и он завышает наши шансы. Сам
    # чекер о деградации сообщает, но в отчёте финала это раньше не всплывало, и
    # разбирать надо было чужой лог. Называем такие вузы поимённо здесь.
    empty_ext_final = sorted(name for name, codes in external_map.items() if not codes)
    if empty_ext_final:
        print(f'  ВНИМАНИЕ: внешних вузов с ПУСТЫМ набором согласий '
              f'{len(empty_ext_final)}: {", ".join(empty_ext_final)}. Ушедшие туда '
              f'останутся в наших списках и завысят Основной ВП.')
    silent_ext_final = sorted(name for name in external_map
                              if external_map[name] and name not in gone_by_ext)
    if silent_ext_final:
        print(f'  Внешних вузов без единого пересечения с нашими кодами: '
              f'{len(silent_ext_final)} ({", ".join(silent_ext_final[:8])}'
              f'{" …" if len(silent_ext_final) > 8 else ""}) — само по себе норма, '
              f'но если вуз крупный, это повод проверить его чекер.')
else:
    print('  ВНИМАНИЕ: external_consents_by_vuz нет — выполни «БЛОК: СОГЛАСИЯ '
          'ВНЕШНИХ ВУЗОВ» перед этим блоком, шаг пропущен.')

# ===== Контроль кодов-исключений (очистка их не трогает) =====
for keep_code in sorted(KEEP_CODES_FINAL):
    rows_by_vuz = {}
    for vuz, vuz_data in final_data.items():
        rows = sum(int((data['df']['Код'].astype(str) == keep_code).sum())
                   for data in vuz_data.values())
        if rows:
            rows_by_vuz[vuz] = rows
    if rows_by_vuz:
        print(f'\nКод-исключение {keep_code} оставлен: ' + ', '.join(
            f'{vuz} — {rows} строк' for vuz, rows in rows_by_vuz.items()))
    else:
        print(f'\nВНИМАНИЕ: кода-исключения {keep_code} нет ни в одном списке.')

# ===== Шаг 4: пересчёт № и ВП по каждому вузу отдельно =====
print('\nШаг 4: пересчёт ВП (deferred acceptance по всем программам вуза)...')
free_places_final = {}   # вуз -> {ключ программы: мест не занято}
bad_priority_final = collections.Counter()   # вуз -> строк с нечитаемым приоритетом
for vuz, vuz_data in final_data.items():
    # Индекс после удалений. Порядок строк здесь — ранг конкурса вуза, именно по
    # нему считается ВП; № проставляется ниже, уже после сортировки таблиц.
    for data in vuz_data.values():
        data['df'] = data['df'].reset_index(drop=True)

    # Кандидат-предлагающая отложенная приёмка (как в блоках вузов).
    # no_general_codes — коды, которые проходят по квоте или БВИ и место общего
    # конкурса не занимают (заполняет блок вуза; у Политеха — розыгрыш квот).
    # Строки таких абитуриентов из таблиц не убираем: они в списке есть, и их надо
    # видеть, — но в отложенную приёмку не пускаем, иначе место считалось бы дважды.
    locked_codes = set()
    for data in vuz_data.values():
        locked_codes.update(str(code) for code in data.get('no_general_codes', ()))
    prog_rank = {}
    prog_budget = {}
    cand_prefs = {}
    for key, data in vuz_data.items():
        df = data['df']
        prog_budget[key] = int(data.get('budget_da', data.get('budget', 0)) or 0)
        prog_rank[key] = {code: i for i, code in enumerate(df['Код'].astype(str))}
        for code, priority in zip(df['Код'].astype(str), df['Приоритет']):
            if code in locked_codes:
                continue
            # Приоритет может прийти пустым, прочерком или NaN: вузы перекраивают
            # таблицы, а у строк со статусом «зачислен» приоритет иногда не
            # печатается вовсе. Голый int() на таком значении ронял ValueError'ом
            # весь шаг 4 — то есть и Основной ВП, и pkl, и Excel. Нечитаемый
            # приоритет = «худший», абитуриент идёт в конец своих предпочтений.
            try:
                priority_value = int(str(priority).strip())
            except (TypeError, ValueError):
                priority_value = 99
                bad_priority_final[vuz] += 1
            cand_prefs.setdefault(code, []).append((priority_value, key))
    for code, pairs in cand_prefs.items():
        cand_prefs[code] = [key for _, key in sorted(pairs, key=lambda x: x[0])]

    admitted = {key: {} for key in vuz_data}
    next_choice = {code: 0 for code in cand_prefs}
    pending = collections.deque(cand_prefs)
    while pending:
        code = pending.popleft()
        prefs = cand_prefs[code]
        while next_choice[code] < len(prefs):
            key = prefs[next_choice[code]]
            budget = prog_budget[key]
            if budget <= 0:
                next_choice[code] += 1
                continue
            rank = prog_rank[key][code]
            held = admitted[key]
            if len(held) < budget:
                held[code] = rank
                break
            worst = max(held, key=held.get)
            if rank < held[worst]:
                del held[worst]
                held[code] = rank
                next_choice[worst] += 1
                pending.append(worst)
                break
            next_choice[code] += 1

    enrolled = set()
    for key, data in vuz_data.items():
        codes_adm = set(admitted[key])
        df = data['df']
        df['Основной ВП'] = df['Код'].astype(str).isin(codes_adm).map(
            {True: 'да', False: 'нет'})
        enrolled.update(codes_adm)

    # Свободные места после Основного ВП. «Проходной ВП» здесь посчитать нельзя:
    # после отложенной приёмки у программы со свободными местами не остаётся
    # незачисленных кандидатов (тот, кого не взяли нигде, дошёл бы до этого места
    # и занял его), поэтому прежний расчёт давал «нет» во всех строках всех вузов.
    # Оставляем «нет» и показываем сам остаток: если он больше нуля, проходной
    # балл программы ниже нашего порога отбора — Таня проходит с запасом.
    free_places = {key: max(0, prog_budget[key]
                            - int((vuz_data[key]['df']['Основной ВП'] == 'да').sum()))
                   for key in vuz_data}
    free_places_final[vuz] = free_places
    for key, data in vuz_data.items():
        data['df']['Проходной ВП'] = 'нет'

    # Сортировка таблиц: по убыванию «Сумма», затем «Сумма без ИД». Только после
    # расчёта ВП — иначе строки БВИ (сумма 0) и равнобалльные уехали бы со своих
    # конкурсных позиций и сдвинули места. № = номер строки в готовой таблице.
    for data in vuz_data.values():
        df = _sort_rows_final(data['df'])
        df['№'] = range(1, len(df) + 1)
        data['df'] = df

    rows_now = sum(len(d['df']) for d in vuz_data.values())
    print(f'  {vuz}: строк было {rows_before_final[vuz]}, стало {rows_now} '
          f'(удалено {rows_before_final[vuz] - rows_now}).')

if bad_priority_final:
    print('  ВНИМАНИЕ: приоритет не прочитался у строк: ' + '; '.join(
        f'{vuz} — {count}' for vuz, count in bad_priority_final.items()) +
        '. Им проставлен приоритет 99 (в конец предпочтений).')

# ===== Вывод: те же таблицы, только целевые программы =====
BASE_COLS_FINAL = ['№', '№_orig', 'Реальный рейтинг', 'Код', 'Сумма', 'Сумма без ИД',
                   'Оценки', 'ИД', 'Согласие', 'Приоритет', 'БВИ', 'Квота',
                   'Основной ВП', 'ОВП сайта', 'Проходной ВП', 'Хватает балла',
                   'Вердикт вуза', 'Статус', 'Примечание']

display(Markdown('# ФИНАЛЬНАЯ СТАТИСТИКА (после очистки, ВП пересчитан)'))
for vuz, vuz_data in final_data.items():
    print(SEPARATOR_FINAL)
    display(Markdown(f'## {vuz}'))
    for key in final_targets[vuz]:
        data = vuz_data[key]
        df = data['df']
        count_main = (df['Основной ВП'] == 'да').sum()
        free = free_places_final.get(vuz, {}).get(key, 0)
        title = data.get('competition') or str(key)
        form = data.get('form')
        display(Markdown(f"### {title}" + (f" ({form})" if form and str(form) not in title else '')))
        places = data.get('budget_da', data.get('budget', '—'))
        display(Markdown(
            f"**Мест общего конкурса:** {places}  |  "
            f"**Порог отбора:** {data.get('threshold', '—')}  |  "
            f"**Строк после очистки:** {len(df)}  |  "
            f"**Данные на:** {data.get('update_date', '—')}"
        ))
        display(Markdown(
            f'**Основной ВП (да):** {count_main}  |  '
            f'**Мест не занято абитуриентами выше порога:** {free} '
            f'(если больше нуля — проходной балл ниже нашего порога отбора)'
        ))
        display(df[[c for c in BASE_COLS_FINAL if c in df.columns]]
                .style.hide(axis='index'))
        print('-' * 70)
    print(SEPARATOR_FINAL)
    print()

# ===== Сводка очистки =====
if cleanup_log_final:
    print('Сводка очистки:')
    summary = collections.defaultdict(lambda: [0, 0])
    for vuz, reason, n_codes, n_rows in cleanup_log_final:
        summary[(vuz, reason)][0] += n_codes
        summary[(vuz, reason)][1] += n_rows
    for (vuz, reason), (n_codes, n_rows) in summary.items():
        print(f'  {vuz} — {reason}: {n_rows} строк')

# ===== Сохранение: pickle + Excel =====
with open(PICKLE_FINAL, 'wb') as f:
    pickle.dump({'data': final_data, 'targets': final_targets}, f)
print(f'\nСохранено: {PICKLE_FINAL}')

try:
    with pd.ExcelWriter(EXCEL_FINAL, engine='openpyxl') as writer:
        for vuz, vuz_data in final_data.items():
            sheet = vuz[:31]
            row_cursor = 0
            for key in final_targets[vuz]:
                data = vuz_data[key]
                df = data['df']
                out_df = df[[c for c in BASE_COLS_FINAL if c in df.columns]]
                title = data.get('competition') or str(key)
                places = data.get('budget_da', data.get('budget', '—'))
                header = pd.DataFrame({'A': [
                    title,
                    f"Мест общего конкурса: {places} | Порог: {data.get('threshold', '—')} | "
                    f"Строк: {len(df)} | Основной ВП (да): {(df['Основной ВП'] == 'да').sum()} | "
                    f"Мест не занято абитуриентами выше порога: "
                    f"{free_places_final.get(vuz, {}).get(key, 0)} | "
                    f"Данные на: {data.get('update_date', '—')}",
                ]})
                header.to_excel(writer, sheet_name=sheet, startrow=row_cursor,
                                index=False, header=False)
                row_cursor += 2
                out_df.to_excel(writer, sheet_name=sheet, startrow=row_cursor, index=False)
                row_cursor += len(out_df) + 3
    print(f'Сохранено: {EXCEL_FINAL} (лист = вуз, программы друг под другом).')
except Exception as error:
    print(f'ВНИМАНИЕ: Excel не сохранился ({error}) — pickle сохранён, '
          f'таблицы выведены выше.')

print(f'\nВремя работы финальной статистики: {time.time() - start_time_final:.1f} сек.')

In [ ]:
# -*- coding: utf-8 -*-
# ===== БЛОК: ПРЕДСКАЗАТЕЛЬНАЯ МОДЕЛЬ ШАНСОВ НА ПОСТУПЛЕНИЕ =====
# ЗАКЛЮЧИТЕЛЬНЫЙ блок серии. Запускать ПОСЛЕ «ФИНАЛЬНОЙ СТАТИСТИКИ»: он берёт из
# памяти ядра результат её очистки (final_data / final_targets). Если переменных
# в памяти нет (перезапуск ядра) — читает final_cleaned.pkl из текущей папки.
# Ничего не качает из сети и не привязан к конкретному диску: работает там, где
# запущен Jupyter (у тебя I:\MyJupProject).
#
# ЧТО СЧИТАЕТ. Для каждого целевого вуза и каждой целевой программы — вероятность,
# что Таня поступит на бюджет (в %). На выходе по каждому вузу таблица из пяти
# столбцов ТЗ: Программы | Бюджетные места | Минимальный проходной балл | Наши
# баллы | Шансы поступления.
#
# НАШИ БАЛЛЫ (ВИ без ИД): математика 90, русский 91, физика 86, информатика 78.
#   • где в ВИ идёт физика  -> 90+91+86 = 267;
#   • где только информатика -> 90+91+78 = 259.
# ИД: золотая медаль / красный аттестат + значок ГТО, но не больше 10. Почти все
# вузы дают 10 за медаль (медаль 10 + ГТО 1 = всё равно 10). Финуниверситет за
# медаль даёт 3 -> у нас там ИД = 3 + 1(ГТО) = 4. Итог наш конкурсный балл:
#   277 (физика) / 269 (информатика); Финуниверситет 271 (физика) / 263 (инф.).
# Эти же числа блок вуза уже положил в data['threshold'] по каждой программе —
# модель берёт их оттуда, а класс ApplicantProfile служит прозрачной формулой и
# страховкой, если у какой-то программы порог не задан.
#
# СТОЛБЕЦ «МИНИМАЛЬНЫЙ ПРОХОДНОЙ БАЛЛ» (правка 24.07.2026). Блоки вузов качают
# списки только до нашего порога отбора — кто ниже, того в данных нет вообще.
# Поэтому минимум балла среди занявших места — настоящий проходной ТОЛЬКО когда
# видимые заняли все бюджетные места. Если мест занято меньше, остаток уходит
# тем, кто ниже порога, и граница лежит ниже него — в таком случае пишем
# «ниже 269 (наш балл выше)», а не число. До правки столбец брал минимум всегда и
# врал вверх: СТАНКИН 09.03.02 (53 места, занято 1) показывал 271 при пороге 269,
# МИИТ и Политех — ровно 277 на программах с горой свободных мест.
#
# КАК СЧИТАЕТСЯ ШАНС (теория вероятностей, а не сравнение «наш балл vs проходной»).
# Наивное «наш балл ниже минимального проходного -> 0%» ошибается: сильные
# абитуриенты часто оседают в приоритетных программах и освобождают эту (у
# Политеха мест реально свободно больше, чем кажется), а места иногда заняты
# льготниками БВИ независимо от баллов (МТУСИ). Поэтому:
#   1. Таня получает бюджетное место в программе на B мест, если среди тех, кто
#      РЕАЛЬНО займёт здесь место и при этом стоит В ОЧЕРЕДИ ВЫШЕ неё, окажется
#      меньше B человек.
#   2. «Выше в очереди» = БВИ (абсолютный приоритет, вне баллов) ИЛИ конкурсный
#      балл выше нашего; равный балл — тай-брейк, вес 0.5.
#   3. «Реально займёт место» — вероятность held по сигналам из очищенных списков:
#      согласие сюда и Основной ВП сюда = сидит намертво; Основной ВП сюда без
#      согласия = посадил расчёт (deferred acceptance), но может ещё уйти; Основной
#      ВП в ДРУГОЙ программе = уйдёт туда, освободит место; и т.д. «Основной ВП» уже
#      вобрал в себя приоритеты и согласия (его считает финальная статистика), а
#      «да» приравнивает абитуриента к 1-му приоритету, даже если в строке стоит 29.
#   4. Число занявших место конкурентов выше нас — сумма независимых испытаний с
#      разными вероятностями (пуассон-биномиальное распределение). Шанс Тани =
#      P(таких конкурентов < B). Считается точной свёрткой, масштабируется на любое
#      число абитуриентов и мест; вероятностные веса вынесены в AdmissionModel и
#      настраиваются. Чем больше проставлено согласий «да», тем ближе held к 0/1 и
#      тем острее (ближе к 0% или 100%) становится вывод.
# Крайние значения: 100% — только когда мест заведомо хватает (сильнее нас меньше,
# чем мест); 0% — когда места уже заняты намертво определившимися сильнее нас
# (согласие+ВП или БВИ). Между ними — расчётный процент.

import os
import pickle
import time

import pandas as pd
from IPython.display import display, Markdown


# ===== Наш профиль абитуриента (прозрачная формула «наших баллов») =====
class ApplicantProfile:
    """Баллы ВИ и ИД Тани и расчёт итогового конкурсного балла по программе.

    Держит формулу «наших баллов» в одном месте: сумму ВИ считает по тому, какой
    третий предмет засчитывается (физика или информатика), а бонус ИД — по вузу
    (Финуниверситет даёт за медаль 3 балла вместо 10). Основной источник итогового
    балла — data['threshold'] из блока вуза; этот класс дублирует его формулой и
    ловит рассинхрон, если у какой-то программы порог не проставлен.
    """

    ID_CAP = 10               # больше 10 за ИД не начисляют нигде
    GTO_BONUS = 1             # значок ГТО
    MEDAL_DEFAULT = 10        # медаль/красный аттестат — почти во всех вузах
    MEDAL_BY_VUZ = {'Финуниверситет': 3}   # исключение по условиям приёма

    def __init__(self, math=90, russian=91, physics=86, informatics=78,
                 has_medal=True, has_gto=True, code='2033142'):
        self.math = math
        self.russian = russian
        self.physics = physics
        self.informatics = informatics
        self.has_medal = has_medal
        self.has_gto = has_gto
        # Уникальный код ЕПГУ. Нужен, чтобы отличить НАШУ строку от чужой с таким
        # же баллом: подано ли согласие именно нами — это разница между «место уже
        # наше» и «место будет нашим, если сегодня подадим». Тот же код зашит в
        # KEEP_CODES_FINAL финальной статистики.
        self.code = str(code)

    def subjects_sum(self, uses_physics):
        """Сумма трёх ВИ: мат + рус + (физика | информатика)."""
        third = self.physics if uses_physics else self.informatics
        return self.math + self.russian + third

    def id_bonus(self, vuz):
        """Балл за ИД в конкретном вузе (медаль + ГТО, но не больше ID_CAP)."""
        medal = self.MEDAL_BY_VUZ.get(vuz, self.MEDAL_DEFAULT) if self.has_medal else 0
        gto = self.GTO_BONUS if self.has_gto else 0
        return min(self.ID_CAP, medal + gto)

    def score_for(self, vuz, uses_physics):
        """Итоговый конкурсный балл (ВИ + ИД) для программы этого вуза."""
        return self.subjects_sum(uses_physics) + self.id_bonus(vuz)

    def resolve_score(self, vuz, threshold):
        """Наш балл по программе: берём порог из данных, при его отсутствии —
        считаем формулой. Порог 277/271 -> физика, 269/263 -> информатика."""
        if threshold is not None and not pd.isna(threshold):
            return int(threshold)
        # Порога нет — определить предмет по вузу нельзя, берём физику как более
        # частый для наших IT-направлений случай и помечаем это допущением.
        return self.score_for(vuz, uses_physics=True)


# ===== Вероятностная модель занятости мест =====
class AdmissionModel:
    """Веса «реально займёт место» и пуассон-биномиальный расчёт шанса.

    Веса подобраны по смыслу сигналов в очищенных списках; вынесены в атрибуты,
    чтобы модель настраивалась без правки логики.
    """

    # held — вероятность, что абитуриент в итоге займёт место ИМЕННО здесь.
    #
    # Наборов весов ДВА, потому что 04.08.2026 — последний день подачи согласия, и
    # ответ зависит от того, чем этот день кончится:
    #   «если все» — прежний набор. Согласие ещё донесут все, кто может, поэтому
    #                конкурентом считается каждый, кто проходит сюда по баллам.
    #   «сейчас»   — считаем по УЖЕ поданным согласиям. Кто согласия не подал, того
    #                09.08 на бюджет зачислить нельзя, значит место он не занимает
    #                (вес падает с 0.90 до 0.15 — не до нуля: сегодня ещё донесут).
    #                Зеркально растёт вес тех, кто согласие СЮДА подал, но по
    #                расчёту сидит не здесь: после закрытия согласий они первые в
    #                очереди на освободившиеся места (0.25 -> 0.60).
    #
    # Это ровно те две границы, которые публикует сам МИРЭА (minScore — проходной
    # среди подавших согласие, minScoreByAll — если согласятся все). Разрыв между
    # двумя сценариями показывает, насколько вывод устойчив: сходятся — решение
    # безопасное, расходятся — зависит от чужих действий сегодня.
    SCENARIOS = ('сейчас', 'если все')
    DEFAULT_SCENARIO = 'если все'
    WEIGHTS = {
        # ключи: БВИ с ВП сюда; ВП сюда + согласие сюда; только ВП; только
        # согласие сюда; ни ВП, ни согласия
        'если все': {'bvi': 0.98, 'locked': 0.97, 'main_only': 0.90,
                     'wants_only': 0.25, 'elsewhere': 0.05},
        'сейчас':   {'bvi': 0.98, 'locked': 0.97, 'main_only': 0.15,
                     'wants_only': 0.60, 'elsewhere': 0.02},
    }

    def is_bvi(self, row):
        """БВИ / поступление без ВИ: явная отметка или ноль баллов за ВИ."""
        if 'БВИ' in row and str(row['БВИ']).strip().lower() == 'да':
            return True
        raw = row.get('Сумма без ИД')
        try:
            return float(raw) == 0
        except (TypeError, ValueError):
            return False

    def held_probability(self, row, is_bvi, scenario=None):
        """Вероятность, что абитуриент из строки займёт место в этой программе."""
        weights = self.WEIGHTS.get(scenario or self.DEFAULT_SCENARIO,
                                   self.WEIGHTS[self.DEFAULT_SCENARIO])
        main_here = str(row.get('Основной ВП')).strip() == 'да'
        consent_here = str(row.get('Согласие')).strip() == 'да'
        if main_here and is_bvi:
            return weights['bvi']
        if main_here and consent_here:
            return weights['locked']
        if main_here:
            return weights['main_only']
        if consent_here:
            return weights['wants_only']
        return weights['elsewhere']

    @staticmethod
    def poisson_binomial_leq(probs, k):
        """P(число успехов <= k) для суммы независимых Бернулли с разными probs.

        Точная свёртка distribution (динамика по числу успехов), O(n*k). Пустой
        список -> 0 успехов гарантированно.
        """
        if k < 0:
            return 0.0
        dp = [1.0]   # dp[j] — вероятность ровно j успехов
        for p in probs:
            nxt = [0.0] * (len(dp) + 1)
            for j, val in enumerate(dp):
                nxt[j] += val * (1.0 - p)
                nxt[j + 1] += val * p
            dp = nxt
        return sum(dp[:min(k + 1, len(dp))])


# ===== Прогноз по одной программе =====
class ProgramForecast:
    """Расчёт минимального проходного балла и шанса для одной программы."""

    def __init__(self, data, our_score, model, our_code=None):
        self.our_code = str(our_code) if our_code is not None else None
        self.df = data['df'].copy()
        self.df['_score'] = pd.to_numeric(self.df['Сумма'], errors='coerce')
        # ВНИМАНИЕ на `or`: budget_da == 0 — это законный ответ вуза «мест общего
        # конкурса не осталось», а не «поля нет». Прежняя запись `budget_da or
        # budget` на нуле откатывалась к исходному ПЛАНУ, и «мест не осталось»
        # превращалось в «шанс 100%» — конкуренция считалась по несуществующим
        # местам. Наличие ключа проверяем явно, как делает финал_статистика.py.
        budget_da = data.get('budget_da')
        self.budget = int(budget_da if budget_da is not None
                          else (data.get('budget') or 0))
        self.our_score = int(our_score)
        self.model = model
        # Проходные баллы, опубликованные самим вузом (МИРЭА: minScore — минимум
        # среди реально проходящих сейчас, minScoreByAll — если согласятся все).
        # Считаются по ПОЛНОМУ списку, поэтому свободны от главной грабли проекта:
        # наши таблицы обрезаны порогом 277/269, и любой минимум по ним врёт вверх.
        site_min = data.get('min_score')
        self.site_min_score = (int(site_min) if site_min is not None
                               and not pd.isna(site_min) else None)
        site_min_all = data.get('min_score_all')
        self.site_min_score_all = (int(site_min_all) if site_min_all is not None
                                   and not pd.isna(site_min_all) else None)
        # Порог отбора вуза (277/269, у Финуниверситета 271/263) — списки блоков
        # обрезаны по нему, ниже данных нет. Нужен, чтобы честно сказать «граница
        # ниже порога» вместо выдуманного числа (см. min_passing_score).
        threshold = data.get('threshold')
        self.threshold = (int(threshold)
                          if threshold is not None and not pd.isna(threshold) else None)

    def _competitors(self, scenario=None):
        """Кандидаты, стоящие в очереди ВЫШЕ нас, с их held-вероятностями.

        Возвращает (probs, locked_ahead, strong_ahead, equal_ahead):
          probs         — held каждого конкурента выше/равного (равный с весом 0.5);
          locked_ahead  — сколько СИЛЬНЕЕ нас сидят намертво (согласие+ВП или БВИ);
          strong_ahead  — сколько строго сильнее нас (БВИ или балл выше);
          equal_ahead   — сколько с равным баллом (тай-брейк).
        """
        probs = []
        locked_ahead = strong_ahead = equal_ahead = 0
        for _, row in self.df.iterrows():
            score = row['_score']
            is_bvi = self.model.is_bvi(row)
            if pd.isna(score) and not is_bvi:
                continue
            # Непустая «Квота» — абитуриент проходит по квоте (место квоты из КЦП
            # уже вычтено) и за место общего конкурса не борется. Считать его
            # конкурентом — значит дважды занять одно место и занизить наш шанс.
            # Колонку заполняет блок Политеха; у остальных вузов её нет.
            if str(row.get('Квота') or '').strip():
                continue
            strong = is_bvi or (score > self.our_score)
            equal = (not is_bvi) and (score == self.our_score)
            if not (strong or equal):
                continue
            held = self.model.held_probability(row, is_bvi, scenario)
            probs.append(held if strong else held * 0.5)
            if strong:
                strong_ahead += 1
                main_here = str(row.get('Основной ВП')).strip() == 'да'
                consent_here = str(row.get('Согласие')).strip() == 'да'
                if main_here and (is_bvi or consent_here):
                    locked_ahead += 1
            else:
                equal_ahead += 1
        return probs, locked_ahead, strong_ahead, equal_ahead

    def min_passing_score(self):
        """Минимальный проходной по общему конкурсу — наименьший балл среди
        зачисленных по Основному ВП, кроме БВИ/льготников (у них балла нет).

        ВАЖНО: списки в блоках вузов обрезаны нашим порогом отбора, всех, кто
        ниже, мы не видим. Поэтому минимум по видимой части — проходной балл
        ТОЛЬКО когда видимые заняли все места. Если мест занято меньше, чем
        есть, остаток раздаётся тем, кто ниже порога, и настоящая граница лежит
        ниже него: минимум по обрезанному списку тогда врёт вверх (у СТАНКИНа
        09.03.02 при 53 местах и одном занявшем показывал 271 при пороге 269 —
        число выше наших баллов рядом с шансом 99%). Проверяем занятость мест
        ДО взятия минимума; прежняя пометка 'недобор' была тем же случаем, но
        срабатывала, только если ни один зачисленный не имел балла.

        Возвращает (значение, пометка): число — граница видна в наших данных;
        None с пометкой 'ниже порога' — мест хватило всем видимым, граница ниже
        порога отбора; None с пометкой 'вне конкурса' — места забрали БВИ/квоты.
        """
        # Если вуз публикует проходной сам — берём его: он посчитан по полному
        # списку, а не по нашему обрезанному. Это единственное честное число в
        # этой колонке после 30.07, когда списки обрезаны с ОБЕИХ сторон: снизу
        # нашим порогом, сверху — удалёнными зачисленными по квотам и БВИ.
        if self.site_min_score is not None:
            return self.site_min_score, 'сайт'
        if self.budget <= 0:
            return None, 'вне конкурса'
        admitted = self.df[self.df['Основной ВП'] == 'да']
        if len(admitted) < self.budget:
            return None, 'ниже порога'
        real = [r['_score'] for _, r in admitted.iterrows()
                if not self.model.is_bvi(r) and pd.notna(r['_score']) and r['_score'] > 0]
        if real:
            return int(min(real)), None
        return None, 'вне конкурса'

    def chance_by_site_cutoff(self, scenario=None):
        """Шанс по проходному баллу самого вуза, если вуз его публикует.

        Возвращает число или None (вуз проходной не публикует — считаем моделью).

        Почему это надёжнее нашей вероятностной модели. Проходной вуза уже учёл
        главное: кто из подавших согласие оседает на СВОЕЙ приоритетной программе
        и здешнее место не занимает. Наша модель этого по МИРЭА не знает, потому
        что «Основной ВП» там берётся сайтовый iHP — модель «согласились все».
        Из-за этого выходило противоречие: «минимальный проходной 260» рядом с
        «шанс 0%» при нашем балле 277.

        Что означает результат: «проходим, ЕСЛИ согласие подано сюда». Если
        согласие уже подано и балл выше проходного — место наше сейчас (100%).
        Если согласия нет, то же число читается как «пройдём, если подадим».
        Потолок 99% там, где согласия ещё нет: сегодня его донесут и другие, и
        проходной может подрасти до границы «если согласятся все».
        """
        cutoff = (self.site_min_score_all if scenario == 'если все'
                  else self.site_min_score)
        if cutoff is None:
            return None
        if self.budget <= 0:
            return 0
        ours_with_consent = False
        if (self.our_code and 'Согласие' in self.df.columns
                and 'Код' in self.df.columns):
            ours = self.df[self.df['Код'].astype(str) == self.our_code]
            ours_with_consent = bool(
                len(ours) and (ours['Согласие'].astype(str).str.strip() == 'да').any())
        if self.our_score > cutoff:
            return 100 if ours_with_consent else 99
        if self.our_score == cutoff:
            return 50
        return 0

    def chance_percent(self, scenario=None):
        """Шанс на бюджет в процентах (0..100) для заданного сценария согласий."""
        if self.budget <= 0:
            return 0
        by_site = self.chance_by_site_cutoff(scenario)
        if by_site is not None:
            return by_site
        probs, locked_ahead, strong_ahead, equal_ahead = self._competitors(scenario)
        # Места уже заняты намертво определившимися сильнее нас — не пройти.
        if locked_ahead >= self.budget:
            return 0
        # Сильнее (и равных) нас меньше, чем мест, — место достанется точно.
        if strong_ahead + equal_ahead < self.budget:
            return 100
        probability = self.model.poisson_binomial_leq(probs, self.budget - 1)
        # 100% резервируем за гарантией выше; здесь потолок 99%, а низ — как есть
        # (почти безнадёжный расклад честно показываем близким к нулю).
        return min(99, round(probability * 100))


# ===== Прогноз по вузу (таблица пяти столбцов ТЗ) =====
class VuzForecast:
    """Строит итоговую таблицу по одному вузу."""

    # Пять столбцов ТЗ идут первыми и в этом порядке.
    COLUMNS = ['Программы', 'Бюджетные места', 'Минимальный проходной балл',
               'Наши баллы', 'Шансы поступления']
    # Столбцы сверх ТЗ (04.08.2026). «Шансы поступления» — одно число, а в день
    # закрытия согласий одного числа мало: важно, держится ли вывод в обоих
    # предельных сценариях. Расходятся сильно — значит исход зависит от того, кто
    # донесёт согласие сегодня, и на такую программу опираться нельзя.
    EXTRA_COLUMNS = ['Шанс по текущим согласиям', 'Шанс если согласятся все',
                     'Проходной если согласятся все']

    def __init__(self, vuz, vuz_data, target_keys, profile, model):
        self.vuz = vuz
        self.vuz_data = vuz_data
        self.target_keys = target_keys
        self.profile = profile
        self.model = model

    def _min_pass_text(self, forecast):
        value, note = forecast.min_passing_score()
        if value is not None:
            return f'{value} (по данным вуза)' if note == 'сайт' else str(value)
        if note == 'ниже порога':
            # Не пишем «мест хватает»: у СТАНКИНа сайт места как раз раздал, но
            # тем, кто ниже нашего порога, — в наших данных их нет. Точный факт
            # один: граница лежит ниже порога, то есть ниже наших баллов.
            return (f'ниже {forecast.threshold} (наш балл выше)' if forecast.threshold
                    else 'ниже нашего порога (наш балл выше)')
        return {'вне конкурса': 'вне конкурса (БВИ/квоты)'}.get(note, '—')

    def build_table(self):
        rows = []
        for key in self.target_keys:
            data = self.vuz_data[key]
            our_score = self.profile.resolve_score(self.vuz, data.get('threshold'))
            forecast = ProgramForecast(data, our_score, self.model,
                                       our_code=getattr(self.profile, 'code', None))
            chance_now = forecast.chance_percent('сейчас')
            chance_all = forecast.chance_percent('если все')
            rows.append({
                'Программы': data.get('competition') or str(key),
                'Бюджетные места': forecast.budget,
                'Минимальный проходной балл': self._min_pass_text(forecast),
                'Наши баллы': our_score,
                # «Шансы поступления» по ТЗ — осторожная оценка, поэтому берём
                # худший из двух сценариев, а оба показываем рядом.
                'Шансы поступления': f'{min(chance_now, chance_all)}%',
                'Шанс по текущим согласиям': f'{chance_now}%',
                'Шанс если согласятся все': f'{chance_all}%',
                'Проходной если согласятся все': (
                    str(forecast.site_min_score_all)
                    if forecast.site_min_score_all is not None else '—'),
            })
        return pd.DataFrame(rows, columns=self.COLUMNS + self.EXTRA_COLUMNS)


# ===== Оркестратор =====
class AdmissionPredictor:
    """Собирает данные (из памяти или из pickle), печатает таблицы, пишет Excel."""

    PICKLE_CANDIDATES = ('final_cleaned.pkl', 'final_cleaned_partial.pkl')
    EXCEL_OUT = 'шансы_поступления.xlsx'

    # Старше этого возраста источник считаем протухшим и говорим об этом вслух.
    MAX_SOURCE_AGE_HOURS = 6

    def __init__(self, final_data, final_targets, profile=None, model=None,
                 source_note=None):
        self.final_data = final_data
        self.final_targets = final_targets
        self.profile = profile or ApplicantProfile()
        self.model = model or AdmissionModel()
        # Откуда взяты данные и какого они возраста. Раньше модель молча считала
        # по pickle двенадцатидневной давности и рисовала «Шансы поступления» без
        # единого намёка на возраст — в день подачи согласия это опаснее ошибки в
        # формуле.
        self.source_note = source_note or 'final_data из памяти ядра'

    @classmethod
    def from_context(cls, namespace):
        """Взять final_data/final_targets из памяти ядра, иначе из pickle рядом."""
        if namespace.get('final_data') and namespace.get('final_targets'):
            return cls(namespace['final_data'], namespace['final_targets'])
        for name in cls.PICKLE_CANDIDATES:
            if os.path.exists(name):
                with open(name, 'rb') as file:
                    dump = pickle.load(file)
                age_hours = (time.time() - os.path.getmtime(name)) / 3600.0
                stamp = time.strftime('%d.%m.%Y %H:%M',
                                      time.localtime(os.path.getmtime(name)))
                note = f'файл «{name}» от {stamp} (возраст {age_hours:.1f} ч)'
                print(f'final_data нет в памяти — загружен {note}.')
                if age_hours > cls.MAX_SOURCE_AGE_HOURS:
                    print(f'  ВНИМАНИЕ: источник старше {cls.MAX_SOURCE_AGE_HOURS} ч. '
                          f'Списки вузов меняются каждый день, а после зачисления по '
                          f'квотам и БВИ поменялось ещё и число мест — перед решением '
                          f'перегони блоки вузов и «ФИНАЛЬНУЮ СТАТИСТИКУ».')
                return cls(dump['data'], dump['targets'], source_note=note)
        raise SystemExit('Нет ни final_data в памяти, ни final_cleaned.pkl рядом — '
                         'сначала выполни блок «ФИНАЛЬНАЯ СТАТИСТИКА».')

    def _profile_line(self):
        physics = self.profile.subjects_sum(uses_physics=True)
        inform = self.profile.subjects_sum(uses_physics=False)
        return (f'**Наши баллы ВИ:** математика {self.profile.math}, русский '
                f'{self.profile.russian}, физика {self.profile.physics}, '
                f'информатика {self.profile.informatics}. '
                f'Сумма ВИ: {physics} (с физикой) / {inform} (с информатикой). '
                f'**ИД:** медаль + ГТО = 10 (в Финуниверситете 4). Итоговый балл '
                f'уже с ИД в столбце «Наши баллы».')

    def _freshness_line(self):
        """Даты актуальности списков по каждому вузу — как их отдал сам вуз.

        Форматы дат у вузов разные ('2026-08-03 20:44:59', '04.08.2026 01:51',
        '12:50 23.07.2026'), поэтому не разбираем их, а показываем как есть:
        человеку видно сразу, а разбор чужого формата — лишний источник вранья.
        """
        parts = []
        for vuz, vuz_data in self.final_data.items():
            dates = {str(d.get('update_date')) for d in vuz_data.values()
                     if d.get('update_date')}
            parts.append(f'{vuz}: ' + ('; '.join(sorted(dates)[:2]) if dates
                                       else 'дата не проставлена'))
        return ('**Списки актуальны на:** ' + '  |  '.join(parts)
                if parts else '**Списки актуальны на:** дат в данных нет.')

    def run(self, to_excel=True):
        display(Markdown('# ШАНСЫ НА ПОСТУПЛЕНИЕ (предсказательная модель)'))
        display(Markdown(self._profile_line()))
        display(Markdown(f'**Источник данных:** {self.source_note}.'))
        display(Markdown(self._freshness_line()))
        tables = {}
        for vuz, vuz_data in self.final_data.items():
            keys = [k for k in self.final_targets.get(vuz, []) if k in vuz_data]
            if not keys:
                continue
            table = VuzForecast(vuz, vuz_data, keys, self.profile, self.model).build_table()
            tables[vuz] = table
            display(Markdown(f'## {vuz}'))
            display(table.style.hide(axis='index'))
        if to_excel:
            self._save_excel(tables)
        return tables

    def _excel_name(self):
        """Имя файла выгрузки с предохранителем от пробных прогонов.

        Тот же переключатель, что у финальной статистики: `FINAL_DRY_RUN=1`.
        Причина та же — 29.07.2026 пробный прогон на старых данных затёр боевую
        выгрузку, второй копии не было. Модель пишет один файл на весь конвейер,
        и прогон по неполному составу вузов затирал бы его так же молча.
        """
        dry = os.environ.get('FINAL_DRY_RUN', '').strip().lower() \
            not in ('', '0', 'нет', 'false')
        if not dry:
            return self.EXCEL_OUT
        name = self.EXCEL_OUT.replace('.xlsx', '_dryrun.xlsx')
        print(f'FINAL_DRY_RUN=1: пробный прогон, пишем в «{name}», '
              f'боевой файл не трогаем.')
        return name

    def _save_excel(self, tables):
        target = self._excel_name()
        try:
            with pd.ExcelWriter(target, engine='openpyxl') as writer:
                for vuz, table in tables.items():
                    table.to_excel(writer, sheet_name=vuz[:31], index=False)
            print(f'\nСохранено: {target} (лист = вуз).')
        except Exception as error:   # noqa: BLE001 — Excel не критичен, таблицы выше
            print(f'\nВНИМАНИЕ: Excel не сохранился ({error}); таблицы показаны выше.')


# ===== Запуск =====
_predictor = AdmissionPredictor.from_context(globals())
chances_by_vuz = _predictor.run()